# Phase 5 — Concurrency, Async and Performance

**Outcome:** choose threads, processes, subinterpreters or asyncio for a workload
and justify it with a benchmark; explain the asyncio event loop as precisely as
Node's; profile and optimize CPU and memory.

| Unit | Topic |
| --- | --- |
| 5.1 | Concurrency fundamentals, threads and the GIL |
| 5.2 | Multiprocessing and subinterpreters |
| 5.3 | asyncio internals: the event loop |
| 5.4 | asyncio in production |
| 5.5 | Performance and memory optimization |
| Project | 10× faster sales report |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- **Timings are measurements, not facts.** Every benchmark prints the numbers it
  measured on your machine and the ratio between them; no cell asserts a speedup.
  Your numbers will differ with CPU, core count, load and Python build.
- **One knob for workload sizes.** Every unit setup defines
  `DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))`. Benchmarks
  multiply their sizes by it, and child processes inherit the variable. The
  default keeps every cell under about 20 seconds; start Jupyter with
  `PHASE5_DEMO_SCALE=4` for steadier numbers.
- **Workers live in files.** Process pools, `ProcessPoolExecutor` and
  subinterpreter pools need worker functions that a fresh interpreter can import
  (Unit 5.2 explains why). Cells starting with `%%writefile` save such modules
  under `notebooks/_scratch/phase5/` (git-ignored); demos that need a clean
  process (start methods, the main guard, daemon threads at exit) run as scripts
  in a child interpreter.
- **Optional builds and tools are detected, never assumed.** The free-threaded
  build (`python3.15t`), subinterpreters, the JIT, `uvloop` and the profilers are
  checked first; when one is missing its cell prints a skip message.
- **No public network.** HTTP demos talk to a local server on `127.0.0.1`
  (the shop API from Unit 4.6) or to `httpx.MockTransport`.
- *Predict the output* questions hide the answer in a collapsible block: decide
  first, then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check
  cell. Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.14*, …).

## What you need from Phases 1–4

Phase 5 is about doing several things at once. These earlier ideas come back
constantly; re-read the unit if one feels shaky.

| Earlier idea | Unit | What Phase 5 adds |
| --- | --- | --- |
| Generators, `yield`, iterator protocol | 1.13 | Coroutines are generator-like objects driven by an event loop (5.3) |
| Closures, decorators, `functools` | 1.11–1.12 | Passing work to threads and pools; `Timer` and `timer` measure it |
| Context managers, `ExitStack` | 1.14 | `with lock:`, `with executor:`, cleanup on shutdown |
| Reference counting, the GIL's place in CPython, the free-threaded build | 3.1 | What the GIL protects, when it is released, and what changes without it |
| Bytecode, `dis`, the eval breaker and the switch interval | 3.2 | Why `counter += 1` is not atomic; where thread switches can happen |
| Modules, `sys.path`, `__main__` | 3.3 | Why process workers must live in importable modules |
| `pickle` | 4.5 | Every argument sent to a process is pickled: constraints and cost |
| `httpx`, the local shop API, NumPy | 4.6 | Concurrent downloads; NumPy arrays in shared memory |

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 5.1 — Concurrency Fundamentals, Threads and the GIL

**Objectives**

- Tell concurrency from parallelism and CPU-bound from I/O-bound work, and pick
  processes, threads or coroutines from a comparison table and real latency
  numbers.
- Create, name, join and stop threads; know what a daemon thread is.
- Explain what the GIL protects, why it exists and when it is released, and
  prove it with CPU-bound and I/O-bound benchmarks.
- Find race conditions (in `dis` output and in check-then-act code) and fix them
  with the right primitive: `Lock`, `RLock`, `Semaphore`, `Event`, `Condition`,
  `Barrier`, `Timer` or a queue.
- Use `ThreadPoolExecutor` and `Future` well: submit, map, as_completed, wait,
  sizing, shutdown and error handling.
- Run the free-threaded build (`python3.15t`) and measure what it changes.

**JS bridge**

| JavaScript / Node | Python | The difference that bites |
| --- | --- | --- |
| One thread runs all your JavaScript; the event loop interleaves callbacks | Many OS threads can run Python code; in the default build the GIL lets one of them execute bytecode at a time | Python threads are **preemptive**: a switch can happen between two bytecodes of the same line, so shared state needs locks |
| No data races on ordinary objects (only one thread touches them) | Every thread sees the same heap | `counter += 1` from two threads can lose updates |
| libuv thread pool (4 threads by default) runs `fs`, `dns.lookup`, `crypto.pbkdf2`, `zlib` for you | `ThreadPoolExecutor`: you create it, size it and submit work, including your own Python functions | Nothing is offloaded implicitly |
| `worker_threads`: separate V8 isolate, separate heap, messages via structured clone | `threading.Thread` shares one heap; subinterpreters (5.2.6) are the isolate analogue | A Python thread is *not* a Worker |
| `Atomics`, `Atomics.wait` / `notify` on a `SharedArrayBuffer` | `Lock`, `Condition`, `Event`, `Semaphore` | Python's primitives work on any object, not just a shared buffer |
| `Promise` | `concurrent.futures.Future` | `future.result()` **blocks** the calling thread; there is no `await` on it outside asyncio |

**How this unit runs.** Benchmarks print what they measured and the ratio; they
never assert a speedup. I/O demos call the local shop API from Unit 4.6 (its
`/slow?seconds=` route sleeps on the server, like a slow upstream service).
Demos that need a fresh process (daemon threads at exit, the default
`threading.excepthook` output, the free-threaded build) run as scripts in a
child interpreter via `run_child(source, python=...)`.

**The local shop API (copied unchanged from Unit 4.6).** A
`BaseHTTPRequestHandler` with a few routes, served by `ThreadingHTTPServer` in a
daemon thread on `127.0.0.1:18046` (any free port if that one is busy).

In [2]:
import http.server
import json
import random
import threading
import time
import urllib.parse

SHOP_API_PORT = 18046  # fixed so outputs (URLs in messages) are stable; falls back to any free port


def demo_orders() -> list[dict[str, object]]:
    """40 seeded orders as an API would return them, with 3 invalid records planted."""
    generator = random.Random(46)
    cities = ["Pune", "Mumbai", "Bengaluru", "Hubli & Dharwad"]
    prices = {"TEA-500": "240.00", "MUG-02": "199.00", "JAR-750": "350.50", "KETTLE-9": "899.00"}
    orders: list[dict[str, object]] = []
    for order_id in range(1, 41):
        sku = generator.choice(sorted(prices))
        orders.append(
            {
                "order_id": order_id,
                "customer": f"customer-{order_id:03d}",
                "city": generator.choice(cities),
                "sku": sku,
                "quantity": generator.randint(1, 5),
                "unit_price": prices[sku],
                "placed_at": f"2026-11-{generator.randint(1, 7):02d}"
                f"T{generator.randint(9, 20):02d}:00:00+05:30",
            }
        )
    orders[6]["quantity"] = -2  # invalid: not positive
    del orders[17]["city"]  # invalid: missing field
    orders[29]["unit_price"] = "two hundred"  # invalid: not a number
    return orders


PRODUCTS_HTML = """<!doctype html>
<html><head><title>Chai Corner - Products</title></head>
<body>
  <h1>Products</h1>
  <ul id="catalogue">
    <li class="product" data-sku="TEA-500">
      <a href="/products/tea-500">Masala tea 500 g</a> <span class="price">₹240.00</span>
    </li>
    <li class="product" data-sku="MUG-02">
      <a href="/products/mug-02">Steel mug</a> <span class="price">₹199.00</span>
    </li>
    <li class="product sold-out" data-sku="JAR-750">
      <a href="/products/jar-750">Pickle jar 750 ml</a> <span class="price">₹350.50</span>
    </li>
  </ul>
  <p class="note">Prices include GST &amp; free delivery above ₹500.</p>
</body></html>
"""
ROBOTS_TXT = "User-agent: *\nDisallow: /admin/\nCrawl-delay: 1\n\nUser-agent: BadBot\nDisallow: /\n"


class ShopAPIHandler(http.server.BaseHTTPRequestHandler):
    """A tiny local HTTP API used by every HTTP demo (no external network)."""

    protocol_version = "HTTP/1.1"  # keep-alive, so clients can reuse connections
    hits: dict[str, int] = {}
    connections = 0

    def setup(self) -> None:
        """Count each new TCP connection."""
        type(self).connections += 1  # one per TCP connection, not per request
        super().setup()

    def log_message(self, format: str, *args: object) -> None:  # keep the notebook quiet
        """Silence the default request log."""
        pass

    def _send(self, status: int, body: object, content_type: str = "application/json") -> None:
        """Send status and a JSON or text body."""
        data = body.encode() if isinstance(body, str) else json.dumps(body).encode()
        try:
            self.send_response(status)
            self.send_header("Content-Type", f"{content_type}; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except BrokenPipeError, ConnectionResetError:  # the client gave up (a timeout demo)
            self.close_connection = True

    def _count(self, key: str) -> int:
        """Increment and return the hit counter for key."""
        self.hits[key] = self.hits.get(key, 0) + 1
        return self.hits[key]

    def do_GET(self) -> None:
        """Route GET requests to the demo endpoints."""
        url = urllib.parse.urlsplit(self.path)
        query = dict(urllib.parse.parse_qsl(url.query))
        parts = url.path.strip("/").split("/")
        fail_first = int(query.get("fail", 0))
        if url.path == "/rates":
            base = query.get("base", "INR")
            self._send(
                200,
                {
                    "base": base,
                    "date": "2026-11-07",
                    "rates": {"USD": 0.012, "EUR": 0.011, "GBP": 0.0094},
                },
            )
        elif url.path == "/orders":
            if "key" in query and self._count(query["key"]) <= fail_first:
                self._send(503, {"error": "service busy, try again"})
            else:
                self._send(200, demo_orders())
        elif parts[0] == "flaky" and len(parts) == 2:
            attempt = self._count(parts[1])
            if attempt <= fail_first:
                self._send(503, {"error": "busy", "attempt": attempt})
            else:
                self._send(200, {"ok": True, "attempt": attempt})
        elif url.path == "/hits":
            self._send(
                200, {"key": query.get("key"), "hits": self.hits.get(query.get("key", ""), 0)}
            )
        elif url.path == "/connections":
            self._send(200, {"connections": type(self).connections})
        elif url.path == "/reset":
            self.hits.clear()
            self._send(200, {"reset": True})
        elif url.path == "/slow":
            time.sleep(float(query.get("seconds", "2")))
            self._send(200, {"slow": True})
        elif parts[0] == "status" and len(parts) == 2:
            self._send(int(parts[1]), {"status": int(parts[1])})
        elif url.path == "/headers":
            self._send(
                200,
                {
                    name.lower(): value
                    for name, value in self.headers.items()
                    if name.lower() in ("user-agent", "x-request-id", "authorization")
                },
            )
        elif url.path == "/products.html":
            self._send(200, PRODUCTS_HTML, "text/html")
        elif url.path == "/robots.txt":
            self._send(200, ROBOTS_TXT, "text/plain")
        else:
            self._send(404, {"error": f"no route {url.path}"})

    def do_POST(self) -> None:
        """Echo the posted JSON back."""
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"null")
        self._send(201 if self.path == "/orders" else 200, {"received": payload})


def start_shop_api() -> str:
    """Start the local API once (in a daemon thread) and return its base URL."""
    server = globals().get("_SHOP_API_SERVER")
    if server is None:
        try:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", SHOP_API_PORT), ShopAPIHandler)
        except OSError:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), ShopAPIHandler)
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True).start()
        globals()["_SHOP_API_SERVER"] = server
    return f"http://127.0.0.1:{server.server_address[1]}"

Unit setup, part 1: imports, `DEMO_SCALE`, the scratch folder, the API's base
URL (`SHOP_API`) and the error and interpreter helpers shared since Phase 3.

In [3]:
import asyncio
import concurrent.futures
import dis
import functools
import hashlib
import inspect
import itertools
import json
import os
import platform
import queue
import random
import re
import shutil
import sqlite3
import subprocess
import sys
import sysconfig
import textwrap
import threading
import time
import timeit
import traceback
import urllib.request
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))  # the one knob for workload sizes
REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE5_DIR = scratch_dir / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1"}
SHOP_API = start_shop_api()


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" for the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return f"{type(error).__name__}: {error}"
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"

Unit setup, part 2: `Timer` (the timing context manager from Unit 1.14),
`find_free_threaded_python()` (from Unit 3.1) and `run_child()`, which runs a
script in a child interpreter and hides machine-specific paths.

In [4]:
class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        """Remember the label and where to report the elapsed time."""
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        """Start the clock."""
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        """Stop the clock and report the elapsed milliseconds."""
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def find_free_threaded_python() -> str | None:
    """Path of an installed free-threaded 3.15 interpreter, or None (shared since Unit 3.1)."""
    candidates: list[str] = []
    if uv_path := shutil.which("uv"):
        found = subprocess.run(
            [uv_path, "python", "find", "3.15t"], capture_output=True, text=True, timeout=60
        )
        if found.returncode == 0:
            candidates.append(found.stdout.strip())
    if on_path := shutil.which("python3.15t"):
        candidates.append(on_path)
    probe = "import sysconfig; print(sysconfig.get_config_var('Py_GIL_DISABLED'))"
    for candidate in candidates:
        answer = subprocess.run([candidate, "-c", probe], capture_output=True, text=True)
        if answer.stdout.strip() == "1":
            return candidate
    return None


def run_child(
    source: str, python: str = sys.executable, extra_env: dict[str, str] | None = None
) -> str:
    """Run dedented source as a script in a child interpreter; return its stdout + stderr.

    Paths of the scratch folder and interpreters are hidden so outputs stay machine-neutral.
    """
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    completed = subprocess.run(
        [python, str(script)], capture_output=True, text=True, timeout=120,
        env={**CHILD_ENV, **(extra_env or {})}, cwd=PHASE5_DIR,
    )  # fmt: skip
    output = (completed.stdout + completed.stderr).rstrip()
    for path, label in ((str(PHASE5_DIR), "<scratch>"), (str(Path(python).parent), "<python>")):
        output = output.replace(path, label)
    return re.sub(r"/[^\s\"']*/lib/python3\.\d+t?/", "<stdlib>/", output)


FREE_THREADED_PYTHON = find_free_threaded_python()
print(interpreter_label())
print(f"{DEMO_SCALE=}  {os.process_cpu_count()=}  {SHOP_API=}")
print(f"free-threaded interpreter: {'found' if FREE_THREADED_PYTHON else 'not installed'}")

CPython 3.15.1, default (GIL) build, x86_64
DEMO_SCALE=1  os.process_cpu_count()=16  SHOP_API='http://127.0.0.1:18046'
free-threaded interpreter: found


### 5.1.1 Core concepts

- **Concurrency** is about *structure*: several tasks are in progress at the same
  time and their steps interleave. One core is enough.
- **Parallelism** is about *execution*: several tasks run at the same instant on
  different cores.

Node gives you concurrency (one thread, many callbacks in flight) and gets
parallelism only inside libuv's thread pool or from `worker_threads`. CPython's
default build gives threads concurrency but, for Python bytecode, not
parallelism.

```mermaid
gantt
    dateFormat x
    axisFormat %L ms
    section Concurrency (1 core)
    task A :a1, 0, 5
    task B :b1, 5, 10
    task A :a2, 10, 15
    task B :b2, 15, 20
    section Parallelism (2 cores)
    task A (core 1) :0, 20
    task B (core 2) :0, 20
```

<details><summary>ASCII fallback</summary>

```text
concurrency, 1 core:   AAAAABBBBBAAAAABBBBB   (interleaved: both in progress)
parallelism, 2 cores:  AAAAAAAAAAAAAAAAAAAA   core 1
                       BBBBBBBBBBBBBBBBBBBB   core 2 (both running at once)
```

</details>

The next cell makes this visible. Two CPU-bound threads each record a timestamp
on every loop iteration for 60 ms. Afterwards the timeline is cut into 1 ms
slots: `A` or `B` means only that thread ran in the slot, `*` means both ran
(parallelism). Under the GIL you should see runs of `A` and `B` (about 5 ms
each, the switch interval) and almost no `*`.

In [5]:
def record_activity(stop_at: float, samples: list[float]) -> None:
    """Spin until stop_at, appending a timestamp on every iteration."""
    while (now := time.perf_counter()) < stop_at:
        samples.append(now)


def activity_timeline(duration: float = 0.06, slot: float = 0.001) -> str:
    """Run two spinning threads and draw which one ran in each time slot."""
    start = time.perf_counter()
    samples: dict[str, list[float]] = {"A": [], "B": []}
    threads = [
        threading.Thread(target=record_activity, args=(start + duration, samples[name]))
        for name in samples
    ]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    slot_count = int(duration / slot)
    busy = {
        name: {int((sample_time - start) / slot) for sample_time in times}
        for name, times in samples.items()
    }
    cells = []
    for index in range(slot_count):
        ran = [name for name in samples if index in busy[name]]
        cells.append("*" if len(ran) == 2 else (ran[0] if ran else "."))
    return "".join(cells)


print(f"[{interpreter_label()}]")
print(activity_timeline())

[CPython 3.15.1, default (GIL) build, x86_64]
AAAAAAAAAA*BBBB*AAAA*BBBB*AAAAAAAAAA*BBBB*AAAA*BBBBBBBBBB*AA


**What to notice:** long runs of one letter, then the other. Both threads were
*in progress* the whole time (concurrency), but at any moment only one executed
Python code. A few `*` slots can appear where one thread hands over in the
middle of a millisecond. The free-threaded build (5.1.10) prints mostly `*`.

#### CPU-bound vs I/O-bound

| | CPU-bound | I/O-bound |
| --- | --- | --- |
| Time is spent | executing instructions | waiting for a disk, network or another process |
| Examples | parsing, hashing, image resizing, pure-Python loops | HTTP calls, database queries, reading files, `sleep` |
| More threads help? | only if the work runs outside the GIL (C code that releases it) or on a free-threaded build | yes: waiting threads release the GIL |
| Usual tool | processes, subinterpreters, NumPy, free-threaded threads | threads or asyncio |

You can **measure** which kind a function is: compare the CPU time its thread
used (`time.thread_time()`) with the wall-clock time (`time.perf_counter()`).
A ratio near 1 means CPU-bound; near 0 means the thread mostly waited.

In [6]:
def busy_sum(count: int) -> int:
    """Pure-Python CPU work. `% 1000` keeps every value a small int, so each step costs the same."""
    return sum(number % 1000 for number in range(count))


def fetch_slow_page(seconds: float = 0.2) -> bytes:
    """I/O work: the local API sleeps `seconds` before answering."""
    with urllib.request.urlopen(f"{SHOP_API}/slow?seconds={seconds}", timeout=10) as response:
        return response.read()


def cpu_share(work: Callable[[], object]) -> float:
    """CPU time used by this thread divided by wall-clock time, for one call of work."""
    wall_start, cpu_start = time.perf_counter(), time.thread_time()
    work()
    return (time.thread_time() - cpu_start) / (time.perf_counter() - wall_start)


print(f"busy_sum:        CPU/wall = {cpu_share(lambda: busy_sum(2_000_000 * DEMO_SCALE)):.2f}")
print(f"fetch_slow_page: CPU/wall = {cpu_share(fetch_slow_page):.2f}")

busy_sum:        CPU/wall = 0.99


fetch_slow_page: CPU/wall = 0.01


#### Latency numbers

Choosing a concurrency model starts with knowing what you wait for. The classic
table ("latency numbers every programmer should know", rounded; real hardware
varies by a factor of a few):

| Operation | Approximate time | Scaled: if 1 ns were 1 second |
| --- | --- | --- |
| L1 cache reference | 1 ns | 1 s |
| Branch mispredict | 3 ns | 3 s |
| L2 cache reference | 4 ns | 4 s |
| Uncontended mutex lock/unlock (C) | 20 ns | 20 s |
| Main memory reference | 100 ns | under 2 minutes |
| Read 1 MB sequentially from memory | 10 µs | 3 hours |
| Random read from an NVMe SSD | 20–100 µs | 6–28 hours |
| Round trip inside one data centre | 500 µs | 6 days |
| Read 1 MB sequentially from an SSD | 50–200 µs | 14–56 hours |
| Disk seek (spinning disk) | 5 ms | 2 months |
| Packet round trip, Europe ↔ US west coast | 150 ms | 5 years |

Python adds its own costs on top. The next cell measures a few of them **on this
machine** with `timeit` and one-off timings: a function call, an uncontended
`threading.Lock`, starting a thread, creating and running a coroutine, a local
HTTP round trip, reading 1 MB from the page cache and starting a child
interpreter. Notice the six orders of magnitude between the first and last rows.

In [7]:
def per_call_seconds(statement: Callable[[], object], number: int) -> float:
    """Best-of-3 average seconds per call of statement."""
    return min(timeit.repeat(statement, number=number, repeat=3)) / number


def human_time(seconds: float) -> str:
    """Format seconds as ns, µs or ms."""
    for unit, factor in (("ns", 1e9), ("µs", 1e6), ("ms", 1e3)):
        if seconds * factor < 1000:
            return f"{seconds * factor:,.1f} {unit}"
    return f"{seconds:.2f} s"


def start_and_join_thread() -> None:
    """Create, start and join a thread that does nothing."""
    worker = threading.Thread(target=lambda: None)
    worker.start()
    worker.join()


lock_for_timing = threading.Lock()
megabyte_file = PHASE5_DIR / "one_megabyte.bin"
megabyte_file.write_bytes(os.urandom(1024 * 1024))


def lock_round_trip() -> None:
    """Acquire and release an uncontended lock."""
    with lock_for_timing:
        pass


def coroutine_round_trip() -> None:
    """Create a coroutine and run it to completion without an event loop."""
    coroutine = asyncio.sleep(0)
    try:
        coroutine.send(None)
    except StopIteration:
        pass

The table itself. The child-interpreter row runs `python -c pass` five times, so
this cell takes a fraction of a second.

In [8]:
def start_child_interpreter() -> None:
    """Start a fresh interpreter that does nothing."""
    subprocess.run([sys.executable, "-c", "pass"], check=True)


def local_http_round_trip() -> None:
    """GET a tiny JSON document from the local API (new TCP connection each time)."""
    with urllib.request.urlopen(f"{SHOP_API}/rates", timeout=5) as response:
        response.read()


measurements = [
    ("Python function call", per_call_seconds(lambda: None, 200_000)),
    ("threading.Lock acquire + release", per_call_seconds(lock_round_trip, 200_000)),
    ("coroutine create + run (no loop)", per_call_seconds(coroutine_round_trip, 50_000)),
    ("read 1 MB file (page cache)", per_call_seconds(megabyte_file.read_bytes, 200)),
    ("thread start + join", per_call_seconds(start_and_join_thread, 500)),
    ("local HTTP round trip", per_call_seconds(local_http_round_trip, 50)),
    ("child interpreter start", per_call_seconds(start_child_interpreter, 5)),
]
print(f"[{interpreter_label()}] (varies by machine)")
for label, seconds in measurements:
    print(f"{label:<36}{human_time(seconds):>14}")

[CPython 3.15.1, default (GIL) build, x86_64] (varies by machine)
Python function call                       12.8 ns
threading.Lock acquire + release           41.5 ns
coroutine create + run (no loop)          273.0 ns
read 1 MB file (page cache)                41.4 µs
thread start + join                        35.4 µs
local HTTP round trip                     403.4 µs
child interpreter start                    10.3 ms


#### Processes vs threads vs coroutines

| | Process | Thread | Coroutine (asyncio) |
| --- | --- | --- | --- |
| Memory | separate address space (copy or fresh interpreter) | shared heap, own stack | shared heap; a suspended coroutine is a small heap object |
| Start cost | milliseconds | tens of microseconds | about a microsecond |
| How many | a few per core | hundreds to a few thousand | hundreds of thousands |
| Runs Python bytecode in parallel | yes (one GIL each) | no in the default build; yes on a free-threaded build | no (one thread) |
| Who decides when to switch | the OS scheduler | the OS, plus the GIL handoff every switch interval (preemptive) | your code, at each `await` (cooperative) |
| Shared state | none by default; pass messages or use shared memory | everything; needs locks | everything, but switches only at `await`, so fewer races |
| A crash affects | only that process | the whole process | the whole event loop |
| Best for | CPU-bound pure Python | blocking I/O, C code that releases the GIL, legacy blocking libraries | many concurrent network connections |
| Node equivalent | `child_process.fork`, `cluster` | libuv pool (C++ only), `worker_threads` (but isolated) | `async`/`await` on the event loop |

The cell below checks the "start cost" and "how many" rows: it starts 200 threads
that each sleep 10 ms, then runs 200 coroutines that each `await asyncio.sleep`
for 10 ms (top-level `await` works in Jupyter because the kernel already runs an
event loop). It also prints the size of one suspended coroutine object.

In [9]:
def sleep_briefly() -> None:
    """A thread body that waits 10 ms."""
    time.sleep(0.01)


with Timer("200 threads (start, sleep 10 ms, join)"):
    many_threads = [threading.Thread(target=sleep_briefly) for _ in range(200)]
    for thread in many_threads:
        thread.start()
    for thread in many_threads:
        thread.join()

with Timer("200 coroutines (sleep 10 ms, gather)"):
    await asyncio.gather(*(asyncio.sleep(0.01) for _ in range(200)))

suspended = asyncio.sleep(1)
print(f"one coroutine object: {sys.getsizeof(suspended)} bytes [{interpreter_label()}]")
suspended.close()  # never awaited: close it so Python does not warn
print(
    f"thread stack size setting: {threading.stack_size()} "
    "(0 = the OS default, 8 MiB virtual on Linux)"
)

200 threads (start, sleep 10 ms, join) took 27.91 ms
200 coroutines (sleep 10 ms, gather) took 12.28 ms
one coroutine object: 256 bytes [CPython 3.15.1, default (GIL) build, x86_64]
thread stack size setting: 0 (0 = the OS default, 8 MiB virtual on Linux)


Both finish in roughly 10–30 ms, but the threads pay for 200 OS threads (each
with its own stack reserved in virtual memory) while the coroutines are a few
hundred bytes each on the heap. That is why a server holding 10 000 open
connections uses asyncio (Units 5.3–5.4), not 10 000 threads.

### 5.1.2 `threading.Thread`

| API | What it does |
| --- | --- |
| `Thread(target=f, args=(...), kwargs={...}, name="...", daemon=...)` | describe a thread; nothing runs yet |
| `thread.start()` | ask the OS to start it; `run()` then calls `target(*args, **kwargs)` |
| `thread.join(timeout=None)` | wait until it finishes (or the timeout passes; check `is_alive()` afterwards) |
| `thread.is_alive()` | `True` between `start()` and the end of `run()` |
| `thread.name`, `thread.ident`, `thread.native_id` | label you choose; Python's id; the OS thread id (what `top -H` shows) |
| `thread.daemon` | daemon threads are killed when the interpreter exits |
| `threading.current_thread()`, `main_thread()`, `get_ident()`, `get_native_id()` | who am I? |
| `threading.enumerate()`, `active_count()` | all live `Thread` objects |

**Creation, `start` and `join`.** A thread function cannot return a value to
the caller: `join()` returns `None`. The usual workaround is to pass a container
the thread writes into (or use a `ThreadPoolExecutor`, 5.1.8).

In [10]:
def greet_customer(customer_name: str, greetings: list[str]) -> None:
    """Thread body: append a greeting that names the thread running it."""
    current_thread = threading.current_thread()
    greetings.append(f"{current_thread.name} greeted {customer_name}")


greetings: list[str] = []
greeter = threading.Thread(
    target=greet_customer, args=("Asha",), kwargs={"greetings": greetings}, name="greeter-1"
)
print(f"before start: {greeter.is_alive()=}, {greeter.ident=}")
greeter.start()
join_result = greeter.join()
print(f"after join:   {greeter.is_alive()=}, {join_result=}")
print(f"{greetings=}")

before start: greeter.is_alive()=False, greeter.ident=None
after join:   greeter.is_alive()=False, join_result=None
greetings=['greeter-1 greeted Asha']


**Names and IDs.** `ident` is Python's identifier (reused after a thread ends);
`native_id` is the kernel's thread id. The main thread's `native_id` equals the
process id on Linux. Exact numbers change on every run, so the cell prints
comparisons and leaves the raw values for you to look at.

In [11]:
ids_seen: dict[str, tuple[int | None, int | None]] = {}


def remember_ids() -> None:
    """Record this thread's Python ident and OS native id."""
    current_thread = threading.current_thread()
    ids_seen[current_thread.name] = (threading.get_ident(), threading.get_native_id())


workers = [
    threading.Thread(target=remember_ids, name=f"id-probe-{probe_number}")
    for probe_number in range(2)
]
for worker in workers:
    worker.start()
for worker in workers:
    worker.join()
main = threading.main_thread()
print(f"{main.name=}, main native_id == os.getpid(): {main.native_id == os.getpid()}")
print(f"probe threads have distinct native ids: {len({ids[1] for ids in ids_seen.values()}) == 2}")
print(f"raw ids (vary per run): {ids_seen}")
print(f"default name of an unnamed thread: {threading.Thread(target=print).name!r}")

main.name='MainThread', main native_id == os.getpid(): True
probe threads have distinct native ids: True
raw ids (vary per run): {'id-probe-0': (140589907273408, 151387), 'id-probe-1': (140589915666112, 151388)}
default name of an unnamed thread: 'Thread-1859 (print)'


`threading.enumerate()` lists every live thread in this process. In a notebook
that includes the kernel's own threads (heartbeat, I/O publishing, control) and
the shop API's server thread; your list depends on the ipykernel version.

In [12]:
print(f"{threading.active_count()=}")
for live_thread in threading.enumerate():
    print(f"  {live_thread.name:<28} daemon={live_thread.daemon}")

threading.active_count()=10
  MainThread                   daemon=False
  IOPub                        daemon=True
  Heartbeat                    daemon=True
  Thread-2 (_watch_pipe_fd)    daemon=True
  Thread-3 (_watch_pipe_fd)    daemon=True
  Control                      daemon=True
  Shell channel                daemon=True
  IPythonHistorySavingThread   daemon=True
  Thread-1                     daemon=True
  Thread-5 (serve_forever)     daemon=True


**Lifecycle errors.** A `Thread` object runs at most once: a second `start()`
raises `RuntimeError`. `join(timeout)` returns when the timeout passes even if
the thread is still running, so always check `is_alive()` afterwards.

In [13]:
sleeper = threading.Thread(target=time.sleep, args=(0.3,), name="sleeper")
sleeper.start()
sleeper.join(timeout=0.05)
print(f"after join(timeout=0.05): {sleeper.is_alive()=}")
print(raised_message(sleeper.start))
sleeper.join()
print(f"after a full join:        {sleeper.is_alive()=}")

after join(timeout=0.05): sleeper.is_alive()=True
RuntimeError: threads can only be started once


after a full join:        sleeper.is_alive()=False


**Daemon threads.** The interpreter waits for every non-daemon thread before it
exits; daemon threads are simply stopped (no `finally`, no `with` cleanup). A
new thread inherits `daemon` from the thread that creates it (the main thread is
not a daemon). This only shows at interpreter exit, so the demo runs as a script
in a child process: the main thread returns immediately in both cases.

In [14]:
DAEMON_DEMO = """
import threading, time

def background_job():
    try:
        time.sleep(0.3)
        print("background job finished")
    finally:
        print("background job cleanup ran")

daemon = {daemon}
threading.Thread(target=background_job, daemon=daemon).start()
print(f"main thread done (daemon={{daemon}})")
"""
print(f"{threading.Thread(target=print).daemon=} (inherited from the main thread)")
for daemon_flag in (False, True):
    print(f"--- child run with daemon={daemon_flag}")
    print(run_child(DAEMON_DEMO.format(daemon=daemon_flag)))

threading.Thread(target=print).daemon=False (inherited from the main thread)
--- child run with daemon=False


main thread done (daemon=False)
background job finished
background job cleanup ran
--- child run with daemon=True
main thread done (daemon=True)


With `daemon=False` the process waits 0.3 s and the job finishes; with
`daemon=True` the process exits at once and **neither print runs**, not even the
`finally`. Use daemon threads only for work that is safe to lose (a heartbeat, a
cache warmer), and give important workers a stop flag instead (5.1.9).

**Subclassing vs `target`.** You can subclass `Thread` and override `run()`;
this is handy when the thread carries state (here, its result). Passing a
`target` keeps the work a plain function you can also call and test without
threads.

In [15]:
class PriceFetcher(threading.Thread):
    """A thread subclass that fetches exchange rates and keeps the result."""

    def __init__(self, base_currency: str) -> None:
        """Name the thread after the base currency it fetches rates for."""
        super().__init__(name=f"rates-{base_currency}")
        self.base_currency = base_currency
        self.rates: dict[str, float] = {}

    def run(self) -> None:
        """Called in the new thread by start()."""
        url = f"{SHOP_API}/rates?base={self.base_currency}"
        with urllib.request.urlopen(url, timeout=5) as reply:
            self.rates = json.load(reply)["rates"]


fetcher = PriceFetcher("INR")
fetcher.start()
fetcher.join()
print(f"subclass: {fetcher.name} -> {fetcher.rates}")


def fetch_rates(base_currency: str, results: dict[str, dict[str, float]]) -> None:
    """The same work as a plain function, usable with or without threads."""
    with urllib.request.urlopen(f"{SHOP_API}/rates?base={base_currency}", timeout=5) as reply:
        results[base_currency] = json.load(reply)["rates"]


rate_results: dict[str, dict[str, float]] = {}
plain = threading.Thread(target=fetch_rates, args=("INR", rate_results))
plain.start()
plain.join()
print(f"target:   {plain.name} -> {rate_results}")

subclass: rates-INR -> {'USD': 0.012, 'EUR': 0.011, 'GBP': 0.0094}
target:   Thread-1862 (fetch_rates) -> {'INR': {'USD': 0.012, 'EUR': 0.011, 'GBP': 0.0094}}


| Style | Use it when |
| --- | --- |
| `Thread(target=func, args=...)` | almost always: the work stays a testable function |
| `class Worker(Thread): def run(self)` | the thread is a long-lived object with its own state and methods (a poller, a server loop) |
| `ThreadPoolExecutor().submit(func, ...)` | you want return values and exceptions back (5.1.8): the default for new code |

> **Note:** override `run()`, never `start()`. And call `super().__init__()` in a
> subclass's `__init__`, or `start()` fails with
> `RuntimeError: thread.__init__() not called`.

### 5.1.3 The Global Interpreter Lock

**What it protects.** The GIL is a lock inside the interpreter. In the default
build a thread must hold it to execute bytecode or touch Python objects. It
protects CPython's *own* state: reference counts (incremented and decremented
with plain, non-atomic instructions), the internals of `dict`, `list` and other
built-ins, the memory allocator, the import system. It does **not** protect
your program's invariants across several bytecodes (5.1.4).

**Why it exists.** Reference counting (Unit 3.1) touches a counter on almost
every operation; making every one of those atomic or locked would slow
single-threaded code. One big lock made CPython simple and fast for one thread,
and made C extensions easy to write (they could assume no other Python code runs
concurrently). PEP 703's free-threaded build removes it at a cost (5.1.10).

**When it is released.**

| Situation | What happens |
| --- | --- |
| Blocking I/O: `socket.recv`, `file.read`, `select`, `subprocess` waits | the C code releases the GIL around the system call |
| `time.sleep(seconds)` | released for the whole sleep |
| C extensions doing long work: `hashlib` (data over 2 KiB), `zlib`, `bz2`, `lzma`, NumPy, `sqlite3` queries | released if the extension says so (`Py_BEGIN_ALLOW_THREADS`) |
| A pure-Python loop | **not released voluntarily**; after the switch interval another thread asks for it and the eval loop hands it over at the next check |

```mermaid
sequenceDiagram
    participant A as Thread A (CPU loop)
    participant G as GIL
    participant B as Thread B (waiting)
    A->>G: holds the GIL, runs bytecode
    B->>G: waits up to the switch interval (5 ms), then sets "drop request"
    G-->>A: eval breaker: A drops the GIL at the next check
    G->>B: B takes the GIL and runs
    A->>G: A waits for its turn
```

<details><summary>ASCII fallback</summary>

```text
Thread A: [run bytecode .......5 ms.......] drop -> wait ............ [run ...
Thread B: wait ......... set drop request -> [run bytecode ...5 ms...] drop
            a waiting thread asks after `sys.getswitchinterval()` seconds
```

</details>

**The switch interval.** `sys.getswitchinterval()` is 0.005 s by default. The
next cell reuses `count_thread_switches` from Unit 3.2.10: two CPU-bound threads
count how often the running thread changes in 0.2 s, at two intervals.

In [16]:
def count_thread_switches(switch_interval: float, duration: float = 0.2) -> int:
    """Two CPU-bound threads; count how often the running thread changes."""
    state = {"owner": "", "switches": 0}
    deadline = time.perf_counter() + duration

    def spin(name: str) -> None:
        """Spin until the deadline, counting each change of the running thread."""
        while time.perf_counter() < deadline:
            if state["owner"] != name:
                state["owner"] = name
                state["switches"] += 1

    original_interval = sys.getswitchinterval()
    sys.setswitchinterval(switch_interval)
    try:
        threads = [threading.Thread(target=spin, args=(name,)) for name in ("A", "B")]
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
    finally:
        sys.setswitchinterval(original_interval)
    return state["switches"]


print(f"{sys.getswitchinterval()=}  [{interpreter_label()}]")
for interval in (0.005, 0.0005):
    print(
        f"switch interval {interval * 1000:>4} ms -> "
        f"{count_thread_switches(interval):>5} switches in 0.2 s"
    )

sys.getswitchinterval()=0.005  [CPython 3.15.1, default (GIL) build, x86_64]


switch interval  5.0 ms ->    33 switches in 0.2 s


switch interval  0.5 ms ->   337 switches in 0.2 s


About 40 switches in 0.2 s at 5 ms (one every 5 ms), about ten times more at
0.5 ms. A shorter interval makes threads more responsive but adds handoff
overhead; leave it alone in real code.

#### Benchmark: CPU-bound work in threads

The same total work (`busy_sum` over 8 million numbers × `DEMO_SCALE`) split
across 1, 2 and 4 threads. With the GIL only one thread executes bytecode at a
time, so expect the wall time to stay about the same (or get slightly worse
from handoffs).

In [17]:
def run_in_threads(work: Callable[[int], object], items: list[int]) -> float:
    """Run work(item) for every item, one thread per item; return wall seconds."""
    threads = [threading.Thread(target=work, args=(item,)) for item in items]
    start = time.perf_counter()
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    return time.perf_counter() - start


total_numbers = 8_000_000 * DEMO_SCALE
print(f"[{interpreter_label()}] busy_sum over {total_numbers:,} numbers in total")
cpu_baseline = 0.0
for thread_count in (1, 2, 4):
    seconds = run_in_threads(busy_sum, [total_numbers // thread_count] * thread_count)
    cpu_baseline = cpu_baseline or seconds
    print(
        f"{thread_count} thread(s): {seconds:.3f} s   "
        f"speedup vs 1 thread: {cpu_baseline / seconds:.2f}x"
    )

[CPython 3.15.1, default (GIL) build, x86_64] busy_sum over 8,000,000 numbers in total
1 thread(s): 0.164 s   speedup vs 1 thread: 1.00x


2 thread(s): 0.170 s   speedup vs 1 thread: 0.97x
4 thread(s): 0.179 s   speedup vs 1 thread: 0.92x


#### Benchmark: I/O-bound work in threads

Now 8 requests to the local API, each of which waits 0.1 s on the server. While
a thread waits on its socket it has released the GIL, so the waits overlap.

In [18]:
request_count = 8
print(f"{request_count} requests, each waiting 0.1 s on the server")
io_baseline = 0.0
for thread_count in (1, 2, 4, 8):
    per_thread = request_count // thread_count
    seconds = run_in_threads(
        lambda repeat: [fetch_slow_page(0.1) for _ in range(repeat)], [per_thread] * thread_count
    )
    io_baseline = io_baseline or seconds
    print(
        f"{thread_count} thread(s): {seconds:.3f} s   "
        f"speedup vs 1 thread: {io_baseline / seconds:.2f}x"
    )

8 requests, each waiting 0.1 s on the server


1 thread(s): 0.826 s   speedup vs 1 thread: 1.00x


2 thread(s): 0.414 s   speedup vs 1 thread: 1.99x


4 thread(s): 0.211 s   speedup vs 1 thread: 3.91x
8 thread(s): 0.111 s   speedup vs 1 thread: 7.41x


Close to linear: 8 threads take about as long as one request. Same threads, same
GIL, opposite result; the difference is *where the time goes*.

**`sleep` releases the GIL.** While the main thread sleeps for 50 ms, a
background thread keeps counting. If `sleep` held the GIL the count would stay
at zero.

In [19]:
progress = {"count": 0}
stop_counting = threading.Event()


def count_until_stopped() -> None:
    """CPU loop that counts until the event is set."""
    while not stop_counting.is_set():
        progress["count"] += 1


counter_thread = threading.Thread(target=count_until_stopped)
counter_thread.start()
before_sleep = progress["count"]
time.sleep(0.05)  # the main thread releases the GIL here
during_sleep = progress["count"] - before_sleep
stop_counting.set()
counter_thread.join()
print(f"the other thread counted {during_sleep:,} times while the main thread slept 50 ms")

the other thread counted 1,467,233 times while the main thread slept 50 ms


**C extensions release it too.** `hashlib` releases the GIL while it hashes
buffers larger than 2 KiB, so hashing four 32 MiB buffers in four threads runs
on four cores. Hashing 100 000 tiny 64-byte strings spends its time in Python
(calls, loop) with the GIL held, so threads do not help there.

In [20]:
big_buffers = [os.urandom(32 * 1024 * 1024) for _ in range(4)]
tiny_messages = [os.urandom(64) for _ in range(100_000)]


def hash_big(index: int) -> None:
    """Hash one large buffer DEMO_SCALE times (the GIL is released inside)."""
    for _ in range(DEMO_SCALE):
        hashlib.sha256(big_buffers[index]).digest()


def hash_tiny(index: int) -> None:
    """Hash a quarter of the tiny messages (mostly Python overhead)."""
    for _ in range(DEMO_SCALE):
        for message in tiny_messages[index::4]:
            hashlib.sha256(message).digest()


def all_four(work: Callable[[int], None]) -> None:
    """Do the four pieces of work one after another (the 1-thread baseline)."""
    for index in range(4):
        work(index)


for label, work in (("4 x 32 MiB buffers", hash_big), ("100 000 x 64 B messages", hash_tiny)):
    sequential = run_in_threads(all_four, [work])
    threaded = run_in_threads(work, [0, 1, 2, 3])
    print(
        f"{label:<24} 1 thread {sequential:.3f} s | 4 threads "
        f"{threaded:.3f} s | {sequential / threaded:.2f}x"
    )
big_buffers.clear()  # free 128 MiB

4 x 32 MiB buffers       1 thread 0.061 s | 4 threads 0.017 s | 3.64x
100 000 x 64 B messages  1 thread 0.025 s | 4 threads 0.030 s | 0.82x


> **Note:** "Is my workload CPU-bound in Python, or CPU-bound in C code that
> releases the GIL?" decides whether threads help. NumPy, `hashlib`, `zlib`,
> image codecs and database drivers often release it; your own `for` loops never
> do (on the default build).

**Under the hood.** Whether this interpreter has a GIL is a build-time property
(`Py_GIL_DISABLED`) plus a runtime switch on free-threaded builds
(`sys._is_gil_enabled()`, *new in 3.13*).

In [21]:
print(f"{sysconfig.get_config_var('Py_GIL_DISABLED')=}")
print(f"{sys._is_gil_enabled()=}")
print(f"{sys.getswitchinterval()=}")

sysconfig.get_config_var('Py_GIL_DISABLED')=0
sys._is_gil_enabled()=True
sys.getswitchinterval()=0.005


### 5.1.4 Race conditions

A **race condition** is a bug whose outcome depends on how threads happen to
interleave. It needs shared mutable state and at least one writer, and it
usually hides in a *read–modify–write* or *check-then-act* sequence.

**`counter += 1` is not atomic.** One line of Python, several bytecodes: load
the value, add one, store the result. A thread switch between the load and the
store means another thread's increment is overwritten.

In [22]:
stock_counter = 0


def add_one() -> None:
    """Increment a module-level counter."""
    global stock_counter
    stock_counter += 1


print(f"# dis on {interpreter_label()} (instruction names change between versions)")
dis.dis(add_one)

# dis on CPython 3.15.1, default (GIL) build, x86_64 (instruction names change between versions)
  4           RESUME                   0

  7           LOAD_GLOBAL              0 (stock_counter)
              LOAD_SMALL_INT           1
              BINARY_OP               13 (+=)
              STORE_GLOBAL             0 (stock_counter)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


Read the listing as three steps: `LOAD_GLOBAL stock_counter` (read), `BINARY_OP
... (+=)` (modify), `STORE_GLOBAL stock_counter` (write). Two threads can both
read 41, both compute 42, and both store 42: one increment is lost.

**Demo 1: plain `+=` on the default build.** Four threads add 100 000 each.

In [23]:
class PlainCounter:
    """A counter with no locking at all."""

    def __init__(self) -> None:
        """Start at zero."""
        self.value = 0


def add_many(counter: PlainCounter, times: int) -> None:
    """Increment counter.value `times` times with +=."""
    for _ in range(times):
        counter.value += 1


plain_counter = PlainCounter()
run_in_threads(lambda _: add_many(plain_counter, 100_000), [0, 1, 2, 3])
print(f"[{interpreter_label()}] expected 400,000, got {plain_counter.value:,}")

[CPython 3.15.1, default (GIL) build, x86_64] expected 400,000, got 400,000


On CPython 3.15's default build this usually prints the right total, and that
is the trap. The eval loop only hands the GIL over at certain instructions (loop
back-edges and calls), which here fall *between* increments, never inside one.
That is an implementation detail, not a guarantee: other Python versions,
a function call inside the loop, or the free-threaded build (5.1.10 runs this
exact code there) lose updates.

**Demo 2: make the window visible.** `time.sleep(0)` between the read and the
write gives up the GIL, the way any I/O or call into slow code would. Now
updates are lost on every build.

In [24]:
class YieldingCounter:
    """Read, give up the GIL, then write: the classic lost-update race."""

    def __init__(self) -> None:
        """Start at zero."""
        self.value = 0

    def increment(self) -> None:
        """Read, yield the GIL, then write: two threads can lose an update."""
        current = self.value  # read
        time.sleep(0)  # another thread can run here
        self.value = current + 1  # write (may overwrite someone else's update)


yielding_counter = YieldingCounter()
run_in_threads(lambda _: [yielding_counter.increment() for _ in range(500)], [0, 1, 2, 3])
print(f"expected 2,000, got {yielding_counter.value:,}")

expected 2,000, got 503


**Check-then-act.** The check is correct when it runs, but the world changes
before the action. Five customers try to buy the last three phone cases; each
checks stock, then (after a pause that stands for a database round trip)
decrements it.

In [25]:
stock = {"phone-case": 3}
sales: list[str] = []


def buy(customer: str) -> None:
    """Check stock, pause, then sell: racy."""
    if stock["phone-case"] > 0:  # check
        time.sleep(0.01)  # e.g. charge the card
        stock["phone-case"] -= 1  # act
        sales.append(customer)


run_in_threads(buy, ["Asha", "Ravi", "Meena", "John", "Fatima"])
print(f"sold {len(sales)} of 3 cases; stock is now {stock['phone-case']}")

sold 5 of 3 cases; stock is now -2


All five checks saw `3 > 0` before anyone decremented, so the shop oversold and
stock went negative. The same pattern hides in lazy initialization: `if key not
in cache: cache[key] = load(key)` can call `load` once per thread.

In [26]:
cache: dict[str, str] = {}
load_calls: list[str] = []


def load_profile(user_id: str) -> str:
    """Expensive load (counts how often it runs)."""
    load_calls.append(threading.current_thread().name)
    time.sleep(0.01)
    return f"profile of {user_id}"


def get_profile(user_id: str) -> str:
    """Racy lazy cache."""
    if user_id not in cache:
        cache[user_id] = load_profile(user_id)
    return cache[user_id]


run_in_threads(get_profile, ["u-42"] * 4)
print(f"load_profile ran {len(load_calls)} times for one key")

load_profile ran 4 times for one key


#### What is atomic?

"Atomic" here means: no other thread can observe a half-done state. In CPython,
single operations on built-in containers are atomic. The default build gets that
from the GIL; the free-threaded build keeps the same guarantees with per-object
locks. This is documented in the
[FAQ](https://docs.python.org/3/faq/library.html#what-kinds-of-global-value-mutation-are-thread-safe),
but it is a CPython property, not a language promise.

| Atomic (one operation on a built-in) | Not atomic (several steps) |
| --- | --- |
| `items.append(x)`, `items.pop()`, `items[i] = x` | `counter += 1`, `items[i] += 1` |
| `mapping[key] = value`, `mapping.get(key)`, `mapping.pop(key, None)` | `if key not in mapping: mapping[key] = value` |
| `mapping.setdefault(key, value)` | `mapping[key] = mapping.get(key, 0) + 1` |
| `deque.append(x)`, `deque.popleft()` | iterating a list or dict while another thread changes it |
| `name = other_object` (rebinding) | keeping two objects consistent with each other |
| `next(it)` on built-in iterators such as `itertools.count` | `next(generator)` from two threads (raises `ValueError`) |

Rule of thumb: if correctness depends on two steps happening together, use a
lock. The next cell shows the atomic side: 4 threads × 50 000 `append`s always
give exactly 200 000 items, and `setdefault` picks exactly one winner.

In [27]:
shared_items: list[int] = []
run_in_threads(lambda _: [shared_items.append(1) for _ in range(50_000)], [0, 1, 2, 3])
print(f"{len(shared_items)=:,} (expected 200,000)")

winners: dict[str, str] = {}
claims: list[str] = []
run_in_threads(
    lambda name: claims.append(winners.setdefault("seat-1A", name)), ["Asha", "Ravi", "Meena"]
)
print(f"setdefault winner seen by every thread: {set(claims) == {winners['seat-1A']}}")

len(shared_items)=200,000 (expected 200,000)
setdefault winner seen by every thread: True


**Generators are not thread-safe.** Two threads calling `next()` on the same
generator at the same moment get `ValueError: generator already executing`.
*New in 3.15:* `threading.serialize_iterator(iterator)` wraps any iterator (or
generator) with a lock around `__next__`, and `threading.synchronized_iterator`
does the same as a decorator for generator functions. The cell is guarded so it
prints a skip message on older kernels.

In [28]:
def order_numbers(count: int) -> Iterator[int]:
    """A generator with a pause inside, so two threads overlap in next()."""
    for number in range(count):
        time.sleep(0.0005)
        yield number


def drain(source: Iterator[int], taken: list[int], errors: list[str]) -> None:
    """Consume the shared iterator, recording values and errors."""
    try:
        for number in source:
            taken.append(number)
    except ValueError as error:
        errors.append(str(error))


def share_between_two_threads(shared_source: Iterator[int]) -> str:
    """Let two threads drain one iterator; summarize what they got."""
    taken: list[int] = []
    errors: list[str] = []
    run_in_threads(lambda _: drain(shared_source, taken, errors), [0, 1])
    return (
        f"items taken={len(taken)}, no duplicates={len(set(taken)) == len(taken)}, errors={errors}"
    )


print(f"bare generator:     {share_between_two_threads(order_numbers(200))}")
if hasattr(threading, "serialize_iterator"):
    print(
        "serialize_iterator: "
        f"{share_between_two_threads(threading.serialize_iterator(order_numbers(200)))}"
    )
else:
    print("serialize_iterator: SKIP (new in 3.15)")

bare generator:     items taken=200, no duplicates=True, errors=['generator already executing']


serialize_iterator: items taken=200, no duplicates=True, errors=[]


### 5.1.5 Synchronization primitives

| Primitive | What it does | Typical use | JS analogue |
| --- | --- | --- | --- |
| `Lock` | one holder at a time; not reentrant | protect a read–modify–write | `Atomics` on a `SharedArrayBuffer`, or a promise-based mutex library |
| `RLock` | like `Lock`, but the holding thread may re-acquire it | recursive or layered code that locks the same object | — |
| `Semaphore(n)` | at most `n` holders | cap concurrent calls to a service | `p-limit` |
| `BoundedSemaphore(n)` | a semaphore that raises if released more than acquired | the same, with a bug check: prefer it | — |
| `Event` | a flag threads can wait for | "start now", "stop now" | a promise that resolves once |
| `Condition` | a lock plus wait/notify on a predicate | "wait until stock > 0" | `Atomics.wait` / `notify` |
| `Barrier(n)` | blocks until `n` threads arrive | phases that must start together | `Promise.all` used as a rendezvous |
| `Timer(delay, f)` | a thread that calls `f` after `delay` unless cancelled | timeouts, debouncing | `setTimeout` / `clearTimeout` |

All of them are context managers where it makes sense (`with lock:`), which
guarantees the release even when the body raises.

#### `Lock`

The lost-update counter from 5.1.4, fixed: the read and the write happen while
the lock is held, so no other thread can interleave.

In [29]:
class LockedYieldingCounter:
    """The yielding counter from 5.1.4 with its read-modify-write under a lock."""

    def __init__(self) -> None:
        """Start at zero with a lock that guards the read-modify-write."""
        self.value = 0
        self._lock = threading.Lock()

    def increment(self) -> None:
        """Read and write under the lock, so no update is lost."""
        with self._lock:
            current = self.value
            time.sleep(0)  # other threads may run, but cannot enter this block
            self.value = current + 1


locked_counter = LockedYieldingCounter()
run_in_threads(lambda _: [locked_counter.increment() for _ in range(500)], [0, 1, 2, 3])
print(f"expected 2,000, got {locked_counter.value:,}")

expected 2,000, got 2,000


`acquire()` can also be non-blocking or time-limited, and `locked()` reports the
state. A plain `Lock` is **not reentrant**: the thread holding it blocks forever
if it tries again. Here the second attempt uses `blocking=False` so the cell
does not hang.

In [30]:
inventory_lock = threading.Lock()
print(f"{inventory_lock.acquire()=}  {inventory_lock.locked()=}")
print(f"second acquire, same thread, non-blocking: {inventory_lock.acquire(blocking=False)}")
from_other_thread: list[bool] = []
run_in_threads(lambda _: from_other_thread.append(inventory_lock.acquire(timeout=0.05)), [0])
print(f"acquire(timeout=0.05) from another thread: {from_other_thread[0]}")
inventory_lock.release()
print(f"after release: {inventory_lock.locked()=}")

inventory_lock.acquire()=True  inventory_lock.locked()=True
second acquire, same thread, non-blocking: False
acquire(timeout=0.05) from another thread: False
after release: inventory_lock.locked()=False


#### `RLock`

A reentrant lock counts how many times its owner acquired it and frees it only
after the same number of releases. It fits recursive code: totalling a category
tree locks the tree at every level. With a plain `Lock` the recursive call would
deadlock; the `timeout` in the plain-lock version turns that into a `False`.

In [31]:
category_tree = {"Tea": {"Green": {}, "Black": {"Assam": {}, "Darjeeling": {}}}}


def count_categories(tree: dict[str, dict], lock: threading.Lock | threading.RLock) -> int:
    """Count nodes, taking the lock at every level of recursion."""
    if not lock.acquire(timeout=0.1):
        raise TimeoutError("could not re-acquire the lock (it is not reentrant)")
    try:
        return sum(1 + count_categories(children, lock) for children in tree.values())
    finally:
        lock.release()


print(f"RLock: {count_categories(category_tree, threading.RLock())} categories")
print(f"Lock:  {raised_message(lambda: count_categories(category_tree, threading.Lock()))}")

RLock: 5 categories
Lock:  TimeoutError: could not re-acquire the lock (it is not reentrant)


#### `Semaphore`

A semaphore holds a counter: `acquire()` decrements it (blocking at zero),
`release()` increments it. Ten threads call a payment gateway that allows three
concurrent requests; the semaphore keeps the peak at three.

In [32]:
gateway_slots = threading.Semaphore(3)
in_flight = {"now": 0, "peak": 0}
in_flight_lock = threading.Lock()


def charge_card(order_id: int) -> None:
    """Call the 'gateway' while holding one of the three slots."""
    with gateway_slots:
        with in_flight_lock:
            in_flight["now"] += 1
            in_flight["peak"] = max(in_flight["peak"], in_flight["now"])
        time.sleep(0.02)  # the network call
        with in_flight_lock:
            in_flight["now"] -= 1


seconds = run_in_threads(charge_card, list(range(10)))
print(f"peak concurrent charges: {in_flight['peak']} (limit 3); 10 charges took {seconds:.2f} s")

peak concurrent charges: 3 (limit 3); 10 charges took 0.08 s


#### `BoundedSemaphore`

A plain `Semaphore` accepts extra `release()` calls silently, so a bug that
releases twice raises the limit for good. `BoundedSemaphore` raises
`ValueError` instead. Prefer it whenever releases should match acquires.

In [33]:
def acquirable(semaphore: threading.Semaphore) -> int:
    """How many non-blocking acquires succeed (then release them all again)."""
    count = 0
    while semaphore.acquire(blocking=False):
        count += 1
    for _ in range(count):
        semaphore.release()
    return count


loose = threading.Semaphore(2)
loose.release()  # a bug: one release too many
print(f"Semaphore(2) after an extra release allows {acquirable(loose)} holders")

strict = threading.BoundedSemaphore(2)
print(f"BoundedSemaphore(2) extra release: {raised_message(strict.release)}")

Semaphore(2) after an extra release allows 3 holders
BoundedSemaphore(2) extra release: ValueError: Semaphore released too many times


#### `Event`

An event is a boolean flag with `set()`, `clear()`, `is_set()` and
`wait(timeout)`. Workers block on `wait()` until the main thread sets it: all
three start at (almost) the same moment. `wait(timeout)` returns `False` if the
timeout passes first.

In [34]:
shop_open = threading.Event()
start_times: dict[str, float] = {}


def cashier(name: str) -> None:
    """Wait for the shop to open, then record the start time."""
    shop_open.wait()
    start_times[name] = time.perf_counter()


cashiers = [
    threading.Thread(target=cashier, args=(name,)) for name in ("till-1", "till-2", "till-3")
]
for till in cashiers:
    till.start()
time.sleep(0.05)
print(f"before set: {shop_open.is_set()=}, started={len(start_times)}")
shop_open.set()
for till in cashiers:
    till.join()
spread_ms = (max(start_times.values()) - min(start_times.values())) * 1000
print(f"after set: started={len(start_times)}, spread {spread_ms:.2f} ms")
shop_open.clear()
print(f"after clear, wait(timeout=0.02) returns {shop_open.wait(timeout=0.02)}")

before set: shop_open.is_set()=False, started=0
after set: started=3, spread 0.19 ms
after clear, wait(timeout=0.02) returns False


#### `Condition`

A condition variable combines a lock with "wait until something is true".
`wait_for(predicate)` releases the lock while waiting and re-checks the
predicate after every `notify`. Here two packers wait for stock; a delivery
thread adds items and calls `notify_all()`.

In [35]:
stock_arrived = threading.Condition()
warehouse: list[str] = []
packed: list[str] = []


def packer(name: str, wanted: int) -> None:
    """Wait until enough items exist, then take them."""
    with stock_arrived:
        stock_arrived.wait_for(lambda: len(warehouse) >= wanted, timeout=2)
        taken = [warehouse.pop() for _ in range(min(wanted, len(warehouse)))]
        packed.append(f"{name} packed {len(taken)}")


def deliver(items: list[str]) -> None:
    """Add items one by one, waking the waiters each time."""
    for item in items:
        time.sleep(0.01)
        with stock_arrived:
            warehouse.append(item)
            stock_arrived.notify_all()


run_in_threads(
    lambda job: job(),
    [lambda: packer("packer-A", 2), lambda: packer("packer-B", 3), lambda: deliver(list("abcde"))],
)
print(f"{sorted(packed)}, left in warehouse: {warehouse}")

['packer-A packed 2', 'packer-B packed 3'], left in warehouse: []


> **Note:** always wait in a loop (or with `wait_for`): a thread can wake up
> without its condition being true (another thread took the stock first, or a
> spurious wake-up). `notify()` wakes one waiter, `notify_all()` wakes all.

#### `Barrier`

A barrier for `n` parties blocks every caller of `wait()` until `n` have
arrived, then releases them together; `wait()` returns a distinct index
`0 … n-1` and an optional `action` runs once per round. If a party never
arrives, `wait(timeout)` breaks the barrier and every waiter gets
`BrokenBarrierError`.

In [36]:
report_ready = threading.Barrier(3, action=lambda: print("action: all 3 parts loaded, merging"))
arrival_indexes: list[int] = []


def load_part(seconds: float) -> None:
    """Load a part of a report, then wait for the others."""
    time.sleep(seconds)
    arrival_indexes.append(report_ready.wait())


run_in_threads(load_part, [0.01, 0.03, 0.02])
print(f"indexes returned by wait(): {sorted(arrival_indexes)}")

lonely = threading.Barrier(2)
print(f"only 1 of 2 parties arrives: {raised_error_name(lambda: lonely.wait(timeout=0.05))}")
print(f"{lonely.broken=}")

action: all 3 parts loaded, merging
indexes returned by wait(): [0, 1, 2]


only 1 of 2 parties arrives: BrokenBarrierError
lonely.broken=True


#### `Timer`

`threading.Timer(delay, function, args)` is a `Thread` subclass that waits
`delay` seconds, then calls the function, unless `cancel()` was called first.

In [37]:
reminders: list[str] = []
fires = threading.Timer(0.05, reminders.append, args=("payment reminder sent",))
cancelled = threading.Timer(0.05, reminders.append, args=("this never happens",))
fires.start()
cancelled.start()
cancelled.cancel()  # before the delay is over
fires.join()
cancelled.join()
print(f"{reminders=}  {isinstance(fires, threading.Thread)=}")

reminders=['payment reminder sent']  isinstance(fires, threading.Thread)=True


#### Deadlocks and prevention

A **deadlock** is a set of threads each waiting for a lock another one holds.
It needs all four (Coffman) conditions: mutual exclusion, hold-and-wait, no
preemption, and a **circular wait**. Remove any one and deadlock is impossible.

| Prevention | How | Breaks |
| --- | --- | --- |
| Lock ordering | every thread takes locks in one global order (for example by id) | circular wait |
| Try-lock with back-off | `acquire(timeout=...)`; on failure release everything, wait a random moment, retry | hold-and-wait |
| One coarser lock | protect both resources with a single lock | hold-and-wait (at the cost of concurrency) |
| Don't hold a lock while calling unknown code | callbacks, I/O and other objects' methods may take locks of their own | hidden cycles |
| Share by message passing | hand work to one owner thread through a `Queue` (5.1.6) | mutual exclusion on shared data |

The demo: one thread updates the *inventory* then the *ledger*, another the
ledger then the inventory. A `Barrier` makes both take their first lock before
either asks for its second, so the circular wait happens every time; the
`acquire(timeout=0.2)` turns the hang into a reported failure.

In [38]:
def lock_pair_naive(
    first: threading.Lock, second: threading.Lock, both_hold_one: threading.Barrier
) -> str:
    """Take first, wait until the other thread holds its first lock, then try second."""
    with first:
        both_hold_one.wait()
        if not second.acquire(timeout=0.2):
            return "gave up: deadlock"
        second.release()
        return "done"


inventory_lock, ledger_lock = threading.Lock(), threading.Lock()
rendezvous = threading.Barrier(2)
outcomes: list[str] = []
run_in_threads(
    lambda locks: outcomes.append(lock_pair_naive(*locks, rendezvous)),
    [(inventory_lock, ledger_lock), (ledger_lock, inventory_lock)],
)
print(f"{outcomes=}")

outcomes=['gave up: deadlock', 'done']


Both threads held one lock and waited for the other: a circular wait. The first
thread to time out gave up and released its lock, and only then could the other
finish. Without the timeout both would wait forever (and the kernel would hang).

**Fix with try-lock and back-off.** If the second lock is not free soon, release
the first, sleep a short *random* time so the two threads stop colliding, and
try again. (Lab 5.1 fixes a deadlock the other way, with lock ordering.)

In [39]:
def lock_pair_backoff(
    first: threading.Lock, second: threading.Lock, both_hold_one: threading.Barrier, seed: int
) -> str:
    """Acquire both locks or neither; retry with random back-off."""
    backoff = random.Random(seed)
    for attempt in range(1, 20):
        with first:
            if attempt == 1:
                both_hold_one.wait()  # force the collision on the first try
            if second.acquire(timeout=0.05):
                second.release()
                return f"done on attempt {attempt}"
        time.sleep(backoff.uniform(0.0, 0.05))  # first is released here
    return "gave up"


rendezvous = threading.Barrier(2)
outcomes = []
run_in_threads(
    lambda job: outcomes.append(lock_pair_backoff(job[0], job[1], rendezvous, seed=job[2])),
    [(inventory_lock, ledger_lock, 1), (ledger_lock, inventory_lock, 2)],
)
print(f"{sorted(outcomes)=}")

sorted(outcomes)=['done on attempt 1', 'done on attempt 2']


### 5.1.6 Thread-safe queues

The `queue` module gives thread-safe queues with the locking built in, which is
usually the simplest correct design: threads share a queue instead of sharing
data structures.

| Class / method | Behaviour |
| --- | --- |
| `queue.Queue(maxsize=0)` | FIFO; `maxsize > 0` makes `put` block when full (back-pressure) |
| `queue.LifoQueue` | LIFO (a stack) |
| `queue.PriorityQueue` | smallest item first (a heap); put `(priority, tie_breaker, item)` tuples |
| `put(item, block=True, timeout=None)`, `put_nowait` | `queue.Full` when it cannot add |
| `get(block=True, timeout=None)`, `get_nowait` | `queue.Empty` when nothing arrives |
| `task_done()`, `join()` | consumers mark items finished; `join()` waits until every `put` has a `task_done` |
| `qsize()`, `empty()`, `full()` | snapshots only: another thread may change the answer immediately |
| `shutdown(immediate=False)` | *new in 3.13*: later `put`s raise `queue.ShutDown`; `get` raises it once the queue is drained |

**FIFO, LIFO and priority order** for the same three items:

In [40]:
for queue_class in (queue.Queue, queue.LifoQueue):
    tickets = queue_class()
    for ticket in ("T-1 login bug", "T-2 typo", "T-3 outage"):
        tickets.put(ticket)
    print(f"{queue_class.__name__:<10} -> {[tickets.get() for _ in range(3)]}")

by_priority: queue.PriorityQueue[tuple[int, int, str]] = queue.PriorityQueue()
arrival = itertools.count()  # tie-breaker: equal priorities keep arrival order
for priority, ticket in (
    (3, "T-1 login bug"),
    (5, "T-2 typo"),
    (1, "T-3 outage"),
    (3, "T-4 slow page"),
):
    by_priority.put((priority, next(arrival), ticket))
print(f"PriorityQueue -> {[by_priority.get()[2] for _ in range(4)]}")

Queue      -> ['T-1 login bug', 'T-2 typo', 'T-3 outage']
LifoQueue  -> ['T-3 outage', 'T-2 typo', 'T-1 login bug']
PriorityQueue -> ['T-3 outage', 'T-1 login bug', 'T-4 slow page', 'T-2 typo']


The tie-breaker matters: without it, two equal priorities would compare the
next tuple element, and two items that cannot be compared (dicts, custom objects)
raise `TypeError` inside the queue.

**Bounded queues and timeouts.** `maxsize` makes a fast producer wait for a slow
consumer instead of filling memory. The non-blocking and timeout forms raise
`queue.Full` and `queue.Empty`.

In [41]:
small_buffer: queue.Queue[str] = queue.Queue(maxsize=2)
small_buffer.put("line 1")
small_buffer.put("line 2")
print(
    f"{small_buffer.full()=}  put_nowait: "
    f"{raised_error_name(lambda: small_buffer.put_nowait('line 3'))}"
)
print(f"put with timeout: {raised_error_name(lambda: small_buffer.put('line 3', timeout=0.02))}")
print(f"{small_buffer.get()=}  {small_buffer.get()=}")
print(
    f"get with timeout on empty queue: {raised_error_name(lambda: small_buffer.get(timeout=0.02))}"
)

small_buffer.full()=True  put_nowait: Full
put with timeout: Full
small_buffer.get()='line 1'  small_buffer.get()='line 2'
get with timeout on empty queue: Empty


**`task_done` and `join`.** Every `put` increments an "unfinished tasks"
counter; every `task_done()` decrements it; `queue.join()` blocks until it is
zero. That lets the producer wait until the work is *processed*, not just taken
off the queue.

In [42]:
resize_jobs: queue.Queue[str] = queue.Queue()
resized: list[str] = []


def resize_worker() -> None:
    """Process jobs forever (a daemon thread); mark each one done."""
    while True:
        image_name = resize_jobs.get()
        time.sleep(0.01)
        resized.append(image_name)
        resize_jobs.task_done()


threading.Thread(target=resize_worker, daemon=True, name="resizer").start()
for image_name in ("a.png", "b.png", "c.png"):
    resize_jobs.put(image_name)
print(f"right after put: {resize_jobs.unfinished_tasks=}")
resize_jobs.join()
print(f"after join:      {resize_jobs.unfinished_tasks=}, {resized=}")

right after put: resize_jobs.unfinished_tasks=3


after join:      resize_jobs.unfinished_tasks=0, resized=['a.png', 'b.png', 'c.png']


That worker never ends: it is a daemon, so it would simply be killed at exit.

**Sentinel shutdown.** To stop consumers cleanly, put one *sentinel* value per
consumer after the real work. A unique `object()` is safer than `None`, which
could be real data. Each consumer exits when it takes a sentinel, so `join()` on
the threads returns.

In [43]:
STOP = object()  # sentinel: compared by identity
log_lines: queue.Queue[object] = queue.Queue()
error_counts: dict[str, int] = {}
counts_lock = threading.Lock()


def count_errors() -> None:
    """Consume log lines until the sentinel arrives."""
    while (line := log_lines.get()) is not STOP:
        if " ERROR " in line:
            reader = threading.current_thread().name
            with counts_lock:
                error_counts[reader] = error_counts.get(reader, 0) + 1
        log_lines.task_done()
    log_lines.task_done()  # the sentinel counts as a task too


consumers = [
    threading.Thread(target=count_errors, name=f"log-reader-{reader_number}")
    for reader_number in (1, 2)
]
for consumer in consumers:
    consumer.start()
for number in range(1, 101):
    log_lines.put(
        f"2026-11-16 10:00:{number % 60:02d} "
        f"{'ERROR' if number % 10 == 0 else 'INFO'} request {number}"
    )
for _ in consumers:
    log_lines.put(STOP)
for consumer in consumers:
    consumer.join()
print(
    f"total errors: {sum(error_counts.values())}; "
    f"threads alive: {[consumer.is_alive() for consumer in consumers]}"
)

total errors: 10; threads alive: [False, False]


**`Queue.shutdown()`** (*new in 3.13*) replaces the sentinel dance: after
`shutdown()`, `put` raises `queue.ShutDown`, and `get` keeps returning the
remaining items, then raises `queue.ShutDown` once the queue is empty.
`shutdown(immediate=True)` drops the remaining items instead.

In [44]:
if sys.version_info >= (3, 13):
    closing_queue: queue.Queue[str] = queue.Queue()
    for order_id in ("A-1", "A-2"):
        closing_queue.put(order_id)
    closing_queue.shutdown()
    print(f"put after shutdown: {raised_error_name(lambda: closing_queue.put('A-3'))}")
    print(
        f"drain: {closing_queue.get()}, {closing_queue.get()}, "
        f"then {raised_error_name(closing_queue.get)}"
    )
else:
    print("SKIP: Queue.shutdown is new in 3.13")

put after shutdown: ShutDown
drain: A-1, A-2, then ShutDown


### 5.1.7 Thread-local state

`threading.local()` creates an object whose attributes are **separate for every
thread**: each thread sees only what it set itself. Web frameworks used this for
"the current request" (Flask's `request`); in asyncio code the equivalent is
`contextvars` (Unit 5.3), and in Node it is `AsyncLocalStorage`.

Three request-handling threads store their user on a shared `local` object; a
deeply nested function reads it without the value being passed around.

In [45]:
request_context = threading.local()
audit_log: list[str] = []


def write_audit(action: str) -> None:
    """Deep in the call stack: read the user of *this* thread's request."""
    audit_log.append(f"{request_context.user} {action}")


def handle_request(user: str) -> None:
    """Set the per-thread context, then do work that needs it."""
    request_context.user = user
    time.sleep(0.01)  # other threads set their own user meanwhile
    write_audit("viewed orders")


run_in_threads(handle_request, ["asha", "ravi", "meena"])
print(sorted(audit_log))
print(f"main thread: {raised_message(lambda: request_context.user)}")

['asha viewed orders', 'meena viewed orders', 'ravi viewed orders']
main thread: AttributeError: '_thread._local' object has no attribute 'user'


The main thread never set `user`, so it gets `AttributeError`. Subclass `local`
to give every thread defaults: `__init__` runs once **per thread**, the first
time that thread touches the object.

A real use: a `sqlite3` connection may only be used by the thread that created
it (`check_same_thread=True`), so each worker thread keeps its own connection.

In [46]:
class PerThreadDatabase(threading.local):
    """One in-memory SQLite connection per thread, created on first use."""

    def __init__(self) -> None:
        """Open a private in-memory database and record the owning thread."""
        self.connection = sqlite3.connect(":memory:")
        self.owner = threading.current_thread().name


shared_connection = sqlite3.connect(":memory:")
print("shared connection used from another thread:")
run_in_threads(
    lambda _: print(" ", raised_error_name(lambda: shared_connection.execute("select 1"))), [0]
)

per_thread_db = PerThreadDatabase()
connections_seen: list[sqlite3.Connection] = []


def query_in_thread(_: object) -> None:
    """Use this thread's own connection (created on first access)."""
    connections_seen.append(per_thread_db.connection)
    per_thread_db.connection.execute("select 1")


run_in_threads(query_in_thread, [0, 1])
print(f"each worker got its own connection: {connections_seen[0] is not connections_seen[1]}")
print(f"the main thread's is a third one: {per_thread_db.connection not in connections_seen}")
print(f"main thread owner: {per_thread_db.owner}")

shared connection used from another thread:
  ProgrammingError
each worker got its own connection: True
the main thread's is a third one: True
main thread owner: MainThread


> **Warning:** thread pools reuse threads, so thread-local values **leak from
> one task to the next** on the same worker. Reset them at the start of each
> task (or use `contextvars`, which executors and asyncio copy per task).

In [47]:
task_local = threading.local()


def tag_task(task_name: str) -> str:
    """Report what the previous task on this thread left behind, then tag it."""
    leftover = getattr(task_local, "task", None)
    task_local.task = task_name
    return f"{task_name} saw leftover {leftover!r}"


with concurrent.futures.ThreadPoolExecutor(max_workers=1) as single_worker:
    print(list(single_worker.map(tag_task, ["task-1", "task-2", "task-3"])))

['task-1 saw leftover None', "task-2 saw leftover 'task-1'", "task-3 saw leftover 'task-2'"]


### 5.1.8 `concurrent.futures.ThreadPoolExecutor`

An executor owns a pool of worker threads and a work queue. You hand it
callables; it hands back `Future` objects, which hold the eventual result **or
exception**. This is the high-level API to reach for first.

| API | What it does |
| --- | --- |
| `ThreadPoolExecutor(max_workers=None, thread_name_prefix="")` | pool; threads start lazily as work arrives |
| `executor.submit(fn, *args, **kwargs)` | schedule one call; returns a `Future` immediately |
| `executor.map(fn, *iterables, timeout=None, buffersize=None)` | results **in input order**; raises a task's exception when you reach it |
| `as_completed(futures, timeout=None)` | yields futures **in completion order** |
| `wait(futures, timeout=None, return_when=ALL_COMPLETED)` | returns `(done, not_done)` sets; also `FIRST_COMPLETED`, `FIRST_EXCEPTION` |
| `future.result(timeout)`, `exception(timeout)` | block for the outcome; `result()` re-raises the task's exception |
| `future.done()`, `running()`, `cancelled()`, `cancel()` | state; only a *pending* future can be cancelled |
| `future.add_done_callback(fn)` | call `fn(future)` when it finishes |
| `executor.shutdown(wait=True, cancel_futures=False)` | stop accepting work; `with` calls it for you |

**`submit` and `Future`.** `slow_echo` waits on the local API for the given
time, then returns its first argument. (In JS terms: `submit` is like calling an
async function; the `Future` is the promise. But `result()` *blocks* the
calling thread instead of being awaited.)

In [48]:
def slow_echo(value: str, seconds: float) -> str:
    """Wait `seconds` on the local API, then return value."""
    fetch_slow_page(seconds)
    return value


with concurrent.futures.ThreadPoolExecutor(max_workers=2, thread_name_prefix="shop") as executor:
    invoice_future = executor.submit(slow_echo, "invoice-17", 0.2)
    time.sleep(0.02)
    print(f"just after submit: done={invoice_future.done()}, running={invoice_future.running()}")
    print(f"result(timeout=5) -> {invoice_future.result(timeout=5)!r}")
    print(
        f"afterwards:        done={invoice_future.done()}, exception={invoice_future.exception()}"
    )
    print(
        f"worker thread name: {executor.submit(lambda: threading.current_thread().name).result()}"
    )

just after submit: done=False, running=True


result(timeout=5) -> 'invoice-17'
afterwards:        done=True, exception=None
worker thread name: shop_0


**Cancelling and callbacks.** With one worker, the first task runs and the
others wait in the queue. `cancel()` succeeds only for a task that has not
started. A done-callback runs in the worker thread that finished the task (or
immediately in the caller if the future is already done).

In [49]:
finished: list[str] = []
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    running_job = executor.submit(slow_echo, "job-1", 0.1)
    queued_job = executor.submit(slow_echo, "job-2", 0.1)
    doomed_job = executor.submit(slow_echo, "job-3", 0.1)
    for job in (running_job, queued_job, doomed_job):
        job.add_done_callback(
            lambda done: finished.append("cancelled" if done.cancelled() else done.result())
        )
    time.sleep(0.02)
    print(f"cancel running job-1: {running_job.cancel()}")
    print(f"cancel queued job-3:  {doomed_job.cancel()}")
print(f"callbacks saw: {finished}")

cancel running job-1: False
cancel queued job-3:  True


callbacks saw: ['cancelled', 'job-1', 'job-2']


**`map`: input order, not finish order.** The three tasks finish in the order
0.1 s, 0.2 s, 0.3 s, but `map` yields results in the order you passed them. The
total time is the slowest task, not the sum. `timeout` applies to the whole
iteration and raises `TimeoutError` (the built-in, since 3.11) when exceeded.

In [50]:
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    with Timer("map over 0.3 s, 0.1 s, 0.2 s tasks"):
        print(list(executor.map(slow_echo, ["slow", "fast", "medium"], [0.3, 0.1, 0.2])))
    too_slow = executor.map(slow_echo, ["late"], [0.3], timeout=0.05)
    print(f"with timeout=0.05: {raised_error_name(lambda: list(too_slow))}")

['slow', 'fast', 'medium']
map over 0.3 s, 0.1 s, 0.2 s tasks took 304.04 ms
with timeout=0.05: TimeoutError


**`map` with `buffersize`** (*new in 3.14*). Plain `map` submits a task for
every input item straight away, so it reads the whole input iterable up front
(an infinite generator would hang). `buffersize=n` keeps at most `n` tasks in
flight and reads more input only as you consume results.

In [51]:
def numbered_rows(limit: int, consumed: list[int]) -> Iterator[int]:
    """Yield 0..limit-1, recording how many rows were read."""
    for row_number in range(limit):
        consumed.append(row_number)
        yield row_number


if sys.version_info >= (3, 14):
    for buffer_size in (None, 4):
        consumed_rows: list[int] = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
            squares = executor.map(
                lambda row_number: row_number * row_number,
                numbered_rows(10_000, consumed_rows),
                buffersize=buffer_size,
            )
            first_square = next(squares)
            print(
                f"buffersize={buffer_size}: first result {first_square}, "
                f"rows read so far: {len(consumed_rows):,}"
            )
else:
    print("SKIP: Executor.map(buffersize=) is new in 3.14")

buffersize=None: first result 0, rows read so far: 10,000
buffersize=4: first result 0, rows read so far: 5


**`as_completed`: finish order.** Use it when you want to handle each result as
soon as it is ready (show progress, start follow-up work). Keep a dict from
future to input so you know which task finished.

In [52]:
delays = {"report-A": 0.3, "report-B": 0.1, "report-C": 0.2}
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    future_to_name = {
        executor.submit(slow_echo, name, seconds): name for name, seconds in delays.items()
    }
    print(
        "completion order:",
        [future.result() for future in concurrent.futures.as_completed(future_to_name)],
    )
    stragglers = [executor.submit(slow_echo, "late", 0.3)]
    waiting = concurrent.futures.as_completed(stragglers, timeout=0.05)
    print(f"as_completed(timeout=0.05): {raised_error_name(lambda: list(waiting))}")

completion order: ['report-B', 'report-C', 'report-A']
as_completed(timeout=0.05): TimeoutError


**`wait`.** Returns two sets, `done` and `not_done`, when the condition is met.
`FIRST_COMPLETED` is handy for "take the fastest mirror"; `FIRST_EXCEPTION`
stops waiting as soon as one task fails.

In [53]:
def fail_after(seconds: float) -> str:
    """Raise after a short wait."""
    time.sleep(seconds)
    raise ConnectionError("mirror unreachable")


with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    mirrors = [
        executor.submit(slow_echo, f"mirror-{mirror_number}", seconds)
        for mirror_number, seconds in enumerate((0.3, 0.1, 0.2))
    ]
    done, not_done = concurrent.futures.wait(
        mirrors, return_when=concurrent.futures.FIRST_COMPLETED
    )
    print(
        f"FIRST_COMPLETED: done={[future.result() for future in done]}, "
        f"still running={len(not_done)}"
    )

    mixed = [executor.submit(fail_after, 0.05), executor.submit(slow_echo, "slow", 0.3)]
    done, not_done = concurrent.futures.wait(mixed, return_when=concurrent.futures.FIRST_EXCEPTION)
    print(
        f"FIRST_EXCEPTION: done={len(done)} "
        f"({next(iter(done)).exception()!r}), not done={len(not_done)}"
    )

    done, not_done = concurrent.futures.wait(mirrors + mixed)  # ALL_COMPLETED
    print(f"ALL_COMPLETED: done={len(done)}, not done={len(not_done)}")

FIRST_COMPLETED: done=['mirror-1'], still running=2
FIRST_EXCEPTION: done=1 (ConnectionError('mirror unreachable')), not done=1


ALL_COMPLETED: done=5, not done=0


**Sizing.** The default `max_workers` is `min(32, (os.process_cpu_count() or 1)
+ 4)` (*changed in 3.13*: it used `os.cpu_count()` before). The right number
depends on the work:

| Work | Size the pool | Why |
| --- | --- | --- |
| I/O-bound (HTTP, database) | about the concurrency you want: throughput × latency (Little's law), capped by what the server allows | waiting threads cost little |
| C code that releases the GIL (hashing, compression, NumPy) | about the number of cores | more threads than cores just queue up |
| Pure-Python CPU work (default build) | threads do not help | use processes (5.2) or a free-threaded build |

The cell prints the default (via the private `_max_workers` attribute, for
demonstration only) and times 16 requests of 0.05 s each with 1, 4 and 16
workers.

In [54]:
print(
    f"{os.process_cpu_count()=}, default "
    f"max_workers={concurrent.futures.ThreadPoolExecutor()._max_workers}"
)
for worker_count in (1, 4, 16):
    with concurrent.futures.ThreadPoolExecutor(max_workers=worker_count) as executor:
        with Timer(f"16 requests x 0.05 s, {worker_count:>2} workers", report=None) as timing:
            list(executor.map(fetch_slow_page, [0.05] * 16))
    print(
        f"{timing.label}: {timing.elapsed_seconds:.2f} s "
        f"(ideal {0.05 * -(-16 // worker_count):.2f} s)"
    )

os.process_cpu_count()=16, default max_workers=20


16 requests x 0.05 s,  1 workers: 0.86 s (ideal 0.80 s)


16 requests x 0.05 s,  4 workers: 0.23 s (ideal 0.20 s)
16 requests x 0.05 s, 16 workers: 0.07 s (ideal 0.05 s)


**Shutdown.** Leaving the `with` block calls `shutdown(wait=True)`: it waits for
every submitted task. `shutdown(wait=False, cancel_futures=True)` cancels
everything still queued and returns at once (running tasks still finish).
After shutdown, `submit` raises `RuntimeError`.

In [55]:
executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
backlog = [executor.submit(slow_echo, f"email-{email_number}", 0.05) for email_number in range(10)]
time.sleep(0.02)
executor.shutdown(wait=False, cancel_futures=True)
print(
    f"cancelled: {sum(future.cancelled() for future in backlog)}, ran "
    f"or running: {sum(not future.cancelled() for future in backlog)}"
)
print(f"submit after shutdown: {raised_message(lambda: executor.submit(print))}")
started = [future.result() for future in backlog if not future.cancelled()]  # let them finish

cancelled: 9, ran or running: 1
submit after shutdown: RuntimeError: cannot schedule new futures after shutdown


**Exceptions.** A task's exception is stored in its future. `exception()`
returns it; `result()` re-raises the *same* object, with the worker's traceback
attached. The danger is the opposite case: a fire-and-forget `submit` whose
future nobody looks at swallows the error silently.

In [56]:
def parse_quantity(text: str) -> int:
    """Parse a quantity; fails on bad input."""
    return int(text)


with concurrent.futures.ThreadPoolExecutor() as executor:
    bad = executor.submit(parse_quantity, "three")
    stored = bad.exception()
    print(f"exception(): {stored!r}")
    try:
        bad.result()
    except ValueError as error:
        last_frame = traceback.extract_tb(error.__traceback__)[-1].name
        print(
            "result() re-raised the same object: "
            f"{error is stored}; traceback ends in {last_frame!r}"
        )
    executor.submit(parse_quantity, "seven")  # fire and forget: the ValueError vanishes
print("no error was printed for 'seven': always check futures (or use map / as_completed)")

exception(): ValueError("invalid literal for int() with base 10: 'three'")
result() re-raised the same object: True; traceback ends in 'parse_quantity'
no error was printed for 'seven': always check futures (or use map / as_completed)


### 5.1.9 Errors and shutdown

An exception that escapes a thread's `run()` does **not** reach the thread that
called `join()`. The thread dies, `threading.excepthook` is called, and the rest
of the program carries on. The default hook prints `Exception in thread NAME:`
and the traceback to stderr. Seen from a script in a child process:

In [57]:
print(
    run_child("""
    import threading

    def import_prices():
        raise KeyError("sku-404")

    worker = threading.Thread(target=import_prices, name="price-importer")
    worker.start()
    worker.join()
    print("main thread: join() returned normally, worker.is_alive() =", worker.is_alive())
""")
)

main thread: join() returned normally, worker.is_alive() = False
Exception in thread price-importer:
Traceback (most recent call last):
  File "<stdlib>/threading.py", line 1218, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "<stdlib>/threading.py", line 1160, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<scratch>/child_script.py", line 4, in import_prices
    raise KeyError("sku-404")
KeyError: 'sku-404'


**A custom `threading.excepthook`** receives one `ExceptHookArgs` argument with
`exc_type`, `exc_value`, `exc_traceback` and `thread`. Install one to log thread
failures properly (or to collect them, as here). Restore the previous hook
afterwards; `threading.__excepthook__` always holds the original.

In [58]:
thread_failures: list[str] = []


def collect_failure(hook_args: threading.ExceptHookArgs) -> None:
    """Record which thread failed and why, instead of printing a traceback."""
    thread_name = hook_args.thread.name if hook_args.thread else "?"
    thread_failures.append(f"{thread_name}: {hook_args.exc_type.__name__}: {hook_args.exc_value}")


def import_prices() -> None:
    """A thread body that fails."""
    raise KeyError("sku-404")


previous_hook = threading.excepthook
threading.excepthook = collect_failure
try:
    failing = threading.Thread(target=import_prices, name="price-importer")
    failing.start()
    failing.join()
finally:
    threading.excepthook = previous_hook
print(f"{thread_failures=}")
print(f"{threading.__excepthook__ is not collect_failure=}")

thread_failures=["price-importer: KeyError: 'sku-404'"]
threading.__excepthook__ is not collect_failure=True


**Daemon threads at shutdown, again.** A daemon thread is frozen mid-step when
the interpreter exits: buffered data is lost and `finally` blocks never run. The
child script below starts two writers that append a line every 10 ms and write
a footer in `finally`. One is a daemon; the other is stopped by a flag the main
thread sets before exiting. Then the parent reads both files.

In [59]:
WRITERS_DEMO = """
    import threading, time

    stop = threading.Event()

    def write_lines(path, stop_flag):
        with open(path, "w") as output:
            try:
                for number in range(1000):
                    if stop_flag is not None and stop_flag.is_set():
                        break
                    output.write(f"line {number}\\n")
                    time.sleep(0.01)
            finally:
                output.write("FOOTER\\n")

    threading.Thread(target=write_lines, args=("daemon.txt", None), daemon=True).start()
    flagged = threading.Thread(target=write_lines, args=("flagged.txt", stop))
    flagged.start()
    time.sleep(0.05)
    stop.set()  # ask the non-daemon writer to finish
    print("main thread exiting")
"""
print(run_child(WRITERS_DEMO))
for name in ("daemon.txt", "flagged.txt"):
    lines = (PHASE5_DIR / name).read_text().splitlines()
    print(f"{name:<12} {len(lines)} lines, last line: {lines[-1] if lines else '(empty file)'!r}")

main thread exiting
daemon.txt   0 lines, last line: '(empty file)'
flagged.txt  6 lines, last line: 'FOOTER'


The daemon's file has no footer (and may even be empty: its buffered lines were
never flushed). The flagged writer stopped at a clean point and closed its file.

**Stop flags.** Python has no `thread.kill()`: killing a thread could leave
locks held and data half-written. Stopping is cooperative: the thread checks a
flag. Use an `Event` and wait on it with a timeout instead of `time.sleep`, so
the thread wakes up the moment the flag is set.

In [60]:
def poll_with_sleep(stop: threading.Event) -> None:
    """Responds to stop only after the current sleep ends."""
    while not stop.is_set():
        time.sleep(0.3)


def poll_with_wait(stop: threading.Event) -> None:
    """Event.wait returns as soon as stop is set."""
    while not stop.wait(timeout=0.3):
        pass


for poller in (poll_with_sleep, poll_with_wait):
    stop_flag = threading.Event()
    poller_thread = threading.Thread(target=poller, args=(stop_flag,))
    poller_thread.start()
    time.sleep(0.05)
    with Timer(f"{poller.__name__}: time from set() to exit", report=None) as stop_timing:
        stop_flag.set()
        poller_thread.join()
    print(f"{stop_timing.label}: {stop_timing.elapsed_seconds * 1000:.0f} ms")

poll_with_sleep: time from set() to exit: 250 ms
poll_with_wait: time from set() to exit: 1 ms


> **Note:** `KeyboardInterrupt` (Ctrl+C) and every signal handler run only in
> the **main** thread. A worker blocked in `queue.get()` or `lock.acquire()`
> never sees Ctrl+C, so the main thread must set the stop flag (or put
> sentinels) when it catches `KeyboardInterrupt`, then join the workers.

### 5.1.10 Free-threaded Python

| | |
| --- | --- |
| [PEP 703](https://peps.python.org/pep-0703/) | Making the GIL optional: a separate *free-threaded* build of CPython. Experimental in 3.13. |
| [PEP 779](https://peps.python.org/pep-0779/) | Criteria for "supported" status. Since 3.14 the free-threaded build is officially supported (phase II), still a separate build, not the default. |
| Executable | `python3.15t` (the `t` suffix); install with `uv python install 3.15t` |
| Detect the build | `sysconfig.get_config_var("Py_GIL_DISABLED") == 1`, `sys.abiflags` contains `t` |
| Detect the GIL at run time | `sys._is_gil_enabled()` (*new in 3.13*); can be `True` on a free-threaded build |
| Force it | `PYTHON_GIL=0` / `PYTHON_GIL=1`, or `-X gil=0` / `-X gil=1` |

**How it works without the GIL** (Unit 3.1 showed the memory side): biased
reference counting (the owning thread updates its count cheaply; other threads
use atomic operations), immortal objects, per-object locks inside `dict`, `list`
and friends (so the "atomic" table in 5.1.4 still holds), and the `mimalloc`
allocator. The price is some single-threaded speed and memory.

**Detect and probe.** The next cell runs the same probe in this kernel's
interpreter and in `python3.15t` (and once more with `PYTHON_GIL=1`), or prints
a skip message when no free-threaded interpreter is installed.

In [61]:
BUILD_PROBE = """
    import sys, sysconfig
    print("Py_GIL_DISABLED:", sysconfig.get_config_var("Py_GIL_DISABLED"))
    print("sys.abiflags:", repr(sys.abiflags))
    print("extension suffix:", sysconfig.get_config_var("EXT_SUFFIX"))
    print("sys._is_gil_enabled():", sys._is_gil_enabled())
    print("threads inherit contextvars:", bool(sys.flags.thread_inherit_context))
"""
print(f"--- this kernel [{interpreter_label()}]")
print(run_child(BUILD_PROBE))
if FREE_THREADED_PYTHON is None:
    print("SKIP: no free-threaded interpreter found (install one with: uv python install 3.15t)")
else:
    print("--- python3.15t")
    print(run_child(BUILD_PROBE, python=FREE_THREADED_PYTHON))
    print("--- python3.15t with PYTHON_GIL=1")
    print(run_child(BUILD_PROBE, python=FREE_THREADED_PYTHON, extra_env={"PYTHON_GIL": "1"}))

--- this kernel [CPython 3.15.1, default (GIL) build, x86_64]
Py_GIL_DISABLED: 0
sys.abiflags: ''
extension suffix: .cpython-315-x86_64-linux-gnu.so
sys._is_gil_enabled(): True
threads inherit contextvars: False
--- python3.15t
Py_GIL_DISABLED: 1
sys.abiflags: 't'
extension suffix: .cpython-315t-x86_64-linux-gnu.so
sys._is_gil_enabled(): False
threads inherit contextvars: True
--- python3.15t with PYTHON_GIL=1
Py_GIL_DISABLED: 1
sys.abiflags: 't'
extension suffix: .cpython-315t-x86_64-linux-gnu.so
sys._is_gil_enabled(): True
threads inherit contextvars: True


**Compatibility.**

- **Pure-Python packages** run unchanged.
- **C extensions** need a separate wheel (ABI tag `cp315t`; note the different
  extension suffix above) **and** must declare that they are safe without the
  GIL (the `Py_mod_gil` slot). Importing an extension that does not declare it
  **re-enables the GIL** for the whole process and emits a `RuntimeWarning`.
  Check `sys._is_gil_enabled()` after your imports in production.
- **Your own code**: races that the GIL happened to hide become real. The plain
  `counter.value += 1` from 5.1.4, unchanged, in both builds:

In [62]:
PLAIN_INCREMENT_RACE = """
    import sys, threading

    class PlainCounter:
        value = 0

    counter = PlainCounter()

    def add_many(times):
        for _ in range(times):
            counter.value += 1

    threads = [threading.Thread(target=add_many, args=(100_000,)) for _ in range(4)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    print(f"GIL enabled: {sys._is_gil_enabled()!s:<5}  expected 400,000, got {counter.value:,}")
"""
print(run_child(PLAIN_INCREMENT_RACE))
if FREE_THREADED_PYTHON is None:
    print("SKIP: no free-threaded interpreter found")
else:
    print(run_child(PLAIN_INCREMENT_RACE, python=FREE_THREADED_PYTHON))

GIL enabled: True   expected 400,000, got 400,000
GIL enabled: False  expected 400,000, got 135,130


**Benchmark.** Each thread runs the same pure-Python loop (1 000 000 iterations
× `DEMO_SCALE`); 1, 2, 4 and 8 threads run at once. *Throughput speedup* =
threads × (time for one thread) ÷ (time for all threads): 1.0 means no gain,
`n` means perfect scaling. The last line compares single-thread speed between
the builds. Expect about 1.0× everywhere on the default build, and scaling up
to your core count on 3.15t.

In [63]:
SCALING_BENCHMARK = """
    import json, os, threading, time

    LOOPS = 1_000_000 * int(os.environ.get("PHASE5_DEMO_SCALE", "1"))

    def spin():
        total = 0
        for number in range(LOOPS):
            total += number * number

    def wall_seconds(thread_count):
        threads = [threading.Thread(target=spin) for _ in range(thread_count)]
        start = time.perf_counter()
        for thread in threads:
            thread.start()
        for thread in threads:
            thread.join()
        return time.perf_counter() - start

    print(json.dumps({count: wall_seconds(count) for count in (1, 2, 4, 8)}))
"""

`scaling_table` runs that script in a child interpreter (this kernel's, then
`python3.15t` if installed) and parses its JSON line; the loop prints one row per
build with the time and throughput speedup for each thread count:

In [64]:
def scaling_table(python: str) -> dict[str, float]:
    """Run the benchmark in a child interpreter; return {thread count: seconds}."""
    return json.loads(run_child(SCALING_BENCHMARK, python=python).splitlines()[-1])


builds = {"default build": scaling_table(sys.executable)}
if FREE_THREADED_PYTHON is None:
    print("SKIP: no free-threaded interpreter found; showing the default build only")
else:
    builds["free-threaded"] = scaling_table(FREE_THREADED_PYTHON)
print(f"CPU cores available: {os.process_cpu_count()}")
for build_name, seconds in builds.items():
    cells = [
        f"{count}T {seconds[count]:.2f}s ({int(count) * seconds['1'] / seconds[count]:.1f}x)"
        for count in seconds
    ]
    print(f"{build_name:<14} " + " | ".join(cells))
if "free-threaded" in builds:
    print(
        "single-thread time, free-threaded / default: "
        f"{builds['free-threaded']['1'] / builds['default build']['1']:.2f}"
    )

CPU cores available: 16
default build  1T 0.03s (1.0x) | 2T 0.07s (1.0x) | 4T 0.14s (1.0x) | 8T 0.30s (0.9x)
free-threaded  1T 0.04s (1.0x) | 2T 0.04s (1.9x) | 4T 0.04s (3.8x) | 8T 0.07s (4.5x)
single-thread time, free-threaded / default: 1.27


On a machine with 4 or more cores the free-threaded row should approach 4× at
four threads, while the default row stays near 1×. The single-thread ratio shows
the cost of running without the GIL: usually a little above 1.0.

| Choose the free-threaded build when | Stay on the default build when |
| --- | --- |
| CPU-bound pure-Python work must run in parallel **and** share large in-memory data | the work is I/O-bound (threads or asyncio already work) |
| every C extension you need ships `cp315t` wheels | a key dependency has no free-threaded wheel (the GIL would come back anyway) |
| you can test for races you never saw before | single-thread speed matters most |

### 5.1.11 JS bridge

**libuv's thread pool.** Node runs your JavaScript on one thread. Network sockets
never use a thread: libuv watches them with `epoll` / `kqueue` / IOCP. Work the
OS cannot do asynchronously goes to libuv's **thread pool**: 4 threads by
default, set with the `UV_THREADPOOL_SIZE` environment variable before startup
(maximum 1024). It runs `fs.*` calls, `dns.lookup`, `crypto.pbkdf2`,
`crypto.scrypt`, `crypto.randomBytes` (async form), async `zlib`, and C++ addon
work. Your JavaScript itself never runs in that pool.

**`worker_threads`.** Each `Worker` is a new V8 isolate with its **own heap,
own event loop and own globals**. Workers communicate by `postMessage`
(structured clone copies), transfer `ArrayBuffer`s, or share memory explicitly
through a `SharedArrayBuffer` with `Atomics`.

| Node | Python | Precise difference |
| --- | --- | --- |
| libuv pool runs C++ work for built-in modules | `ThreadPoolExecutor` runs **any** callable, including Python code | Python code in the pool still takes turns on the GIL (default build); C code that releases it runs in parallel, like libuv's pool |
| `UV_THREADPOOL_SIZE=4` (process-wide, set before start) | `max_workers=` per executor; asyncio's default executor is `min(32, cpus + 4)` | you can have several pools of different sizes |
| `fs.readFile(cb)` goes to the pool automatically | `open().read()` blocks the calling thread; in asyncio use `await asyncio.to_thread(...)` | nothing is offloaded unless you ask |
| `worker_threads`: separate heap, message passing | `threading.Thread`: **one shared heap**, no copying | threads can race on any object; Workers cannot |
| `worker_threads` (isolation + parallelism) | subinterpreters (5.2.6): own GIL, own modules, message passing | the closest match to a Worker |
| `child_process.fork`, `cluster` | `multiprocessing`, `ProcessPoolExecutor` (5.2) | Python pickles messages; Node uses JSON/structured clone over IPC |
| `SharedArrayBuffer` + `Atomics` | `multiprocessing.shared_memory` + locks (5.2.5) | |

**The pool, side by side.** In Node, four `crypto.pbkdf2()` calls run in
parallel on the libuv pool. In Python, `hashlib.pbkdf2_hmac` releases the GIL
while OpenSSL works, so a `ThreadPoolExecutor` of 4 gets the same parallelism.

In [65]:
def derive_key(password: bytes) -> str:
    """PBKDF2-HMAC-SHA256 with many iterations (the GIL is released inside)."""
    iterations = 300_000 * DEMO_SCALE
    return hashlib.pbkdf2_hmac("sha256", password, b"per-user-salt", iterations).hex()[:12]


passwords = [b"asha-pass", b"ravi-pass", b"meena-pass", b"john-pass"]
with Timer("4 keys, one after another") as one_by_one:
    sequential_keys = [derive_key(password) for password in passwords]
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as key_pool:
    with Timer("4 keys, pool of 4 threads") as pooled:
        pooled_keys = list(key_pool.map(derive_key, passwords))
print(
    f"same keys: {pooled_keys == sequential_keys}; speedup "
    f"{one_by_one.elapsed_seconds / pooled.elapsed_seconds:.2f}x"
)

4 keys, one after another took 123.92 ms
4 keys, pool of 4 threads took 37.61 ms
same keys: True; speedup 3.30x


**From asyncio** (Unit 5.3 explains the event loop), `asyncio.to_thread(f, *args)`
runs `f` in the loop's default thread pool and gives back an awaitable. That is
the closest thing to `util.promisify(crypto.pbkdf2)`: blocking or GIL-releasing
work leaves the event loop thread free. The kernel already runs a loop, so the
cell uses top-level `await`.

In [66]:
with Timer("asyncio.gather of 4 to_thread calls"):
    async_keys = await asyncio.gather(
        *(asyncio.to_thread(derive_key, password) for password in passwords)
    )
print(f"same keys again: {async_keys == sequential_keys}")

asyncio.gather of 4 to_thread calls took 39.54 ms
same keys again: True


#### Predict the output

**Predict the output.**

```python
import threading

def work():
    print("working in", threading.current_thread().name)

worker = threading.Thread(target=work(), name="worker")
worker.start()
worker.join()
```

<details><summary>Answer</summary>

`working in MainThread`. `target=work()` **calls** `work` immediately, in the
main thread, and passes its return value (`None`) as the target. The `worker`
thread then starts with no target and does nothing. Pass the function itself:
`target=work`. (In JS the same slip is `setTimeout(work(), 0)`.)

</details>

Run the next cell to check your prediction.

In [67]:
def work() -> None:
    """Print the name of the thread running this function."""
    print("working in", threading.current_thread().name)


worker = threading.Thread(target=work(), name="worker")
worker.start()
worker.join()

working in MainThread


**Predict the output.**

```python
from concurrent.futures import ThreadPoolExecutor
import time

def nap(seconds):
    time.sleep(seconds)
    return seconds

with ThreadPoolExecutor(max_workers=3) as executor:
    print(list(executor.map(nap, [0.3, 0.1, 0.2])))
```

<details><summary>Answer</summary>

`[0.3, 0.1, 0.2]`. The tasks *finish* in the order 0.1, 0.2, 0.3, but `map`
always yields results in **input order**. Use `as_completed` for finish order.
The whole call takes about 0.3 s (the longest task), not 0.6 s.

</details>

Run the next cell to check your prediction.

In [68]:
def nap(seconds: float) -> float:
    """Sleep for seconds, then return them."""
    time.sleep(seconds)
    return seconds


with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    print(list(executor.map(nap, [0.3, 0.1, 0.2])))

[0.3, 0.1, 0.2]


**Predict the output.**

```python
import threading

def explode():
    raise ValueError("boom")

worker = threading.Thread(target=explode)
worker.start()
worker.join()
print("joined")
```

<details><summary>Answer</summary>

The thread's `ValueError` does **not** propagate to `join()`. The default
`threading.excepthook` prints `Exception in thread Thread-N (explode):` and the
traceback to stderr, then the main thread prints `joined`. The proof cell
installs a temporary hook that prints one line instead of the full traceback.

</details>

Run the next cell to check your prediction.

In [69]:
def explode() -> None:
    """Raise a ValueError inside the thread."""
    raise ValueError("boom")


def one_line_hook(hook_args: threading.ExceptHookArgs) -> None:
    """Print one line for an uncaught thread exception instead of a traceback."""
    print(f"excepthook: {hook_args.exc_type.__name__}: {hook_args.exc_value}")


saved_hook, threading.excepthook = threading.excepthook, one_line_hook
try:
    worker = threading.Thread(target=explode)
    worker.start()
    worker.join()
    print("joined")
finally:
    threading.excepthook = saved_hook

excepthook: ValueError: boom
joined


### Lab 5.1

Six exercises; replace each `raise NotImplementedError`, then run the check cell.
Until you finish, the checks print FAIL; that is expected. The first setup cell
below imports what the exercises need and defines `run_threads` (used by the
checks):

In [70]:
import concurrent.futures
import functools
import inspect
import itertools
import json
import math
import os
import queue
import subprocess
import sys
import threading
import time
from collections.abc import Callable, Iterator
from contextlib import contextmanager
from dataclasses import dataclass, field

import httpx


def run_threads(work: Callable[[object], object], items: list[object]) -> None:
    """Run work(item) in one thread per item and join them all (a helper for the checks)."""
    threads = [threading.Thread(target=work, args=(item,)) for item in items]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()


print("Lab 5.1 imports ready; helper: run_threads")

Lab 5.1 imports ready; helper: run_threads


`UnsafeCounter` for exercise 1: it reads the value, gives up the GIL with
`time.sleep(0)`, then writes, so concurrent increments get lost:

In [71]:
class UnsafeCounter:
    """Exercise 1: reads, gives up the GIL, then writes, so concurrent updates get lost."""

    def __init__(self) -> None:
        """Start at zero."""
        self.value = 0

    def increment(self) -> None:
        """Read, yield the GIL, then write, so concurrent updates get lost."""
        current = self.value
        time.sleep(0)  # let another thread run between the read and the write
        self.value = current + 1


print("Lab 5.1 helper ready: UnsafeCounter")

Lab 5.1 helper ready: UnsafeCounter


The third setup cell defines the bank-transfer pieces of exercise 5: `Account`
(an account with its own lock), `transfer_naive` (locks source, then target),
`acquire_or_fail` and `DeadlockError` (a timed acquire, so a deadlock fails fast
instead of hanging the notebook):

In [72]:
class DeadlockError(RuntimeError):
    """Exercise 5: raised when a lock cannot be acquired in time."""


def acquire_or_fail(lock: threading.Lock, timeout: float = 0.5) -> None:
    """Acquire lock or raise DeadlockError after timeout seconds (so a deadlock cannot hang)."""
    if not lock.acquire(timeout=timeout):
        raise DeadlockError(f"could not acquire a lock within {timeout} s")


@dataclass(eq=False)
class Account:
    """Exercise 5: a bank account with its own lock."""

    account_id: int
    balance: int
    lock: threading.Lock = field(default_factory=threading.Lock, repr=False)


def transfer_naive(
    source: Account,
    target: Account,
    amount: int,
    pause: Callable[[], None] = lambda: time.sleep(0.05),
) -> None:
    """Exercise 5: lock source, pause, lock target, move money. Deadlocks on opposite transfers."""
    acquire_or_fail(source.lock)
    try:
        pause()  # widen the window so the deadlock shows every time
        acquire_or_fail(target.lock)
        try:
            source.balance -= amount
            target.balance += amount
        finally:
            target.lock.release()
    finally:
        source.lock.release()


print("Lab 5.1 helpers ready: Account, transfer_naive, acquire_or_fail, DeadlockError")

Lab 5.1 helpers ready: Account, transfer_naive, acquire_or_fail, DeadlockError


#### Exercise 1 — prove a race condition and fix it

1. `hammer(counter, thread_count, increments_per_thread)` starts `thread_count`
   threads, each calling `counter.increment()` `increments_per_thread` times,
   joins them all and returns `counter.value`. Run on an `UnsafeCounter`, it
   should return **less** than `thread_count × increments_per_thread`: that
   proves the race.
2. `LockedCounter` has a `value` attribute starting at 0 and an `increment()`
   that does the same read / `time.sleep(0)` / write as `UnsafeCounter`, but
   inside `with self._lock:` (a `threading.Lock` created in `__init__`).

In [73]:
def hammer(counter: object, thread_count: int, increments_per_thread: int) -> int:
    """Increment counter from several threads at once; return the final value."""
    raise NotImplementedError


class LockedCounter:
    """UnsafeCounter's read / sleep(0) / write, made safe with a Lock."""

    def __init__(self) -> None:
        """Set up the value and whatever protects it."""
        raise NotImplementedError

    def increment(self) -> None:
        """Add one to the value without losing concurrent updates."""
        raise NotImplementedError

Check exercise 1:

In [74]:
def threads_left_behind() -> int:
    """How many more threads are alive after hammer() than before."""
    before = threading.active_count()
    hammer(LockedCounter(), 3, 10)
    return threading.active_count() - before


run_checks(
    [
        (
            "race proven: UnsafeCounter loses updates",
            lambda: hammer(UnsafeCounter(), 4, 200) < 800,
            True,
        ),
        ("LockedCounter: 4 threads x 200", lambda: hammer(LockedCounter(), 4, 200), 800),
        ("LockedCounter: 1 thread x 50", lambda: hammer(LockedCounter(), 1, 50), 50),
        (
            "LockedCounter keeps a threading.Lock",
            lambda: any(
                isinstance(value, type(threading.Lock()))
                for value in vars(LockedCounter()).values()
            ),
            True,
        ),
        ("hammer joins every thread it starts", threads_left_behind, 0),
    ]
)

FAIL  race proven: UnsafeCounter loses updates: raised NotImplementedError: 
FAIL  LockedCounter: 4 threads x 200: raised NotImplementedError: 
FAIL  LockedCounter: 1 thread x 50: raised NotImplementedError: 
FAIL  LockedCounter keeps a threading.Lock: raised NotImplementedError: 
FAIL  hammer joins every thread it starts: raised NotImplementedError: 
0/5 checks passed


#### Exercise 2 — producer–consumer with `Queue`

`process_with_workers(items, handle, worker_count)` returns
`[handle(item) for item in items]`, computed by `worker_count` consumer threads:

- put `(index, item)` pairs on a `queue.Queue`;
- start `worker_count` threads named `consumer-1`, `consumer-2`, …; each takes
  pairs, stores `handle(item)` at its index and calls `task_done()`;
- shut down with **one sentinel per consumer**, then join every consumer;
- return the results in input order.

In [75]:
def process_with_workers[T, R](
    items: list[T], handle: Callable[[T], R], worker_count: int
) -> list[R]:
    """Run handle over items with worker_count consumer threads fed by a Queue."""
    raise NotImplementedError

Check exercise 2:

In [76]:
def square_slowly(number: int) -> int:
    """A handler that takes a few milliseconds."""
    time.sleep(0.005)
    return number * number


def consumer_names_used(worker_count: int) -> tuple[int, bool]:
    """How many distinct threads ran the handler, and whether all were named consumer-N."""
    seen: set[str] = set()

    def record(number: int) -> int:
        """Remember which thread handled the item, then return it unchanged."""
        seen.add(threading.current_thread().name)
        time.sleep(0.01)
        return number

    process_with_workers(list(range(20)), record, worker_count)
    return len(seen), all(name.startswith("consumer-") for name in seen)


def consumers_still_alive() -> list[str]:
    """Names of consumer threads alive after process_with_workers returns."""
    process_with_workers(list(range(10)), square_slowly, 3)
    return [thread.name for thread in threading.enumerate() if thread.name.startswith("consumer-")]

The checks: results in input order, any handler, empty input, all four named
consumers busy, none left alive afterwards, and a real `queue.Queue` underneath:

In [77]:
run_checks(
    [
        (
            "results in input order",
            lambda: process_with_workers(list(range(12)), square_slowly, 4),
            [number * number for number in range(12)],
        ),
        (
            "works with any handler",
            lambda: process_with_workers(["tea", "mug"], str.upper, 2),
            ["TEA", "MUG"],
        ),
        ("empty input", lambda: process_with_workers([], square_slowly, 3), []),
        ("all 4 consumers took work, named consumer-N", lambda: consumer_names_used(4), (4, True)),
        ("no consumer thread left alive", consumers_still_alive, []),
        ("uses queue.Queue", lambda: "Queue(" in inspect.getsource(process_with_workers), True),
    ]
)

FAIL  results in input order: raised NotImplementedError: 
FAIL  works with any handler: raised NotImplementedError: 
FAIL  empty input: raised NotImplementedError: 
FAIL  all 4 consumers took work, named consumer-N: raised NotImplementedError: 
FAIL  no consumer thread left alive: raised NotImplementedError: 
FAIL  uses queue.Queue: expected True, got False
0/6 checks passed


#### Exercise 3 — a thread-pool downloader

`download_all(client, urls, max_workers)` downloads every URL with
`client.get(url)` on a `ThreadPoolExecutor(max_workers)`, collects results with
`as_completed`, and returns a dict from each URL to:

- the number of bytes in the body, for a successful (2xx) response;
- the exception class name otherwise: call `response.raise_for_status()` and
  catch `httpx.HTTPError`, so a 404 gives `"HTTPStatusError"` and a refused
  connection gives `"ConnectError"`.

The checks use an `httpx.Client` with a `MockTransport` that waits 50 ms per
request and records how many requests were in flight at once.

In [78]:
def download_all(
    client: httpx.Client, urls: list[str], max_workers: int = 8
) -> dict[str, int | str]:
    """Download urls concurrently; map each to its byte count or error class name."""
    raise NotImplementedError

Check exercise 3:

In [79]:
def lab_client(latency: float = 0.05) -> tuple[httpx.Client, dict[str, int]]:
    """A client whose transport sleeps per request and tracks the peak in-flight count."""
    stats = {"now": 0, "peak": 0}
    stats_lock = threading.Lock()

    def handler(request: httpx.Request) -> httpx.Response:
        """Answer a mock request after the latency, tracking the in-flight peak."""
        with stats_lock:
            stats["now"] += 1
            stats["peak"] = max(stats["peak"], stats["now"])
        try:
            time.sleep(latency)
            path = request.url.path
            if path == "/broken":
                raise httpx.ConnectError("connection refused", request=request)
            if path.startswith("/files/"):
                return httpx.Response(200, content=b"x" * int(path.rsplit("/", 1)[1]))
            return httpx.Response(404)
        finally:
            with stats_lock:
                stats["now"] -= 1

    return httpx.Client(transport=httpx.MockTransport(handler)), stats


def peak_in_flight(url_count: int, max_workers: int) -> int:
    """Download url_count files and return the peak number of concurrent requests."""
    client, stats = lab_client()
    download_all(
        client,
        [f"http://lab.test/files/{file_number}" for file_number in range(url_count)],
        max_workers,
    )
    return stats["peak"]

The checks: byte counts per URL, errors turned into class names, the concurrency
cap (more than one request in flight, never more than `max_workers`), and the
required `ThreadPoolExecutor` + `as_completed`:

In [80]:
FILE_URLS = ["http://lab.test/files/10", "http://lab.test/files/2048"]
BAD_URLS = ["http://lab.test/missing", "http://lab.test/broken"]
run_checks(
    [
        (
            "byte counts",
            lambda: download_all(lab_client()[0], FILE_URLS, 4),
            {FILE_URLS[0]: 10, FILE_URLS[1]: 2048},
        ),
        (
            "errors become class names",
            lambda: download_all(lab_client()[0], BAD_URLS, 4),
            {BAD_URLS[0]: "HTTPStatusError", BAD_URLS[1]: "ConnectError"},
        ),
        ("12 URLs, 4 workers: concurrent but capped", lambda: 1 < peak_in_flight(12, 4) <= 4, True),
        ("1 worker: one request at a time", lambda: peak_in_flight(4, 1), 1),
        ("no URLs", lambda: download_all(lab_client()[0], [], 4), {}),
        (
            "uses ThreadPoolExecutor and as_completed",
            lambda: all(
                word in inspect.getsource(download_all)
                for word in ("ThreadPoolExecutor", "as_completed")
            ),
            True,
        ),
    ]
)

FAIL  byte counts: raised NotImplementedError: 
FAIL  errors become class names: raised NotImplementedError: 
FAIL  12 URLs, 4 workers: concurrent but capped: raised NotImplementedError: 
FAIL  1 worker: one request at a time: raised NotImplementedError: 
FAIL  no URLs: raised NotImplementedError: 
FAIL  uses ThreadPoolExecutor and as_completed: expected True, got False
0/6 checks passed


#### Exercise 4 — a semaphore-capped connection pool

`ConnectionPool(size, connect)` hands out at most `size` connections at a time:

- `connection()` is a context manager (use `@contextmanager`) that blocks while
  `size` connections are in use (a `threading.BoundedSemaphore(size)`), then
  yields an idle connection, or a new one from `connect()` if none is idle;
  on exit (even after an exception) the connection goes back to the idle list
  and the semaphore is released;
- attributes `created` (how many times `connect()` ran), `in_use` (connections
  currently handed out) and `peak_in_use` (the highest `in_use` seen). Protect
  the counters and the idle list with a `Lock`.

In [81]:
class ConnectionPool:
    """At most `size` connections in use; idle connections are reused."""

    def __init__(self, size: int, connect: Callable[[], object]) -> None:
        """Prepare the semaphore, the idle list and the counters."""
        raise NotImplementedError

    @contextmanager
    def connection(self) -> Iterator[object]:
        """Lend a connection for the duration of a with block."""
        raise NotImplementedError
        yield  # (makes this a generator; remove when you implement it)

Check exercise 4:

In [82]:
def make_pool(size: int) -> ConnectionPool:
    """A pool whose connections are numbered strings."""
    numbers = itertools.count(1)
    return ConnectionPool(size, lambda: f"conn-{next(numbers)}")


def busy_pool(size: int, users: int) -> tuple[int, int, int]:
    """`users` threads each hold a connection for 20 ms; return (peak, created, in_use)."""
    pool = make_pool(size)

    def use(_: object) -> None:
        """Hold one pooled connection for 20 ms."""
        with pool.connection():
            time.sleep(0.02)

    run_threads(use, list(range(users)))
    return pool.peak_in_use, pool.created, pool.in_use


def sequential_reuse() -> list[object]:
    """Five uses one after another should reuse a single connection."""
    pool = make_pool(3)
    handed_out = []
    for _ in range(5):
        with pool.connection() as connection:
            handed_out.append(connection)
    return handed_out


def released_after_error() -> tuple[int, object]:
    """An exception inside the with block must still return the connection."""
    pool = make_pool(1)
    try:
        with pool.connection():
            raise ValueError("query failed")
    except ValueError:
        pass
    with pool.connection() as connection:  # would block forever if the slot leaked
        return pool.in_use, connection

The checks: 12 users through a pool of 3 never see more than 3 connections,
sequential users reuse one connection, an exception still returns it, and the cap
is a `BoundedSemaphore`:

In [83]:
run_checks(
    [
        (
            "12 users, size 3: peak 3, at most 3 created, all returned",
            lambda: busy_pool(3, 12),
            (3, 3, 0),
        ),
        ("sequential uses reuse one connection", sequential_reuse, ["conn-1"] * 5),
        ("exception releases the connection", released_after_error, (1, "conn-1")),
        (
            "uses a BoundedSemaphore",
            lambda: any(
                isinstance(value, threading.BoundedSemaphore)
                for value in vars(make_pool(2)).values()
            ),
            True,
        ),
    ]
)

FAIL  12 users, size 3: peak 3, at most 3 created, all returned: raised NotImplementedError: 
FAIL  sequential uses reuse one connection: raised NotImplementedError: 
FAIL  exception releases the connection: raised NotImplementedError: 
FAIL  uses a BoundedSemaphore: raised NotImplementedError: 
0/4 checks passed


#### Exercise 5 — reproduce and fix a deadlock

`transfer_naive` (setup cell) locks the source account, pauses 50 ms, then locks
the target. `acquire_or_fail` raises `DeadlockError` after 0.5 s instead of
hanging.

1. `run_opposite_transfers(transfer)` creates `Account(1, 100)` and
   `Account(2, 100)`, runs `transfer(account_1, account_2, 10)` and
   `transfer(account_2, account_1, 20)` in two threads at the same time, and
   returns `{"outcomes": sorted list of "ok" or the exception class name per
   thread, "balances": (account_1.balance, account_2.balance)}`. With
   `transfer_naive` the outcomes include `"DeadlockError"`: that reproduces it.
2. `transfer_ordered(source, target, amount, pause=...)` fixes it with **lock
   ordering**: acquire the two locks (with `acquire_or_fail`) in order of
   `account_id`, call `pause()` between the two acquisitions, move the money,
   release both.

In [84]:
def run_opposite_transfers(transfer: Callable[[Account, Account, int], None]) -> dict[str, object]:
    """Run two opposite transfers at once; report outcomes and final balances."""
    raise NotImplementedError


def transfer_ordered(
    source: Account,
    target: Account,
    amount: int,
    pause: Callable[[], None] = lambda: time.sleep(0.05),
) -> None:
    """Move amount from source to target, locking accounts in account_id order."""
    raise NotImplementedError

Check exercise 5:

In [85]:
class RecordingLock:
    """A lock that records the account id each time it is acquired."""

    def __init__(self, account_id: int, log: list[int]) -> None:
        """Wrap a real lock that logs account_id on every acquire."""
        self._lock, self._account_id, self._log = threading.Lock(), account_id, log

    def acquire(self, blocking: bool = True, timeout: float = -1) -> bool:
        """Acquire the wrapped lock and log the account id on success."""
        acquired = self._lock.acquire(blocking, timeout)
        if acquired:
            self._log.append(self._account_id)
        return acquired

    def release(self) -> None:
        """Release the wrapped lock."""
        self._lock.release()


def lock_order_used() -> list[int]:
    """Account ids in the order transfer_ordered locked them, transferring 2 -> 1."""
    log: list[int] = []
    first = Account(1, 50, lock=RecordingLock(1, log))
    second = Account(2, 50, lock=RecordingLock(2, log))
    transfer_ordered(second, first, 5, pause=lambda: None)
    return log

The checks: the naive version deadlocks, the ordered version completes both
transfers with the right balances, and it locks the lower `account_id` first:

In [86]:
run_checks(
    [
        (
            "naive transfers deadlock",
            lambda: "DeadlockError" in run_opposite_transfers(transfer_naive)["outcomes"],
            True,
        ),
        (
            "ordered transfers both succeed",
            lambda: run_opposite_transfers(transfer_ordered)["outcomes"],
            ["ok", "ok"],
        ),
        (
            "ordered transfers move the money",
            lambda: run_opposite_transfers(transfer_ordered)["balances"],
            (110, 90),
        ),
        ("lower account_id is locked first", lock_order_used, [1, 2]),
    ]
)

FAIL  naive transfers deadlock: raised NotImplementedError: 
FAIL  ordered transfers both succeed: raised NotImplementedError: 
FAIL  ordered transfers move the money: raised NotImplementedError: 
FAIL  lower account_id is locked first: raised NotImplementedError: 
0/4 checks passed


#### Exercise 6 — GIL vs free-threaded CPU benchmark

`measure_scaling(python_path, thread_count=4, loops=1_000_000)` runs a benchmark
**in a child interpreter** (`subprocess.run([python_path, "-c", source], ...)`).
The child script you write times a pure-Python loop of `loops` iterations in one
thread, then `thread_count` threads each running the same loop at once, and
prints one JSON line. Return a dict with exactly these keys:

- `"gil_enabled"`: the child's `sys._is_gil_enabled()`;
- `"one_thread_seconds"`, `"all_threads_seconds"`: the two wall times;
- `"speedup"`: `thread_count * one_thread_seconds / all_threads_seconds`.

The checks run it on this kernel's interpreter and, if installed, on
`python3.15t` (`FREE_THREADED_PYTHON` from the unit setup). Speedups are only
printed, except one loose sanity check: the default build cannot reach 1.5×.

In [87]:
def measure_scaling(
    python_path: str, thread_count: int = 4, loops: int = 1_000_000
) -> dict[str, object]:
    """Benchmark 1 vs thread_count CPU-bound threads in a child interpreter."""
    raise NotImplementedError

Check exercise 6 (the free-threaded check prints a skip line when 3.15t is missing):

In [88]:
@functools.cache
def default_build_scaling() -> dict[str, object]:
    """Benchmark this kernel's interpreter once (exceptions are not cached)."""
    return measure_scaling(sys.executable)


def free_threaded_reports_no_gil() -> bool:
    """True if 3.15t reports the GIL disabled, or if no 3.15t is installed."""
    if FREE_THREADED_PYTHON is None:
        print("SKIP  free-threaded interpreter not installed")
        return True
    return measure_scaling(FREE_THREADED_PYTHON)["gil_enabled"] is False


def speedup_formula_holds() -> bool:
    """speedup == 4 * one / all for the default-build result."""
    result = default_build_scaling()
    expected = 4 * result["one_thread_seconds"] / result["all_threads_seconds"]
    return math.isclose(result["speedup"], expected, rel_tol=1e-6)

The checks: the four keys, the GIL flag of this kernel, the speedup formula, the
one loose sanity bound for the default build, and the GIL flag of 3.15t:

In [89]:
run_checks(
    [
        (
            "returns the four keys",
            lambda: sorted(default_build_scaling()),
            ["all_threads_seconds", "gil_enabled", "one_thread_seconds", "speedup"],
        ),
        (
            "default build reports the GIL enabled",
            lambda: default_build_scaling()["gil_enabled"],
            True,
        ),
        ("speedup = threads x one / all", speedup_formula_holds, True),
        (
            "default build: no real speedup (< 1.5x)",
            lambda: default_build_scaling()["speedup"] < 1.5,
            True,
        ),
        ("3.15t reports the GIL disabled", free_threaded_reports_no_gil, True),
    ]
)

FAIL  returns the four keys: raised NotImplementedError: 
FAIL  default build reports the GIL enabled: raised NotImplementedError: 
FAIL  speedup = threads x one / all: raised NotImplementedError: 
FAIL  default build: no real speedup (< 1.5x): raised NotImplementedError: 
FAIL  3.15t reports the GIL disabled: raised NotImplementedError: 
0/5 checks passed


Once the checks pass, this cell prints your measurements side by side. Explain
them in one sentence each: why the default build stays near 1×, and what limits
the free-threaded build (core count, the single-thread overhead, other load on
the machine).

In [90]:
try:
    results = {"default build": default_build_scaling()}
    if FREE_THREADED_PYTHON is not None:
        results["free-threaded"] = measure_scaling(FREE_THREADED_PYTHON)
    print(f"CPU cores available: {os.process_cpu_count()}")
    for build_name, result in results.items():
        print(
            f"{build_name:<14} GIL={result['gil_enabled']!s:<5} "
            f"1 thread {result['one_thread_seconds']:.3f} s, "
            f"4 threads {result['all_threads_seconds']:.3f} s, speedup {result['speedup']:.2f}x"
        )
except NotImplementedError:
    print("Finish exercise 6 first.")

Finish exercise 6 first.


### Recap — Unit 5.1

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| Concurrency vs parallelism | Interleaving needs one core; running at the same instant needs several (and no GIL in the way) | one event loop vs `worker_threads` |
| CPU- vs I/O-bound | Measure `thread_time` / wall time; threads help I/O and GIL-releasing C code | libuv pool helps `fs`/`crypto` |
| `threading.Thread` | `target=` a function, `start`, `join`; no return value; daemon threads die at exit | `new Worker()` (but isolated) |
| The GIL | Protects CPython's internals, not your invariants; released for I/O, `sleep` and cooperative C code; handed over every 5 ms | — |
| Race conditions | `+=` and check-then-act are several bytecodes; built-in single operations are atomic | `Atomics` |
| Primitives | `Lock`, `RLock`, `(Bounded)Semaphore`, `Event`, `Condition`, `Barrier`, `Timer`; always `with` | `p-limit`, `Atomics.wait` |
| Deadlocks | Break circular wait with lock ordering, or hold-and-wait with try-lock + back-off | — |
| Queues | `Queue`/`LifoQueue`/`PriorityQueue`; `task_done` + `join`; sentinels or `shutdown()` (3.13) | message channels |
| `threading.local` | Per-thread attributes; leaks between tasks in a pool | `AsyncLocalStorage` |
| `ThreadPoolExecutor` | `submit` → `Future`; `map` keeps order; `as_completed` finish order; check every future | `Promise.all`, `Promise.race` |
| Errors and shutdown | `threading.excepthook`; no `kill()`; stop with an `Event` and `wait(timeout)` | `worker.terminate()` exists in Node |
| Free-threaded build | `python3.15t`, `sys._is_gil_enabled()`; real thread parallelism, hidden races exposed | `worker_threads` parallelism |

**Further reading**

- [`threading`](https://docs.python.org/3/library/threading.html),
  [`queue`](https://docs.python.org/3/library/queue.html),
  [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html)
- [`sys.setswitchinterval`](https://docs.python.org/3/library/sys.html#sys.setswitchinterval),
  [FAQ: what kinds of global value mutation are thread-safe?](https://docs.python.org/3/faq/library.html#what-kinds-of-global-value-mutation-are-thread-safe)
- [Python support for free threading](https://docs.python.org/3/howto/free-threading-python.html),
  [C API extension support for free threading](https://docs.python.org/3/howto/free-threading-extensions.html)
- [PEP 703](https://peps.python.org/pep-0703/) (making the GIL optional),
  [PEP 779](https://peps.python.org/pep-0779/) (supported status for free-threaded Python)

## Unit 5.2 — Multiprocessing and Subinterpreters

**Objectives**

- Run CPU-bound Python in parallel with processes, and know what each one costs
  to start.
- Pick a start method (`fork`, `spawn`, `forkserver`) and write code that works
  with all three, including from a notebook.
- Use `Process`, `Pool` and `ProcessPoolExecutor`, and tune `chunksize`,
  `initializer` and `max_tasks_per_child`.
- Know what can be pickled, what pickling costs, and when parallel code is slower
  than serial code.
- Move data between processes with queues, pipes, shared values, managers and
  shared memory (with NumPy).
- Use subinterpreters (`concurrent.interpreters`, `InterpreterPoolExecutor`):
  parallelism inside one process.
- Choose a concurrency model with a decision matrix, Amdahl's law and a
  benchmark.

**JS bridge**

| Node | Python | The difference that bites |
| --- | --- | --- |
| `child_process.fork(file)` | `multiprocessing.Process(target=func)` | Python starts the child at a *function*, but that function must be importable by the child |
| `cluster` (fork workers that share a port) | `ProcessPoolExecutor`, `multiprocessing.Pool` | pools hand out tasks; they do not share sockets |
| `process.send` / `'message'` (JSON or structured clone over IPC) | `Queue`, `Pipe` (pickle over a pipe) | pickle can rebuild arbitrary classes, and cannot send lambdas, locks or open files |
| `worker_threads` (isolate per thread, own heap) | subinterpreters: own GIL, own modules, one process | the closest match to a Worker |
| `SharedArrayBuffer` + typed arrays | `multiprocessing.shared_memory` + NumPy arrays | you must `close()` and `unlink()` the block yourself |
| `os.availableParallelism()` | `os.process_cpu_count()` (*new in 3.13*) | both respect CPU affinity |

**How this unit runs.** A process started with `spawn` or `forkserver` is a
fresh interpreter: it can only run a function it can **import**. Functions
typed into a notebook cell live in the kernel's `__main__`, which a child cannot
import (5.2.2 shows the failure). So each section first saves its worker
functions to a small module under `notebooks/_scratch/phase5/` with
`%%writefile`, then imports it with `fresh_import()`. Demos that need a clean
parent process (start-method costs, the main guard, Ctrl+C handling) run as
scripts in a child interpreter with `run_child()`.

Unit setup, part 1: imports, `DEMO_SCALE`, the scratch folder (also put on
`sys.path` so worker modules saved there are importable here and in every child
process), feature flags for NumPy and subinterpreters, and the error and
interpreter helpers.

In [91]:
import concurrent.futures
import functools
import importlib
import importlib.util
import itertools
import math
import multiprocessing
import operator
import os
import pickle
import pickletools
import platform
import random
import re
import signal
import subprocess
import sys
import sysconfig
import textwrap
import threading
import time
from collections import Counter
from collections.abc import Callable, Iterable
from multiprocessing import shared_memory
from pathlib import Path
from types import ModuleType

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))  # the one knob for workload sizes
REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE5_DIR = scratch_dir / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT / "notebooks")  # %%writefile cells use paths relative to notebooks/
if str(PHASE5_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE5_DIR))  # child processes receive a copy of sys.path
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1"}
HAS_NUMPY = importlib.util.find_spec("numpy") is not None
HAS_INTERPRETERS = importlib.util.find_spec("concurrent.interpreters") is not None


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" (memory addresses hidden), or "no error"."""
    try:
        thunk()
    except Exception as error:
        return re.sub(r" at 0x[0-9a-f]+", "", f"{type(error).__name__}: {error}")
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"

Unit setup, part 2: `Timer` (Unit 1.14), `run_child()` (Unit 5.1, now also
hiding process ids) and `fresh_import(module_name)`, which re-imports a module
from disk after a `%%writefile` cell has changed it.

In [92]:
class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        """Remember the label and where to report the elapsed time."""
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        """Start the clock."""
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        """Stop the clock and report the elapsed milliseconds."""
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def run_child(
    source: str, python: str = sys.executable, extra_env: dict[str, str] | None = None
) -> str:
    """Run dedented source as a script in a child interpreter; return its stdout + stderr."""
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    completed = subprocess.run(
        [python, str(script)], capture_output=True, text=True, timeout=120,
        env={**CHILD_ENV, **(extra_env or {})}, cwd=PHASE5_DIR,
    )  # fmt: skip
    output = (completed.stdout + completed.stderr).rstrip()
    for path, label in ((str(PHASE5_DIR), "<scratch>"), (str(Path(python).parent), "<python>")):
        output = output.replace(path, label)
    output = re.sub(r"pid=\d+", "pid=<pid>", output)
    return re.sub(r"/[^\s\"']*/lib/python3\.\d+t?/", "<stdlib>/", output)


def fresh_import(module_name: str) -> ModuleType:
    """Import a module saved by a %%writefile cell, always re-reading the file from disk."""
    for cached in (PHASE5_DIR / "__pycache__").glob(f"{module_name}.*.pyc"):
        cached.unlink()  # a stale .pyc could survive a quick edit
    importlib.invalidate_caches()
    sys.modules.pop(module_name, None)
    return importlib.import_module(module_name)


print(interpreter_label())
print(f"{DEMO_SCALE=}  {os.process_cpu_count()=}  {multiprocessing.get_start_method()=}")
print(
    f"numpy: {'available' if HAS_NUMPY else 'missing'}, "
    f"subinterpreters: {'available' if HAS_INTERPRETERS else 'missing'}"
)

CPython 3.15.1, default (GIL) build, x86_64
DEMO_SCALE=1  os.process_cpu_count()=16  multiprocessing.get_start_method()='forkserver'
numpy: available, subinterpreters: available


### 5.2.1 Processes

A process has its **own address space** and runs its **own interpreter with its
own GIL**. Nothing is shared by accident: no races on ordinary objects, and
CPU-bound Python code really runs in parallel. The price: starting a process
takes milliseconds, and every argument and result must be serialized (pickled)
to cross the boundary.

The worker module for 5.2.1–5.2.4. `%%writefile` saves the cell's text to a
file instead of running it; the module-level `ORDERS` list will show that each
process has its own copy of module state.

In [93]:
%%writefile _scratch/phase5/p5_basics.py
"""Worker functions for Units 5.2.1-5.2.4. Child processes import this module by name."""

import os
import sys
import time

ORDERS: list[str] = []  # module-level state: every process has its own copy
PRICE_TABLE: dict[str, float] = {}


def busy_sum(count: int) -> int:
    """Pure-Python CPU work (the same loop as Unit 5.1)."""
    return sum(number % 1000 for number in range(count))


def add_order(order_id: str) -> tuple[int, int]:
    """Append to this process's ORDERS; return (pid, number of orders it holds)."""
    ORDERS.append(order_id)
    return os.getpid(), len(ORDERS)


def square(number: int) -> int:
    """A tiny task."""
    return number * number


def power(base: int, exponent: int) -> int:
    """Two arguments, for starmap."""
    return base**exponent


def slow_square(number: int) -> int:
    """Square after a delay that makes later numbers finish first."""
    time.sleep(0.05 * (3 - number % 3))
    return number * number

Overwriting _scratch/phase5/p5_basics.py


The rest of the module: helpers for exit codes (`nap_then_exit`), a pool
initializer and the task that uses what it loaded (`load_price_table`,
`price_with_gst`), and two `Pool` helpers (`pid_after_nap`, `silence_stderr`).
`%%writefile -a` **appends** to the same file instead of replacing it:

In [94]:
%%writefile -a _scratch/phase5/p5_basics.py


def nap_then_exit(seconds: float, exit_code: int) -> None:
    """Sleep, then end the process with exit_code."""
    time.sleep(seconds)
    sys.exit(exit_code)


def load_price_table(gst_percent: float) -> None:
    """Pool initializer: runs once in each worker process."""
    PRICE_TABLE.update({"TEA-500": 240.0, "MUG-02": 199.0, "gst": gst_percent})
    PRICE_TABLE["loaded_by"] = os.getpid()


def price_with_gst(sku: str) -> tuple[str, float, bool]:
    """Use the table the initializer loaded in this worker."""
    total = round(PRICE_TABLE[sku] * (1 + PRICE_TABLE["gst"] / 100), 2)
    return sku, total, PRICE_TABLE["loaded_by"] == os.getpid()


def pid_after_nap(_: object) -> int:
    """Return the worker's pid after a short nap."""
    time.sleep(0.01)
    return os.getpid()


def silence_stderr() -> None:
    """Initializer that sends this worker's stderr to /dev/null (keeps one demo short)."""
    os.dup2(os.open(os.devnull, os.O_WRONLY), 2)

Appending to _scratch/phase5/p5_basics.py


Import it. `fresh_import` re-reads the file, so editing and re-running the cell
above always takes effect.

In [95]:
p5_basics = fresh_import("p5_basics")
print(f"{p5_basics.__name__} loaded; {p5_basics.square(12)=}")

p5_basics loaded; p5_basics.square(12)=144


**Separate memory.** One worker process receives three orders; each call
appends to *its* `ORDERS`. The parent's `p5_basics.ORDERS` never changes, and
the worker's process id differs from the kernel's.

In [96]:
with concurrent.futures.ProcessPoolExecutor(max_workers=1) as one_worker:
    replies = list(one_worker.map(p5_basics.add_order, ["A-1", "A-2", "A-3"]))
worker_pids = {pid for pid, _ in replies}
print(f"worker saw order counts {[count for _, count in replies]}")
print(f"worker pid differs from the kernel's: {worker_pids != {os.getpid()}}")
print(f"the kernel's p5_basics.ORDERS: {p5_basics.ORDERS}")

worker saw order counts [1, 2, 3]
worker pid differs from the kernel's: True
the kernel's p5_basics.ORDERS: []


**Separate GILs.** The 5.1.3 benchmark again: the same total `busy_sum` work,
now split across 1 and 4 *processes*. Each process has its own GIL, so on a
machine with at least 4 cores this should approach 4× (minus start-up and
pickling).

In [97]:
total_numbers = 16_000_000 * DEMO_SCALE
with Timer("1 process (this kernel)") as serial_run:
    p5_basics.busy_sum(total_numbers)
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as pool:
    with Timer("4 worker processes, including pool start-up") as parallel_run:
        list(pool.map(p5_basics.busy_sum, [total_numbers // 4] * 4))
print(
    f"[{interpreter_label()}] speedup "
    f"{serial_run.elapsed_seconds / parallel_run.elapsed_seconds:.2f}x"
)

1 process (this kernel) took 322.78 ms


4 worker processes, including pool start-up took 116.63 ms
[CPython 3.15.1, default (GIL) build, x86_64] speedup 2.77x


**Creation cost.** How long does it take to start a process and wait for it to
finish, for each start method (explained in 5.2.2), compared with a thread? The
measurement runs as a script in a fresh child interpreter, because `fork` from
this multi-threaded kernel would be unsafe (5.2.2 shows why).

In [98]:
CREATION_COST = """
    import multiprocessing, threading, time

    def do_nothing():
        pass

    def average_ms(start_one, repeats=5):
        start = time.perf_counter()
        for _ in range(repeats):
            start_one()
        return (time.perf_counter() - start) / repeats * 1000

    def thread_once():
        worker = threading.Thread(target=do_nothing)
        worker.start()
        worker.join()

    if __name__ == "__main__":
        print(f"{'thread':<11} {average_ms(thread_once):8.3f} ms")
        for method in multiprocessing.get_all_start_methods():
            context = multiprocessing.get_context(method)

            def process_once():
                child = context.Process(target=do_nothing)
                child.start()
                child.join()

            process_once()  # warm-up (starts the fork server once)
            print(f"{method:<11} {average_ms(process_once):8.3f} ms")
"""
print(f"[{interpreter_label()}] start + join, average of 5 (varies by machine):")
print(run_child(CREATION_COST))

[CPython 3.15.1, default (GIL) build, x86_64] start + join, average of 5 (varies by machine):


thread         0.137 ms
forkserver     1.644 ms
fork           1.091 ms
spawn         23.395 ms


A thread costs tens of microseconds. `fork` copies the parent (cheap thanks to
copy-on-write pages) and costs about a millisecond; `forkserver` forks from a
small, already-running server process; `spawn` starts a whole new interpreter
and re-imports modules, the slowest. That is why you **reuse** processes in a
pool instead of starting one per task.

### 5.2.2 Start methods

| Method | How the child starts | Speed | Safe with threads in the parent? | Default on |
| --- | --- | --- | --- | --- |
| `fork` | `os.fork()`: a copy of the parent, mid-program, with all its memory | fastest | **no**: locks held by other threads are copied in their locked state | nothing since 3.14 (was Linux/POSIX until 3.13) |
| `spawn` | a fresh interpreter that imports the parent's `__main__` module, then unpickles the target | slowest | yes | Windows, macOS (since 3.8) |
| `forkserver` | a single-threaded server process, started once, forks a child for each request | fast after the first | yes | **POSIX except macOS, since 3.14** |

Query and choose with `get_start_method()`, `get_all_start_methods()` and
`get_context(method)`, which returns an object with the whole `multiprocessing`
API bound to that method. Prefer a context over `set_start_method()`, which may
be called only once per program.

In [99]:
print(f"{multiprocessing.get_start_method()=}")
print(f"{multiprocessing.get_all_start_methods()=}")
spawn_context = multiprocessing.get_context("spawn")
with concurrent.futures.ProcessPoolExecutor(max_workers=1, mp_context=spawn_context) as spawn_pool:
    print(f"spawn pool: {spawn_pool.submit(p5_basics.square, 9).result()=}")
print(f"{spawn_context.Process.__name__=}")

multiprocessing.get_start_method()='forkserver'
multiprocessing.get_all_start_methods()=['forkserver', 'fork', 'spawn']


spawn pool: spawn_pool.submit(p5_basics.square, 9).result()=81
spawn_context.Process.__name__='SpawnProcess'


**Why `fork` and threads do not mix.** `fork()` copies only the calling thread.
If another thread held a lock at that moment, the child gets the lock in its
locked state and nobody left to release it. Python warns
(`DeprecationWarning: This process ... is multi-threaded, use of fork() may lead
to deadlocks`). The script below makes it happen: a thread holds a lock, the
main thread forks, and the child tries the lock with a timeout.

In [100]:
FORK_WITH_THREADS = """
    import multiprocessing, threading, time, warnings

    warnings.simplefilter("always")  # show the DeprecationWarning
    inventory_lock = threading.Lock()

    def hold_lock():
        with inventory_lock:
            time.sleep(1)

    def child_wants_lock():
        got_it = inventory_lock.acquire(timeout=0.5)
        print(f"child: acquired the lock copied from the parent: {got_it}")

    if __name__ == "__main__":
        threading.Thread(target=hold_lock).start()
        time.sleep(0.05)  # the other thread now holds the lock
        child = multiprocessing.get_context("fork").Process(target=child_wants_lock)
        child.start()
        child.join()
"""
if "fork" in multiprocessing.get_all_start_methods():
    print(run_child(FORK_WITH_THREADS))
else:
    print("SKIP: fork is not available on this platform")

child: acquired the lock copied from the parent: False
<stdlib>/multiprocessing/popen_fork.py:76: DeprecationWarning: This process (pid=<pid>) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


The child waits half a second and gives up: the lock will never be released in
its copy of the world. Real programs deadlock this way inside `logging`,
`malloc` or a database driver. A notebook kernel always runs extra threads,
which is one reason `forkserver` became the default.

**The main guard.** With `spawn` and `forkserver`, the child imports the
parent's main module (as `__mp_main__`) to find the target function. Without an
`if __name__ == "__main__":` guard, that import runs the process-starting code
again inside the child, and `multiprocessing` stops it with a `RuntimeError`.
The output is trimmed to the last lines.

In [101]:
NO_GUARD = """
    import multiprocessing

    def greet():
        print("hello from the child")

    context = multiprocessing.get_context("spawn")
    child = context.Process(target=greet)
    child.start()
    child.join()
    print(f"parent: child exit code {child.exitcode}")
"""
WITH_GUARD = """
    import multiprocessing

    def greet():
        print("hello from the child")

    if __name__ == "__main__":
        context = multiprocessing.get_context("spawn")
        child = context.Process(target=greet)
        child.start()
        child.join()
        print(f"parent: child exit code {child.exitcode}")
"""
print("--- without the guard (key lines of the output)")
key_phrases = ("RuntimeError", "bootstrapping phase", "if __name__", "parent:")
for line in run_child(NO_GUARD).splitlines():
    if any(phrase in line for phrase in key_phrases):
        print(line.strip())
print("--- with the guard")
print(run_child(WITH_GUARD))

--- without the guard (key lines of the output)
parent: child exit code 1
raise RuntimeError('''
RuntimeError:
current process has finished its bootstrapping phase.
if __name__ == '__main__':
--- with the guard
hello from the child
parent: child exit code 0


#### Notebook caveat: worker functions must live in importable modules

Functions are pickled **by reference**: the pickle stores the module name and
the function name, never the code. A function typed into a notebook cell
belongs to the kernel's `__main__` module. A child interpreter's `__main__` is a
different module, so the lookup fails. First the mechanism, by hand: pickle a
notebook function and try to load it in a fresh interpreter.

In [102]:
def tax_for(amount: float) -> float:
    """Defined in a notebook cell, so it belongs to the kernel's __main__ module."""
    return round(amount * 0.18, 2)


payload = pickle.dumps(tax_for)
print(
    "the pickle names the function: "
    f"{b'__main__' in payload and b'tax_for' in payload}, {len(payload)} bytes"
)
loader = subprocess.run(
    [sys.executable, "-c", "import pickle, sys; pickle.loads(sys.stdin.buffer.read())"],
    input=payload, capture_output=True, env=CHILD_ENV,
)  # fmt: skip
print(f"loading it in a fresh interpreter: {loader.stderr.decode().strip().splitlines()[-1]}")

the pickle names the function: True, 35 bytes
loading it in a fresh interpreter: AttributeError: module '__main__' has no attribute 'tax_for'


A process pool hits exactly that error inside the worker, which then dies, and
the pool reports `BrokenProcessPool`. (The `silence_stderr` initializer hides
the worker's own traceback to keep the output short.) The same call with the
function from the importable `p5_basics` module works.

In [103]:
with concurrent.futures.ProcessPoolExecutor(
    max_workers=1, initializer=p5_basics.silence_stderr
) as pool:
    print(
        "notebook function: "
        f"{raised_message(lambda: pool.submit(tax_for, 100.0).result(timeout=60))}"
    )
with concurrent.futures.ProcessPoolExecutor(max_workers=1) as pool:
    print(f"module function:   {pool.submit(p5_basics.square, 100).result()=}")

notebook function: BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.


module function:   pool.submit(p5_basics.square, 100).result()=10000


| Option | Works because | Trade-off |
| --- | --- | --- |
| Save workers to a module (`%%writefile`, or a real `.py` file in your project) and import it | the child imports the module by name | the approach used in this phase |
| Run the code as a script (`python script.py`) | the child imports the script as `__mp_main__`; it needs the main guard | not interactive |
| `get_context("fork")` | the child is a copy of the kernel, function included | unsafe with the kernel's threads (above); not on Windows |
| Third-party pickling (`cloudpickle`, used by Dask, Ray, joblib) | pickles functions by value | an extra dependency |

### 5.2.3 Process APIs

#### `Process`

`multiprocessing.Process` mirrors `threading.Thread` (`target`, `args`, `name`,
`daemon`, `start`, `join`, `is_alive`) and adds process controls:

| Attribute / method | Meaning |
| --- | --- |
| `pid` | the OS process id (after `start()`) |
| `exitcode` | `None` while running, `0` on success, `n` after `sys.exit(n)`, `-signal` if killed by a signal |
| `terminate()` / `kill()` | send `SIGTERM` / `SIGKILL` (no `finally` blocks run in the child) |
| `interrupt()` | send `SIGINT`, so the child gets `KeyboardInterrupt` (*new in 3.14*) |
| `close()` | release the `Process` object's resources once it has finished |

Three children: one exits normally, one calls `sys.exit(3)`, one is terminated
while it sleeps.

In [104]:
children = {
    "finishes": multiprocessing.Process(
        target=p5_basics.nap_then_exit, args=(0.05, 0), name="finishes"
    ),
    "exits with 3": multiprocessing.Process(target=p5_basics.nap_then_exit, args=(0.05, 3)),
    "terminated": multiprocessing.Process(target=p5_basics.nap_then_exit, args=(10, 0)),
}
for child in children.values():
    child.start()
print(
    f"right after start: exitcode={children['finishes'].exitcode}, "
    f"has pid={children['finishes'].pid is not None}"
)
children["terminated"].terminate()
for label, child in children.items():
    child.join()
    print(f"{label:<13} exitcode={child.exitcode:>4}  alive={child.is_alive()}")
    child.close()
print(f"-signal.SIGTERM = {-signal.SIGTERM}")

right after start: exitcode=None, has pid=True
finishes      exitcode=   0  alive=False
exits with 3  exitcode=   3  alive=False
terminated    exitcode= -15  alive=False
-signal.SIGTERM = -15


#### `Pool`

`multiprocessing.Pool(processes)` is the original pool API:

| Method | Returns | Order |
| --- | --- | --- |
| `map(func, iterable, chunksize)` | a list | input order |
| `imap(func, iterable, chunksize)` | a lazy iterator | input order |
| `imap_unordered(func, iterable, chunksize)` | a lazy iterator | **completion** order |
| `starmap(func, iterable_of_tuples)` | a list of `func(*args)` | input order |
| `apply_async`, `map_async` | `AsyncResult` objects | — |

> **Warning:** `Pool.__exit__` calls `terminate()`, not `close()` + `join()`.
> Collect every result *inside* the `with` block.

In [105]:
with multiprocessing.Pool(processes=3) as pool:
    print(f"map:            {pool.map(p5_basics.slow_square, range(6))}")
    print(f"imap_unordered: {list(pool.imap_unordered(p5_basics.slow_square, range(6)))}")
    print(f"starmap:        {pool.starmap(p5_basics.power, [(2, 10), (3, 4), (10, 3)])}")

map:            [0, 1, 4, 9, 16, 25]


imap_unordered: [4, 1, 0, 16, 25, 9]
starmap:        [1024, 81, 1000]


`slow_square` sleeps longer for numbers divisible by 3, so `imap_unordered`
yields them last (the exact order can vary slightly between runs).

**`chunksize`.** Each task is pickled and sent through a pipe; `chunksize=n`
sends `n` items per message, so the per-message overhead is paid once per chunk.
For many tiny tasks that is the difference between slow and fast. (Lab 5.2
draws the full curve.)

In [106]:
tiny_items = range(20_000 * DEMO_SCALE)
with multiprocessing.Pool(processes=4) as pool:
    pool.map(p5_basics.square, range(8))  # start the workers before timing
    for chunk_size in (1, 2_000):
        with Timer(f"Pool.map of {len(tiny_items):,} tiny tasks, chunksize={chunk_size:>5}"):
            pool.map(p5_basics.square, tiny_items, chunksize=chunk_size)

Pool.map of 20,000 tiny tasks, chunksize=    1 took 2802.74 ms
Pool.map of 20,000 tiny tasks, chunksize= 2000 took 25.65 ms


#### `ProcessPoolExecutor`

The `concurrent.futures` version: the same `submit` / `map` / `Future` API as
`ThreadPoolExecutor` (5.1.8), plus process-specific options:

| Option | Effect |
| --- | --- |
| `max_workers` | defaults to `os.process_cpu_count()` |
| `mp_context` | a start-method context (`get_context("spawn")`) |
| `initializer`, `initargs` | a function run once in each new worker (load a model, open a connection, set up logging) |
| `max_tasks_per_child` | replace a worker after it has run `n` tasks (contains memory leaks in long-running pools; *new in 3.11*) |
| `map(..., chunksize=n)` | the same batching as `Pool.map` |
| `terminate_workers()`, `kill_workers()` | stop all workers now (*new in 3.14*) |

**`initializer`.** Each worker loads the price table once; every task then uses
the worker's own copy. The third field shows the table was loaded in the same
process that ran the task.

In [107]:
with concurrent.futures.ProcessPoolExecutor(
    max_workers=2, initializer=p5_basics.load_price_table, initargs=(18.0,)
) as pool:
    for sku, price_with_tax, loaded_here in pool.map(
        p5_basics.price_with_gst, ["TEA-500", "MUG-02", "TEA-500"]
    ):
        print(f"{sku:<8} ₹{price_with_tax:>7.2f}  table loaded in this worker: {loaded_here}")
print(f"the kernel's PRICE_TABLE stays empty: {p5_basics.PRICE_TABLE}")

TEA-500  ₹ 283.20  table loaded in this worker: True
MUG-02   ₹ 234.82  table loaded in this worker: True
TEA-500  ₹ 283.20  table loaded in this worker: True
the kernel's PRICE_TABLE stays empty: {}


**`max_tasks_per_child`.** With one worker and a limit of 2 tasks, six tasks run
in three different processes.

In [108]:
with concurrent.futures.ProcessPoolExecutor(max_workers=1, max_tasks_per_child=2) as pool:
    task_pids = list(pool.map(p5_basics.pid_after_nap, range(6)))
print(f"distinct worker processes for 6 tasks: {len(set(task_pids))}")
print(
    "pattern (same letter = same process): "
    f"{''.join(chr(65 + sorted(set(task_pids)).index(pid)) for pid in task_pids)}"
)

distinct worker processes for 6 tasks: 3
pattern (same letter = same process): AABBCC


### 5.2.4 Data transfer

Every argument, return value and exception that crosses a process boundary is
**pickled** in the sender and unpickled in the receiver (Unit 4.5 covered
`pickle`). Two consequences: some objects cannot cross at all, and big objects
are expensive to send.

**Pickling constraints.**

| Can be pickled | Cannot be pickled |
| --- | --- |
| `None`, numbers, strings, bytes, tuples, lists, dicts, sets of picklable items | lambdas and functions defined inside functions |
| module-level functions and classes (by reference: module + name) | open files, sockets, database connections |
| instances of module-level classes (dataclasses included) | locks, threads, generators, coroutines |
| `functools.partial` of a module-level function | functions in the notebook's `__main__` (they pickle, but a child cannot load them: 5.2.2) |

In [109]:
def make_discount(percent: int) -> Callable[[float], float]:
    """Return a nested function (not picklable)."""

    def apply(amount: float) -> float:
        """Apply the captured discount percentage to amount."""
        return amount * (100 - percent) / 100

    return apply


candidates = {
    "tuple of numbers and text": ("TEA-500", 2, 240.0),
    "module-level function": p5_basics.square,
    "functools.partial(p5_basics.power, 2)": functools.partial(p5_basics.power, 2),
    "lambda": lambda amount: amount * 2,
    "nested function": make_discount(10),
    "threading.Lock()": threading.Lock(),
    "generator": (number for number in range(3)),
    "open file": open(os.devnull),
}


def pickle_outcome(candidate: object) -> str:
    """ "OK" if candidate pickles, else the error."""
    return raised_message(lambda: pickle.dumps(candidate)).replace("no error", "OK")


for label, candidate in candidates.items():
    print(f"{label:<40} {pickle_outcome(candidate)}")
candidates["open file"].close()

tuple of numbers and text                OK
module-level function                    OK
functools.partial(p5_basics.power, 2)    OK
lambda                                   PicklingError: Can't pickle <function <lambda>>: it's not found as __main__.<lambda>
nested function                          PicklingError: Can't pickle local object <function make_discount.<locals>.apply>
threading.Lock()                         TypeError: cannot pickle '_thread.lock' object
generator                                TypeError: cannot pickle 'generator' object
open file                                TypeError: cannot pickle 'TextIOWrapper' instances


#### Under the hood: what a pickled function contains

`pickletools.dis` disassembles a pickle. A function is just two strings, its
module and its name, plus `STACK_GLOBAL` ("import this and look it up"). The
code itself is never sent: the receiver must be able to import it.

In [110]:
os.environ["PYTHON_COLORS"] = "0"  # the kernel sets FORCE_COLOR=1; this wins over it
try:
    pickletools.dis(pickle.dumps(p5_basics.square))
finally:
    del os.environ["PYTHON_COLORS"]

    0: \x80 PROTO      5
    2: \x95 FRAME      24
   11: \x8c SHORT_BINUNICODE 'p5_basics'
   22: \x94 MEMOIZE    (as 0)
   23: \x8c SHORT_BINUNICODE 'square'
   31: \x94 MEMOIZE    (as 1)
   32: \x93 STACK_GLOBAL
   33: \x94 MEMOIZE    (as 2)
   34: .    STOP
highest protocol among opcodes = 4


**Pickling cost.** A list of a million integers: `sys.getsizeof` counts only
the list's pointer array, the pickle holds every value, and sending it to a
worker means pickle → pipe → unpickle (and the same back for the result).

In [111]:
order_ids = list(range(1_000_000 * DEMO_SCALE))
with Timer("pickle.dumps(1M ints)"):
    order_payload = pickle.dumps(order_ids)
with Timer("pickle.loads(1M ints)"):
    pickle.loads(order_payload)
print(
    f"sys.getsizeof(list): {sys.getsizeof(order_ids) / 1e6:.1f} "
    f"MB, pickle: {len(order_payload) / 1e6:.1f} MB"
)

with concurrent.futures.ProcessPoolExecutor(max_workers=1) as pool:
    pool.submit(len, []).result()  # start the worker before timing
    with Timer("len() of the list in a worker (send 1M ints)"):
        pool.submit(len, order_ids).result()
with Timer("len() of the list in the kernel"):
    len(order_ids)

pickle.dumps(1M ints) took 7.72 ms


pickle.loads(1M ints) took 24.40 ms
sys.getsizeof(list): 8.0 MB, pickle: 4.9 MB


len() of the list in a worker (send 1M ints) took 31.90 ms
len() of the list in the kernel took 0.00 ms


**When parallel is slower.** Squaring a number takes about 30 ns; sending it to
a worker and getting the answer back takes tens of microseconds. With tiny
tasks the pool spends its time on overhead, and even large chunks only pay off
when the work per item outweighs pickling it.

In [112]:
numbers = list(range(200_000 * DEMO_SCALE))
with Timer("sequential, in the kernel"):
    [p5_basics.square(number) for number in numbers]
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as pool:
    list(pool.map(p5_basics.square, range(8)))  # start the workers
    with Timer("4 processes, chunksize=1 (only the first 10% of the items!)"):
        list(pool.map(p5_basics.square, numbers[: len(numbers) // 10]))
    with Timer("4 processes, chunksize=10_000"):
        list(pool.map(p5_basics.square, numbers, chunksize=10_000))

sequential, in the kernel took 9.69 ms


4 processes, chunksize=1 (only the first 10% of the items!) took 769.00 ms
4 processes, chunksize=10_000 took 32.10 ms


Both parallel versions lose. Rules of thumb:

- Make each task **coarse**: seconds of work, not microseconds (batch items, or
  use `chunksize`).
- **Send little, compute a lot**: pass a file name or a row range, not the data;
  return a summary, not every intermediate value.
- For big arrays, share memory instead of pickling (5.2.5).
- Measure. A parallel version that is not faster is pure complexity.

### 5.2.5 IPC

| Tool | What it is | Use it for |
| --- | --- | --- |
| `multiprocessing.Queue` | many-to-many FIFO; items are pickled and sent by a background feeder thread | work and result streams |
| `Pipe(duplex=True)` | two `Connection` ends; `send` / `recv` pickled objects | a conversation between exactly two processes |
| `Value(typecode, initial)`, `Array(typecode, size)` | a C value / array in shared memory, with an optional lock | counters, small numeric buffers |
| `Manager()` | a server process holding real Python objects; others get **proxies** | shared dicts and lists (slow: every access is a round trip) |
| `shared_memory.SharedMemory` | a named block of raw shared memory | large arrays without copying (with NumPy) |

The worker module for this section:

In [113]:
%%writefile _scratch/phase5/p5_ipc.py
"""Worker functions for Unit 5.2.5 (inter-process communication)."""

import os
from multiprocessing import shared_memory
from multiprocessing.connection import Connection
from multiprocessing.managers import DictProxy, ListProxy
from multiprocessing.queues import Queue
from multiprocessing.sharedctypes import Synchronized, SynchronizedArray


def produce_orders(order_queue: Queue[dict[str, int] | None], count: int) -> None:
    """Put count orders on a multiprocessing.Queue, then a None sentinel."""
    for number in range(1, count + 1):
        order_queue.put({"order_id": number, "producer_pid": os.getpid()})
    order_queue.put(None)


def price_service(connection: Connection) -> None:
    """Answer price requests over one end of a Pipe until told to stop."""
    prices = {"TEA-500": 240.0, "MUG-02": 199.0}
    while (request := connection.recv()) != "stop":
        connection.send({"sku": request, "price": prices.get(request)})
    connection.close()


def add_visits(counter: Synchronized[int], times: int, use_lock: bool) -> None:
    """Increment a shared multiprocessing.Value, with or without its lock."""
    for _ in range(times):
        if use_lock:
            with counter.get_lock():
                counter.value += 1
        else:
            counter.value += 1  # read and write are separate steps: updates get lost

Overwriting _scratch/phase5/p5_ipc.py


The second half appends the shared-memory workers: `fill_squares` (a slice of a
shared `Array`), `record_stock` (writes through `Manager` proxies) and
`square_in_place` (a NumPy view on a named `SharedMemory` block). The
annotations name the real proxy and wrapper types the parent passes in:

In [114]:
%%writefile -a _scratch/phase5/p5_ipc.py


def fill_squares(shared_array: SynchronizedArray[int], start: int, stop: int) -> None:
    """Write index squared into a slice of a shared multiprocessing.Array."""
    for index in range(start, stop):
        shared_array[index] = index * index


def record_stock(stock: DictProxy[str, int], log: ListProxy[str], sku: str, quantity: int) -> None:
    """Write through Manager proxies (a dict and a list)."""
    stock[sku] = quantity
    log.append(f"set {sku}")


def square_in_place(block_name: str, length: int) -> None:
    """Attach to a shared float64 block by name and square every value in place."""
    import numpy as np

    block = shared_memory.SharedMemory(name=block_name, track=False)
    try:
        values = np.ndarray((length,), dtype=np.float64, buffer=block.buf)
        values **= 2
        del values  # drop the view before closing the block
    finally:
        block.close()

Appending to _scratch/phase5/p5_ipc.py


Import it:

In [115]:
p5_ipc = fresh_import("p5_ipc")
print(f"{p5_ipc.__name__} loaded")

p5_ipc loaded


**`Queue`.** A producer process puts five orders and a `None` sentinel; the
parent reads until the sentinel (the same pattern as 5.1.6, across processes).

In [116]:
order_queue = multiprocessing.Queue()
producer = multiprocessing.Process(target=p5_ipc.produce_orders, args=(order_queue, 5))
producer.start()
received = []
while (order := order_queue.get(timeout=30)) is not None:
    received.append(order)
producer.join()
print(f"received order ids {[order['order_id'] for order in received]}")
print(f"all from the producer process: {len({order['producer_pid'] for order in received}) == 1}")

received order ids [1, 2, 3, 4, 5]
all from the producer process: True


**`Pipe`.** A two-way channel: the parent asks a price service in another
process and gets answers back. `poll(timeout)` checks whether a message is
waiting.

In [117]:
parent_end, child_end = multiprocessing.Pipe()
service = multiprocessing.Process(target=p5_ipc.price_service, args=(child_end,))
service.start()
for sku in ("TEA-500", "SPOON-1"):
    parent_end.send(sku)
    print(f"asked {sku}: {parent_end.recv()}")
print(f"anything else waiting? {parent_end.poll(timeout=0.05)}")
parent_end.send("stop")
service.join()
print(f"service exit code: {service.exitcode}")

asked TEA-500: {'sku': 'TEA-500', 'price': 240.0}
asked SPOON-1: {'sku': 'SPOON-1', 'price': None}
anything else waiting? False
service exit code: 0


**`Value` and `Array`.** Shared C storage. `Value` comes with a lock, but
`counter.value += 1` is still a read followed by a write: the 5.1.4 race, now
between processes. Wrap it in `with counter.get_lock():`. `Array` slices can be
filled by different processes without a lock when they do not overlap.

In [118]:
for use_lock in (False, True):
    visits = multiprocessing.Value("i", 0)
    workers = [
        multiprocessing.Process(target=p5_ipc.add_visits, args=(visits, 5_000, use_lock))
        for _ in range(4)
    ]
    for worker in workers:
        worker.start()
    for worker in workers:
        worker.join()
    print(f"use_lock={use_lock!s:<5}: expected 20,000 visits, got {visits.value:,}")

squares = multiprocessing.Array("q", 8)  # 8 signed 64-bit integers, zeroed
halves = [
    multiprocessing.Process(target=p5_ipc.fill_squares, args=(squares, start, start + 4))
    for start in (0, 4)
]
for half in halves:
    half.start()
for half in halves:
    half.join()
print(f"Array filled by two processes: {list(squares)}")

use_lock=False: expected 20,000 visits, got 19,638


use_lock=True : expected 20,000 visits, got 20,000


Array filled by two processes: [0, 1, 4, 9, 16, 25, 36, 49]


**`Manager`.** A manager runs a server process that owns ordinary Python
objects; other processes get proxies that forward every operation to it. That
makes it flexible (any picklable values, nested structures) and slow. The timing
compares 1 000 writes to a managed dict with 1 000 writes to a plain dict.

In [119]:
with multiprocessing.Manager() as manager:
    stock, change_log = manager.dict(), manager.list()
    writers = [
        multiprocessing.Process(target=p5_ipc.record_stock, args=(stock, change_log, sku, quantity))
        for sku, quantity in (("TEA-500", 40), ("MUG-02", 12))
    ]
    for writer in writers:
        writer.start()
    for writer in writers:
        writer.join()
    print(f"stock={dict(sorted(stock.items()))}, log={sorted(change_log)}")
    with Timer("1,000 writes to a Manager dict (a round trip each)"):
        for number in range(1_000):
            stock[f"sku-{number}"] = number
plain_dict: dict[str, int] = {}
with Timer("1,000 writes to a plain dict"):
    for number in range(1_000):
        plain_dict[f"sku-{number}"] = number

stock={'MUG-02': 12, 'TEA-500': 40}, log=['set MUG-02', 'set TEA-500']
1,000 writes to a Manager dict (a round trip each) took 13.38 ms
1,000 writes to a plain dict took 0.12 ms


#### `shared_memory` with NumPy

`SharedMemory(create=True, size=n)` allocates a named block that any process
can attach to by name. Wrapping its buffer in a NumPy array gives both processes
a view of the **same bytes**: nothing is pickled except the block's name. The
owner must `close()` its view and `unlink()` the block when done (otherwise the
memory stays allocated until reboot on some systems). The worker attaches with
`track=False` (*new in 3.13*) so only the owner's resource tracker manages the
block.

In [120]:
if not HAS_NUMPY:
    print("SKIP: numpy is not installed")
else:
    import numpy as np

    readings = np.arange(1.0, 9.0)  # 8 float64 values
    block = shared_memory.SharedMemory(create=True, size=readings.nbytes)
    try:
        shared_readings = np.ndarray(readings.shape, dtype=np.float64, buffer=block.buf)
        shared_readings[:] = readings
        squarer = multiprocessing.Process(
            target=p5_ipc.square_in_place, args=(block.name, len(readings))
        )
        squarer.start()
        squarer.join()
        print(f"after the child squared them: {shared_readings.tolist()}")
        print(
            f"sent to the child: the name {block.name!r:.12}... "
            f"({len(pickle.dumps(block.name))} bytes pickled)"
        )
        print(f"pickling the array instead would send {len(pickle.dumps(readings))} bytes")
        del shared_readings  # release the view before closing
    finally:
        block.close()
        block.unlink()

after the child squared them: [1.0, 4.0, 9.0, 16.0, 25.0, 36.0, 49.0, 64.0]
sent to the child: the name 'psm_ae83976... (27 bytes pickled)
pickling the array instead would send 191 bytes


Eight floats is a toy; the same code with 100 million floats still sends only
the name. Lab 5.2 splits one shared array across several workers.

### 5.2.6 Subinterpreters

A **subinterpreter** is a second, isolated Python interpreter inside the *same
process*: its own modules, its own `sys`, its own `__main__`, and (since
[PEP 684](https://peps.python.org/pep-0684/), 3.12) **its own GIL**. So several
interpreters can run Python bytecode in parallel on different threads, with
process-like isolation and thread-like start-up cost.

| | |
| --- | --- |
| [PEP 734](https://peps.python.org/pep-0734/) | `concurrent.interpreters` in the standard library (*new in 3.14*) |
| `interpreters.create()` | a new interpreter; `close()` it when done |
| `interp.exec(source)` | run code in its `__main__`; errors come back as `ExecutionFailed` |
| `interp.call(func, *args)` | call a function there and return its result |
| `interp.call_in_thread(func, *args)` | the same in a new thread (returns the `Thread`) |
| `interp.prepare_main(name=value)` | bind shareable objects in its `__main__` |
| `interpreters.create_queue()` | a queue that works across interpreters |
| `interpreters.is_shareable(obj)` | `None`, `bool`, numbers, `str`, `bytes`, tuples of those, queues |
| `InterpreterPoolExecutor` | `concurrent.futures` pool of interpreters (*new in 3.14*) |

> **Note:** output printed *inside* a subinterpreter goes to the process's real
> stdout, which the notebook does not show. The demos send results back through
> return values and queues instead.

**Basics.** Create an interpreter, pass it a queue, let it report its id, call
a function in it, and show that changing `sys` there does not affect the main
interpreter.

In [121]:
if not HAS_INTERPRETERS:
    print("SKIP: concurrent.interpreters is not available (new in 3.14)")
else:
    from concurrent import interpreters

    worker_interpreter = interpreters.create()
    try:
        reports = interpreters.create_queue()
        worker_interpreter.prepare_main(reports=reports, greeting="namaste")
        worker_interpreter.exec(
            "from concurrent import interpreters as i; import sys; "
            "sys.course_marker = True; reports.put((greeting, i.get_current().id))"
        )
        print(f"from the subinterpreter: {reports.get(timeout=10)}")
        print(f"main interpreter id: {interpreters.get_main().id}")
        print(f"call(len, 'namaste') = {worker_interpreter.call(len, 'namaste')}")
        print(f"main interpreter sees sys.course_marker: {hasattr(sys, 'course_marker')}")
        print(f"interpreters alive: {len(interpreters.list_all())}")
        samples = (42, "text", (1, 2), [1, 2], {"a": 1})
        shareable = {repr(value): interpreters.is_shareable(value) for value in samples}
        print(f"is_shareable: {shareable}")
    finally:
        worker_interpreter.close()

from the subinterpreter: ('namaste', 1)
main interpreter id: 0
call(len, 'namaste') = 7
main interpreter sees sys.course_marker: False
interpreters alive: 2
is_shareable: {'42': True, "'text'": True, '(1, 2)': True, '[1, 2]': False, "{'a': 1}": False}


**A subinterpreter has its own `sys.path`.** It starts with the default search
path (the working directory and the standard library), not the kernel's
modified `sys.path`, so our scratch folder is missing. Callables and arguments
are passed as shareable objects or pickled, and a function pickled by reference
must be importable *inside the subinterpreter*. The fix is a tiny function that
uses only built-ins, run once in each new interpreter: here with `call`, in a
pool as the `initializer`.

In [122]:
def add_to_sys_path(folder: str) -> None:
    """Initializer for a fresh subinterpreter: make `folder` importable there."""
    import sys

    if folder not in sys.path:
        sys.path.insert(0, folder)


if not HAS_INTERPRETERS:
    print("SKIP: concurrent.interpreters is not available (new in 3.14)")
else:
    path_test = interpreters.create()
    try:
        try:
            path_test.call(p5_basics.pid_after_nap, None)
        except interpreters.NotShareableError as error:
            cause = str(error.__cause__).splitlines()[0]
            print(f"before: {type(error).__name__}: {error} ({cause})")
        path_test.call(add_to_sys_path, str(PHASE5_DIR))
        same_process = path_test.call(p5_basics.pid_after_nap, None) == os.getpid()
        print(f"after add_to_sys_path: it runs, in the kernel's own process: {same_process}")
    finally:
        path_test.close()

before: NotShareableError: func not shareable (ModuleNotFoundError: No module named 'p5_basics')
after add_to_sys_path: it runs, in the kernel's own process: True


**One GIL per interpreter.** The same four `busy_sum` jobs on a pool of four
threads (one shared GIL) and on a pool of four interpreters (four GILs, each
interpreter initialized with `add_to_sys_path`). Unlike processes, the
interpreter pool starts in milliseconds and stays in one process.

In [123]:
if not HAS_INTERPRETERS:
    print("SKIP: concurrent.interpreters is not available (new in 3.14)")
else:
    jobs = [4_000_000 * DEMO_SCALE] * 4
    print(f"[{interpreter_label()}] 4 x busy_sum({jobs[0]:,})")
    pools = {
        "ThreadPoolExecutor": lambda: concurrent.futures.ThreadPoolExecutor(max_workers=4),
        "InterpreterPoolExecutor": lambda: concurrent.futures.InterpreterPoolExecutor(
            max_workers=4, initializer=add_to_sys_path, initargs=(str(PHASE5_DIR),)
        ),
    }
    for pool_name, make_pool in pools.items():
        with make_pool() as pool:
            list(pool.map(p5_basics.busy_sum, [10] * 4))  # start the workers
            with Timer(f"{pool_name:<24}"):
                list(pool.map(p5_basics.busy_sum, jobs))
    with Timer("create + close one interpreter", report=None) as interpreter_cost:
        interpreters.create().close()
    print(f"create + close one interpreter: {interpreter_cost.elapsed_seconds * 1000:.1f} ms")

[CPython 3.15.1, default (GIL) build, x86_64] 4 x busy_sum(4,000,000)


ThreadPoolExecutor       took 354.01 ms
InterpreterPoolExecutor  took 107.80 ms
create + close one interpreter: 12.6 ms


**Compatibility.** Every C extension imported in a subinterpreter must support
multi-phase initialization and a per-interpreter GIL. Most of the standard
library does; many third-party extensions do not yet. NumPy, for example,
refuses to load:

In [124]:
if not HAS_INTERPRETERS or not HAS_NUMPY:
    print("SKIP: needs both concurrent.interpreters and numpy")
else:
    numpy_test = interpreters.create()
    try:
        numpy_test.exec("import numpy")
        print("numpy imported in a subinterpreter")
    except interpreters.ExecutionFailed as failure:
        first_sentence = next(line for line in failure.excinfo.msg.splitlines() if line.strip())
        print(f"ExecutionFailed wrapping {failure.excinfo.type.__name__}: {first_sentence}")
    finally:
        numpy_test.close()

ExecutionFailed wrapping ImportError: IMPORTANT: PLEASE READ THIS FOR ADVICE ON HOW TO SOLVE THIS ISSUE!


| | Threads | Subinterpreters | Processes |
| --- | --- | --- | --- |
| Parallel Python bytecode | no (default build) | yes | yes |
| Isolation | none | modules, globals, `sys` | everything (address space) |
| Start-up | ~10 µs | ~ms | ~ms to tens of ms |
| Passing data | shared objects | shareable values, queues, pickling | pickling, shared memory |
| C extensions | all | only those that support it | all |
| A crash (segfault) takes down | the process | the process | only that worker |

### 5.2.7 Choosing a model

**Decision matrix.**

| Workload | Threads | Processes | Subinterpreters | asyncio (5.3–5.4) | Free-threaded threads |
| --- | --- | --- | --- | --- | --- |
| Many network calls (HTTP, DB) | good up to hundreds | wasteful | wasteful | **best** for thousands | good |
| Blocking library with no async API | **best** | works | depends on C extensions | via `to_thread` | good |
| CPU-bound pure Python | useless (GIL) | **best default** | good if extensions allow | useless | **good** |
| CPU-bound in NumPy / hashlib / zlib | **good** (they release the GIL) | works, pays pickling | NumPy not supported yet | via `to_thread` | good |
| Large shared in-memory data, CPU-bound | GIL-bound | shared memory only | shareable values only | — | **best** |
| Isolation from crashes and leaks | none | **best** (`max_tasks_per_child`) | partial | none | none |

The worker module for the last two sections:

In [125]:
%%writefile _scratch/phase5/p5_patterns.py
"""Worker functions for Units 5.2.7-5.2.8."""

import time
from collections import Counter

from p5_basics import busy_sum  # noqa: F401  (re-exported for the benchmark)


def wait_for_io(seconds: float) -> float:
    """Stand-in for a network call: the worker just waits."""
    time.sleep(seconds)
    return seconds


def revenue_by_city(records: list[tuple[str, int]]) -> Counter[str]:
    """Map step: total revenue per city in one chunk of sales records."""
    totals: Counter[str] = Counter()
    for city, amount in records:
        totals[city] += amount
    return totals

Overwriting _scratch/phase5/p5_patterns.py


Import it:

In [126]:
p5_patterns = fresh_import("p5_patterns")
print(f"{p5_patterns.__name__} loaded")

p5_patterns loaded


**Justify it with a benchmark.** The same two jobs on every model: CPU-bound
(8 × `busy_sum`) and I/O-bound (8 × a 0.1 s wait), with 4 workers each. One
table, measured on this machine.

In [127]:
cpu_jobs, io_jobs = [3_000_000 * DEMO_SCALE] * 8, [0.1] * 8
executors: dict[str, Callable[[], concurrent.futures.Executor]] = {
    "threads": lambda: concurrent.futures.ThreadPoolExecutor(4),
    "processes": lambda: concurrent.futures.ProcessPoolExecutor(4),
}
if HAS_INTERPRETERS:
    executors["subinterpreters"] = lambda: concurrent.futures.InterpreterPoolExecutor(
        4, initializer=add_to_sys_path, initargs=(str(PHASE5_DIR),)
    )

measured: dict[str, dict[str, float]] = {"sequential": {}}
for job_name, worker, jobs in (
    ("cpu", p5_patterns.busy_sum, cpu_jobs),
    ("io", p5_patterns.wait_for_io, io_jobs),
):
    with Timer("", report=None) as serial_timer:
        [worker(job) for job in jobs]
    measured["sequential"][job_name] = serial_timer.elapsed_seconds
    for model, make_executor in executors.items():
        with make_executor() as pool:
            list(pool.map(worker, jobs[:4]))  # warm up the workers
            with Timer("", report=None) as model_timer:
                list(pool.map(worker, jobs))
        measured.setdefault(model, {})[job_name] = model_timer.elapsed_seconds
print(f"[{interpreter_label()}], 4 workers per pool")
print(f"{'model':<16}{'CPU-bound':>12}{'I/O-bound':>12}")
for model, times in measured.items():
    print(f"{model:<16}{times['cpu']:>11.3f}s{times['io']:>11.3f}s")

[CPython 3.15.1, default (GIL) build, x86_64], 4 workers per pool
model              CPU-bound   I/O-bound
sequential            0.518s      0.802s
threads               0.517s      0.201s
processes             0.206s      0.205s
subinterpreters       0.210s      0.203s


Threads win nothing on the CPU column and everything on the I/O column;
processes and subinterpreters win on CPU. For I/O, all pools get about 4×
(4 workers waiting at once), which is why the cheapest option, threads (or
asyncio), is the right one there.

#### Amdahl's law

If a fraction `p` of a program's run time can be parallelized and the rest
`1 - p` cannot, the best possible speedup on `N` workers is

```text
S(N) = 1 / ((1 - p) + p / N)        and as N grows without limit, S → 1 / (1 - p)
```

A program that is 90% parallel can never run more than 10× faster, however many
cores you add. The table and chart show how quickly the serial part dominates.

In [128]:
def amdahl(parallel_fraction: float, workers: float) -> float:
    """Amdahl's law: the speedup limit for a parallel fraction on `workers` workers."""
    return 1 / ((1 - parallel_fraction) + parallel_fraction / workers)


worker_counts = [1, 2, 4, 8, 16, 64, math.inf]
print(
    "p \\ N  "
    + "".join(
        f"{('inf' if worker_count == math.inf else worker_count):>7}"
        for worker_count in worker_counts
    )
)
for fraction in (0.5, 0.75, 0.9, 0.95, 0.99):
    print(
        f"{fraction:<7}"
        + "".join(f"{amdahl(fraction, worker_count):>7.1f}" for worker_count in worker_counts)
    )
print()
for worker_count in (1, 2, 4, 8, 16, 32):
    speedup = amdahl(0.9, worker_count)
    print(f"p=0.9, N={worker_count:>2} {'#' * round(speedup * 4):<40} {speedup:.1f}x")

p \ N        1      2      4      8     16     64    inf
0.5        1.0    1.3    1.6    1.8    1.9    2.0    2.0
0.75       1.0    1.6    2.3    2.9    3.4    3.8    4.0
0.9        1.0    1.8    3.1    4.7    6.4    8.8   10.0
0.95       1.0    1.9    3.5    5.9    9.1   15.4   20.0
0.99       1.0    2.0    3.9    7.5   13.9   39.3  100.0

p=0.9, N= 1 ####                                     1.0x
p=0.9, N= 2 #######                                  1.8x
p=0.9, N= 4 ############                             3.1x
p=0.9, N= 8 ###################                      4.7x
p=0.9, N=16 ##########################               6.4x
p=0.9, N=32 ###############################          7.8x


Run it backwards to estimate the parallel fraction of a real run: from a
measured speedup `S` on `N` workers, `p = (1 - 1/S) / (1 - 1/N)`. Process
start-up and pickling count as serial time, so measured `p` is lower than the
"obvious" one.

In [129]:
for model in ("processes", "subinterpreters"):
    if model in measured:
        speedup = measured["sequential"]["cpu"] / measured[model]["cpu"]
        implied_fraction = (1 - 1 / speedup) / (1 - 1 / 4)
        print(
            f"{model:<16} CPU speedup on 4 workers {speedup:.2f}x -> "
            f"implied parallel fraction p = {implied_fraction:.2f}"
        )

processes        CPU speedup on 4 workers 2.51x -> implied parallel fraction p = 0.80
subinterpreters  CPU speedup on 4 workers 2.46x -> implied parallel fraction p = 0.79


### 5.2.8 Patterns

#### Map-reduce

1. **Split** the input into chunks.
2. **Map**: each worker turns its chunk into a small partial result.
3. **Reduce**: combine the partial results in the parent.

Only the partial results travel back, so the reduce step stays cheap. Here:
revenue per city over 400 000 sales records, with `Counter` partials that add up
with `+`. (Lab 5.2 does word counts the same way.)

In [130]:
sales_random = random.Random(52)
cities = ["Pune", "Mumbai", "Bengaluru", "Chennai", "Kolkata"]
sales = [
    (sales_random.choice(cities), sales_random.randint(100, 5_000))
    for _ in range(400_000 * DEMO_SCALE)
]

with Timer("sequential map + reduce"):
    serial_totals = p5_patterns.revenue_by_city(sales)
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as pool:
    with Timer("parallel: 16 chunks on 4 workers"):
        chunks = [
            sales[start : start + len(sales) // 16]
            for start in range(0, len(sales), len(sales) // 16)
        ]
        partials = pool.map(p5_patterns.revenue_by_city, chunks)
        parallel_totals = functools.reduce(operator.add, partials, Counter())
print(
    f"same totals: {parallel_totals == serial_totals}; top city: {parallel_totals.most_common(1)}"
)

sequential map + reduce took 26.20 ms


parallel: 16 chunks on 4 workers took 49.41 ms
same totals: True; top city: [('Chennai', 204450646)]


On this workload the parallel version is often *not* faster: summing one number
per record is cheap compared with pickling the record to a worker. The pattern
pays off when the map step is expensive per item (parsing, hashing, matching),
and costs least when workers read their own input (a file name or a byte range)
instead of receiving it.

#### Chunking

How you split matters as much as how many workers you use:

| Too few chunks (1 per worker) | Too many chunks (1 per item) | About 4 chunks per worker |
| --- | --- | --- |
| one slow chunk leaves the others idle at the end | per-chunk overhead dominates | balances load and overhead |

`itertools.batched` (*new in 3.12*) cuts an iterable into chunks of a fixed
size; to get a fixed *number* of nearly equal chunks, compute the size first.

In [131]:
def split_evenly[T](items: list[T], chunk_count: int) -> list[list[T]]:
    """Split items into chunk_count chunks whose sizes differ by at most one."""
    size, extra = divmod(len(items), chunk_count)
    chunks, start = [], 0
    for index in range(chunk_count):
        stop = start + size + (1 if index < extra else 0)
        chunks.append(items[start:stop])
        start = stop
    return chunks


ten_orders = [f"order-{order_number}" for order_number in range(10)]
print(
    "batched(size 4):        "
    f"{[len(batch) for batch in itertools.batched(ten_orders, 4, strict=False)]}"
)
print(f"split_evenly(4 chunks): {[len(chunk) for chunk in split_evenly(ten_orders, 4)]}")
print(f"workers x 4 chunks for {os.process_cpu_count()} CPUs: {os.process_cpu_count() * 4} chunks")

batched(size 4):        [4, 4, 2]
split_evenly(4 chunks): [3, 3, 2, 2]
workers x 4 chunks for 16 CPUs: 64 chunks


#### `KeyboardInterrupt` handling

Pressing Ctrl+C in a terminal sends `SIGINT` to the whole process group: the
parent *and* every worker. Without care, each worker prints its own
`KeyboardInterrupt` traceback and the pool can hang. The usual recipe:

1. Workers ignore `SIGINT` (an `initializer` that sets `SIG_IGN`): stopping is
   the parent's job.
2. The parent catches `KeyboardInterrupt`, cancels pending work and stops the
   workers: `executor.terminate_workers()` (*new in 3.14*; it also cancels
   pending futures), or `pool.terminate()` for a `multiprocessing.Pool`.
3. Exit with a clear message (and exit code 130, the shell convention for
   Ctrl+C).

The script below follows that recipe. The notebook starts it, waits for two
results, then sends `SIGINT` to the parent process only (POSIX).

In [132]:
CTRL_C_DEMO = """
    import signal, sys, time
    from concurrent.futures import ProcessPoolExecutor

    def ignore_sigint():
        signal.signal(signal.SIGINT, signal.SIG_IGN)  # Ctrl+C is the parent's job

    def slow_task(number):
        time.sleep(0.2)
        return number

    if __name__ == "__main__":
        executor = ProcessPoolExecutor(max_workers=2, initializer=ignore_sigint)
        try:
            print("started", flush=True)
            for result in executor.map(slow_task, range(100)):
                print(f"got {result}", flush=True)
        except KeyboardInterrupt:
            print("KeyboardInterrupt: cancelling pending tasks and stopping workers")
            executor.terminate_workers()
            print("workers stopped; exiting cleanly")
            sys.exit(130)
"""
if not hasattr(signal, "SIGINT") or os.name != "posix":
    print("SKIP: this demo sends POSIX signals")
else:
    (PHASE5_DIR / "ctrl_c_demo.py").write_text(textwrap.dedent(CTRL_C_DEMO), encoding="utf-8")
    demo = subprocess.Popen(
        [sys.executable, "ctrl_c_demo.py"], cwd=PHASE5_DIR, env=CHILD_ENV,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )  # fmt: skip
    lines_before = [demo.stdout.readline().strip() for _ in range(3)]  # started, got 0, got 1
    demo.send_signal(signal.SIGINT)
    rest, _ = demo.communicate(timeout=30)
    print("\n".join(lines_before + rest.strip().splitlines()))
    print(f"exit code: {demo.returncode}")

started
got 0
got 1
KeyboardInterrupt: cancelling pending tasks and stopping workers
workers stopped; exiting cleanly
exit code: 130


#### `os.process_cpu_count()`

`os.cpu_count()` reports the CPUs in the machine. `os.process_cpu_count()`
(*new in 3.13*) reports how many **this process may use**: it respects CPU
affinity (`taskset`, container limits that pin CPUs). The `-X cpu_count=N` option
and the `PYTHON_CPU_COUNT` variable (*new in 3.13*) override **both** functions,
which is handy for testing pool sizing. Pools default to `process_cpu_count()`,
and so should your own sizing. The child script pins itself to two CPUs (Linux
only), then runs again with `PYTHON_CPU_COUNT=3`.

In [133]:
AFFINITY_DEMO = """
    import os
    print(f"os.cpu_count()={os.cpu_count()}  os.process_cpu_count()={os.process_cpu_count()}")
    if hasattr(os, "sched_setaffinity"):
        os.sched_setaffinity(0, set(sorted(os.sched_getaffinity(0))[:2]))
        print(f"pinned to 2 CPUs: os.process_cpu_count()={os.process_cpu_count()}")
    else:
        print("SKIP: sched_setaffinity is not available on this platform")
"""
print(run_child(AFFINITY_DEMO))
print("--- with PYTHON_CPU_COUNT=3")
print(run_child(AFFINITY_DEMO.split("if hasattr")[0], extra_env={"PYTHON_CPU_COUNT": "3"}))

os.cpu_count()=16  os.process_cpu_count()=16
pinned to 2 CPUs: os.process_cpu_count()=2
--- with PYTHON_CPU_COUNT=3
os.cpu_count()=3  os.process_cpu_count()=3


#### Predict the output

**Predict the output.**

```python
# p5_basics.add_order appends to the module-level list ORDERS
# and returns (pid, len(ORDERS)).
with ProcessPoolExecutor(max_workers=1) as pool:
    counts = [count for _, count in pool.map(p5_basics.add_order, ["B-1", "B-2"])]
print(counts, p5_basics.ORDERS)
```

<details><summary>Answer</summary>

`[1, 2] []`. The single worker process keeps its own copy of `ORDERS` across
both tasks (it is reused), so it counts 1 then 2. The kernel's module was never
touched: processes do not share memory. (A fresh pool starts from an empty
list again.)

</details>

Run the next cell to check your prediction.

In [134]:
with concurrent.futures.ProcessPoolExecutor(max_workers=1) as pool:
    counts = [count for _, count in pool.map(p5_basics.add_order, ["B-1", "B-2"])]
print(counts, p5_basics.ORDERS)

[1, 2] []


**Predict the output.**

```python
import pickle

def outer():
    def helper():
        return 1
    return helper

for candidate in (len, outer, outer()):
    try:
        pickle.dumps(candidate)
        print("ok", end=" ")
    except Exception as error:
        print(type(error).__name__, end=" ")
```

<details><summary>Answer</summary>

`ok ok PicklingError`. `len` and `outer` are found by name (`builtins.len`,
`__main__.outer`), so they pickle. The nested `helper` has the qualified name
`outer.<locals>.helper`, which cannot be looked up, so pickling fails, and it
could never be sent to a process. (Whether `__main__.outer` could be *loaded*
by a child is a separate question: 5.2.2.)

</details>

Run the next cell to check your prediction.

In [135]:
def outer() -> Callable[[], int]:
    """Return a nested function (which cannot be pickled by name)."""

    def helper() -> int:
        """Return 1."""
        return 1

    return helper


for candidate in (len, outer, outer()):
    try:
        pickle.dumps(candidate)
        print("ok", end=" ")
    except Exception as error:
        print(type(error).__name__, end=" ")
print()

ok ok PicklingError 


**Predict the output.**

```python
# 90% of a job parallelizes perfectly. What speedup on 16 cores, and the most on any number?
print(round(1 / ((1 - 0.9) + 0.9 / 16), 1), round(1 / (1 - 0.9), 1))
```

<details><summary>Answer</summary>

`6.4 10.0`. Amdahl's law: the 10% serial part costs 0.1 of the original time on
any number of cores, so the speedup can never exceed 1 / 0.1 = 10. Sixteen
cores give 1 / (0.1 + 0.05625) = 6.4.

</details>

Run the next cell to check your prediction.

In [136]:
print(round(1 / ((1 - 0.9) + 0.9 / 16), 1), round(1 / (1 - 0.9), 1))

6.4 10.0


### Lab 5.2

Four exercises. Worker functions go in `%%writefile` cells (child processes must
import them); edit the cell, run it, then run the cell after it, which
re-imports the module with `fresh_import`. Replace each
`raise NotImplementedError`, then run the check cell. Until you finish, the
checks print FAIL; that is expected.

The first setup cell writes eight 8 MiB files for exercise 1; the second builds
a seeded 100 000-line corpus for exercise 4.

> **Warning:** after editing a worker module, re-run *both* the `%%writefile`
> cell and the cell after it. `fresh_import` puts a new module object in
> `sys.modules`; a function still taken from the old module object can no
> longer be pickled (`PicklingError: ... it's not the same object as ...`),
> because pickle checks that the name leads back to that exact object.

In [137]:
import concurrent.futures
import hashlib
import importlib.util
import inspect
import math
import multiprocessing
import random
import re
import sys
import time
from collections import Counter
from collections.abc import Iterable
from multiprocessing import shared_memory
from pathlib import Path

LAB52_DIR = PHASE5_DIR / "lab52"
LAB52_DIR.mkdir(parents=True, exist_ok=True)
lab_random = random.Random(52)

HASH_FILES = [LAB52_DIR / f"export-{number}.bin" for number in range(8)]
for path in HASH_FILES:  # 8 files of 8 MiB x DEMO_SCALE, same bytes every run
    path.write_bytes(lab_random.randbytes(8 * 1024 * 1024 * DEMO_SCALE))

if HAS_NUMPY:
    import numpy as np
print(f"{len(HASH_FILES)} files of {HASH_FILES[0].stat().st_size // 2**20} MiB")

8 files of 8 MiB


The corpus for exercise 4 (100 000 seeded lines × `DEMO_SCALE`), and
`shm_segments()`, which lists the shared-memory blocks currently in `/dev/shm`
so exercise 3 can prove that none were left behind:

In [138]:
WORDS = "chai masala the shop's order delivered late early pune mumbai".split()
WORDS += "refund invoice gst customer happy isn't tea mug kettle rain".split()
CORPUS = [
    " ".join(lab_random.choices(WORDS, k=12)).capitalize() for _ in range(100_000 * DEMO_SCALE)
]


def shm_segments() -> set[str]:
    """Names of POSIX shared-memory blocks made by multiprocessing (Linux; empty elsewhere)."""
    folder = Path("/dev/shm")
    return {entry.name for entry in folder.glob("psm_*")} if folder.exists() else set()


print(f"{len(CORPUS):,} corpus lines; first: {CORPUS[0]!r}")

100,000 corpus lines; first: "Gst pune pune invoice chai isn't mumbai late mumbai refund refund mumbai"


#### Exercise 1 — parallel hashing with processes vs threads, with the result explained

1. In `lab52_hashing.py`, `sha256_file(path)` returns `(path.name, hex digest)`,
   reading the file in 1 MiB blocks and feeding them to one `hashlib.sha256()`.
2. `hash_files(paths, executor_class, max_workers=4)` hashes every file with
   `executor_class(max_workers=max_workers)` (it will be called with
   `ThreadPoolExecutor` and `ProcessPoolExecutor`) and returns
   `{file name: digest}`.
3. `benchmark_hashing(paths, max_workers=4)` returns the seconds taken for
   `"sequential"` (a plain loop), `"threads"` and `"processes"`.

In [139]:
%%writefile _scratch/phase5/lab52_hashing.py
"""Lab 5.2 exercise 1 worker."""

import hashlib  # noqa: F401 (for your solution)
from pathlib import Path


def sha256_file(path: Path) -> tuple[str, str]:
    """Return (file name, SHA-256 hex digest), reading 1 MiB blocks."""
    raise NotImplementedError

Overwriting _scratch/phase5/lab52_hashing.py


Re-import the worker module, then write the two functions:

In [140]:
hashing_workers = fresh_import("lab52_hashing")


def hash_files(
    paths: list[Path], executor_class: type[concurrent.futures.Executor], max_workers: int = 4
) -> dict[str, str]:
    """Hash files with executor_class(max_workers); return {file name: digest}."""
    raise NotImplementedError


def benchmark_hashing(paths: list[Path], max_workers: int = 4) -> dict[str, float]:
    """Seconds for "sequential", "threads" and "processes"."""
    raise NotImplementedError

Check exercise 1:

In [141]:
EXPECTED_DIGESTS = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in HASH_FILES}
run_checks(
    [
        (
            "sha256_file returns (name, digest)",
            lambda: hashing_workers.sha256_file(HASH_FILES[0]),
            (HASH_FILES[0].name, EXPECTED_DIGESTS[HASH_FILES[0].name]),
        ),
        (
            "threads: right digests",
            lambda: hash_files(HASH_FILES, concurrent.futures.ThreadPoolExecutor),
            EXPECTED_DIGESTS,
        ),
        (
            "processes: right digests",
            lambda: hash_files(HASH_FILES, concurrent.futures.ProcessPoolExecutor),
            EXPECTED_DIGESTS,
        ),
        (
            "benchmark: three positive timings",
            lambda: sorted(
                name for name, seconds in benchmark_hashing(HASH_FILES).items() if seconds > 0
            ),
            ["processes", "sequential", "threads"],
        ),
    ]
)

FAIL  sha256_file returns (name, digest): raised NotImplementedError: 
FAIL  threads: right digests: raised NotImplementedError: 
FAIL  processes: right digests: raised NotImplementedError: 
FAIL  benchmark: three positive timings: raised NotImplementedError: 
0/4 checks passed


**Explain the result.** Run the benchmark, then write down which model won and
why before opening the answer.

<details><summary>Answer</summary>

Threads usually match or beat processes here. `hashlib` releases the GIL while
it hashes each 1 MiB block, and file reads release it too, so threads already
hash on several cores at once, with no start-up cost and nothing to pickle
(each task sends only a path and returns a short string). Processes get the
same parallelism but pay for starting workers and importing the module. With
much bigger files the two converge; with many tiny files, threads win clearly;
if the hashing were pure Python (no GIL release), only processes would scale.

</details>

In [142]:
try:
    for model, seconds in benchmark_hashing(HASH_FILES).items():
        print(f"{model:<11} {seconds:.3f} s")
except NotImplementedError:
    print("Finish exercise 1 first.")

Finish exercise 1 first.


#### Exercise 2 — a `chunksize` tuning chart

The worker `tiny_task` is given (it is deliberately cheap).

1. `time_chunksizes(items, chunksizes, processes=4)` creates **one**
   `multiprocessing.Pool(processes)`, warms it up, then times
   `pool.map(tiny_task, items, chunksize=c)` for each `c` and returns
   `{chunksize: seconds}`.
2. `render_chart(timings, width=40)` returns one line per chunksize, in
   ascending chunksize order: `f"{chunksize:>6} | {bar} {seconds:.3f} s"` where
   `bar` is `"#"` repeated `max(1, round(width * seconds / slowest))` times.

In [143]:
%%writefile _scratch/phase5/lab52_chunks.py
"""Lab 5.2 exercise 2 worker (given)."""


def tiny_task(number: int) -> int:
    """A deliberately tiny task."""
    return (number * 7) % 11

Overwriting _scratch/phase5/lab52_chunks.py


Re-import the worker module, then write the two functions:

In [144]:
chunk_workers = fresh_import("lab52_chunks")


def time_chunksizes(
    items: list[int], chunksizes: list[int], processes: int = 4
) -> dict[int, float]:
    """Seconds for Pool.map(tiny_task, items, chunksize=c) for each c, using one pool."""
    raise NotImplementedError


def render_chart(timings: dict[int, float], width: int = 40) -> list[str]:
    """A text bar chart, one line per chunksize in ascending order."""
    raise NotImplementedError

Check exercise 2:

In [145]:
run_checks(
    [
        (
            "chart lines",
            lambda: render_chart({10: 0.25, 1: 0.5, 100: 0.05}, width=20),
            [
                "     1 | #################### 0.500 s",
                "    10 | ########## 0.250 s",
                "   100 | ## 0.050 s",
            ],
        ),
        (
            "tiny bars still show one #",
            lambda: render_chart({1: 1.0, 1000: 0.001}, width=10)[1],
            "  1000 | # 0.001 s",
        ),
        (
            "one key per chunksize",
            lambda: sorted(time_chunksizes(list(range(2_000)), [1, 50, 500])),
            [1, 50, 500],
        ),
        (
            "positive timings",
            lambda: all(
                seconds > 0 for seconds in time_chunksizes(list(range(500)), [1, 100]).values()
            ),
            True,
        ),
    ]
)

FAIL  chart lines: raised NotImplementedError: 
FAIL  tiny bars still show one #: raised NotImplementedError: 
FAIL  one key per chunksize: raised NotImplementedError: 
FAIL  positive timings: raised NotImplementedError: 
0/4 checks passed


Once the checks pass, draw the real chart for 40 000 tasks (this takes a few
seconds, most of it at `chunksize=1`). Where does the curve flatten on your
machine?

In [146]:
try:
    chart_timings = time_chunksizes(list(range(40_000 * DEMO_SCALE)), [1, 10, 100, 1_000, 10_000])
    print("\n".join(render_chart(chart_timings)))
except NotImplementedError:
    print("Finish exercise 2 first.")

Finish exercise 2 first.


#### Exercise 3 — shared-memory array workers

1. In `lab52_shared.py`, `scale_slice(block_name, length, start, stop, factor)`
   attaches to the shared block with `SharedMemory(name=block_name,
   track=False)`, views it as a `float64` NumPy array of `length` values,
   multiplies `values[start:stop]` by `factor` in place, and closes the block
   (delete the array view first).
2. `parallel_scale(values, factor, workers=4)` creates a `SharedMemory` block,
   copies `values` into it (as `float64`), starts one `multiprocessing.Process`
   per slice (split as evenly as possible), joins them, returns a **copy** of
   the result, and always closes and unlinks the block. The input array must
   not change.

In [147]:
%%writefile _scratch/phase5/lab52_shared.py
"""Lab 5.2 exercise 3 worker."""

from multiprocessing import shared_memory  # noqa: F401 (for your solution)


def scale_slice(block_name: str, length: int, start: int, stop: int, factor: float) -> None:
    """Multiply values[start:stop] of the shared float64 array by factor, in place."""
    raise NotImplementedError

Overwriting _scratch/phase5/lab52_shared.py


Re-import the worker module, then write `parallel_scale`:

In [148]:
shared_workers = fresh_import("lab52_shared")


def parallel_scale(values: np.ndarray, factor: float, workers: int = 4) -> np.ndarray:
    """Scale a copy of values in shared memory with one process per slice."""
    raise NotImplementedError

Check exercise 3 (prints a skip message without NumPy):

In [149]:
def segments_left_behind() -> list[str]:
    """Shared-memory blocks that exist after parallel_scale but did not before."""
    before = shm_segments()
    parallel_scale(np.arange(1_000.0), 2.0)
    return sorted(shm_segments() - before)


def input_unchanged() -> list[float]:
    """parallel_scale must not modify its argument."""
    original = np.arange(5.0)
    parallel_scale(original, 3.0)
    return original.tolist()

The checks: values scaled correctly (also with an uneven split and one worker),
`float64` out, the input untouched, no shared-memory block left in `/dev/shm`, and
real `Process` + `SharedMemory` underneath:

In [150]:
if not HAS_NUMPY:
    print("SKIP: numpy is not installed")
else:
    run_checks(
        [
            (
                "scales every value",
                lambda: parallel_scale(np.arange(6.0), 2.5).tolist(),
                [0.0, 2.5, 5.0, 7.5, 10.0, 12.5],
            ),
            (
                "uneven split: 10 values, 4 workers",
                lambda: parallel_scale(np.arange(10.0), 10.0, 4).tolist(),
                [10.0 * index for index in range(10)],
            ),
            (
                "one worker",
                lambda: parallel_scale(np.arange(3.0), -1.0, 1).tolist(),
                [0.0, -1.0, -2.0],
            ),
            ("returns float64", lambda: str(parallel_scale(np.arange(4), 2.0).dtype), "float64"),
            ("input array unchanged", input_unchanged, [0.0, 1.0, 2.0, 3.0, 4.0]),
            ("no shared memory left behind", segments_left_behind, []),
            (
                "uses Process and SharedMemory",
                lambda: all(
                    word in inspect.getsource(parallel_scale)
                    for word in ("Process(", "SharedMemory(")
                ),
                True,
            ),
        ]
    )

FAIL  scales every value: raised NotImplementedError: 
FAIL  uneven split: 10 values, 4 workers: raised NotImplementedError: 
FAIL  one worker: raised NotImplementedError: 
FAIL  returns float64: raised NotImplementedError: 
FAIL  input array unchanged: raised NotImplementedError: 
FAIL  no shared memory left behind: raised NotImplementedError: 
FAIL  uses Process and SharedMemory: expected True, got False
0/7 checks passed


#### Exercise 4 — map-reduce word counts, measured speedup vs Amdahl's prediction

1. In `lab52_wordcount.py`, `count_words(lines)` (the map step) returns a
   `Counter` of the words in a list of lines: lowercase each line and take
   `re.findall(r"[a-z']+", line)`.
2. `merge_counts(partials)` (the reduce step) adds an iterable of `Counter`s.
3. `parallel_word_count(lines, workers=4, chunks_per_worker=4)` splits `lines`
   into `workers * chunks_per_worker` nearly equal chunks, maps them with a
   `ProcessPoolExecutor(workers)` and reduces with `merge_counts`.
4. `amdahl_speedup(parallel_fraction, workers)` returns
   `1 / ((1 - p) + p / workers)`; raise `ValueError` unless `0 <= p <= 1`.
5. `speedup_report(lines, workers=4)` returns a dict with
   - `"sequential_seconds"`: time to run `count_words` on every chunk in the
     kernel (the map time) plus time to `merge_counts` them (the reduce time);
   - `"parallel_seconds"`: time for `parallel_word_count(lines, workers)`;
   - `"measured_speedup"`: sequential ÷ parallel;
   - `"parallel_fraction"`: map time ÷ sequential time (the part that can run
     in parallel);
   - `"predicted_speedup"`: `amdahl_speedup(parallel_fraction, workers)`.

In [151]:
%%writefile _scratch/phase5/lab52_wordcount.py
"""Lab 5.2 exercise 4 worker."""

import re  # noqa: F401 (for your solution)
from collections import Counter


def count_words(lines: list[str]) -> Counter[str]:
    """Map step: count the lowercase words in a chunk of lines."""
    raise NotImplementedError

Overwriting _scratch/phase5/lab52_wordcount.py


Re-import the worker module, then write the four functions:

In [152]:
wordcount_workers = fresh_import("lab52_wordcount")


def merge_counts(partials: Iterable[Counter[str]]) -> Counter[str]:
    """Reduce step: add the partial counts."""
    raise NotImplementedError


def parallel_word_count(
    lines: list[str], workers: int = 4, chunks_per_worker: int = 4
) -> Counter[str]:
    """Map-reduce word count with a process pool."""
    raise NotImplementedError


def amdahl_speedup(parallel_fraction: float, workers: int) -> float:
    """Amdahl's law."""
    raise NotImplementedError


def speedup_report(lines: list[str], workers: int = 4) -> dict[str, float]:
    """Measured speedup next to Amdahl's prediction."""
    raise NotImplementedError

Check exercise 4:

In [153]:
SAMPLE_LINES = ["The shop's tea", "the TEA was late", ""]


def report_consistent() -> bool:
    """The report's numbers agree with each other."""
    report = speedup_report(CORPUS[:20_000], 4)
    return (
        math.isclose(
            report["measured_speedup"], report["sequential_seconds"] / report["parallel_seconds"]
        )
        and math.isclose(
            report["predicted_speedup"], amdahl_speedup(report["parallel_fraction"], 4)
        )
        and 0 < report["parallel_fraction"] <= 1
    )

The map-reduce checks: the map step, the reduce step, and the parallel total equal
to the sequential one:

In [154]:
run_checks(
    [
        (
            "count_words (map step)",
            lambda: dict(wordcount_workers.count_words(SAMPLE_LINES)),
            {"the": 2, "shop's": 1, "tea": 2, "was": 1, "late": 1},
        ),
        (
            "merge_counts (reduce step)",
            lambda: dict(merge_counts([Counter(a=1, b=2), Counter(b=3), Counter()])),
            {"a": 1, "b": 5},
        ),
        (
            "parallel result equals sequential",
            lambda: (
                parallel_word_count(CORPUS[:20_000])
                == wordcount_workers.count_words(CORPUS[:20_000])
            ),
            True,
        ),
    ]
)

FAIL  count_words (map step): raised NotImplementedError: 
FAIL  merge_counts (reduce step): raised NotImplementedError: 
FAIL  parallel result equals sequential: raised NotImplementedError: 
0/3 checks passed


The Amdahl checks: the formula on known values, rejection of a fraction outside
0..1, and a `speedup_report` whose numbers agree with each other:

In [155]:
run_checks(
    [
        ("amdahl_speedup(0.9, 16)", lambda: round(amdahl_speedup(0.9, 16), 2), 6.4),
        (
            "amdahl_speedup at p=1 and p=0",
            lambda: (amdahl_speedup(1.0, 8), amdahl_speedup(0.0, 8)),
            (8.0, 1.0),
        ),
        (
            "p outside 0..1 is rejected",
            lambda: raised_error_name(lambda: amdahl_speedup(1.5, 4)),
            "ValueError",
        ),
        (
            "speedup_report keys",
            lambda: sorted(speedup_report(CORPUS[:20_000], 4)),
            [
                "measured_speedup",
                "parallel_fraction",
                "parallel_seconds",
                "predicted_speedup",
                "sequential_seconds",
            ],
        ),
        ("speedup_report is consistent", report_consistent, True),
    ]
)

FAIL  amdahl_speedup(0.9, 16): raised NotImplementedError: 
FAIL  amdahl_speedup at p=1 and p=0: raised NotImplementedError: 
FAIL  p outside 0..1 is rejected: expected 'ValueError', got 'NotImplementedError'
FAIL  speedup_report keys: raised NotImplementedError: 
FAIL  speedup_report is consistent: raised NotImplementedError: 
0/5 checks passed


Once the checks pass, compare the measurement with the prediction on the full
corpus. Expect the measured speedup to fall short of Amdahl's number: the law
assumes the parallel part splits perfectly, but here every chunk of lines must
be pickled to a worker and every `Counter` pickled back, and the pool must
start. That overhead behaves like extra serial time. The last number turns the
measurement back into a parallel fraction (`p = (1 - 1/S) / (1 - 1/N)`, from
5.2.7): the gap between it and the profiled `p` is the cost of going parallel.

In [156]:
try:
    for workers in (2, 4):
        report = speedup_report(CORPUS, workers)
        effective_p = (1 - 1 / report["measured_speedup"]) / (1 - 1 / workers)
        print(
            f"{workers} workers: sequential {report['sequential_seconds']:.3f} s, "
            f"parallel {report['parallel_seconds']:.3f} s"
        )
        print(
            f"  measured {report['measured_speedup']:.2f}x, "
            f"Amdahl predicts {report['predicted_speedup']:.2f}x"
        )
        print(
            f"  p from the profile {report['parallel_fraction']:.3f}, "
            f"p implied by the measurement {effective_p:.2f}"
        )
except NotImplementedError:
    print("Finish exercise 4 first.")

Finish exercise 4 first.


### Recap — Unit 5.2

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| Processes | Own memory and own GIL: real parallelism for Python code; milliseconds to start | `child_process.fork` |
| Start methods | `forkserver` is the POSIX default since 3.14; `fork` is unsafe with threads; `spawn` needs the main guard | — |
| Importable workers | Functions are pickled by name; notebook functions live in `__main__` and fail in children | — |
| `Process`, `Pool`, `ProcessPoolExecutor` | Reuse workers; `chunksize` for small tasks; `initializer` per worker; `max_tasks_per_child` against leaks | `cluster`, worker pools |
| Data transfer | Everything crosses as a pickle: coarse tasks, small messages, or parallel gets slower | structured clone |
| IPC | `Queue`, `Pipe`, `Value`/`Array` (+ lock), `Manager` (slow proxies), `shared_memory` (zero-copy with NumPy) | `process.send`, `SharedArrayBuffer` |
| Subinterpreters | `concurrent.interpreters` (3.14): own GIL inside one process; C extensions must opt in | `worker_threads` |
| Choosing a model | Decision matrix + a benchmark on your workload | — |
| Amdahl's law | `S = 1 / ((1 - p) + p / N)`; the serial part caps the speedup | — |
| Patterns | Map-reduce with `Counter`s, ~4 chunks per worker, workers ignore SIGINT, size by `os.process_cpu_count()` | `os.availableParallelism()` |

**Further reading**

- [`multiprocessing`](https://docs.python.org/3/library/multiprocessing.html)
  (read *Programming guidelines*),
  [`multiprocessing.shared_memory`](https://docs.python.org/3/library/multiprocessing.shared_memory.html)
- [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html)
  (`ProcessPoolExecutor`, `InterpreterPoolExecutor`),
  [`concurrent.interpreters`](https://docs.python.org/3/library/concurrent.interpreters.html)
- [`pickle`: what can be pickled](https://docs.python.org/3/library/pickle.html#what-can-be-pickled-and-unpickled),
  [`os.process_cpu_count`](https://docs.python.org/3/library/os.html#os.process_cpu_count)
- [PEP 371](https://peps.python.org/pep-0371/) (the `multiprocessing` package),
  [PEP 684](https://peps.python.org/pep-0684/) (a per-interpreter GIL),
  [PEP 734](https://peps.python.org/pep-0734/) (multiple interpreters in the standard library)

## Unit 5.3 — asyncio Internals: the Event Loop

**Objectives**

- Explain why one thread can serve thousands of connections (the C10k problem)
  and what "non-blocking" means at the operating-system level.
- Use `select`, `epoll` and the `selectors` module directly.
- Build a working event loop from scratch: generators as tasks, a ready queue,
  a timer heap and `selectors` for I/O, then an echo server on top of it.
- Read asyncio's real loop (`_run_once`) and map every part of the toy loop onto it.
- Explain coroutines, awaitables, Futures and Tasks, and exactly what `await` does.
- Predict the order in which callbacks and coroutines run, in Python and in Node,
  and explain every difference.

**JS bridge**

| Node.js | asyncio | The difference that bites |
| --- | --- | --- |
| The event loop starts by itself when the script ends | You start it: `asyncio.run(main())` (Jupyter already runs one) | No loop, no async; and one loop per thread |
| Calling an `async function` runs its body up to the first `await` | Calling an `async def` returns a coroutine object; **nothing runs** | Forgetting `await` silently does nothing (plus a warning) |
| `Promise` | `asyncio.Future` (a result slot with callbacks); `asyncio.Task` drives a coroutine | Futures and Tasks can be **cancelled** |
| Microtask queue (`then`, `await`) drained after every callback | One FIFO ready queue for every callback and task step | No "microtasks run first" rule (5.3.9) |
| `setTimeout(fn, ms)` / `setImmediate(fn)` | `loop.call_later(seconds, fn)` / `loop.call_soon(fn)` | Seconds, not milliseconds |
| libuv thread pool (4 threads) for `fs`, `dns.lookup`, `crypto` | The loop's default `ThreadPoolExecutor` (`asyncio.to_thread`) | Python's pool is bigger (`min(32, cpus + 4)`) and you call it explicitly |
| Every `await` is a trip through the microtask queue | `await` of a coroutine that never suspends **does not yield** | A loop of quick awaits can starve other tasks |

**How this unit runs.** The Jupyter kernel already runs an asyncio event loop,
so cells use **top-level `await`** directly. `asyncio.run()` cannot be called
inside a running loop (5.3.8 shows the error), so demos of `asyncio.run()`,
`asyncio.Runner` and script entry points run as scripts in a child interpreter
with `run_child()`. The toy event loop in 5.3.3 is plain synchronous Python and
runs anywhere.

Unit setup, part 1: imports, `DEMO_SCALE`, the scratch folder for child
scripts, and the error and interpreter helpers used since Phase 3.

In [157]:
import asyncio
import dis
import errno
import gc
import heapq
import importlib.util
import inspect
import itertools
import os
import platform
import re
import select
import selectors
import shutil
import socket
import subprocess
import sys
import sysconfig
import textwrap
import threading
import time
import tracemalloc
import warnings
from collections import deque
from collections.abc import Callable, Coroutine, Generator, Iterable
from pathlib import Path
from typing import Any

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))  # the one knob for workload sizes
REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE5_DIR = scratch_dir / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT / "notebooks")
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1"}
HAS_RESOURCE = importlib.util.find_spec("resource") is not None  # POSIX only


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" (memory addresses hidden), or "no error"."""
    try:
        thunk()
    except Exception as error:
        return re.sub(r" at 0x[0-9a-f]+", "", f"{type(error).__name__}: {error}")
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"

Unit setup, part 2: `Timer` (Unit 1.14), `run_child()` (Unit 5.1; this
version also hides memory addresses, which asyncio prints in debug messages)
and `KERNEL_LOOP`, the loop the kernel runs cells on.

In [158]:
class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        """Remember the label and where to report the elapsed time."""
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        """Start the clock."""
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        """Stop the clock and report the elapsed milliseconds."""
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def run_child(
    source: str, python: str = sys.executable, extra_env: dict[str, str] | None = None
) -> str:
    """Run dedented source as a script in a child interpreter; return its stdout + stderr."""
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    completed = subprocess.run(
        [python, str(script)], capture_output=True, text=True, timeout=120,
        env={**CHILD_ENV, **(extra_env or {})}, cwd=PHASE5_DIR,
    )  # fmt: skip
    output = (completed.stdout + completed.stderr).rstrip()
    for path, label in ((str(PHASE5_DIR), "<scratch>"), (str(Path(python).parent), "<python>")):
        output = output.replace(path, label)
    output = re.sub(r"pid=\d+", "pid=<pid>", output)
    output = re.sub(r"0x[0-9a-f]{6,}", "0x<address>", output)
    return re.sub(r"/[^\s\"']*/lib/python3\.\d+t?/", "<stdlib>/", output)


KERNEL_LOOP = asyncio.get_running_loop()  # the kernel's own loop: top-level await runs on it
print(interpreter_label())
print(f"{DEMO_SCALE=}  kernel loop: {type(KERNEL_LOOP).__name__}")

CPython 3.15.1, default (GIL) build, x86_64
DEMO_SCALE=1  kernel loop: _UnixSelectorEventLoop


### 5.3.1 Why async

**The C10k problem.** In 1999 Dan Kegel asked how one server could handle ten
thousand clients at once. The classic design, one thread (or process) per
connection, runs out of memory and drowns in context switches long before
that: each thread reserves a stack, and most of those threads are just
*waiting* for a slow client. The fix that every modern server uses (nginx,
Node.js, asyncio, Go's runtime) is to stop dedicating a thread to each wait:

| Model | One connection costs | Who switches between connections |
| --- | --- | --- |
| Thread per connection | a thread: a stack reservation (8 MiB virtual by default on Linux) + kernel bookkeeping | the OS scheduler, at any instruction (preemptive) |
| Event loop (asyncio, Node) | a small object: a coroutine frame and a socket | the program, only at `await` (cooperative) |

The next cell compares the two costs on this machine: the stack size the OS
reserves for every new thread, and the memory `tracemalloc` sees for each
parked asyncio task.

In [159]:
if HAS_RESOURCE:
    import resource

    stack_soft_limit, _ = resource.getrlimit(resource.RLIMIT_STACK)
    print(f"thread stack reservation: {stack_soft_limit / 2**20:.0f} MiB of virtual memory")
else:
    print("resource module not available on this OS: skipping the stack-size probe")


async def idle_connection(closed: asyncio.Event) -> None:
    """A connection that waits for its client: it holds no thread while waiting."""
    await closed.wait()


task_count = 10_000 * DEMO_SCALE
closed = asyncio.Event()
tracemalloc.start()
idle_tasks = [asyncio.create_task(idle_connection(closed)) for _ in range(task_count)]
await asyncio.sleep(0)  # let every task start and park on the event
traced_bytes, _ = tracemalloc.get_traced_memory()
tracemalloc.stop()
closed.set()
await asyncio.gather(*idle_tasks)
print(f"{task_count:,} parked tasks: about {traced_bytes / task_count:,.0f} bytes each")

thread stack reservation: 8 MiB of virtual memory


10,000 parked tasks: about 1,048 bytes each


Thousands of waiting tasks cost a few megabytes in total; the same number of
threads would reserve tens of gigabytes of address space and make the OS
schedule all of them. (Numbers vary by Python version and OS.)

**Blocking vs non-blocking I/O.** A *blocking* call (`time.sleep`,
`socket.recv` on a normal socket, `requests.get`) parks the whole thread until
the data arrives. A *non-blocking* call returns at once: either with data or
with "not ready yet". An event loop only makes non-blocking calls, and asks the
OS to tell it when something becomes ready. Here are twenty 50 ms waits, done
blocking (one after another) and non-blocking (all waiting at once on one
thread):

In [160]:
def blocking_fetch(seconds: float) -> None:
    """Stand-in for a blocking network call: the thread can do nothing else."""
    time.sleep(seconds)


async def non_blocking_fetch(seconds: float) -> None:
    """The same wait, but the loop can run other tasks meanwhile."""
    await asyncio.sleep(seconds)


request_count, wait_seconds = 20, 0.05
with Timer(f"{request_count} blocking waits, one after another") as blocking:
    for _ in range(request_count):
        blocking_fetch(wait_seconds)
with Timer(f"{request_count} non-blocking waits on one thread") as non_blocking:
    await asyncio.gather(*(non_blocking_fetch(wait_seconds) for _ in range(request_count)))
print(f"ratio: {blocking.elapsed_seconds / non_blocking.elapsed_seconds:.1f}x")

20 blocking waits, one after another took 1004.98 ms
20 non-blocking waits on one thread took 51.18 ms
ratio: 19.6x


The same difference at the socket level. `socket.socketpair()` gives two
connected sockets. On a blocking socket `recv` waits (here a 0.2 s timeout
stops it); after `setblocking(False)` it returns immediately with
`BlockingIOError` (errno `EAGAIN`: "try again later") when there is no data.

In [161]:
sending_side, receiving_side = socket.socketpair()
receiving_side.settimeout(0.2)  # blocking, but give up after 0.2 s
started = time.perf_counter()
try:
    receiving_side.recv(1024)
except TimeoutError:
    print(f"blocking recv waited {time.perf_counter() - started:.2f} s for nothing")

receiving_side.setblocking(False)
try:
    receiving_side.recv(1024)
except BlockingIOError as error:
    print(f"non-blocking recv returned at once: {errno.errorcode[error.errno]}")

sending_side.send(b"order 42 ready")
print(f"after send: {receiving_side.recv(1024)=}")
sending_side.close()
receiving_side.close()

blocking recv waited 0.20 s for nothing
non-blocking recv returned at once: EAGAIN
after send: receiving_side.recv(1024)=b'order 42 ready'


**Cooperative multitasking.** Threads are *preemptive*: the OS can switch at
any moment. asyncio tasks are *cooperative*: a task runs until it reaches an
`await` that actually suspends, and only then can another task run. A task
that never awaits keeps everyone else waiting. Notice how `greedy` runs all
its steps in one go while the two polite tasks take turns:

In [162]:
switch_trace: list[str] = []


async def polite(name: str, steps: int) -> None:
    """Yield to the loop after every step: other tasks get a turn."""
    for step in range(steps):
        switch_trace.append(f"{name}{step}")
        await asyncio.sleep(0)  # the only place a switch can happen


async def greedy(name: str, steps: int) -> None:
    """Never awaits: nobody else runs until it returns."""
    for step in range(steps):
        switch_trace.append(f"{name}{step}")


await asyncio.gather(polite("p", 3), greedy("g", 3), polite("q", 3))
print(" ".join(switch_trace))

p0 g0 g1 g2 q0 p1 q1 p2 q2


> **Note:** cooperative scheduling is why async code needs no locks around
> `counter += 1` *between* awaits, and also why one slow, CPU-heavy or blocking
> call freezes every connection on the loop (Unit 5.4.6 shows how to detect it).

### 5.3.2 OS foundations

An event loop needs two things from the operating system: **non-blocking
sockets** (calls that return "not ready" instead of waiting) and a way to
**wait for many sockets at once**. The second comes in two styles:

| OS | API | Style | asyncio uses |
| --- | --- | --- | --- |
| Any POSIX, Windows (sockets only) | `select()` | readiness: "tell me which sockets are ready"; scans every socket each call | `SelectSelector` (fallback) |
| POSIX | `poll()` | readiness; no 1024-descriptor limit, still a scan | `PollSelector` |
| Linux | `epoll` | readiness; the kernel keeps the interest list, cost grows with *ready* sockets only | `EpollSelector` (default on Linux) |
| macOS, BSD | `kqueue` | readiness; also files, signals, timers | `KqueueSelector` |
| Windows | IOCP | **completion**: "do this read and tell me when it is done" | `ProactorEventLoop` (the default on Windows) |

Readiness APIs say *a read would not block now*; the program then does the
read. Completion APIs do the read for you and report when it has finished.
Node's libuv wraps the same calls (`epoll`, `kqueue`, IOCP). Which ones does
this machine have?

In [163]:
for api_name in ("select", "poll", "epoll", "kqueue", "devpoll"):
    status = "available" if hasattr(select, api_name) else "missing"
    print(f"select.{api_name:<8} {status}")
print(f"selectors.DefaultSelector -> {selectors.DefaultSelector.__name__}")
print(f"kernel loop selector: {type(KERNEL_LOOP._selector).__name__}")  # CPython internal
print(f"{sys.platform=} (on win32 the default loop is ProactorEventLoop, using IOCP)")

select.select   available
select.poll     available
select.epoll    available
select.kqueue   missing
select.devpoll  missing
selectors.DefaultSelector -> EpollSelector
kernel loop selector: EpollSelector
sys.platform='linux' (on win32 the default loop is ProactorEventLoop, using IOCP)


`select.select(readable, writable, errors, timeout)` is the oldest form: give
it lists of sockets, get back the ones that are ready. With `timeout=0` it only
checks; with `None` it waits until something is ready.

In [164]:
reader_socket, writer_socket = socket.socketpair()
readable, _, _ = select.select([reader_socket], [], [], 0)
print(f"before send: {len(readable)} readable socket(s)")
writer_socket.send(b"ping")
readable, writable, _ = select.select([reader_socket], [writer_socket], [], 0)
print(f"after send: {len(readable)} readable, {len(writable)} writable")
print(f"data: {reader_socket.recv(16)!r}")
reader_socket.close()
writer_socket.close()

before send: 0 readable socket(s)
after send: 1 readable, 1 writable
data: b'ping'


The `selectors` module wraps all of these behind one interface and picks the
best one (`DefaultSelector`). Each registration carries a `data` payload,
usually the callback or task to wake up: exactly what an event loop needs. Here
three "connections" are registered and two of them receive data:

In [165]:
selector = selectors.DefaultSelector()
pairs = {label: socket.socketpair() for label in ("orders", "payments", "emails")}
for label, (receiving, _) in pairs.items():
    receiving.setblocking(False)
    selector.register(receiving, selectors.EVENT_READ, data=label)

pairs["orders"][1].send(b"new order #7")
pairs["emails"][1].send(b"welcome mail")
ready = selector.select(timeout=0)  # [(SelectorKey, event mask), ...]
for key, events in sorted(ready, key=lambda item: item[0].data):
    print(f"{key.data:<9} ready (events={events}): {key.fileobj.recv(64)!r}")
print(f"registered: {sorted(key.data for key in selector.get_map().values())}")
print(f"{selectors.EVENT_READ=} {selectors.EVENT_WRITE=}")
for receiving, sending in pairs.values():
    selector.unregister(receiving)
    receiving.close()
    sending.close()

emails    ready (events=1): b'welcome mail'
orders    ready (events=1): b'new order #7'
registered: ['emails', 'orders', 'payments']
selectors.EVENT_READ=1 selectors.EVENT_WRITE=2


**Write readiness is backpressure at the OS level.** A fresh socket is
writable: the kernel has buffer space. Keep sending without the other side
reading and the buffer fills; `send` then raises `BlockingIOError` and the
socket stops being writable until the reader drains it. asyncio's
`writer.drain()` (Unit 5.4) is built on exactly this signal.

In [166]:
fast_sender, slow_reader = socket.socketpair()
fast_sender.setblocking(False)
slow_reader.setblocking(False)
write_selector = selectors.DefaultSelector()
write_selector.register(fast_sender, selectors.EVENT_WRITE)
print(f"fresh socket writable: {bool(write_selector.select(timeout=0))}")

buffered_bytes = 0
try:
    while True:
        buffered_bytes += fast_sender.send(b"x" * 4096)
except BlockingIOError:
    print(f"the kernel buffered {buffered_bytes:,} bytes, then send raised BlockingIOError")
print(f"writable while full: {bool(write_selector.select(timeout=0))}")

drained_bytes = 0
try:
    while True:
        drained_bytes += len(slow_reader.recv(65536))
except BlockingIOError:
    pass
writable_again = bool(write_selector.select(timeout=0))
print(f"reader drained {drained_bytes:,} bytes; writable again: {writable_again}")
write_selector.close()
fast_sender.close()
slow_reader.close()

fresh socket writable: True
the kernel buffered 180,224 bytes, then send raised BlockingIOError
writable while full: False
reader drained 180,224 bytes; writable again: True


**Non-blocking connect.** Even connecting can wait (a TCP handshake to a far
server). On a non-blocking socket, `connect_ex` returns at once, usually with
`EINPROGRESS`; the socket becomes **writable** when the handshake finishes,
and `SO_ERROR` tells whether it succeeded. (On loopback the handshake may
already be done, so you may see `0`.)

In [167]:
listener = socket.create_server(("127.0.0.1", 0))  # port 0: any free port
port = listener.getsockname()[1]
client = socket.socket()
client.setblocking(False)
result_code = client.connect_ex(("127.0.0.1", port))
print(f"connect_ex returned {errno.errorcode.get(result_code, result_code)}")

connect_selector = selectors.DefaultSelector()
connect_selector.register(client, selectors.EVENT_WRITE)
connect_selector.select(timeout=1)  # wait until writable = handshake finished
print(f"SO_ERROR after the handshake: {client.getsockopt(socket.SOL_SOCKET, socket.SO_ERROR)}")
connection, address = listener.accept()
print(f"server accepted a connection from {address[0]} (the client port varies)")
for item in (connect_selector, client, connection, listener):
    item.close()

connect_ex returned EINPROGRESS
SO_ERROR after the handshake: 0
server accepted a connection from 127.0.0.1 (the client port varies)


### 5.3.3 A toy event loop from scratch

The best way to understand asyncio is to build a small one. We use **plain
generators** as tasks (Unit 1.13): a generator runs until `yield`, hands a
value to whoever called `send()`, and waits there until it is resumed. That is
all a coroutine is. The loop grows in five steps, each a subclass of the last:

| Step | Class | A task can yield | New machinery |
| --- | --- | --- | --- |
| 1 | `RoundRobinLoop` | anything ("let others run") | a **ready queue** (`deque`) |
| 2 | `TimerLoop` | `("sleep", seconds)` | a **timer heap** (`heapq`) |
| 3 | `IOLoop` | `("read", sock)`, `("write", sock)` | a **selector** |
| 4 | (same loop) | `yield from recv(...)` | helpers composed with `yield from`, and an echo server |
| 5 | `ToyLoop` | `("join", task)` | **tasks** with results and waiters |

**Step 1: round-robin.** The ready queue holds `(task, value to send)` pairs.
`run_ready()` resumes only the tasks that were ready when the pass began;
anything scheduled during the pass waits for the next one. asyncio does the
same (`ntodo = len(self._ready)`), and it is what keeps one busy task from
starving the rest.

In [168]:
type ToyCoroutine = Generator[object, object, object]


class RoundRobinLoop:
    """Step 1: a ready queue of generators, resumed in turn until each finishes."""

    def __init__(self) -> None:
        """Start with an empty ready queue and a step counter."""
        self.ready: deque[tuple[ToyCoroutine, object]] = deque()
        self.steps = 0

    def spawn(self, coroutine: ToyCoroutine) -> object:
        """Schedule a new task; it starts on the next pass."""
        self.ready.append((coroutine, None))
        return None

    def run(self) -> None:
        """Run until no task is left."""
        while self.ready:
            self.run_ready()

    def run_ready(self) -> None:
        """Resume every task that was ready when this pass started."""
        for _ in range(len(self.ready)):
            coroutine, value = self.ready.popleft()
            self.steps += 1
            try:
                request = coroutine.send(value)
            except StopIteration as stop:
                self.finish(coroutine, stop.value)
            else:
                self.handle(coroutine, request)

    def handle(self, coroutine: ToyCoroutine, request: object) -> None:
        """Step 1 understands one request: run me again on the next pass."""
        self.ready.append((coroutine, None))

    def finish(self, coroutine: ToyCoroutine, result: object) -> None:
        """A task returned. Step 5 stores the result."""

Two countdown tasks share the loop. Each `yield` hands control back, so their
output interleaves. Nothing here is asynchronous yet: it is just turn-taking.

In [169]:
def countdown(label: str, count: int) -> ToyCoroutine:
    """Print a countdown, yielding to the loop after every number."""
    while count:
        print(f"{label}{count}", end=" ")
        yield
        count -= 1
    return f"{label} done"


round_robin = RoundRobinLoop()
round_robin.spawn(countdown("A", 3))
round_robin.spawn(countdown("B", 2))
round_robin.run()
print(f"\n{round_robin.steps} steps")

A3 B2 A2 B1 A1 
7 steps


**Step 2: timers.** A task that wants to sleep yields `("sleep", seconds)`.
The loop pushes `(wake-up time, tie-breaker, task)` onto a heap, so the
earliest timer is always at index 0 (the counter breaks ties, because
generators cannot be compared). `run()` now has the shape of one asyncio
iteration: work out how long it may wait, wait, move due timers to the ready
queue, run the ready tasks.

In [170]:
class TimerLoop(RoundRobinLoop):
    """Step 2: a task may yield ("sleep", seconds); sleepers wait in a heap."""

    def __init__(self) -> None:
        """Add a heap of (wake time, tie-breaker, coroutine) timers."""
        super().__init__()
        self.timers: list[tuple[float, int, ToyCoroutine]] = []
        self.tie_breaker = itertools.count()

    def handle(self, coroutine: ToyCoroutine, request: object) -> None:
        """Park a coroutine on the timer heap for sleep requests."""
        match request:
            case ("sleep", float() | int() as seconds):
                wake_at = time.monotonic() + seconds
                heapq.heappush(self.timers, (wake_at, next(self.tie_breaker), coroutine))
            case _:
                super().handle(coroutine, request)

    def run(self) -> None:
        """Each iteration: wait, release due timers, run what is ready."""
        while self.ready or self.timers or self.waiting_for_io():
            self.wait_for_events(self.timeout())
            while self.timers and self.timers[0][0] <= time.monotonic():
                _, _, coroutine = heapq.heappop(self.timers)
                self.ready.append((coroutine, None))
            self.run_ready()

    def timeout(self) -> float | None:
        """0 if work is ready, else until the next timer, else forever (None)."""
        if self.ready:
            return 0
        return max(0.0, self.timers[0][0] - time.monotonic()) if self.timers else None

    def wait_for_events(self, timeout: float | None) -> None:
        """Sleep until the next timer is due (no I/O to wait for yet)."""
        time.sleep(timeout or 0)  # nothing else to do until the next timer

    def waiting_for_io(self) -> bool:
        """This loop never waits for I/O."""
        return False

Three sleepers of 0.3, 0.1 and 0.2 s finish in about 0.3 s in total, in
wake-up order, on one thread:

In [171]:
def sleeper(label: str, seconds: float, started: float) -> ToyCoroutine:
    """Sleep on the toy loop, then report when it woke up."""
    yield ("sleep", seconds)
    print(f"{label:<6} woke after {time.perf_counter() - started:.2f} s")
    return label


sleep_started = time.perf_counter()
timer_loop = TimerLoop()
for label, seconds in (("slow", 0.3), ("fast", 0.1), ("medium", 0.2)):
    timer_loop.spawn(sleeper(label, seconds, sleep_started))
timer_loop.run()
print(f"total {time.perf_counter() - sleep_started:.2f} s for 0.6 s of sleeping")

fast   woke after 0.10 s


medium woke after 0.20 s
slow   woke after 0.30 s
total 0.30 s for 0.6 s of sleeping


**Step 3: I/O.** A task that wants to read yields `("read", sock)`. The loop
registers the socket with a selector, using the task as the `data` payload.
`wait_for_events` now calls `selector.select(timeout)` instead of `sleep`: it
returns early when a socket is ready, or when the next timer is due. Each
registration is one-shot (unregistered when it fires); a task that wants to
read again yields again.

In [172]:
class IOLoop(TimerLoop):
    """Step 3: a task may yield ("read", sock) or ("write", sock)."""

    def __init__(self) -> None:
        """Add a selector for sockets that coroutines wait on."""
        super().__init__()
        self.selector = selectors.DefaultSelector()

    def handle(self, coroutine: ToyCoroutine, request: object) -> None:
        """Register the coroutine with the selector for read or write requests."""
        match request:
            case ("read", socket.socket() as sock):
                self.selector.register(sock, selectors.EVENT_READ, coroutine)
            case ("write", socket.socket() as sock):
                self.selector.register(sock, selectors.EVENT_WRITE, coroutine)
            case _:
                super().handle(coroutine, request)

    def waiting_for_io(self) -> bool:
        """True while any socket is registered with the selector."""
        return bool(self.selector.get_map())

    def wait_for_events(self, timeout: float | None) -> None:
        """select() instead of sleep(): wake up early when a socket is ready."""
        if not self.waiting_for_io():
            return super().wait_for_events(timeout)
        for key, _ in self.selector.select(timeout):
            self.selector.unregister(key.fileobj)  # one-shot registration
            self.ready.append((key.data, None))

A reader waits on a socket while a second task sleeps 0.1 s and then writes
to the other end. The reader is woken by the selector, not by polling:

In [173]:
def read_when_ready(sock: socket.socket, started: float) -> ToyCoroutine:
    """Wait for readability, then read."""
    yield ("read", sock)
    print(f"reader got {sock.recv(64)!r} after {time.perf_counter() - started:.2f} s")


def write_later(sock: socket.socket, delay: float) -> ToyCoroutine:
    """Sleep, then send."""
    yield ("sleep", delay)
    sock.send(b"invoice #12 paid")


io_left, io_right = socket.socketpair()
io_left.setblocking(False)
io_loop = IOLoop()
io_loop.spawn(read_when_ready(io_left, time.perf_counter()))
io_loop.spawn(write_later(io_right, 0.1))
io_loop.run()
io_left.close()
io_right.close()

reader got b'invoice #12 paid' after 0.10 s


**Step 4: an echo server.** First two helpers that wait and then act, the toy
versions of `await sock.recv()` and `await sock.sendall()`. A task calls them
with `yield from`, which passes every request up to the loop and returns the
helper's result. `yield from` is the direct ancestor of `await` (PEP 380, then
PEP 492).

In [174]:
def recv(sock: socket.socket, size: int) -> Generator[object, object, bytes]:
    """Wait until sock is readable, then read: a toy `await sock.recv(size)`."""
    yield ("read", sock)
    return sock.recv(size)


def sendall(sock: socket.socket, data: bytes) -> Generator[object, object]:
    """Send everything, waiting for writability whenever the buffer is full."""
    while data:
        yield ("write", sock)
        data = data[sock.send(data) :]

The server task accepts connections (a listening socket is *readable* when a
client is waiting) and spawns a handler task per client. Each handler echoes
what it receives in upper case until the client closes (`recv` returns
`b""`). Each client connects without blocking, sends two messages and reads
the replies.

In [175]:
def echo_server(loop: IOLoop, listener: socket.socket, client_count: int) -> ToyCoroutine:
    """Accept client_count connections, one handler task each."""
    for _ in range(client_count):
        yield ("read", listener)
        connection, _ = listener.accept()
        connection.setblocking(False)
        loop.spawn(echo_handler(connection))
    listener.close()


def echo_handler(connection: socket.socket) -> ToyCoroutine:
    """Echo in upper case until the client closes its end."""
    while data := (yield from recv(connection, 1024)):
        yield from sendall(connection, data.upper())
    connection.close()


def echo_client(
    port: int, name: str, messages: list[str], replies: dict[str, list[str]]
) -> ToyCoroutine:
    """Connect without blocking, send each message, collect the echo."""
    sock = socket.socket()
    sock.setblocking(False)
    sock.connect_ex(("127.0.0.1", port))
    yield ("write", sock)  # writable = connected
    for message in messages:
        yield from sendall(sock, f"{name}: {message}".encode())
        replies[name].append((yield from recv(sock, 1024)).decode())
        yield ("sleep", 0.01)
    sock.close()

One loop, one thread: a server, two handlers and two clients all make progress
together.

In [176]:
echo_listener = socket.create_server(("127.0.0.1", 0))
echo_listener.setblocking(False)
echo_port = echo_listener.getsockname()[1]
echo_replies: dict[str, list[str]] = {"asha": [], "ben": []}
echo_loop = IOLoop()
echo_loop.spawn(echo_server(echo_loop, echo_listener, client_count=2))
echo_loop.spawn(echo_client(echo_port, "asha", ["hi", "two chai"], echo_replies))
echo_loop.spawn(echo_client(echo_port, "ben", ["hello", "one coffee"], echo_replies))
with Timer("echo session"):
    echo_loop.run()
for name, replies in echo_replies.items():
    print(f"{name}: {replies}")
print(f"about {echo_loop.steps} task steps, all on {threading.current_thread().name}")

echo session took 21.26 ms
asha: ['ASHA: HI', 'ASHA: TWO CHAI']
ben: ['BEN: HELLO', 'BEN: ONE COFFEE']
about 31 task steps, all on MainThread


**Step 5: tasks with results.** So far a task's return value is lost, and no
task can wait for another. `ToyLoop.spawn` returns a `ToyTask` holding the
result and a list of *waiters*. A task yields `("join", other_task)`; if the
other task has finished it is resumed at once with the result, otherwise it is
parked in `waiters` until `finish()` puts it back on the ready queue, with
the result as the value to send in. That is a Future with done-callbacks, and
`send(result)` is how `await task` gets its value.

In [177]:
class ToyTask:
    """What asyncio calls a Task: a running coroutine plus its result and waiters."""

    def __init__(self, coroutine: ToyCoroutine) -> None:
        """Wrap a coroutine with a result and the coroutines waiting for it."""
        self.coroutine = coroutine
        self.done = False
        self.result: object = None
        self.waiters: list[ToyCoroutine] = []

`ToyLoop` adds the bookkeeping: `spawn` returns the task, a `("join", task)`
request either resumes at once or parks the joiner, and `finish` wakes the
waiters:

In [178]:
class ToyLoop(IOLoop):
    """Step 5: spawn() returns a ToyTask; a task may yield ("join", other_task)."""

    def __init__(self) -> None:
        """Add a table from coroutine to its ToyTask."""
        super().__init__()
        self.tasks: dict[ToyCoroutine, ToyTask] = {}

    def spawn(self, coroutine: ToyCoroutine) -> ToyTask:
        """Schedule a coroutine and return its ToyTask handle."""
        task = self.tasks[coroutine] = ToyTask(coroutine)
        super().spawn(coroutine)
        return task

    def handle(self, coroutine: ToyCoroutine, request: object) -> None:
        """Resume the joiner now if the task is done, else queue it as a waiter."""
        match request:
            case ("join", ToyTask() as other) if other.done:
                self.ready.append((coroutine, other.result))
            case ("join", ToyTask() as other):
                other.waiters.append(coroutine)  # resumed by finish()
            case _:
                super().handle(coroutine, request)

    def finish(self, coroutine: ToyCoroutine, result: object) -> None:
        """Record the result and wake every coroutine waiting for this task."""
        task = self.tasks.pop(coroutine)
        task.done, task.result = True, result
        for waiter in task.waiters:  # like a Future's done-callbacks
            self.ready.append((waiter, result))

A checkout task starts a price lookup in the background, does its own 0.1 s
of work meanwhile, then joins the lookup. Both waits overlap:

In [179]:
def price_lookup(sku: str, seconds: float) -> ToyCoroutine:
    """Pretend to call a pricing service."""
    yield ("sleep", seconds)
    return {"chai": 20, "samosa": 15}[sku]


def checkout(loop: ToyLoop, sku: str, quantity: int) -> ToyCoroutine:
    """Overlap our own work with a background lookup, then join it."""
    lookup = loop.spawn(price_lookup(sku, 0.15))
    yield ("sleep", 0.1)  # validate the cart meanwhile
    unit_price = yield ("join", lookup)
    return f"{quantity} x {sku} = Rs {unit_price * quantity}"


toy_loop = ToyLoop()
with Timer("checkout (0.1 s + 0.15 s of waiting)"):
    checkout_task = toy_loop.spawn(checkout(toy_loop, "chai", 3))
    toy_loop.run()
print(f"{checkout_task.done=} {checkout_task.result=}")

checkout (0.1 s + 0.15 s of waiting) took 150.41 ms
checkout_task.done=True checkout_task.result='3 x chai = Rs 60'


#### How asyncio's real loop maps onto the toy loop

asyncio's loop lives in `asyncio/base_events.py` (`BaseEventLoop`) and
`asyncio/selector_events.py`; it is pure Python, while `Task` and `Future`
are implemented in C (`_asyncio`) for speed.

| Toy loop | asyncio | Difference |
| --- | --- | --- |
| `self.ready`: deque of `(task, value)` | `loop._ready`: deque of `Handle` (a callback + arguments) | asyncio queues *callbacks*; resuming a Task is just one kind of callback |
| `self.timers`: heap of `(when, n, task)` | `loop._scheduled`: heap of `TimerHandle` | cancelled timers are dropped lazily |
| `self.selector` | `loop._selector` (`EpollSelector` here) | the payload is a reader/writer callback |
| one pass of `TimerLoop.run()` | `loop._run_once()` | same order: timeout → `select` → due timers → run `ntodo` handles |
| `yield ("sleep", s)` | `await asyncio.sleep(s)`: a Future + `loop.call_later(s, ...)` that resolves it | |
| `yield ("read", sock)` | `await loop.sock_recv(sock, n)`: `loop.add_reader(fd, callback)` | streams and transports wrap this |
| `yield from helper()` | `await coroutine()` | `await` also accepts Futures and `__await__` objects |
| `ToyTask`, `waiters` | `asyncio.Task`, a Future's done-callbacks | |
| `coroutine.send(value)` in `run_ready` | `Task.__step()` calls `coro.send(None)` | the result is read from the Future, not sent in |
| `yield ("join", task)` | `await task` | |

The kernel's loop is a real instance; its private attributes are visible
(CPython internals: fine to look at, never rely on them in code):

In [180]:
print(f"loop class: {' -> '.join(cls.__name__ for cls in type(KERNEL_LOOP).__mro__[:4])}")
ready, scheduled, selector = KERNEL_LOOP._ready, KERNEL_LOOP._scheduled, KERNEL_LOOP._selector
print(f"_ready:     {type(ready).__name__} of Handle, {len(ready)} queued now")
print(f"_scheduled: {type(scheduled).__name__} heap of TimerHandle, {len(scheduled)} now")
print(f"_selector:  {type(selector).__name__} watching {len(selector.get_map())} descriptors")
print(f"Task implemented in: {asyncio.Task.__module__}  Future in: {asyncio.Future.__module__}")

loop class: _UnixSelectorEventLoop -> BaseSelectorEventLoop -> BaseEventLoop -> AbstractEventLoop
_ready:     deque of Handle, 0 queued now
_scheduled: list heap of TimerHandle, 0 now
_selector:  EpollSelector watching 2 descriptors
Task implemented in: _asyncio  Future in: _asyncio


And the heart of `_run_once`, filtered to the lines that match the toy loop's
steps (the full method is about 100 lines; read it with
`inspect.getsource(asyncio.base_events.BaseEventLoop._run_once)`):

In [181]:
run_once_lines = inspect.getsource(asyncio.base_events.BaseEventLoop._run_once).splitlines()
keywords = (
    "heapq.heappop",
    "timeout =",
    "_selector.select",
    "_process_events",
    "ntodo",
    "popleft",
    "_run()",
)
for line_number, line in enumerate(run_once_lines, start=1):
    if any(keyword in line for keyword in keywords):
        print(f"{line_number:3} {line.rstrip()}")

 29                 handle = heapq.heappop(self._scheduled)
 32         timeout = None
 34             timeout = 0
 37             timeout = self._scheduled[0]._when - self.time()
 39                 timeout = MAXIMUM_SELECT_TIMEOUT
 41                 timeout = 0
 43         event_list = self._selector.select(timeout)
 44         self._process_events(event_list)
 57             handle = heapq.heappop(self._scheduled)
 67         ntodo = len(self._ready)
 68         for i in range(ntodo):
 69             handle = self._ready.popleft()
 76                     handle._run()
 84                 handle._run()


### 5.3.4 Native coroutines

`async def` defines a **coroutine function**. Calling it does **not** run the
body: it returns a *coroutine object*, a paused frame waiting for someone to
drive it. In JavaScript, calling an `async function` runs the body
immediately, up to the first `await`. Watch for the missing body print:

In [182]:
async def fetch_price(sku: str) -> int:
    """Pretend to look up a price."""
    print(f"  fetch_price({sku!r}) body runs")
    await asyncio.sleep(0)
    return 250


price_coroutine = fetch_price("chai")
print(
    f"called: {type(price_coroutine).__name__}, state {inspect.getcoroutinestate(price_coroutine)}"
)
print(f"{inspect.iscoroutinefunction(fetch_price)=} {inspect.iscoroutine(price_coroutine)=}")
chai_price = await price_coroutine  # now it runs
print(f"awaited: {chai_price=}, state {inspect.getcoroutinestate(price_coroutine)}")

called: coroutine, state CORO_CREATED
inspect.iscoroutinefunction(fetch_price)=True inspect.iscoroutine(price_coroutine)=True
  fetch_price('chai') body runs
awaited: chai_price=250, state CORO_CLOSED


A coroutine that is created and dropped without ever being awaited is almost
always a bug (a forgotten `await`). CPython warns when such a coroutine is
garbage-collected. The warning is captured here instead of printed:

In [183]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    forgotten = fetch_price("samosa")  # BUG: no await
    del forgotten  # last reference gone: the coroutine is destroyed unstarted
    gc.collect()
for caught in caught_warnings:
    print(f"{caught.category.__name__}: {caught.message}")

**Awaitables.** `await x` accepts three kinds of object:

| Awaitable | What it is | Made by |
| --- | --- | --- |
| coroutine | the object an `async def` call returns | calling a coroutine function |
| `asyncio.Future` | a result slot that callbacks can wait on | `loop.create_future()` |
| `asyncio.Task` | a Future that drives a coroutine | `asyncio.create_task(coro)` |
| any object with `__await__` | `__await__()` must return an iterator (usually a generator) | your own class |

A generator is *not* awaitable (unless decorated by the legacy
`types.coroutine`). `Delay` below is a custom awaitable: its `__await__`
delegates to `asyncio.sleep`'s own `__await__` with `yield from`.

In [184]:
class Delay:
    """A custom awaitable: __await__ returns a generator."""

    def __init__(self, seconds: float) -> None:
        """Remember how long to wait."""
        self.seconds = seconds

    def __await__(self) -> Generator[Any, None, str]:
        """Delegate to asyncio.sleep, then return a message."""
        yield from asyncio.sleep(self.seconds).__await__()
        return f"waited {self.seconds} s"


bare_future = KERNEL_LOOP.create_future()
bare_future.set_result("ready")
candidates = {
    "coroutine": asyncio.sleep(0, result="slept"),
    "Task": asyncio.create_task(asyncio.sleep(0, result="task done")),
    "Future": bare_future,
    "Delay(0.01)": Delay(0.01),
}
for label, candidate in candidates.items():
    print(f"{label:<12} isawaitable={inspect.isawaitable(candidate)}  result={await candidate!r}")
plain_generator = (number for number in range(1))
print(f"{'generator':<12} isawaitable={inspect.isawaitable(plain_generator)}")

coroutine    isawaitable=True  result='slept'
Task         isawaitable=True  result='task done'
Future       isawaitable=True  result='ready'
Delay(0.01)  isawaitable=True  result='waited 0.01 s'
generator    isawaitable=False


**Under the hood.** A coroutine is compiled like a generator. `dis` shows
`RETURN_GENERATOR` (calling the function only builds the object), and each
`await` becomes `GET_AWAITABLE` followed by a `SEND` / `YIELD_VALUE` loop:
the same machinery as `yield from`. The `CO_COROUTINE` flag marks the code
object.

In [185]:
async def two_steps() -> int:
    """One await, then a return."""
    first = await asyncio.sleep(0, result=1)
    return first + 1


interesting = {
    "RETURN_GENERATOR",
    "GET_AWAITABLE",
    "SEND",
    "YIELD_VALUE",
    "END_SEND",
    "RETURN_VALUE",
}
print(
    [
        instruction.opname
        for instruction in dis.get_instructions(two_steps)
        if instruction.opname in interesting
    ]
)
print(f"{bool(two_steps.__code__.co_flags & inspect.CO_COROUTINE)=}")
print(f"{bool(countdown.__code__.co_flags & inspect.CO_GENERATOR)=}  (the toy-loop generator)")

['RETURN_GENERATOR', 'GET_AWAITABLE', 'SEND', 'YIELD_VALUE', 'END_SEND', 'RETURN_VALUE']
bool(two_steps.__code__.co_flags & inspect.CO_COROUTINE)=True
bool(countdown.__code__.co_flags & inspect.CO_GENERATOR)=True  (the toy-loop generator)


**Driving a coroutine with `send`.** Nothing about `async def` needs asyncio.
`coroutine.send(value)` runs it until something deep inside yields; the
yielded value comes out of `send`, and the next `send(reply)` resumes it with
`reply` as the result of that innermost `yield`. Here `Ask` yields a question
out to whoever drives the coroutine, and we answer by hand, exactly as an
event loop would:

In [186]:
class Ask:
    """An awaitable that hands a question to the driver and returns its reply."""

    def __init__(self, question: str) -> None:
        """Remember the question to yield."""
        self.question = question

    def __await__(self) -> Generator[str, object, object]:
        """Yield the question and return whatever is sent back."""
        reply = yield self.question
        return reply


async def quote_order(sku: str, quantity: int) -> str:
    """Needs two answers from outside before it can finish."""
    unit_price = await Ask(f"price of {sku}?")
    gst_percent = await Ask("GST rate?")
    total = unit_price * quantity * (100 + gst_percent) / 100
    return f"{quantity} x {sku} = Rs {total:.2f}"


quote = quote_order("chai", 3)
print(f"{quote.send(None)=}")  # start: run to the first await
print(f"{quote.send(20)=}")  # answer the price; get the next question
try:
    quote.send(18)  # answer the GST rate; the coroutine returns
except StopIteration as stop:
    print(f"finished with {stop.value!r}")
print(f"{inspect.getcoroutinestate(quote)=}")

quote.send(None)='price of chai?'
quote.send(20)='GST rate?'
finished with '3 x chai = Rs 70.80'
inspect.getcoroutinestate(quote)='CORO_CLOSED'


`throw(error)` resumes the coroutine by raising `error` at the suspended
`await` (this is how cancellation is delivered: the loop throws
`CancelledError` in). `close()` throws `GeneratorExit`, running `finally`
blocks.

In [187]:
async def guarded_quote() -> str:
    """Falls back when the price lookup fails."""
    try:
        return f"price {await Ask('price?')}"
    except LookupError as error:
        return f"fallback price ({type(error).__name__}: {error})"
    finally:
        print("  guarded_quote cleanup ran")


first_quote = guarded_quote()
first_quote.send(None)
try:
    first_quote.throw(KeyError("unknown sku"))
except StopIteration as stop:
    print(f"after throw: {stop.value!r}")

second_quote = guarded_quote()
second_quote.send(None)
second_quote.close()
print(f"after close: {inspect.getcoroutinestate(second_quote)}")

  guarded_quote cleanup ran
after throw: "fallback price (KeyError: 'unknown sku')"
  guarded_quote cleanup ran
after close: CORO_CLOSED


### 5.3.5 What `await` does

`await something` does three things:

1. **Suspension up to the Task.** If `something` is a coroutine, Python runs
   it in place (no loop involved). Only when the chain reaches a Future that
   is not done yet does anything suspend: the Future's `__await__` yields
   *the Future itself*, and that yield travels up through every `await` in
   the chain to the Task that is driving the outermost coroutine.
2. **Future callbacks.** The Task receives the Future and registers its own
   wake-up method with `future.add_done_callback(...)`, then returns to the
   loop. The Task is now suspended; the loop runs other things.
3. **Resumption.** When someone calls `future.set_result(value)` (a timer, an
   I/O callback, another task), the Future schedules its callbacks with
   `call_soon`. The Task's wake-up runs, calls `coro.send(None)`, execution
   continues inside the innermost `await`, and the Future's `__await__`
   returns `value`.

```mermaid
sequenceDiagram
    participant Loop as Event loop
    participant Task as Task.__step
    participant Outer as outer()
    participant Inner as inner()
    participant Fut as Future
    Loop->>Task: run handle (Task step)
    Task->>Outer: coro.send(None)
    Outer->>Inner: await inner()
    Inner->>Fut: await future
    Fut-->>Task: yield future (up through every await)
    Task->>Fut: add_done_callback(task wake-up)
    Task-->>Loop: return: the task is suspended
    Note over Loop: runs other callbacks, waits in select()
    Loop->>Fut: set_result(42) from a timer or I/O callback
    Fut->>Loop: call_soon(task wake-up)
    Loop->>Task: wake-up, then __step
    Task->>Outer: coro.send(None)
    Outer->>Inner: resumes inside inner()
    Fut-->>Inner: __await__ returns 42
    Inner-->>Outer: return 42
    Outer-->>Task: StopIteration(value)
    Task->>Task: set_result(value), schedule its own callbacks
```

<details><summary>ASCII version of the diagram</summary>

```text
Loop        Task.__step       outer()        inner()        Future
 | run step     |                |              |              |
 |------------->| send(None)     |              |              |
 |              |--------------->| await inner()|              |
 |              |                |------------->| await future |
 |              |<---------------- yield future (travels up) --|
 |              | add_done_callback(wakeup) ------------------>|
 |<-------------| return (suspended)                           |
 |  ... other callbacks, select() ...                          |
 |---------------------------------------- set_result(42) ---->|
 |<------------------------------------- call_soon(wakeup) ----|
 |------------->| send(None)     |              |              |
 |              |--------------->|------------->| gets 42 <----|
 |              |<-- StopIteration(value) ------| return 42    |
 |              | set_result(value) on the Task                |
```

</details>

The first claim can be checked directly: drive a coroutine that awaits a
Future, and the object that comes out of `send` *is* that Future.

In [188]:
async def wait_for_payment(payment: asyncio.Future[str]) -> str:
    """Suspend until the payment Future has a result."""
    receipt = await payment
    return f"paid: {receipt}"


payment = KERNEL_LOOP.create_future()
payment_coroutine = wait_for_payment(payment)
yielded = payment_coroutine.send(None)
print(f"{yielded is payment=}: the Future itself came out of the coroutine")
payment.set_result("UPI-7781")
try:
    payment_coroutine.send(None)
except StopIteration as stop:
    print(f"resumed, and the coroutine returned {stop.value!r}")

yielded is payment=True: the Future itself came out of the coroutine
resumed, and the coroutine returned 'paid: UPI-7781'


Through a chain of coroutines the Future still comes out at the top, and
every frame in between stays suspended. `cr_await` shows what each
suspended coroutine is waiting on (the innermost one waits on the
Future's iterator, `FutureIter`):

In [189]:
async def inner(future: asyncio.Future[int]) -> int:
    """Await the future directly."""
    return await future


async def middle(future: asyncio.Future[int]) -> int:
    """Await inner and double the result."""
    return await inner(future) * 2


async def outer(future: asyncio.Future[int]) -> int:
    """Await middle and add one."""
    return await middle(future) + 1


chain_future = KERNEL_LOOP.create_future()
chain = outer(chain_future)
print(f"{chain.send(None) is chain_future=}")
link: object = chain
while inspect.iscoroutine(link):
    line_number, waiting_on = link.cr_frame.f_lineno, type(link.cr_await).__name__
    print(f"  {link.__qualname__:<6} suspended at line {line_number}, awaiting {waiting_on}")
    link = link.cr_await
chain_future.set_result(20)
try:
    chain.send(None)
except StopIteration as stop:
    print(f"{stop.value=}  (20 * 2 + 1)")

chain.send(None) is chain_future=True
  outer  suspended at line 13, awaiting coroutine
  middle suspended at line 8, awaiting coroutine
  inner  suspended at line 3, awaiting FutureIter
stop.value=41  (20 * 2 + 1)


**Future callbacks are scheduled, never called inline.** `set_result` puts
each done-callback on the ready queue with `call_soon`; they run on the next
loop iteration. (JS behaves similarly: `.then` callbacks are queued as
microtasks.)

In [190]:
order_ready = KERNEL_LOOP.create_future()
order_ready.add_done_callback(lambda future: print(f"  callback ran with {future.result()!r}"))
order_ready.set_result("order 17")
print("set_result returned; the callback has not run yet")
await asyncio.sleep(0)  # one trip through the loop
print("after one loop iteration")

set_result returned; the callback has not run yet
  callback ran with 'order 17'
after one loop iteration


Now with a real Task. While suspended, the Task remembers the Future it waits
on (`_fut_waiter`, a CPython internal), and the Future holds the Task's
wake-up callback. `asyncio.capture_call_graph()` (*new in 3.14*) shows the
same chain from the outside, as `python -m asyncio pstree <pid>` does for a
running program.

In [191]:
async def courier(parcel: asyncio.Future[str]) -> str:
    """Wait for the parcel, then report the delivery."""
    return f"delivered {await parcel}"


async def shop(parcel: asyncio.Future[str]) -> str:
    """Await the courier."""
    return await courier(parcel)


parcel = KERNEL_LOOP.create_future()
shop_task = asyncio.create_task(shop(parcel), name="shop-task")
await asyncio.sleep(0)  # let the task start and suspend
print(f"{shop_task._fut_waiter is parcel=}  callbacks on the Future: {len(parcel._callbacks)}")
if sys.version_info >= (3, 14):
    graph = asyncio.capture_call_graph(shop_task)
    frame_names = [entry.frame.f_code.co_name for entry in graph.call_stack]
    print(f"call graph of {graph.future.get_name()}: {frame_names}")
else:
    print("asyncio.capture_call_graph needs Python 3.14+: skipped")
parcel.set_result("parcel #5")
print(f"{await shop_task=}")

shop_task._fut_waiter is parcel=True  callbacks on the Future: 1
call graph of shop-task: ['courier', 'shop']
await shop_task='delivered parcel #5'


### 5.3.6 The asyncio loop

asyncio's loop keeps three structures, exactly like the toy loop:

| Structure | Attribute | Holds |
| --- | --- | --- |
| Ready queue | `loop._ready` | `Handle`s to run on the next iteration, FIFO |
| Scheduled-timer heap | `loop._scheduled` | `TimerHandle`s ordered by `when` |
| Selector | `loop._selector` | file descriptors with reader/writer callbacks |

**One `_run_once` iteration:**

1. Drop cancelled timers from the heap when there are many of them.
2. Compute the select timeout: `0` if the ready queue is not empty, otherwise
   the time until the earliest timer, otherwise wait forever.
3. `event_list = self._selector.select(timeout)`, then `_process_events`
   appends the callbacks of every ready socket to the ready queue.
4. Pop every timer whose time has come and append it to the ready queue.
5. Run exactly the `ntodo = len(self._ready)` handles that are queued *now*.
   Handles added while they run wait for the next iteration.

The scheduling API maps onto those structures:

| Method | Puts a handle on | JS analogue |
| --- | --- | --- |
| `loop.call_soon(callback, *args)` | ready queue | `setImmediate` |
| `loop.call_later(delay, callback, *args)` | timer heap, `when = loop.time() + delay` | `setTimeout` (seconds, not ms) |
| `loop.call_at(when, callback, *args)` | timer heap at an absolute loop time | — |
| `loop.time()` | the loop's clock: `time.monotonic()` | `performance.now()` |

The next cell schedules one of each and records when they ran:

In [192]:
schedule_log: list[str] = []
loop_start = KERNEL_LOOP.time()
all_fired = KERNEL_LOOP.create_future()


def record(label: str) -> None:
    """Log the label with milliseconds since loop_start."""
    schedule_log.append(f"{label:<24} at +{(KERNEL_LOOP.time() - loop_start) * 1000:3.0f} ms")


KERNEL_LOOP.call_later(0.05, record, "call_later(0.05)")
KERNEL_LOOP.call_at(loop_start + 0.02, record, "call_at(start + 0.02)")
KERNEL_LOOP.call_soon(record, "call_soon")
KERNEL_LOOP.call_later(0.06, all_fired.set_result, None)
await all_fired
print("\n".join(schedule_log))
clock_gap_us = abs(KERNEL_LOOP.time() - time.monotonic()) * 1e6
print(f"loop.time() and time.monotonic() differ by {clock_gap_us:.0f} us")

call_soon                at +  0 ms
call_at(start + 0.02)    at + 21 ms
call_later(0.05)         at + 51 ms
loop.time() and time.monotonic() differ by 2 us


Every scheduling call returns a handle that can be cancelled before it runs.
A `TimerHandle` also knows its `when()`:

In [193]:
soon_handle = KERNEL_LOOP.call_soon(print, "never printed")
timer_handle = KERNEL_LOOP.call_later(10, print, "never printed either")
due_in = timer_handle.when() - KERNEL_LOOP.time()
print(f"{type(soon_handle).__name__}, {type(timer_handle).__name__} due in {due_in:.2f} s")
soon_handle.cancel()
timer_handle.cancel()
print(f"{soon_handle.cancelled()=} {timer_handle.cancelled()=}")
await asyncio.sleep(0)  # the cancelled handles are skipped

Handle, TimerHandle due in 10.00 s
soon_handle.cancelled()=True timer_handle.cancelled()=True


The ready queue is a plain `deque` of handles; their reprs name the callback.
(The kernel may have queued handles of its own, so the next cell filters on
our function's name, and hides the source location in the repr.)

In [194]:
def pack_order(order_id: str) -> None:
    """A trivial callback."""


for order_id in ("A-1", "A-2", "A-3"):
    KERNEL_LOOP.call_soon(pack_order, order_id)
print(
    [
        re.sub(r" at .*>", ">", repr(handle))
        for handle in KERNEL_LOOP._ready
        if "pack_order" in repr(handle)
    ]
)
await asyncio.sleep(0)
still_queued = sum("pack_order" in repr(handle) for handle in KERNEL_LOOP._ready)
print(f"after one iteration: {still_queued} left")

['<Handle pack_order()>', '<Handle pack_order()>', '<Handle pack_order()>']
after one iteration: 0 left


**The selector part.** `loop.add_reader(fd, callback)` registers a file
descriptor with the loop's selector; when `select()` reports it readable,
`_process_events` puts the callback on the ready queue. Streams, transports
and `sock_recv` are all built on this. Here a socket pair is watched by the
kernel's loop:

In [195]:
watched_socket, peer_socket = socket.socketpair()
watched_socket.setblocking(False)
data_arrived = KERNEL_LOOP.create_future()


def on_readable() -> None:
    """Called by the loop when select() reports watched_socket readable."""
    data_arrived.set_result(watched_socket.recv(64))


KERNEL_LOOP.add_reader(watched_socket.fileno(), on_readable)
registered = watched_socket.fileno() in KERNEL_LOOP._selector.get_map()
print(f"registered with {type(KERNEL_LOOP._selector).__name__}: {registered}")
KERNEL_LOOP.call_later(0.02, peer_socket.send, b"payment confirmed")
print(f"reader callback received {await data_arrived!r}")
KERNEL_LOOP.remove_reader(watched_socket.fileno())
registered = watched_socket.fileno() in KERNEL_LOOP._selector.get_map()
print(f"still registered after remove_reader: {registered}")
watched_socket.close()
peer_socket.close()

registered with EpollSelector: True
reader callback received b'payment confirmed'
still registered after remove_reader: False


**Watching `_run_once` iterate.** A loop subclass can wrap `_run_once` and
print the queue sizes at the start of each iteration. It must run as the
*only* loop of a fresh process (the kernel's loop is busy), so it runs as a
script with `asyncio.Runner(loop_factory=...)`. Read it as: iteration 1 runs
`main`'s first step; later ones run task steps and callbacks; when nothing is
ready the loop sleeps in `select()` until the next timer is due.

In [196]:
print(
    run_child(
        """
        import asyncio


        class TracingLoop(asyncio.SelectorEventLoop):
            iteration = 0

            def _run_once(self):
                self.iteration += 1
                ready, timers = len(self._ready), len(self._scheduled)
                print(f"iteration {self.iteration}: {ready=} {timers=}")
                super()._run_once()


        async def worker(name, delay):
            await asyncio.sleep(delay)
            print(f"    {name} woke up")


        async def main():
            asyncio.get_running_loop().call_soon(print, "    call_soon callback ran")
            await asyncio.gather(worker("fast", 0.01), worker("slow", 0.02))
            print("    main done")


        with asyncio.Runner(loop_factory=TracingLoop) as runner:
            runner.run(main())
        """
    )
)

iteration 1: ready=1 timers=0
iteration 2: ready=3 timers=0
    call_soon callback ran
iteration 3: ready=0 timers=2
iteration 4: ready=1 timers=1
    fast woke up
iteration 5: ready=1 timers=1
iteration 6: ready=0 timers=1
iteration 7: ready=1 timers=0
    slow woke up
iteration 8: ready=1 timers=0
iteration 9: ready=1 timers=0
    main done
iteration 10: ready=1 timers=0
iteration 11: ready=1 timers=0
iteration 12: ready=1 timers=0
iteration 13: ready=1 timers=0
iteration 14: ready=1 timers=0


The last few iterations are `Runner` shutting down (closing async generators
and the default executor). The counts depend on timing, so they can differ
slightly between runs.

### 5.3.7 Futures and Tasks

A **Future** is a slot for a result that does not exist yet, plus a list of
callbacks to run when it arrives. A **Task** is a Future that drives a
coroutine: its result is the coroutine's return value. You rarely create
Futures yourself (libraries do), but every `await` that really suspends ends at
one. The Future API, including its errors:

In [197]:
receipt = KERNEL_LOOP.create_future()
print(f"{receipt.done()=}  result() too early -> {raised_message(receipt.result)}")
receipt.set_result("R-1001")
print(f"{receipt.done()=} {receipt.result()=}")
print(f"set_result twice -> {raised_message(lambda: receipt.set_result('R-1002'))}")

declined = KERNEL_LOOP.create_future()
declined.set_exception(ValueError("card declined"))
print(f"{declined.exception()=}  result() -> {raised_message(declined.result)}")

abandoned = KERNEL_LOOP.create_future()
abandoned.cancel("customer left")
try:
    abandoned.result()
except asyncio.CancelledError as error:  # a BaseException, not an Exception
    print(f"{abandoned.cancelled()=}  result() -> CancelledError: {error}")
print(f"{issubclass(asyncio.Task, asyncio.Future)=}")

receipt.done()=False  result() too early -> InvalidStateError: Result is not set.
receipt.done()=True receipt.result()='R-1001'
set_result twice -> InvalidStateError: invalid state
declined.exception()=ValueError('card declined')  result() -> ValueError: card declined
abandoned.cancelled()=True  result() -> CancelledError: customer left
issubclass(asyncio.Task, asyncio.Future)=True


**`create_task` schedules; it does not run.** The new task's first step is put
on the ready queue and runs the next time the current task yields to the
loop. Give tasks names: they show up in debug messages and call graphs.

In [198]:
task_trace: list[str] = []


async def brew(kind: str) -> str:
    """Log the start, yield once, log the end."""
    task_trace.append(f"{kind}: starts")
    await asyncio.sleep(0)
    task_trace.append(f"{kind}: done")
    return kind


brew_task = asyncio.create_task(brew("masala chai"), name="brew-chai")
task_trace.append(f"created {brew_task.get_name()}, done={brew_task.done()}")
task_trace.append(f"awaited: {await brew_task!r}")
print("\n".join(task_trace))
print(f"{brew_task.get_coro().__qualname__=}")

created brew-chai, done=False
masala chai: starts
masala chai: done
awaited: 'masala chai'


brew_task.get_coro().__qualname__='brew'


**Eager tasks.** An *eager* task runs its first step synchronously inside
`create_task`, up to the first real suspension, like a JS `async function`
call. If the coroutine finishes without suspending (a cache hit), the task is
complete before `create_task` returns and never touches the loop. Per task:
`create_task(coro, eager_start=True)` (*new in 3.14*); for a whole loop:
`loop.set_task_factory(asyncio.eager_task_factory)` (*new in 3.12*).

In [199]:
task_trace.clear()


async def cached_price(label: str) -> int:
    """Return a cached price without ever suspending."""
    task_trace.append(f"{label}: body runs")
    return 99  # never suspends


if sys.version_info >= (3, 14):
    lazy_task = asyncio.create_task(cached_price("lazy"))
    eager_task = asyncio.create_task(cached_price("eager"), eager_start=True)
    task_trace.append(f"after create_task: {lazy_task.done()=} {eager_task.done()=}")
    await lazy_task
    print("\n".join(task_trace))
else:
    print("create_task(eager_start=True) needs Python 3.14+: skipped")

eager: body runs
after create_task: lazy_task.done()=False eager_task.done()=True
lazy: body runs


The eager factory pays off when most tasks finish without suspending. This
benchmark gathers tens of thousands of cache lookups (all hits) with the
default and the eager factory, each in a fresh loop in a child process.
Timings vary; the ratio is the point.

In [200]:
print(
    run_child(
        """
        import asyncio
        import os
        import time

        DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))
        CACHE = {f"sku-{number}": number for number in range(1000)}


        async def lookup(sku):
            if sku in CACHE:
                return CACHE[sku]  # cache hit: never suspends
            await asyncio.sleep(0)
            return -1


        async def main():
            skus = [f"sku-{number % 1000}" for number in range(50_000 * DEMO_SCALE)]
            started = time.perf_counter()
            await asyncio.gather(*(lookup(sku) for sku in skus))
            return time.perf_counter() - started


        timings = {}
        factories = {"default tasks": None, "eager_task_factory": asyncio.eager_task_factory}
        for label, factory in factories.items():
            with asyncio.Runner() as runner:
                runner.get_loop().set_task_factory(factory)
                timings[label] = runner.run(main())
            print(f"{label:<19} {timings[label]:.3f} s")
        print(f"eager speedup: {timings['default tasks'] / timings['eager_task_factory']:.1f}x")
        """
    )
)

default tasks       0.137 s
eager_task_factory  0.067 s
eager speedup: 2.0x


**Keep a reference to every task.** The loop holds only *weak* references to
tasks. A fire-and-forget task waiting on something that nothing else
references can be garbage-collected mid-flight: its `finally` runs at a
random moment and asyncio reports "Task was destroyed but it is pending!".
The next cell provokes it deliberately (with a temporary exception handler so
the report is printed here instead of logged):

In [201]:
loop_reports: list[str] = []
previous_handler = KERNEL_LOOP.get_exception_handler()
KERNEL_LOOP.set_exception_handler(lambda loop, context: loop_reports.append(context["message"]))


async def send_receipt_email() -> None:
    """Waits on a Future that only this coroutine references."""
    reply = KERNEL_LOOP.create_future()
    try:
        await reply
    finally:
        loop_reports.append("send_receipt_email: finally ran")


asyncio.create_task(send_receipt_email())  # no reference kept
await asyncio.sleep(0)  # let it start and park on its Future
gc.collect()  # task -> coroutine -> Future -> task is an unreachable cycle
await asyncio.sleep(0)
KERNEL_LOOP.set_exception_handler(previous_handler)
print("\n".join(loop_reports))

send_receipt_email: finally ran
Task was destroyed but it is pending!


The fix: keep tasks in a set and discard them when they finish (the pattern
from the `asyncio.create_task` docs), or better, start them in a `TaskGroup`
(5.4.1), which holds references for you.

In [202]:
background_tasks: set[asyncio.Task[None]] = set()


def fire_and_forget(coroutine: Coroutine[Any, Any, None]) -> asyncio.Task[None]:
    """Start a task and keep a strong reference until it finishes."""
    task = asyncio.create_task(coroutine)
    background_tasks.add(task)
    task.add_done_callback(background_tasks.discard)
    return task


loop_reports.clear()
kept_task = fire_and_forget(send_receipt_email())
await asyncio.sleep(0)
gc.collect()
print(f"after gc: {kept_task.done()=}, {len(background_tasks)} task(s) referenced")
kept_task.cancel()
await asyncio.sleep(0)
print(f"after cancel: {kept_task.cancelled()=}, {len(background_tasks)} referenced; {loop_reports}")

after gc: kept_task.done()=False, 1 task(s) referenced
after cancel: kept_task.cancelled()=True, 1 referenced; ['send_receipt_email: finally ran']


### 5.3.8 Core API

| Call | What it does |
| --- | --- |
| `asyncio.run(main())` | create a new loop, run `main()` to completion, cancel leftover tasks, shut down async generators and the default executor, close the loop |
| `asyncio.Runner()` (*new in 3.11*) | the same, but several `runner.run(...)` calls share one loop; takes `debug=` and `loop_factory=` |
| `await coro` | run `coro` inside the current task (sequential) |
| `asyncio.create_task(coro)` | run `coro` concurrently as a new task |
| `asyncio.gather(*aws)` | run all concurrently; results in argument order |
| `asyncio.sleep(0)` | yield to the loop once |
| `asyncio.wait_for(aw, timeout)` | `TimeoutError` (and cancel `aw`) if it takes too long |

**`asyncio.run` in Jupyter.** It refuses to start a second loop inside the
kernel's running loop. The coroutine was never started, so it is closed by
hand to avoid the "never awaited" warning:

In [203]:
async def main_entry() -> str:
    """Yield once, then return done."""
    await asyncio.sleep(0)
    return "done"


entry = main_entry()
try:
    asyncio.run(entry)
except RuntimeError as error:
    print(f"{type(error).__name__}: {error}")
finally:
    entry.close()

RuntimeError: asyncio.run() cannot be called from a running event loop


In a script, `asyncio.run` is the one place where sync code enters async code:
the entry point, under the `__main__` guard. Each call builds and closes a
fresh loop, and outside it there is no running loop:

In [204]:
print(
    run_child(
        """
        import asyncio


        async def fetch(name, seconds):
            await asyncio.sleep(seconds)
            return name


        async def main():
            loop = asyncio.get_running_loop()
            print(f"inside asyncio.run: {type(loop).__name__}, debug={loop.get_debug()}")
            return await asyncio.gather(fetch("menu", 0.05), fetch("stock", 0.05))


        if __name__ == "__main__":
            print(asyncio.run(main()))
            print(asyncio.run(main(), debug=True))
            try:
                asyncio.get_running_loop()
            except RuntimeError as error:
                print(f"after asyncio.run: RuntimeError: {error}")
        """
    )
)

inside asyncio.run: _UnixSelectorEventLoop, debug=False
['menu', 'stock']
inside asyncio.run: _UnixSelectorEventLoop, debug=True
['menu', 'stock']
after asyncio.run: RuntimeError: no running event loop


`asyncio.Runner` keeps one loop across several `run()` calls (useful for a
sync program that calls async code at a few points, or for tests). Two
`asyncio.run` calls get two different loops:

In [205]:
print(
    run_child(
        """
        import asyncio


        async def current_loop():
            return asyncio.get_running_loop()


        with asyncio.Runner(debug=False) as runner:
            first, second = runner.run(current_loop()), runner.run(current_loop())
            print(f"Runner: same loop for both calls: {first is second}")
            print(f"Runner: loop closed inside the with block: {first.is_closed()}")
        print(f"Runner: loop closed after the with block: {first.is_closed()}")
        shared = asyncio.run(current_loop()) is asyncio.run(current_loop())
        print(f"two asyncio.run calls share a loop: {shared}")
        """
    )
)

Runner: same loop for both calls: True
Runner: loop closed inside the with block: False
Runner: loop closed after the with block: True
two asyncio.run calls share a loop: False


**Sequential `await` vs `create_task` vs `gather`.** Three 0.2 s lookups.
Awaiting them one by one takes 0.6 s; starting them as tasks first, or
gathering them, overlaps the waits:

In [206]:
async def fetch_menu(section: str, seconds: float = 0.2) -> str:
    """Pretend to fetch one menu section."""
    await asyncio.sleep(seconds)
    return section


sections = ["tea", "snacks", "desserts"]
with Timer("sequential awaits") as sequential_timer:
    sequential_results = []
    for section in sections:  # one at a time
        sequential_results.append(await fetch_menu(section))
with Timer("create_task, then await each") as task_timer:
    menu_tasks = [asyncio.create_task(fetch_menu(section)) for section in sections]
    task_results = []
    for task in menu_tasks:  # all three are already running
        task_results.append(await task)
with Timer("gather") as gather_timer:
    gather_results = await asyncio.gather(*(fetch_menu(section) for section in sections))
print(f"same results: {sequential_results == task_results == gather_results}")
print(
    f"sequential / gather: {sequential_timer.elapsed_seconds / gather_timer.elapsed_seconds:.1f}x"
)

sequential awaits took 602.07 ms
create_task, then await each took 200.77 ms


gather took 200.62 ms
same results: True
sequential / gather: 3.0x


**`sleep(0)` and the "await does not always yield" rule.** Awaiting a
coroutine that never reaches a pending Future runs it inline, like a normal
function call: no other task gets a turn. `asyncio.sleep(0)` is the explicit
"let others run" (it yields a bare `None` to the Task, which re-queues itself
with `call_soon`). In JS, by contrast, *every* `await` goes through the
microtask queue.

In [207]:
yield_trace: list[str] = []


async def no_suspend() -> None:
    """Awaitable, but never suspends."""


async def worker(name: str, pause: Callable[[], Coroutine[Any, Any, None]]) -> None:
    """Log two steps, awaiting pause() after each."""
    for step in range(2):
        yield_trace.append(f"{name}{step}")
        await pause()


await asyncio.gather(worker("A", no_suspend), worker("B", no_suspend))
print(f"await no_suspend():  {' '.join(yield_trace)}")
yield_trace.clear()
await asyncio.gather(worker("A", lambda: asyncio.sleep(0)), worker("B", lambda: asyncio.sleep(0)))
print(f"await sleep(0):      {' '.join(yield_trace)}")

await no_suspend():  A0 A1 B0 B1
await sleep(0):      A0 B0 A1 B1


**`wait_for`** puts a timeout on one awaitable. On timeout it cancels the
inner task (its `except CancelledError` / `finally` runs), waits for it to
finish cancelling, then raises `TimeoutError`. `timeout=None` waits forever.
(Unit 5.4.2 covers `asyncio.timeout`, the context-manager form.)

In [208]:
async def slow_quote() -> int:
    """Sleep a second unless wait_for cancels it first."""
    try:
        await asyncio.sleep(1)
        return 100
    except asyncio.CancelledError:
        print("  slow_quote: cancelled by wait_for")
        raise  # always re-raise CancelledError


try:
    await asyncio.wait_for(slow_quote(), timeout=0.05)
except TimeoutError as error:
    print(f"{type(error).__name__} after 0.05 s")
print(f"{await asyncio.wait_for(fetch_menu('tea', 0.01), timeout=1)=}")
print(f"{await asyncio.wait_for(fetch_menu('snacks', 0.01), timeout=None)=}")

  slow_quote: cancelled by wait_for
TimeoutError after 0.05 s
await asyncio.wait_for(fetch_menu('tea', 0.01), timeout=1)='tea'
await asyncio.wait_for(fetch_menu('snacks', 0.01), timeout=None)='snacks'


**Top-level `await` in Jupyter.** ipykernel runs its message handling as
tasks on its own asyncio loop. A cell *without* `await` runs synchronously
inside the kernel's `shell_main` task (that is why
`asyncio.get_running_loop()` worked in the setup cell). A cell *with* a
top-level `await` is compiled as a coroutine (`%autoawait` is on by default)
and awaited by that task, so it can `await` directly, and tasks created in one
cell keep running after the cell finishes. In a `.py` script none of that
exists: write `async def main()` and call `asyncio.run(main())`.

In [209]:
await asyncio.sleep(0)  # a top-level await: IPython runs this cell as a coroutine
cell_frames = [entry.frame.f_code.co_name for entry in asyncio.capture_call_graph().call_stack]
print(f"running loop: {type(asyncio.get_running_loop()).__name__}")
print(f"this cell runs inside a task: {asyncio.current_task() is not None}")
print(f"innermost frames of this cell's task: {cell_frames[:3]}")
print(
    f"tasks alive on the kernel loop: {len(asyncio.all_tasks())} (ipykernel's own tasks included)"
)

running loop: _UnixSelectorEventLoop
this cell runs inside a task: True
innermost frames of this cell's task: ['<module>', 'run_code', 'run_ast_nodes']
tasks alive on the kernel loop: 3 (ipykernel's own tasks included)


### 5.3.9 JS bridge, in depth

**Node's event loop** (libuv) goes round a fixed cycle of **phases**, each
with its own queue:

| Phase | Runs |
| --- | --- |
| timers | expired `setTimeout` / `setInterval` callbacks |
| pending callbacks | some deferred system callbacks (e.g. TCP errors) |
| idle, prepare | internal |
| poll | wait for I/O (epoll/kqueue/IOCP) and run I/O callbacks |
| check | `setImmediate` callbacks |
| close callbacks | `'close'` events |

On top of that, Node has two queues that are **drained completely after every
single callback** (since Node 11): first the `process.nextTick` queue, then
the **microtask** queue (`Promise` reactions, `await` continuations,
`queueMicrotask`). Microtasks always run before the loop moves on.

**asyncio has no phases and no microtask queue.** One `_run_once` iteration
selects, appends ready I/O callbacks and due timers to **one FIFO ready
queue**, and runs the handles that were queued at the start. A Future
callback, a Task step, a `call_soon` callback and a due timer are all just
handles in that queue.

```mermaid
flowchart LR
    subgraph Node["Node.js (libuv)"]
        direction TB
        T[timers] --> P[pending] --> I[idle / prepare] --> PO[poll: I/O] --> C["check: setImmediate"] --> CL[close] --> T
        M["after EVERY callback: drain nextTick queue, then microtasks"]
    end
    subgraph Py["asyncio"]
        direction TB
        S["select(timeout)"] --> E["append I/O callbacks"] --> D["append due timers"] --> R["run the ntodo handles queued now"] --> S
    end
```

<details><summary>ASCII version of the diagram</summary>

```text
Node.js (libuv)                               asyncio
+--> timers ----> pending ----> idle/prepare   +--> select(timeout)
|                                  |           |        |
|    close <---- check <---- poll (I/O)        |    append ready I/O callbacks
|      |      (setImmediate)                   |        |
+------+                                       |    append due timers
after EVERY callback: drain process.nextTick,  |        |
then drain ALL microtasks (promises, await)    +--- run the handles queued now (FIFO)
```

</details>

| Node.js | asyncio |
| --- | --- |
| libuv thread pool (`UV_THREADPOOL_SIZE`, default 4) runs `fs`, `dns.lookup`, `crypto.pbkdf2`, `zlib` | the loop's default `ThreadPoolExecutor` (`min(32, cpus + 4)` threads) runs whatever you pass to `asyncio.to_thread` / `run_in_executor`; asyncio has **no async file I/O** at all (aiofiles uses threads) |
| network sockets use epoll/kqueue directly | the same |
| `Promise` (always eager, never cancellable) | `Future` (result slot, cancellable) + `Task` (drives a coroutine) |
| `async function` call starts running at once | coroutine call is lazy; `create_task` schedules it |

**Eager Promises vs lazy coroutines**, side by side. In JS, `placeOrder()`
prints immediately and the promise is already running. In Python nothing
happens until the coroutine is awaited or wrapped in a task:

```javascript
async function placeOrder() { console.log("  placeOrder body runs"); return "order-1"; }
console.log("calling placeOrder() ...");
const pending = placeOrder();          // prints "placeOrder body runs" right here
console.log("... returned a Promise that is already settled");
```

In [210]:
async def place_order() -> str:
    """Run the body when awaited (a lazy coroutine)."""
    print("  place_order body runs")
    return "order-1"


print("calling place_order() ...")
pending_order = place_order()
print("... returned a coroutine; nothing ran")
order_task = asyncio.create_task(pending_order)
print(f"create_task scheduled it: done={order_task.done()}")
print(f"{await order_task=}")

calling place_order() ...
... returned a coroutine; nothing ran
create_task scheduled it: done=False
  place_order body runs
await order_task='order-1'


#### Side-by-side ordering demo

The same program in both languages: a script that logs, schedules a 0 ms
timer (which starts a file read), queues a promise reaction and a
`nextTick`, calls an async function, and logs again. In Node, a
`setTimeout(0)` and a `setImmediate` scheduled from the *main script* can run
in either order, so they are scheduled from inside the I/O callback, where
the order is guaranteed.

```javascript
const fs = require("node:fs");

async function work() {
  console.log("work() body");
  await null;
  console.log("work() after await");
}

console.log("script start");
setTimeout(() => {
  console.log("setTimeout 0");
  fs.readFile(__filename, () => {
    console.log("readFile callback");
    setTimeout(() => console.log("setTimeout 0 from I/O"), 0);
    setImmediate(() => console.log("setImmediate from I/O"));
  });
}, 0);
Promise.resolve().then(() => console.log("promise.then"));
process.nextTick(() => console.log("nextTick"));
work();
console.log("script end");
```

Known output (Node v24.21.0, the same on every run):

```text
script start
work() body
script end
nextTick
promise.then
work() after await
setTimeout 0
readFile callback
setImmediate from I/O
setTimeout 0 from I/O
```

The Python version maps each line to its closest equivalent: `setTimeout(0)`
→ `loop.call_later(0, ...)`, `fs.readFile` → `asyncio.to_thread(read_bytes)`
(a thread, like libuv's pool), `setImmediate` → `loop.call_soon`,
`Promise.resolve().then(cb)` → a resolved Future with a done-callback,
`process.nextTick` → `loop.call_soon` (there is no closer match), calling
`work()` → `asyncio.create_task(work())`, `await null` →
`await asyncio.sleep(0)`. It runs as a script so that, like Node, it starts
on a fresh loop:

In [211]:
PYTHON_ORDER_SOURCE = """
import asyncio
from pathlib import Path


async def work():
    print("work() body")
    await asyncio.sleep(0)
    print("work() after await")


def on_file_read(_):
    print("to_thread(read) done callback")
    loop.call_later(0, print, "call_later 0 from I/O")
    loop.call_soon(print, "call_soon from I/O")


def on_timer():
    print("call_later 0")
    reading = asyncio.ensure_future(asyncio.to_thread(Path(__file__).read_bytes))
    reading.add_done_callback(on_file_read)


async def main():
    global loop
    loop = asyncio.get_running_loop()
    print("script start")
    loop.call_later(0, on_timer)
    resolved = loop.create_future()
    resolved.add_done_callback(lambda _: print("future done callback"))
    resolved.set_result(None)
    loop.call_soon(print, "call_soon")
    asyncio.create_task(work())
    print("script end")
    await asyncio.sleep(0.2)


asyncio.run(main())
"""

Run it in a fresh interpreter (so it starts on a new loop, like `node script.js`):

In [212]:
python_order_output = run_child(PYTHON_ORDER_SOURCE)
print(python_order_output)

script start
script end
future done callback
call_soon
work() body
call_later 0
work() after await
to_thread(read) done callback
call_soon from I/O
call_later 0 from I/O


Every difference, line by line (the Python output above is also the same on
every run):

| # | Node | Python | Why |
| --- | --- | --- | --- |
| 2 | `work() body` prints **before** `script end` | `work() body` prints **after** `script end` | JS async functions run synchronously up to the first `await`; `create_task` only schedules the coroutine. (`create_task(..., eager_start=True)` would match Node.) |
| 3–4 | `nextTick` runs before `promise.then` | the Future callback runs before `call_soon` | Node drains the nextTick queue before the microtask queue. asyncio has one FIFO: `set_result` queued the callback first, then `call_soon` queued its print |
| 5–6 | `work() after await` runs **before** `setTimeout 0` | `call_later 0` runs **before** `work() after await` | In Node, the `await` continuation is a microtask, and all microtasks finish before the timers phase. In asyncio, the 0 s timer was already due when the iteration started, so it joined the ready queue in that same iteration, right behind the `work()` step. The continuation after `sleep(0)` is queued only when `work()` yields, so it runs one iteration later |
| 7 | `readFile callback` (poll phase) | `to_thread(read) done callback` | both are I/O completions arriving from a thread pool; asyncio delivers it as a Future callback through the self-pipe and the selector |
| 8–9 | `setImmediate` beats `setTimeout 0` (check phase comes right after poll) | `call_soon` beats `call_later(0)` | Node: phase order. Python: the `call_soon` handle is already in the ready queue when the timer becomes due and is appended after it |

The JS program and its known output, as strings for the live run below:

In [213]:
JS_ORDER_SOURCE = """
const fs = require("node:fs");

async function work() {
  console.log("work() body");
  await null;
  console.log("work() after await");
}

console.log("script start");
setTimeout(() => {
  console.log("setTimeout 0");
  fs.readFile(__filename, () => {
    console.log("readFile callback");
    setTimeout(() => console.log("setTimeout 0 from I/O"), 0);
    setImmediate(() => console.log("setImmediate from I/O"));
  });
}, 0);
Promise.resolve().then(() => console.log("promise.then"));
process.nextTick(() => console.log("nextTick"));
work();
console.log("script end");
"""
JS_ORDER_OUTPUT = """
script start
work() body
script end
nextTick
promise.then
work() after await
setTimeout 0
readFile callback
setImmediate from I/O
setTimeout 0 from I/O
"""

Run the JS version with `node` if it is installed (otherwise print a skip message)
and compare its output with the known output:

In [214]:
node_path = shutil.which("node")
if node_path is None:
    print("node not found on PATH: skipping the live JS run (the known output is above)")
else:
    js_script = PHASE5_DIR / "order_demo.js"
    js_script.write_text(textwrap.dedent(JS_ORDER_SOURCE).lstrip(), encoding="utf-8")
    node_command = [node_path, str(js_script)]
    node_run = subprocess.run(node_command, capture_output=True, text=True, timeout=30)
    version_run = subprocess.run([node_path, "--version"], capture_output=True, text=True)
    node_version = version_run.stdout.strip()
    print(node_run.stdout.rstrip())
    matches = node_run.stdout.strip() == JS_ORDER_OUTPUT.strip()
    print(f"node {node_version}: matches the known output: {matches}")

script start
work() body
script end
nextTick
promise.then
work() after await
setTimeout 0
readFile callback
setImmediate from I/O
setTimeout 0 from I/O
node v24.21.0: matches the known output: True


#### Microtasks vs the ready queue

Because Node drains *all* microtasks before moving to the next phase, a chain
of microtasks that keeps queueing more starves timers and I/O completely:

```javascript
let spins = 0;
setTimeout(() => console.log(`setTimeout ran after ${spins} microtasks`), 0);
function spin() { spins += 1; if (spins < 100000) queueMicrotask(spin); }
spin();
// Output (Node v24.21.0): setTimeout ran after 100000 microtasks
```

asyncio runs only the handles that were queued when the iteration started,
so a self-re-queueing `call_soon` chain is interleaved with timers and I/O:
the 0 s timer gets its turn after a couple of spins.

In [215]:
spin_count = 0
timer_fired = KERNEL_LOOP.create_future()


def spin() -> None:
    """Re-queue itself with call_soon, like the JS queueMicrotask chain."""
    global spin_count
    spin_count += 1
    if spin_count < 100_000 and not timer_fired.done():
        KERNEL_LOOP.call_soon(spin)


KERNEL_LOOP.call_later(0, lambda: timer_fired.set_result(spin_count))
spin()
print(f"call_later(0) ran after {await timer_fired} call_soon spins")

call_later(0) ran after 2 call_soon spins


#### Promise combinators and timers

| JavaScript | asyncio | Note |
| --- | --- | --- |
| `Promise.all(ps)` | `await asyncio.gather(*aws)` or a `TaskGroup` | first failure propagates; `gather` does not cancel the others, `TaskGroup` does (5.4.1) |
| `Promise.allSettled(ps)` | `await asyncio.gather(*aws, return_exceptions=True)` | exceptions come back as values |
| `Promise.race(ps)` | `asyncio.wait(tasks, return_when=FIRST_COMPLETED)` | you can (and should) cancel the losers; Lab 5.3 exercise 4 builds `race()` |
| `Promise.any(ps)` | iterate `asyncio.as_completed(aws)` and return the first success | |
| `setTimeout(fn, ms)` / `clearTimeout(t)` | `handle = loop.call_later(seconds, fn)` / `handle.cancel()` | inside a coroutine, usually just `await asyncio.sleep(seconds)` |
| `setImmediate(fn)` | `loop.call_soon(fn)` | |
| `new Promise(resolve => ...)` | `future = loop.create_future()`; later `future.set_result(value)` | |

`Promise.all` → `gather`: results come back in argument order, whatever the
completion order. If one fails, `gather` raises that exception, and the others
keep running:

In [216]:
async def stock_level(sku: str, seconds: float, fail: bool = False) -> int:
    """Return a fake stock level after a delay, or raise LookupError."""
    await asyncio.sleep(seconds)
    if fail:
        raise LookupError(f"no such sku: {sku}")
    return len(sku) * 10


print(f"{await asyncio.gather(stock_level('chai', 0.03), stock_level('vada', 0.01))=}")
try:
    await asyncio.gather(stock_level("chai", 0.01), stock_level("unknown", 0.01, fail=True))
except LookupError as error:
    print(f"gather raised the first failure: {type(error).__name__}: {error}")

await asyncio.gather(stock_level('chai', 0.03), stock_level('vada', 0.01))=[40, 40]
gather raised the first failure: LookupError: no such sku: unknown


`Promise.allSettled` → `gather(..., return_exceptions=True)`. The helper
reshapes the result into JS's `{status, value | reason}` records:

In [217]:
def settled(results: list[object]) -> list[dict[str, object]]:
    """Shape gather(return_exceptions=True) results like Promise.allSettled."""
    return [
        {"status": "rejected", "reason": repr(result)}
        if isinstance(result, BaseException)
        else {"status": "fulfilled", "value": result}
        for result in results
    ]


settled_results = await asyncio.gather(
    stock_level("chai", 0.01), stock_level("ghost", 0.01, fail=True), return_exceptions=True
)
for record in settled(settled_results):
    print(record)

{'status': 'fulfilled', 'value': 40}
{'status': 'rejected', 'reason': "LookupError('no such sku: ghost')"}


`Promise.race` → `asyncio.wait(..., return_when=FIRST_COMPLETED)`, which
returns `(done, pending)` sets of tasks as soon as one finishes. The bite:
the losers are **still running** unless you cancel them (JS cannot cancel
them at all). Lab 5.3 exercise 4 turns this into a proper `race()`.

In [218]:
mirror_tasks = [
    asyncio.create_task(stock_level(name, seconds), name=name)
    for name, seconds in (("mirror-slow", 0.2), ("mirror-fast", 0.01), ("mirror-mid", 0.1))
]
done, pending = await asyncio.wait(mirror_tasks, return_when=asyncio.FIRST_COMPLETED)
winner = done.pop()
print(f"winner: {winner.get_name()} -> {winner.result()}")
print(f"still running: {sorted(task.get_name() for task in pending if not task.done())}")
for task in pending:
    task.cancel()  # clean up the losers
await asyncio.wait(pending)
print(f"after cleanup: all losers cancelled = {all(task.cancelled() for task in pending)}")

winner: mirror-fast -> 110
still running: ['mirror-mid', 'mirror-slow']
after cleanup: all losers cancelled = True


`setTimeout` → `loop.call_later`; `clearTimeout` → `handle.cancel()`;
`setInterval` → re-arm from the callback (or, in a coroutine, a loop around
`await asyncio.sleep(...)`):

In [219]:
ticks: list[str] = []
interval_done = KERNEL_LOOP.create_future()
interval_start = KERNEL_LOOP.time()


def every_20_ms(remaining: int) -> None:
    """setInterval stand-in: do the work, then schedule the next tick."""
    ticks.append(f"+{(KERNEL_LOOP.time() - interval_start) * 1000:.0f} ms")
    if remaining > 1:
        KERNEL_LOOP.call_later(0.02, every_20_ms, remaining - 1)
    else:
        interval_done.set_result(None)


cancelled_timeout = KERNEL_LOOP.call_later(0.01, ticks.append, "never: cancelled")
cancelled_timeout.cancel()  # clearTimeout
KERNEL_LOOP.call_later(0.02, every_20_ms, 3)  # setTimeout
await interval_done
print(f"ticks: {ticks}")

ticks: ['+21 ms', '+41 ms', '+61 ms']


`setImmediate` → `loop.call_soon`: run a callback on the next loop iteration,
after the code that is running now:

In [220]:
immediate_log: list[str] = []
KERNEL_LOOP.call_soon(immediate_log.append, "call_soon callback")
immediate_log.append("current code continues")
await asyncio.sleep(0)
print(immediate_log)

['current code continues', 'call_soon callback']


#### Promise vs Future/Task

| | JS `Promise` | asyncio `Future` / `Task` |
| --- | --- | --- |
| Start | the executor / async body runs immediately | a Future is passive; a Task starts on the next iteration (unless eager) |
| Settle twice | silently ignored | `InvalidStateError` (5.3.7) |
| Cancel | impossible (use `AbortController` by convention) | `task.cancel()` throws `CancelledError` into the coroutine |
| Continuations | `.then(cb)`: a microtask | `add_done_callback(cb)`: a `call_soon` handle; receives the Future |
| Unhandled failure | `unhandledRejection` event | "Task exception was never retrieved", logged when the Task is garbage-collected |
| Await twice | fine, same value | fine, same value |

`then` vs `add_done_callback`: the callback receives the Future itself, not
its value, and its return value is ignored (no chaining):

In [221]:
settled_order = KERNEL_LOOP.create_future()
settled_order.add_done_callback(
    lambda future: print(f"  done-callback got a {type(future).__name__}: {future.result()!r}")
)
settled_order.set_result("order-9")
print(f"{await settled_order=} {await settled_order=}  (awaiting twice is fine)")

await settled_order='order-9' await settled_order='order-9'  (awaiting twice is fine)
  done-callback got a Future: 'order-9'


#### The libuv thread pool vs asyncio's default executor

Node runs some blocking work (file system, `dns.lookup`, `crypto.pbkdf2`,
`zlib`) on a pool of **4** threads by default. Eight `pbkdf2` calls finish in
two waves:

```javascript
const crypto = require("node:crypto");
const start = performance.now();
for (let i = 0; i < 8; i++) {
  crypto.pbkdf2("pw", "salt", 200000, 32, "sha256", () =>
    console.log(`hash ${i} done at ${Math.round(performance.now() - start)} ms`));
}
// Typical output (Node v24.21.0, 16 cores): hashes 0-3 at ~28 ms, hashes 4-7 at ~60 ms
```

asyncio never moves work to a thread on its own: you ask for it with
`asyncio.to_thread(func, ...)` (or `loop.run_in_executor`). The loop's default
executor is a `ThreadPoolExecutor` with `min(32, os.process_cpu_count() + 4)`
workers, created on first use. `hashlib.pbkdf2_hmac` releases the GIL, so
eight of them run in parallel:

In [222]:
import hashlib


def stretch_password(index: int) -> int:
    """CPU-heavy C code that releases the GIL, like crypto.pbkdf2."""
    hashlib.pbkdf2_hmac("sha256", b"pw", b"salt", 200_000 * DEMO_SCALE)
    return index


pool_started = time.perf_counter()
hash_finish_ms: list[str] = []
for finished in asyncio.as_completed(
    [asyncio.to_thread(stretch_password, index) for index in range(8)]
):
    hash_finish_ms.append(f"{await finished}@{(time.perf_counter() - pool_started) * 1000:.0f}ms")
print(" ".join(hash_finish_ms))
executor = KERNEL_LOOP._default_executor
print(f"default executor: {type(executor).__name__} with {executor._max_workers} workers")

0@28ms 2@28ms 6@28ms 1@28ms 5@28ms 4@29ms 3@29ms 7@29ms
default executor: ThreadPoolExecutor with 20 workers


#### `worker_threads` vs processes and subinterpreters

| Node | Python | Shared memory | Start-up | Use for |
| --- | --- | --- | --- | --- |
| the main event loop | one asyncio loop per thread | — | — | many concurrent I/O waits |
| libuv thread pool | `asyncio.to_thread` / `ThreadPoolExecutor` | yes (one heap) | ~50 µs | blocking libraries, C code that releases the GIL |
| `worker_threads` (separate V8 isolate per thread) | subinterpreters (`InterpreterPoolExecutor`, Unit 5.2.6), or processes | no (messages; `SharedArrayBuffer` / `shared_memory`) | ms | CPU-bound Python code |
| `child_process` / `cluster` | `ProcessPoolExecutor`, `multiprocessing` (Unit 5.2) | no | tens of ms | CPU-bound work, isolation |

From async code, CPU-bound work goes to a process pool with
`await loop.run_in_executor(process_pool, func, *args)` (Unit 5.4.6), the
equivalent of posting a message to a worker and awaiting its reply.

### 5.3.10 Function colors

Bob Nystrom's essay *What Color is Your Function?* (2015) names the problem:
async ("red") functions can only be awaited from other async functions, and
sync ("blue") functions cannot wait for red ones. JS and Python share it.
The rules at the boundaries:

| From | To | How | Cost / risk |
| --- | --- | --- | --- |
| async | fast sync code | just call it | none |
| async | blocking sync code (`time.sleep`, `requests`, heavy CPU) | `await asyncio.to_thread(func)` / `run_in_executor` | a thread (or a process for CPU work); calling it directly freezes the loop |
| async | async | `await` | none |
| sync | async, at the program's entry | `asyncio.run(main())` / `asyncio.Runner` | one loop per call; fails if a loop is already running |
| sync, in another thread | async on a running loop | `asyncio.run_coroutine_threadsafe(coro, loop)` | returns a `concurrent.futures.Future` to block on (5.4.11) |
| sync | async, called by mistake | — | you get a coroutine object, not a result, plus a "never awaited" warning |

Colors also **spread upwards**: once a low-level function needs to `await`
(say, a database call), every caller must become `async` too, up to the
entry point. Decide early which layers are async.

The classic mistake: a sync function calls an async one and gets a coroutine
object instead of a number:

In [223]:
async def get_stock(sku: str) -> int:
    """Return a fixed stock level after one yield."""
    await asyncio.sleep(0)
    return 12


def stock_line(sku: str) -> str:
    """BUG: a sync function cannot await, so it formats a coroutine object."""
    stock = get_stock(sku)
    return f"{sku}: {stock}"


with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    print(re.sub(r" at 0x[0-9a-f]+", "", stock_line("chai")))
    gc.collect()
print([f"{caught.category.__name__}: {caught.message}" for caught in caught_warnings])

chai: <coroutine object get_stock>
["RuntimeWarning: coroutine 'get_stock' was never awaited"]


**Sync code calling into a running loop.** Legacy sync code running in a
worker thread can submit a coroutine to the loop with
`run_coroutine_threadsafe` and block on the returned
`concurrent.futures.Future`. (The kernel's loop is not blocked: the sync code
runs in a thread via `to_thread`.)

In [224]:
def legacy_stock_report(loop: asyncio.AbstractEventLoop) -> str:
    """Sync code in a worker thread asking the event loop for a value."""
    stock_future = asyncio.run_coroutine_threadsafe(get_stock("chai"), loop)
    return f"chai: {stock_future.result(timeout=1)} (computed on the loop thread)"


print(await asyncio.to_thread(legacy_stock_report, KERNEL_LOOP))

chai: 12 (computed on the loop thread)


**Async code calling blocking sync code.** Calling it directly stalls every
other task; `to_thread` keeps the loop free. Three 0.1 s blocking calls:

In [225]:
def legacy_price_lookup(sku: str) -> int:
    """A blocking library call (think requests.get)."""
    time.sleep(0.1)
    return len(sku)


async def price_directly(sku: str) -> int:
    """Call the blocking lookup on the loop thread (blocks the loop)."""
    return legacy_price_lookup(sku)  # blocks the loop


async def price_in_thread(sku: str) -> int:
    """Run the blocking lookup in a worker thread."""
    return await asyncio.to_thread(legacy_price_lookup, sku)


with Timer("blocking calls made directly in async code") as direct_timer:
    await asyncio.gather(*(price_directly(sku) for sku in ("chai", "vada", "idli")))
with Timer("the same calls via asyncio.to_thread") as thread_timer:
    await asyncio.gather(*(price_in_thread(sku) for sku in ("chai", "vada", "idli")))
print(f"ratio: {direct_timer.elapsed_seconds / thread_timer.elapsed_seconds:.1f}x")

blocking calls made directly in async code took 301.08 ms
the same calls via asyncio.to_thread took 101.94 ms
ratio: 3.0x


#### Predict the output

**Predict the output.**

```python
async def say(word):
    print(word)

async def main():
    asyncio.create_task(say("A"))
    print("B")
    await asyncio.sleep(0)
    print("C")

await main()
```

<details><summary>Answer</summary>

`B`, `A`, `C`. `create_task` only puts the new task's first step on the
ready queue. `main` keeps running and prints `B`; at `sleep(0)` it yields,
the loop runs the queued task (`A`), then `main` resumes (`C`). In JS,
calling `say("A")` would print `A` first.

</details>

Run the next cell to check your prediction.

In [226]:
async def say(word: str) -> None:
    """Print the word."""
    print(word)


async def main() -> None:
    """Schedule say('A'), print B, yield once, print C."""
    asyncio.create_task(say("A"))
    print("B")
    await asyncio.sleep(0)
    print("C")


await main()

B
A
C


**Predict the output.**

```python
finished = []

async def step(name, seconds):
    await asyncio.sleep(seconds)
    finished.append(name)
    return name

results = await asyncio.gather(step("slow", 0.02), step("fast", 0.01))
print(finished, results)
```

<details><summary>Answer</summary>

`['fast', 'slow'] ['slow', 'fast']`. The tasks *finish* in time order, so
`finished` is fast-first, but `gather` returns results in *argument* order,
exactly like `Promise.all`.

</details>

Run the next cell to check your prediction.

In [227]:
finished_steps: list[str] = []


async def step(name: str, seconds: float) -> str:
    """Sleep, record the step as finished, return its name."""
    await asyncio.sleep(seconds)
    finished_steps.append(name)
    return name


step_results = await asyncio.gather(step("slow", 0.02), step("fast", 0.01))
print(finished_steps, step_results)

['fast', 'slow'] ['slow', 'fast']


**Predict the output.**

```python
async def compute():
    return 42

coro = compute()
print(type(coro).__name__, inspect.getcoroutinestate(coro))
print(await coro)
coro.send(None)
```

<details><summary>Answer</summary>

`coroutine CORO_CREATED`, then `42`, then
`RuntimeError: cannot reuse already awaited coroutine`. A coroutine object
is single-use, like a generator: once it has run to completion it cannot be
restarted. (A JS Promise, by contrast, can be awaited any number of times;
in Python, wrap the coroutine in a Task if several places need its result.)

</details>

Run the next cell to check your prediction.

In [228]:
async def compute() -> int:
    """Return 42."""
    return 42


single_use = compute()
print(type(single_use).__name__, inspect.getcoroutinestate(single_use))
print(await single_use)
print(raised_message(lambda: single_use.send(None)))

coroutine CORO_CREATED
42
RuntimeError: cannot reuse already awaited coroutine


### Lab 5.3

Six exercises. Replace each `raise NotImplementedError`, then run the check
cell. Until you finish, the checks print FAIL; that is expected. Exercise 2
uses `ToyLoop` from 5.3.3 and exercise 1 uses `Ask` from 5.3.4: run those
sections first.

In [229]:
import contextlib
import functools
from collections.abc import Awaitable


async def outcome(awaitable: Awaitable[object]) -> object:
    """Await and return the result, or "raised ErrorName". Never raises an Exception."""
    try:
        return await awaitable
    except Exception as error:
        return f"raised {type(error).__name__}"


@contextlib.contextmanager
def quiet_unawaited() -> Generator[None]:
    """Hide "coroutine ... was never awaited" warnings caused by unfinished stubs."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", "coroutine .* was never awaited", RuntimeWarning)
        yield
        gc.collect()


def run_toy(main: Callable[[ToyLoop], ToyCoroutine]) -> tuple[object, float]:
    """Run main(loop) on a fresh ToyLoop; return (its result, seconds taken)."""
    loop = ToyLoop()
    started = time.perf_counter()
    main_task = loop.spawn(main(loop))
    loop.run()
    return main_task.result, time.perf_counter() - started


async def run_puzzle(puzzle: Callable[[list[str]], Coroutine[Any, Any, None]]) -> list[str]:
    """Run one ordering puzzle and return its log (stragglers get 50 ms to finish)."""
    log: list[str] = []
    await puzzle(log)
    await asyncio.sleep(0.05)
    return log


print("Lab 5.3 helpers ready: outcome, quiet_unawaited, run_toy, run_puzzle")

Lab 5.3 helpers ready: outcome, quiet_unawaited, run_toy, run_puzzle


#### Exercise 1 — drive a coroutine by hand

Write `drive(coroutine, responder)`. Start the coroutine with `send(None)`.
Every value it yields is a *request*: record it and answer it with
`send(responder(request))`. If `responder` raises an `Exception`, deliver that
exception into the coroutine with `throw(error)` instead. When the coroutine
returns, return `(list of requests, return value)`. An exception the
coroutine does not handle must propagate out of `drive`.

The test coroutines below use `Ask` from 5.3.4.

In [230]:
PRICE_BOOK = {"price:chai": 20, "price:vada": 15, "gst": 5}


def book_responder(request: object) -> object:
    """Answer a request from PRICE_BOOK (KeyError if unknown)."""
    return PRICE_BOOK[str(request)]


async def order_total(sku: str, quantity: int) -> int:
    """Ask for the price and the GST rate, then compute the total."""
    price = await Ask(f"price:{sku}")
    gst_percent = await Ask("gst")
    return price * quantity * (100 + gst_percent) // 100


async def safe_price(sku: str) -> object:
    """Ask for a price; turn a KeyError thrown in into 'unavailable'."""
    try:
        return await Ask(f"price:{sku}")
    except KeyError:
        return "unavailable"


async def strict_price(sku: str) -> object:
    """Ask for a price and let a thrown KeyError escape."""
    return await Ask(f"price:{sku}")  # lets a KeyError escape


async def no_questions() -> str:
    """Return at once without asking anything."""
    return "instant"

Your solution:

In [231]:
def drive(
    coroutine: Coroutine[Any, Any, Any], responder: Callable[[object], object]
) -> tuple[list[object], object]:
    """Run coroutine to completion, answering each yielded request with responder."""
    raise NotImplementedError

Check exercise 1:

In [232]:
with quiet_unawaited():
    run_checks(
        [
            (
                "two requests answered",
                lambda: drive(order_total("chai", 2), book_responder),
                (["price:chai", "gst"], 42),
            ),
            (
                "responder error thrown in and handled",
                lambda: drive(safe_price("dosa"), book_responder),
                (["price:dosa"], "unavailable"),
            ),
            (
                "unhandled error propagates",
                lambda: raised_error_name(lambda: drive(strict_price("dosa"), book_responder)),
                "KeyError",
            ),
            ("no requests at all", lambda: drive(no_questions(), book_responder), ([], "instant")),
        ]
    )

FAIL  two requests answered: raised NotImplementedError: 
FAIL  responder error thrown in and handled: raised NotImplementedError: 
FAIL  unhandled error propagates: expected 'KeyError', got 'NotImplementedError'
FAIL  no requests at all: raised NotImplementedError: 
0/4 checks passed


#### Exercise 2 — add `gather` to the toy loop

Write `toy_gather(loop, coroutines)`, a *generator* for the toy loop: it
spawns every toy coroutine on `loop`, waits for all of them with
`("join", task)`, and returns their results **in input order** (not finishing
order). Callers use it with `results = yield from toy_gather(loop, [...])`.
The waits must overlap: three sleeps of 0.2, 0.1 and 0.15 s take about 0.2 s.

In [233]:
def toy_sleep_then(value: object, seconds: float) -> ToyCoroutine:
    """Sleep on the toy loop, then return value."""
    yield ("sleep", seconds)
    return value


def toy_instant(value: object) -> ToyCoroutine:
    """Return value on the first step, without ever yielding."""
    if False:
        yield
    return value

Your solution:

In [234]:
def toy_gather(
    loop: ToyLoop, coroutines: list[ToyCoroutine]
) -> Generator[object, object, list[object]]:
    """Spawn every coroutine on loop, join them all, return results in input order."""
    raise NotImplementedError

Check exercise 2:

In [235]:
def gather_three(loop: ToyLoop) -> ToyCoroutine:
    """Gather three sleeps of different lengths on the toy loop."""
    specs = [("a", 0.2), ("b", 0.1), ("c", 0.15)]
    return (
        yield from toy_gather(loop, [toy_sleep_then(value, seconds) for value, seconds in specs])
    )


def gather_mixed(loop: ToyLoop) -> ToyCoroutine:
    """Gather a sleeping coroutine and an instant one."""
    return (yield from toy_gather(loop, [toy_sleep_then("later", 0.05), toy_instant("now")]))


def gather_nothing(loop: ToyLoop) -> ToyCoroutine:
    """Gather an empty list."""
    return (yield from toy_gather(loop, []))


three_run = functools.cache(lambda: run_toy(gather_three))
run_checks(
    [
        ("results in input order", lambda: three_run()[0], ["a", "b", "c"]),
        ("sleeps overlap (< 0.35 s, not 0.45 s)", lambda: three_run()[1] < 0.35, True),
        (
            "joining a task that already finished",
            lambda: run_toy(gather_mixed)[0],
            ["later", "now"],
        ),
        ("empty list", lambda: run_toy(gather_nothing)[0], []),
    ]
)

FAIL  results in input order: raised NotImplementedError: 
FAIL  sleeps overlap (< 0.35 s, not 0.45 s): raised NotImplementedError: 
FAIL  joining a task that already finished: raised NotImplementedError: 
FAIL  empty list: raised NotImplementedError: 
0/4 checks passed


#### Exercise 3 — turn 5 s of sequential sleeps into 1 s of concurrent ones

`load_dashboard_sequential()` fetches five widgets one after another, each
taking 1 s. Write `load_dashboard_concurrent()` that returns the same dict
(same keys, same order) in about 1 s. The next cell times the sequential
version once (about 5 s).

In [236]:
WIDGETS = ["sales", "stock", "orders", "returns", "reviews"]


async def fetch_widget(name: str, seconds: float = 1.0) -> str:
    """Pretend to load one dashboard widget."""
    await asyncio.sleep(seconds)
    return f"{name}: ok"


async def load_dashboard_sequential() -> dict[str, str]:
    """Load every widget one after another (the slow version)."""
    return {name: await fetch_widget(name) for name in WIDGETS}


with Timer("load_dashboard_sequential"):
    print(await load_dashboard_sequential())

{'sales': 'sales: ok', 'stock': 'stock: ok', 'orders': 'orders: ok', 'returns': 'returns: ok', 'reviews': 'reviews: ok'}
load_dashboard_sequential took 5006.43 ms


Your solution:

In [237]:
async def load_dashboard_concurrent() -> dict[str, str]:
    """The same dict as load_dashboard_sequential(), in about 1 s."""
    raise NotImplementedError

Check exercise 3:

In [238]:
with Timer("load_dashboard_concurrent", report=None) as dashboard_timer:
    dashboard = await outcome(load_dashboard_concurrent())
finished = isinstance(dashboard, dict)
run_checks(
    [
        (
            "same widgets as the sequential version",
            lambda: dashboard,
            {name: f"{name}: ok" for name in WIDGETS},
        ),
        ("keys in WIDGETS order", lambda: list(dashboard) if finished else dashboard, WIDGETS),
        ("about 1 s (< 1.5 s)", lambda: finished and dashboard_timer.elapsed_seconds < 1.5, True),
    ]
)

FAIL  same widgets as the sequential version: expected {'sales': 'sales: ok', 'stock': 'stock: ok', 'orders': 'orders: ok', 'returns': 'returns: ok', 'reviews': 'reviews: ok'}, got 'raised NotImplementedError'
FAIL  keys in WIDGETS order: expected ['sales', 'stock', 'orders', 'returns', 'reviews'], got 'raised NotImplementedError'
FAIL  about 1 s (< 1.5 s): expected True, got False
0/3 checks passed


#### Exercise 4 — `race(coroutines)`

Write `race(coroutines)`, a better `Promise.race`: run all the coroutines
concurrently, and as soon as the first one finishes, **cancel the others and
wait until they have finished cancelling**, then return the winner's result
(or raise the winner's exception). An empty input raises `ValueError`.
`contender` records its cancellation in `RACE_LOG`.

In [239]:
RACE_LOG: list[str] = []


async def contender(name: str, seconds: float, fail: bool = False) -> str:
    """Sleep, then return the name or raise; log a cancellation."""
    try:
        await asyncio.sleep(seconds)
    except asyncio.CancelledError:
        RACE_LOG.append(f"{name} cancelled")
        raise
    if fail:
        raise ConnectionError(f"{name} failed")
    return name

Your solution:

In [240]:
async def race(coroutines: Iterable[Coroutine[Any, Any, Any]]) -> Any:
    """Return the first result (or raise the first error); cancel and await the rest."""
    raise NotImplementedError

Check exercise 4:

In [241]:
with quiet_unawaited():
    RACE_LOG.clear()
    race_winner = await outcome(
        race([contender("slow", 0.2), contender("fast", 0.01), contender("mid", 0.1)])
    )
    losers_cancelled = sorted(RACE_LOG)
    RACE_LOG.clear()
    race_failure = await outcome(
        race([contender("broken", 0.01, fail=True), contender("steady", 0.2)])
    )
    failure_cancelled = sorted(RACE_LOG)
    race_single = await outcome(race([contender("solo", 0)]))
    race_empty = await outcome(race([]))
run_checks(
    [
        ("fastest result wins", lambda: race_winner, "fast"),
        (
            "losers cancelled before race returns",
            lambda: losers_cancelled,
            ["mid cancelled", "slow cancelled"],
        ),
        ("the winner's exception propagates", lambda: race_failure, "raised ConnectionError"),
        ("losers cancelled after a failure too", lambda: failure_cancelled, ["steady cancelled"]),
        ("one contender", lambda: race_single, "solo"),
        ("no contenders -> ValueError", lambda: race_empty, "raised ValueError"),
    ]
)

FAIL  fastest result wins: expected 'fast', got 'raised NotImplementedError'
FAIL  losers cancelled before race returns: expected ['mid cancelled', 'slow cancelled'], got []
FAIL  the winner's exception propagates: expected 'raised ConnectionError', got 'raised NotImplementedError'
FAIL  losers cancelled after a failure too: expected ['steady cancelled'], got []
FAIL  one contender: expected 'solo', got 'raised NotImplementedError'
FAIL  no contenders -> ValueError: expected 'raised ValueError', got 'raised NotImplementedError'
0/6 checks passed


#### Exercise 5 — eight event-loop ordering puzzles

Each puzzle appends strings to `log`. For each one, predict the final `log`
**before running anything**, using the rules from 5.3.5–5.3.9 (one FIFO
ready queue; `create_task` schedules; awaiting a coroutine that does not
suspend never yields; due timers join the queue after the handles already in
it; done-callbacks run in registration order). Then write your answers in
`predictions()`. The runner awaits the puzzle, then gives stragglers 50 ms.
Each puzzle has its own cell below; running a cell only defines the puzzle.

**Puzzle 1**: a task is created, then the parent appends and yields once.

In [242]:
async def puzzle_1(log: list[str]) -> None:
    """Puzzle 1: create_task, then append in the parent."""

    async def child() -> None:
        """Append "child"."""
        log.append("child")

    asyncio.create_task(child())
    log.append("parent")
    await asyncio.sleep(0)

**Puzzle 2**: a background task, and a coroutine awaited directly.

In [243]:
async def puzzle_2(log: list[str]) -> None:
    """Puzzle 2: a background task vs a coroutine awaited directly."""

    async def child(name: str) -> None:
        """Append name."""
        log.append(name)

    background = asyncio.create_task(child("task"))
    await child("direct")
    log.append("after direct")
    await background

**Puzzle 3**: `call_later(0)`, `call_soon` and a plain append.

In [244]:
async def puzzle_3(log: list[str]) -> None:
    """Puzzle 3: call_later(0) vs call_soon vs a plain append."""
    loop = asyncio.get_running_loop()
    loop.call_later(0, log.append, "later")
    loop.call_soon(log.append, "soon")
    log.append("now")
    await asyncio.sleep(0.01)

**Puzzle 4**: a lazy task and an eager task (*New in 3.14*: `eager_start`).

In [245]:
async def puzzle_4(log: list[str]) -> None:  # needs Python 3.14+ (eager_start)
    """Puzzle 4: a lazy task and an eager task."""

    async def child(name: str) -> None:
        """Append a start and an end entry around one yield."""
        log.append(f"{name} start")
        await asyncio.sleep(0)
        log.append(f"{name} end")

    asyncio.create_task(child("lazy"))
    asyncio.create_task(child("eager"), eager_start=True)
    log.append("parent")
    await asyncio.sleep(0.01)

**Puzzle 5**: a done-callback and the coroutine awaiting the same task.

In [246]:
async def puzzle_5(log: list[str]) -> None:
    """Puzzle 5: a done-callback vs the awaiting coroutine."""

    async def child() -> int:
        """Append "child" and return 1."""
        log.append("child")
        return 1

    task = asyncio.create_task(child())
    task.add_done_callback(lambda _: log.append("callback"))
    await task
    log.append("awaiter")

**Puzzle 6**: `gather` over steps with different delays.

In [247]:
async def puzzle_6(log: list[str]) -> None:
    """Puzzle 6: gather over steps with different delays."""

    async def step(name: str, delay: float) -> str:
        """Sleep, append name, return name."""
        await asyncio.sleep(delay)
        log.append(name)
        return name

    results = await asyncio.gather(step("a", 0.02), step("b", 0), step("c", 0.01))
    log.append("+".join(results))

**Puzzle 7**: cancelling a task that is sleeping.

In [248]:
async def puzzle_7(log: list[str]) -> None:
    """Puzzle 7: cancel a sleeping task, then await it."""

    async def child() -> None:
        """Sleep for a second, logging start and finally."""
        try:
            log.append("child start")
            await asyncio.sleep(1)
        finally:
            log.append("child finally")

    task = asyncio.create_task(child())
    await asyncio.sleep(0)
    task.cancel()
    log.append("cancel requested")
    try:
        await task
    except asyncio.CancelledError:
        log.append("parent saw CancelledError")

**Puzzle 8**: two waiters, a done-callback and `call_soon` around one Future.

In [249]:
async def puzzle_8(log: list[str]) -> None:
    """Puzzle 8: two waiters, a done-callback and call_soon on one Future."""
    loop = asyncio.get_running_loop()
    signal = loop.create_future()

    async def waiter(name: str) -> None:
        """Wait for the signal, then append name."""
        await signal
        log.append(name)

    waiters = [asyncio.create_task(waiter(name)) for name in ("w1", "w2")]
    await asyncio.sleep(0)
    signal.add_done_callback(lambda _: log.append("callback"))
    loop.call_soon(log.append, "call_soon")
    signal.set_result(None)
    log.append("set")
    await asyncio.gather(*waiters)

The runner looks the puzzles up in this table:

In [250]:
PUZZLES = {
    1: puzzle_1,
    2: puzzle_2,
    3: puzzle_3,
    4: puzzle_4,
    5: puzzle_5,
    6: puzzle_6,
    7: puzzle_7,
    8: puzzle_8,
}
print(f"{len(PUZZLES)} puzzles loaded")

8 puzzles loaded


Your predictions:

In [251]:
def predictions() -> dict[int, list[str]]:
    """Your predicted final log for each puzzle: {1: [...], 2: [...], ..., 8: [...]}."""
    raise NotImplementedError

Check exercise 5 (it runs each puzzle and compares the log with your prediction):

In [252]:
puzzle_logs = {}
for number, puzzle in PUZZLES.items():
    puzzle_logs[number] = await run_puzzle(puzzle)
run_checks(
    [
        (f"puzzle {number}", lambda number=number: predictions()[number], log)
        for number, log in puzzle_logs.items()
    ]
)

FAIL  puzzle 1: raised NotImplementedError: 
FAIL  puzzle 2: raised NotImplementedError: 
FAIL  puzzle 3: raised NotImplementedError: 
FAIL  puzzle 4: raised NotImplementedError: 
FAIL  puzzle 5: raised NotImplementedError: 
FAIL  puzzle 6: raised NotImplementedError: 
FAIL  puzzle 7: raised NotImplementedError: 
FAIL  puzzle 8: raised NotImplementedError: 
0/8 checks passed


#### Exercise 6 — an asyncio chat server with streams

Implement `ChatRoom.handle(reader, writer)`, the per-client callback for
`asyncio.start_server`. Protocol (UTF-8, one message per line):

- the first line a client sends is its nickname; every **other** member then
  receives `* {nick} joined`;
- each later line is sent to every other member as `{nick}: {text}`;
- when the client disconnects (`readline()` returns `b""`), the others
  receive `* {nick} left`.

Use `writer.write(...)` followed by `await writer.drain()` (backpressure, as
in 5.3.2). A member that has gone away must not crash the others: catch
`ConnectionError` while sending. `chat_scenario` starts the server on a free
port, connects asha, ben and chen, has them talk, and collects what each one
received. The given code comes first: two client-side helpers, then the
scenario itself.

In [253]:
type ChatClients = dict[str, tuple[asyncio.StreamReader, asyncio.StreamWriter]]


async def send_line(clients: ChatClients, nick: str, line: str) -> None:
    """Send one line as nick, then give the server 50 ms to deliver it."""
    clients[nick][1].write(f"{line}\n".encode())
    with contextlib.suppress(ConnectionError):  # the server may have dropped us
        await clients[nick][1].drain()
    await asyncio.sleep(0.05)  # let the server deliver before the next step


async def lines_received(clients: ChatClients, nick: str) -> list[str]:
    """Every line nick has received, read until 100 ms of silence or EOF."""
    lines = []
    while True:
        try:
            line = await asyncio.wait_for(clients[nick][0].readline(), 0.1)
        except TimeoutError:
            return lines
        if not line:
            return lines
        lines.append(line.decode().rstrip("\n"))

`chat_scenario` starts the server, connects the three clients (each sends its
nickname first), has asha and ben talk, disconnects chen, and returns what each
client received plus any exception your `handle` raised:

In [254]:
async def chat_scenario(room_factory: Callable[[], Any]) -> dict[str, object]:
    """Run a three-client conversation against room_factory().handle."""
    room, errors, clients = room_factory(), [], {}

    async def guarded(reader: asyncio.StreamReader, writer: asyncio.StreamWriter) -> None:
        """Run room.handle, recording its exception instead of logging it."""
        try:
            await room.handle(reader, writer)
        except Exception as error:  # keep failures out of the kernel log
            errors.append(type(error).__name__)
            writer.close()

    server = await asyncio.start_server(guarded, "127.0.0.1", 0)
    port = server.sockets[0].getsockname()[1]
    try:
        for nick in ("asha", "ben", "chen"):
            clients[nick] = await asyncio.open_connection("127.0.0.1", port)
            await send_line(clients, nick, nick)
        await send_line(clients, "asha", "hello")
        await send_line(clients, "ben", "hi asha")
        transcripts = {"chen": await lines_received(clients, "chen")}
        clients["chen"][1].close()
        await asyncio.sleep(0.05)
        transcripts |= {nick: await lines_received(clients, nick) for nick in ("asha", "ben")}
        errors_seen = list(errors)
    finally:
        for _, writer in clients.values():
            writer.close()
        server.close()
        with contextlib.suppress(TimeoutError):
            await asyncio.wait_for(server.wait_closed(), 1)
    return {"transcripts": transcripts, "errors": errors_seen}

Your solution:

In [255]:
class ChatRoom:
    """A chat room served with asyncio streams (protocol in the exercise text)."""

    def __init__(self) -> None:
        """Start with no members."""
        self.members: dict[asyncio.StreamWriter, str] = {}

    async def handle(self, reader: asyncio.StreamReader, writer: asyncio.StreamWriter) -> None:
        """Serve one client: nickname line first, then chat lines until EOF."""
        raise NotImplementedError

Check exercise 6:

In [256]:
chat_result = await outcome(chat_scenario(ChatRoom))
chat_transcripts = chat_result["transcripts"] if isinstance(chat_result, dict) else {}
run_checks(
    [
        (
            "asha's screen",
            lambda: chat_transcripts.get("asha"),
            ["* ben joined", "* chen joined", "ben: hi asha", "* chen left"],
        ),
        (
            "ben's screen",
            lambda: chat_transcripts.get("ben"),
            ["* chen joined", "asha: hello", "* chen left"],
        ),
        ("chen's screen", lambda: chat_transcripts.get("chen"), ["asha: hello", "ben: hi asha"]),
        (
            "no handler errors",
            lambda: chat_result["errors"] if isinstance(chat_result, dict) else chat_result,
            [],
        ),
    ]
)

FAIL  asha's screen: expected ['* ben joined', '* chen joined', 'ben: hi asha', '* chen left'], got None
FAIL  ben's screen: expected ['* chen joined', 'asha: hello', '* chen left'], got None
FAIL  chen's screen: expected ['asha: hello', 'ben: hi asha'], got None
FAIL  no handler errors: expected [], got 'raised ConnectionResetError'
0/4 checks passed


### Recap — Unit 5.3

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| Why async | One thread, many waiting connections: tasks cost kilobytes, threads cost a stack each | the reason Node exists |
| OS foundations | Non-blocking sockets + a readiness API (`epoll`, `kqueue`) or completion API (IOCP); `selectors` wraps them | libuv |
| Toy event loop | Generators + ready queue + timer heap + selector = an event loop | — |
| Native coroutines | `async def` returns a lazy coroutine; `send`/`throw`/`close` drive it | async functions run eagerly |
| Awaitables | coroutines, Futures, Tasks, objects with `__await__` | thenables |
| What `await` does | yields a pending Future up to the Task, which resumes via a done-callback | microtask continuation |
| The asyncio loop | `_ready` deque + `_scheduled` heap + `_selector`; `_run_once` runs the handles queued at its start | phases + microtask queues |
| Futures and Tasks | Future = result slot; Task = Future driving a coroutine; keep references; eager tasks skip the loop | `Promise` |
| Core API | `asyncio.run` / `Runner` at the entry point; `gather` for concurrency; `sleep(0)` to yield; `wait_for` for timeouts | `Promise.all`, `setTimeout` |
| Ordering | One FIFO; `await` of a non-suspending coroutine does not yield | nextTick → microtasks → phases |
| Function colors | Async spreads upward; cross with `asyncio.run`, `to_thread`, `run_coroutine_threadsafe` | the same problem |

**Further reading**

- [`asyncio`](https://docs.python.org/3/library/asyncio.html):
  [event loop](https://docs.python.org/3/library/asyncio-eventloop.html),
  [coroutines and tasks](https://docs.python.org/3/library/asyncio-task.html),
  [futures](https://docs.python.org/3/library/asyncio-future.html),
  [runners](https://docs.python.org/3/library/asyncio-runner.html),
  [call graph introspection](https://docs.python.org/3/library/asyncio-graph.html) (3.14)
- [`selectors`](https://docs.python.org/3/library/selectors.html),
  [`select`](https://docs.python.org/3/library/select.html)
- [PEP 342](https://peps.python.org/pep-0342/) (coroutines via enhanced generators),
  [PEP 380](https://peps.python.org/pep-0380/) (`yield from`),
  [PEP 3156](https://peps.python.org/pep-3156/) (asyncio),
  [PEP 492](https://peps.python.org/pep-0492/) (`async`/`await`)
- Node.js: [The Node.js event loop](https://nodejs.org/en/learn/asynchronous-work/event-loop-timers-and-nexttick);
  Dan Kegel, *The C10K problem*; Bob Nystrom, *What Color is Your Function?*

## Unit 5.4 — asyncio in Production

**Objectives**

- Run groups of tasks safely with `TaskGroup`, and handle several failures
  at once with `ExceptionGroup` and `except*`.
- Put deadlines on work with `asyncio.timeout`, cancel tasks correctly, clean
  up on cancellation, and protect critical steps with `shield`.
- Coordinate tasks with `Event`, `Lock`, `Condition`, `Semaphore` and
  `Barrier`; build worker pools and pipelines on `asyncio.Queue` with
  backpressure and `Queue.shutdown()`.
- Write async context managers, iterators and generators, and close them
  deterministically with `aclosing`.
- Detect a blocked event loop (debug mode, slow-callback warnings, a lag
  monitor) and fix it with threads, processes or async libraries.
- Carry request-scoped state with `contextvars`; use a shared
  `httpx.AsyncClient` with limits and timeouts; rate-limit and retry with
  backoff and jitter.
- Test async code with pytest, `AsyncMock` and `httpx.MockTransport`; speed up
  the loop with `uvloop`; bridge threads, anyio and trio.

**JS bridge**

| Node.js | asyncio | The difference that bites |
| --- | --- | --- |
| `Promise.all` + an `AbortController` you wire up yourself | `asyncio.TaskGroup` | a failure cancels the siblings **automatically** and all errors arrive together in an `ExceptionGroup` |
| `AbortSignal.timeout(ms)` | `async with asyncio.timeout(seconds):` | cancellation is delivered *into* the coroutine as `CancelledError` |
| `controller.abort()` | `task.cancel()` | `CancelledError` is a `BaseException`: `except Exception` does not catch it |
| `p-limit(5)` | `asyncio.Semaphore(5)` | built in |
| `for await (const x of stream)` | `async for x in stream:` | close early-exited generators with `aclosing` |
| `await using` / `Symbol.asyncDispose` | `async with` / `__aexit__` | |
| `AsyncLocalStorage` | `contextvars.ContextVar` | each Task gets a *copy* of the context |
| `undici.Agent({connections})` | `httpx.AsyncClient(limits=httpx.Limits(...))` | create one client and share it |
| blocked-loop detectors (`blocked-at`, `--cpu-prof`) | `asyncio` debug mode, `slow_callback_duration` | built in |
| Jest / Vitest async tests | pytest + anyio plugin (or pytest-asyncio), `AsyncMock` | tests need an event loop: the plugin provides it |

**How this unit runs.** Cells use top-level `await` on the kernel's loop.
Demos that need a fresh loop (`asyncio.run(..., debug=True)`, uvloop, anyio's
`run`) and the pytest run happen in child processes. HTTP demos talk to the
Unit 4.6 shop API on `127.0.0.1` or to `httpx.MockTransport`, never to the
internet. Optional packages are detected first; a missing one prints a skip
message.

Unit setup, part 1: imports, `DEMO_SCALE`, the scratch folder (on `sys.path`
for the worker module in 5.4.6), feature flags for the optional packages, and
the error and interpreter helpers.

In [257]:
import asyncio
import concurrent.futures
import contextlib
import contextvars
import functools
import gc
import importlib
import importlib.util
import inspect
import io
import itertools
import logging
import os
import platform
import random
import re
import subprocess
import sys
import sysconfig
import textwrap
import threading
import time
import warnings
from collections import Counter
from collections.abc import AsyncIterator, Awaitable, Callable, Coroutine, Generator, Iterable
from pathlib import Path
from types import ModuleType
from typing import Any
from unittest.mock import AsyncMock

import httpx

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))  # the one knob for workload sizes
REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE5_DIR = scratch_dir / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT / "notebooks")  # %%writefile cells use paths relative to notebooks/
if str(PHASE5_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE5_DIR))  # child processes receive a copy of sys.path
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1", "PY_COLORS": "0"}
KERNEL_LOOP = asyncio.get_running_loop()


def has_module(name: str) -> bool:
    """True if the module can be imported (checked without importing it)."""
    return importlib.util.find_spec(name) is not None


HAS_UVLOOP = has_module("uvloop") and sys.platform != "win32"  # uvloop does not support Windows
HAS_AIOFILES, HAS_AIOHTTP, HAS_TENACITY = (
    has_module("aiofiles"),
    has_module("aiohttp"),
    has_module("tenacity"),
)
HAS_ANYIO, HAS_TRIO, HAS_PYTEST_ASYNCIO = (
    has_module("anyio"),
    has_module("trio"),
    has_module("pytest_asyncio"),
)


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


def raised_message(thunk: Callable[[], object]) -> str:
    """Call thunk and return "ErrorName: message" (memory addresses hidden), or "no error"."""
    try:
        thunk()
    except Exception as error:
        return re.sub(r" at 0x[0-9a-f]+", "", f"{type(error).__name__}: {error}")
    return "no error"


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"

Unit setup, part 2: `Timer`, `run_child()` (as in Unit 5.3) and
`fresh_import()` (Unit 5.2).

In [258]:
class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        """Remember the label and where to report the elapsed time."""
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        """Start the clock."""
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        """Stop the clock and report the elapsed milliseconds."""
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def run_child(
    source: str, python: str = sys.executable, extra_env: dict[str, str] | None = None
) -> str:
    """Run dedented source as a script in a child interpreter; return its stdout + stderr."""
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    completed = subprocess.run(
        [python, str(script)], capture_output=True, text=True, timeout=120,
        env={**CHILD_ENV, **(extra_env or {})}, cwd=PHASE5_DIR,
    )  # fmt: skip
    output = (completed.stdout + completed.stderr).rstrip()
    for path, label in ((str(PHASE5_DIR), "<scratch>"), (str(Path(python).parent), "<python>")):
        output = output.replace(path, label)
    output = re.sub(r"pid=\d+", "pid=<pid>", output)
    output = re.sub(r"0x[0-9a-f]{6,}", "0x<address>", output)
    return re.sub(r"/[^\s\"']*/lib/python3\.\d+t?/", "<stdlib>/", output)


def fresh_import(module_name: str) -> ModuleType:
    """Import a module saved by a %%writefile cell, always re-reading the file from disk."""
    for cached in (PHASE5_DIR / "__pycache__").glob(f"{module_name}.*.pyc"):
        cached.unlink()
    importlib.invalidate_caches()
    sys.modules.pop(module_name, None)
    return importlib.import_module(module_name)


print(interpreter_label())
print(f"{DEMO_SCALE=}  kernel loop: {type(KERNEL_LOOP).__name__}")
optional = {
    "uvloop": HAS_UVLOOP,
    "aiofiles": HAS_AIOFILES,
    "aiohttp": HAS_AIOHTTP,
    "tenacity": HAS_TENACITY,
    "anyio": HAS_ANYIO,
    "trio": HAS_TRIO,
    "pytest-asyncio": HAS_PYTEST_ASYNCIO,
}
print(
    "optional packages: "
    + ", ".join(f"{name} {'yes' if found else 'no'}" for name, found in optional.items())
)

CPython 3.15.1, default (GIL) build, x86_64
DEMO_SCALE=1  kernel loop: _UnixSelectorEventLoop
optional packages: uvloop yes, aiofiles yes, aiohttp no, tenacity yes, anyio yes, trio no, pytest-asyncio no


The local shop API from Unit 4.6 (the same code, started once per kernel) for
the HTTP demos in 5.4.8–5.4.9. Its `/slow?seconds=` route sleeps on the server,
`/flaky/<key>?fail=N` fails N times with 503 before succeeding, and
`/connections` counts the TCP connections it has accepted.

In [259]:
import http.server
import json
import random
import threading
import time
import urllib.parse

SHOP_API_PORT = 18046  # fixed so outputs (URLs in messages) are stable; falls back to any free port


def demo_orders() -> list[dict[str, object]]:
    """40 seeded orders as an API would return them, with 3 invalid records planted."""
    generator = random.Random(46)
    cities = ["Pune", "Mumbai", "Bengaluru", "Hubli & Dharwad"]
    prices = {"TEA-500": "240.00", "MUG-02": "199.00", "JAR-750": "350.50", "KETTLE-9": "899.00"}
    orders: list[dict[str, object]] = []
    for order_id in range(1, 41):
        sku = generator.choice(sorted(prices))
        orders.append(
            {
                "order_id": order_id,
                "customer": f"customer-{order_id:03d}",
                "city": generator.choice(cities),
                "sku": sku,
                "quantity": generator.randint(1, 5),
                "unit_price": prices[sku],
                "placed_at": f"2026-11-{generator.randint(1, 7):02d}"
                f"T{generator.randint(9, 20):02d}:00:00+05:30",
            }
        )
    orders[6]["quantity"] = -2  # invalid: not positive
    del orders[17]["city"]  # invalid: missing field
    orders[29]["unit_price"] = "two hundred"  # invalid: not a number
    return orders


PRODUCTS_HTML = """<!doctype html>
<html><head><title>Chai Corner - Products</title></head>
<body>
  <h1>Products</h1>
  <ul id="catalogue">
    <li class="product" data-sku="TEA-500">
      <a href="/products/tea-500">Masala tea 500 g</a> <span class="price">₹240.00</span>
    </li>
    <li class="product" data-sku="MUG-02">
      <a href="/products/mug-02">Steel mug</a> <span class="price">₹199.00</span>
    </li>
    <li class="product sold-out" data-sku="JAR-750">
      <a href="/products/jar-750">Pickle jar 750 ml</a> <span class="price">₹350.50</span>
    </li>
  </ul>
  <p class="note">Prices include GST &amp; free delivery above ₹500.</p>
</body></html>
"""
ROBOTS_TXT = "User-agent: *\nDisallow: /admin/\nCrawl-delay: 1\n\nUser-agent: BadBot\nDisallow: /\n"


class ShopAPIHandler(http.server.BaseHTTPRequestHandler):
    """A tiny local HTTP API used by every HTTP demo (no external network)."""

    protocol_version = "HTTP/1.1"  # keep-alive, so clients can reuse connections
    hits: dict[str, int] = {}
    connections = 0

    def setup(self) -> None:
        """Count each new TCP connection."""
        type(self).connections += 1  # one per TCP connection, not per request
        super().setup()

    def log_message(self, format: str, *args: object) -> None:  # keep the notebook quiet
        """Silence the default request log."""
        pass

    def _send(self, status: int, body: object, content_type: str = "application/json") -> None:
        """Send status and a JSON or text body."""
        data = body.encode() if isinstance(body, str) else json.dumps(body).encode()
        try:
            self.send_response(status)
            self.send_header("Content-Type", f"{content_type}; charset=utf-8")
            self.send_header("Content-Length", str(len(data)))
            self.end_headers()
            self.wfile.write(data)
        except BrokenPipeError, ConnectionResetError:  # the client gave up (a timeout demo)
            self.close_connection = True

    def _count(self, key: str) -> int:
        """Increment and return the hit counter for key."""
        self.hits[key] = self.hits.get(key, 0) + 1
        return self.hits[key]

    def do_GET(self) -> None:
        """Route GET requests to the demo endpoints."""
        url = urllib.parse.urlsplit(self.path)
        query = dict(urllib.parse.parse_qsl(url.query))
        parts = url.path.strip("/").split("/")
        fail_first = int(query.get("fail", 0))
        if url.path == "/rates":
            base = query.get("base", "INR")
            self._send(
                200,
                {
                    "base": base,
                    "date": "2026-11-07",
                    "rates": {"USD": 0.012, "EUR": 0.011, "GBP": 0.0094},
                },
            )
        elif url.path == "/orders":
            if "key" in query and self._count(query["key"]) <= fail_first:
                self._send(503, {"error": "service busy, try again"})
            else:
                self._send(200, demo_orders())
        elif parts[0] == "flaky" and len(parts) == 2:
            attempt = self._count(parts[1])
            if attempt <= fail_first:
                self._send(503, {"error": "busy", "attempt": attempt})
            else:
                self._send(200, {"ok": True, "attempt": attempt})
        elif url.path == "/hits":
            self._send(
                200, {"key": query.get("key"), "hits": self.hits.get(query.get("key", ""), 0)}
            )
        elif url.path == "/connections":
            self._send(200, {"connections": type(self).connections})
        elif url.path == "/reset":
            self.hits.clear()
            self._send(200, {"reset": True})
        elif url.path == "/slow":
            time.sleep(float(query.get("seconds", "2")))
            self._send(200, {"slow": True})
        elif parts[0] == "status" and len(parts) == 2:
            self._send(int(parts[1]), {"status": int(parts[1])})
        elif url.path == "/headers":
            self._send(
                200,
                {
                    name.lower(): value
                    for name, value in self.headers.items()
                    if name.lower() in ("user-agent", "x-request-id", "authorization")
                },
            )
        elif url.path == "/products.html":
            self._send(200, PRODUCTS_HTML, "text/html")
        elif url.path == "/robots.txt":
            self._send(200, ROBOTS_TXT, "text/plain")
        else:
            self._send(404, {"error": f"no route {url.path}"})

    def do_POST(self) -> None:
        """Echo the posted JSON back."""
        length = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(length) or b"null")
        self._send(201 if self.path == "/orders" else 200, {"received": payload})


def start_shop_api() -> str:
    """Start the local API once (in a daemon thread) and return its base URL."""
    server = globals().get("_SHOP_API_SERVER")
    if server is None:
        try:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", SHOP_API_PORT), ShopAPIHandler)
        except OSError:
            server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), ShopAPIHandler)
        server.daemon_threads = True
        threading.Thread(target=server.serve_forever, daemon=True).start()
        globals()["_SHOP_API_SERVER"] = server
    return f"http://127.0.0.1:{server.server_address[1]}"

Start it (the first call starts the server; later calls return the same URL):

In [260]:
SHOP_API = start_shop_api()
print(f"{SHOP_API=}")

SHOP_API='http://127.0.0.1:18046'


### 5.4.1 Structured concurrency

**Structured concurrency** means a task's children cannot outlive the block
that started them: when the block exits, every child has finished, and if one
child fails, its siblings are cancelled and the error is reported. Trio
introduced the idea ("nurseries"); asyncio adopted it as `TaskGroup` in 3.11.

`gather` is *not* structured. When one awaitable fails, `gather` raises at
once, but the others keep running in the background as orphans:

In [261]:
progress: list[str] = []


async def job(name: str, seconds: float, fail: bool = False) -> str:
    """Sleep, then succeed or fail; records completion and cancellation."""
    try:
        await asyncio.sleep(seconds)
    except asyncio.CancelledError:
        progress.append(f"{name} cancelled")
        raise
    if fail:
        raise ValueError(f"{name} failed")
    progress.append(f"{name} finished")
    return name


try:
    await asyncio.gather(job("report", 0.05), job("invoice", 0.01, fail=True))
except ValueError as error:
    print(f"gather raised: {error}")
print(f"right after: {progress}  <- report is still running")
await asyncio.sleep(0.06)
print(f"a bit later: {progress}  <- the orphan finished on its own")

gather raised: invoice failed
right after: []  <- report is still running
a bit later: ['report finished']  <- the orphan finished on its own


`async with asyncio.TaskGroup() as group:` starts tasks with
`group.create_task(...)`; the block does not exit until all of them are done.
Results are read from the task objects afterwards:

In [262]:
progress.clear()
async with asyncio.TaskGroup() as group:
    report_task = group.create_task(job("report", 0.02))
    invoice_task = group.create_task(job("invoice", 0.01))
print(f"{report_task.result()=} {invoice_task.result()=}")
print(progress)

report_task.result()='report' invoice_task.result()='invoice'
['invoice finished', 'report finished']


When a child fails, the group cancels the other children, waits for them,
and raises an **`ExceptionGroup`** holding every error that happened (two
jobs failed here at the same moment, so both are reported):

In [263]:
progress.clear()
try:
    async with asyncio.TaskGroup() as group:
        group.create_task(job("report", 0.05))
        group.create_task(job("invoice", 0.01, fail=True))
        group.create_task(job("email", 0.01, fail=True))
except ExceptionGroup as error:
    print(f"{type(error).__name__}: {error}")
    for sub_error in error.exceptions:
        print(f"  {sub_error!r}")
print(progress)

ExceptionGroup: unhandled errors in a TaskGroup (2 sub-exceptions)
  ValueError('invoice failed')
  ValueError('email failed')
['report cancelled']


**`ExceptionGroup`** (*new in 3.11*) is an exception that carries several
exceptions, possibly nested. `split(type)` divides it into matching and
non-matching groups; `subgroup(predicate)` keeps only what matches:

In [264]:
batch_error = ExceptionGroup(
    "order batch failed",
    [
        ValueError("bad quantity"),
        KeyError("sku-9"),
        ExceptionGroup("payments", [ConnectionError("gateway down")]),
    ],
)
print(f"{batch_error}  ->  {len(batch_error.exceptions)} top-level entries")
value_errors, others = batch_error.split(ValueError)
print(f"split: {value_errors.exceptions} | rest: {others.exceptions}")
network = batch_error.subgroup(lambda error: isinstance(error, ConnectionError))
print(f"subgroup keeps the nesting: {network!r}")

order batch failed (3 sub-exceptions)  ->  3 top-level entries
split: (ValueError('bad quantity'),) | rest: (KeyError('sku-9'), ExceptionGroup('payments', [ConnectionError('gateway down')]))
subgroup keeps the nesting: ExceptionGroup('order batch failed', [ExceptionGroup('payments', [ConnectionError('gateway down')])])


**`except*`** handles each kind of error in a group separately; every clause
whose type matches runs (unlike plain `except`, where only the first matching
clause runs). Anything left unhandled is re-raised as a smaller group.

In [265]:
def handle_batch(error_group: ExceptionGroup) -> None:
    """Route each kind of failure to its own handler."""
    try:
        raise error_group
    except* ValueError as input_errors:
        print(f"  fix the input: {[str(error) for error in input_errors.exceptions]}")
    except* LookupError as missing:
        print(f"  unknown items: {[str(error) for error in missing.exceptions]}")
    except* ConnectionError as network_errors:
        print(f"  retry later: {[str(error) for error in network_errors.exceptions]}")


handle_batch(batch_error)

  fix the input: ['bad quantity']
  unknown items: ["'sku-9'"]
  retry later: ['payments (1 sub-exception)']


**`as_completed`** hands back results in *completion* order. Iterated with
plain `for`, it yields awaitables to await. *New in 3.13:* iterated with
`async for`, it yields the **original tasks** you passed in, so you can tell
which one finished:

In [266]:
async def quote(supplier: str, seconds: float) -> str:
    """Return a supplier quote after a delay."""
    await asyncio.sleep(seconds)
    return f"{supplier}: Rs {round(seconds * 1000)}"


for next_quote in asyncio.as_completed([quote("A", 0.03), quote("B", 0.01), quote("C", 0.02)]):
    print(f"for:       {await next_quote}")

if sys.version_info >= (3, 13):
    supplier_tasks = {
        asyncio.create_task(quote(name, seconds)): name
        for name, seconds in (("A", 0.03), ("B", 0.01))
    }
    async for finished_task in asyncio.as_completed(supplier_tasks):
        print(f"async for: task for {supplier_tasks[finished_task]} -> {finished_task.result()}")
else:
    print("async iteration over as_completed needs Python 3.13+: skipped")

for:       B: Rs 10
for:       C: Rs 20


for:       A: Rs 30
async for: task for B -> B: Rs 10


async for: task for A -> A: Rs 30


*New in 3.15:* `TaskGroup.cancel()` stops a group early **without** an error,
for example once the first supplier has answered. The block exits normally
and the remaining children are cancelled:

In [267]:
found_suppliers: list[str] = []


async def search_supplier(group: asyncio.TaskGroup, supplier: str, seconds: float) -> None:
    """Record the supplier after a delay, then cancel the whole group."""
    await asyncio.sleep(seconds)
    found_suppliers.append(supplier)
    group.cancel()  # enough: stop the whole search


if hasattr(asyncio.TaskGroup, "cancel"):
    progress.clear()
    async with asyncio.TaskGroup() as search_group:
        for supplier, seconds in (("A", 0.03), ("B", 0.01), ("C", 0.5)):
            search_group.create_task(search_supplier(search_group, supplier, seconds))
    print(f"block exited normally; found {found_suppliers}")
else:
    print("TaskGroup.cancel() needs Python 3.15+: skipped")

block exited normally; found ['B']


**Under the hood.** The cancellation logic is short enough to read.
`_on_task_done` notices a failed child and calls `_abort`, which cancels
every child that is still running:

In [268]:
abort_source = inspect.getsource(asyncio.TaskGroup._abort)
print(textwrap.dedent(abort_source).strip())
print(f"\n{sys.getsizeof(asyncio.TaskGroup())} bytes for an empty TaskGroup (shallow)")

def _abort(self):
    self._aborting = True

    for t in self._tasks:
        if not t.done():
            t.cancel()

48 bytes for an empty TaskGroup (shallow)


### 5.4.2 Timeouts and cancellation

`async with asyncio.timeout(seconds):` (*new in 3.11*) puts a deadline on a
whole block. When it expires, the task is cancelled; the context manager turns
that cancellation into `TimeoutError` at the `async with`. `expired()` tells
whether it fired; `reschedule()` moves the deadline; `timeout(None)` starts
without one.

In [269]:
try:
    async with asyncio.timeout(0.05) as deadline:
        await asyncio.sleep(1)
except TimeoutError:
    print(f"timed out; {deadline.expired()=}")

async with asyncio.timeout(None) as deadline:  # no deadline yet
    deadline.reschedule(KERNEL_LOOP.time() + 0.5)  # decided later
    await asyncio.sleep(0.01)
print(f"finished in time; {deadline.expired()=}")

timed out; deadline.expired()=True


finished in time; deadline.expired()=False


`asyncio.timeout_at(when)` takes an absolute loop time: one **overall
deadline** for several steps, which per-call timeouts cannot express:

In [270]:
steps_done: list[int] = []
overall_deadline = KERNEL_LOOP.time() + 0.1
try:
    async with asyncio.timeout_at(overall_deadline):
        for step in range(5):
            await asyncio.sleep(0.03)  # each step alone would be fine
            steps_done.append(step)
except TimeoutError:
    print(f"overall deadline hit after steps {steps_done}")

overall deadline hit after steps [0, 1, 2]


**`CancelledError` semantics.** `task.cancel(msg)` schedules a
`CancelledError` to be thrown into the coroutine at its current `await`.
Since 3.8 it is a `BaseException`, so `except Exception` does **not** catch
it; cleanup in `finally` still runs. Cancelling a finished task does nothing
and returns `False`.

In [271]:
print(f"{asyncio.CancelledError.__mro__[1].__name__=}")


async def careful_worker() -> None:
    """Sleep a second, showing that except Exception misses a cancel."""
    try:
        await asyncio.sleep(1)
    except Exception:
        print("  never printed: CancelledError is not an Exception")
    finally:
        print("  finally runs on cancellation")


worker_task = asyncio.create_task(careful_worker())
await asyncio.sleep(0)
print(f"{worker_task.cancel('shutting down')=}")
try:
    await worker_task
except asyncio.CancelledError as error:
    print(f"CancelledError: {error}; {worker_task.cancelled()=}")
print(f"cancel a finished task: {worker_task.cancel()=}")

asyncio.CancelledError.__mro__[1].__name__='BaseException'
worker_task.cancel('shutting down')=True
  finally runs on cancellation
CancelledError: shutting down; worker_task.cancelled()=True
cancel a finished task: worker_task.cancel()=False


> **Warning:** never swallow `CancelledError`. If a coroutine catches it and
> carries on, `cancel()` silently fails: the task finishes normally, and
> `asyncio.timeout` and `TaskGroup` (which count pending cancellations with
> `cancelling()` / `uncancel()`) stop working. Clean up, then re-raise.

In [272]:
async def stubborn_worker() -> str:
    """Swallow the cancellation (the bug this demo shows)."""
    try:
        await asyncio.sleep(1)
    except asyncio.CancelledError:
        return "ignored the cancel"  # BUG
    return "finished"


stubborn_task = asyncio.create_task(stubborn_worker())
await asyncio.sleep(0)
stubborn_task.cancel()
print(f"{await stubborn_task=}  {stubborn_task.cancelled()=}  {stubborn_task.cancelling()=}")

await stubborn_task='ignored the cancel'  stubborn_task.cancelled()=False  stubborn_task.cancelling()=1


**Cleanup.** A cancelled coroutine may still `await` during its cleanup (roll
back, close a connection), as long as it re-raises afterwards:

In [273]:
cleanup_log: list[str] = []


async def upload_report() -> None:
    """Upload, rolling back and re-raising if cancelled."""
    try:
        cleanup_log.append("upload: started")
        await asyncio.sleep(1)
    except asyncio.CancelledError:
        cleanup_log.append("upload: cancelled, rolling back")
        await asyncio.sleep(0.01)  # async cleanup is allowed
        raise
    finally:
        cleanup_log.append("upload: connection closed")


upload_task = asyncio.create_task(upload_report())
await asyncio.sleep(0.02)
upload_task.cancel()
await asyncio.gather(upload_task, return_exceptions=True)
print(cleanup_log)

['upload: started', 'upload: cancelled, rolling back', 'upload: connection closed']


**`shield`.** `asyncio.shield(task)` protects an operation from the
*caller's* cancellation: the caller stops waiting, but the shielded task keeps
running. Use it for steps that must not be interrupted halfway (committing a
payment). Keep a reference to the inner task, or it may be collected.

In [274]:
payment_log: list[str] = []


async def charge_card() -> str:
    """Commit a card charge after 50 ms."""
    await asyncio.sleep(0.05)
    payment_log.append("charge: committed")
    return "TXN-81"


async def checkout() -> str:
    """Charge under a short timeout, shielding the charge itself."""
    charge_task = asyncio.create_task(charge_card())
    try:
        async with asyncio.timeout(0.01):
            return await asyncio.shield(charge_task)
    except TimeoutError:
        payment_log.append("checkout: stopped waiting; the charge continues")
        return await charge_task  # here we choose to wait for it after all


print(f"{await checkout()=}")
print(payment_log)

await checkout()='TXN-81'
['checkout: stopped waiting; the charge continues', 'charge: committed']


### 5.4.3 Coordination

asyncio has the same primitives as `threading` (Unit 5.1.5), but for tasks on
one loop. They are **not thread-safe**, and waiting on them is an `await`.

| Primitive | Use it to |
| --- | --- |
| `Event` | let many tasks wait until something happens (`set()` wakes all) |
| `Lock` | make a read-`await`-write sequence atomic |
| `Condition` | wait until a predicate on shared state becomes true |
| `Semaphore` / `BoundedSemaphore` | allow at most N tasks into a section |
| `Barrier` (*new in 3.11*) | make N tasks wait for each other before continuing |

**`Event`:** customers wait until the store opens.

In [275]:
store_open = asyncio.Event()
arrivals: list[str] = []


async def customer(name: str) -> None:
    """Wait for the store to open, then arrive."""
    await store_open.wait()
    arrivals.append(name)


customers = [asyncio.create_task(customer(name)) for name in ("asha", "ben", "chen")]
await asyncio.sleep(0.01)
print(f"before set(): {arrivals} {store_open.is_set()=}")
store_open.set()
await asyncio.gather(*customers)
print(f"after set():  {arrivals}")

before set(): [] store_open.is_set()=False
after set():  ['asha', 'ben', 'chen']


**`Lock`.** Between awaits a task cannot be interrupted, so `balance += 10`
needs no lock. But a read, then an `await`, then a write is a race: every task
reads the old balance before anyone writes. A `Lock` makes the sequence
atomic:

In [276]:
class Wallet:
    """A balance that coroutines update."""

    def __init__(self) -> None:
        """Start with a zero balance."""
        self.balance = 0


async def deposit(wallet: Wallet, amount: int, lock: asyncio.Lock | None = None) -> None:
    """Add amount with a read, a yield and a write, optionally under lock."""

    async def read_then_write() -> None:
        """Read the balance, yield, then write the new balance."""
        current = wallet.balance
        await asyncio.sleep(0)  # e.g. a database round trip
        wallet.balance = current + amount

    if lock is None:
        await read_then_write()
    else:
        async with lock:
            await read_then_write()


for label, lock in (("no lock", None), ("with Lock", asyncio.Lock())):
    wallet = Wallet()
    await asyncio.gather(*(deposit(wallet, 10, lock) for _ in range(100)))
    print(f"{label:<9}: balance {wallet.balance} (expected 1000)")

no lock  : balance 10 (expected 1000)
with Lock: balance 1000 (expected 1000)


**`Condition`:** buyers wait until there is enough stock;
`wait_for(predicate)` re-checks after every `notify_all()`.

In [277]:
stock = {"chai": 0}
stock_changed = asyncio.Condition()
served: list[str] = []


async def buyer(name: str, quantity: int) -> None:
    """Wait until enough chai is in stock, then take it."""
    async with stock_changed:
        await stock_changed.wait_for(lambda: stock["chai"] >= quantity)
        stock["chai"] -= quantity
        served.append(f"{name} got {quantity}")


async def restock(amount: int) -> None:
    """Add stock and wake every waiting buyer."""
    async with stock_changed:
        stock["chai"] += amount
        stock_changed.notify_all()


buyers = [asyncio.create_task(buyer("asha", 2)), asyncio.create_task(buyer("ben", 3))]
for amount in (2, 3):
    await asyncio.sleep(0.01)
    await restock(amount)
    await asyncio.sleep(0.01)
    print(f"after restocking {amount}: served={served} left={stock['chai']}")
await asyncio.wait(buyers)
print(f"all buyers done: {all(buyer_task.done() for buyer_task in buyers)}")

after restocking 2: served=['asha got 2'] left=0
after restocking 3: served=['asha got 2', 'ben got 3'] left=0
all buyers done: True


**`Semaphore`:** at most three calls to a payment gateway at once.
`BoundedSemaphore` also catches a `release()` without a matching `acquire()`:

In [278]:
gateway_slots = asyncio.Semaphore(3)
gateway_load = {"now": 0, "peak": 0}


async def call_gateway(payment_id: int) -> int:
    """Call the gateway inside the semaphore, tracking the peak load."""
    async with gateway_slots:
        gateway_load["now"] += 1
        gateway_load["peak"] = max(gateway_load["peak"], gateway_load["now"])
        await asyncio.sleep(0.01)
        gateway_load["now"] -= 1
    return payment_id


with Timer("10 payments, 3 at a time"):
    await asyncio.gather(*(call_gateway(payment_id) for payment_id in range(10)))
print(f"peak concurrent calls: {gateway_load['peak']}")
print(f"extra release on BoundedSemaphore -> {raised_message(asyncio.BoundedSemaphore(1).release)}")

10 payments, 3 at a time took 42.10 ms
peak concurrent calls: 3
extra release on BoundedSemaphore -> ValueError: BoundedSemaphore released too many times


**`Barrier`:** three delivery riders get ready at different speeds; nobody
leaves until all three are ready. `wait()` returns each task's arrival index.

In [279]:
all_ready = asyncio.Barrier(3)
rider_log: list[str] = []


async def rider(name: str, preparation_seconds: float) -> None:
    """Prepare, wait at the barrier for all riders, then leave."""
    await asyncio.sleep(preparation_seconds)
    rider_log.append(f"{name} ready")
    arrival_index = await all_ready.wait()
    rider_log.append(f"{name} leaves (arrived #{arrival_index})")


await asyncio.gather(rider("asha", 0.03), rider("ben", 0.01), rider("chen", 0.02))
print("\n".join(rider_log))
print(f"{all_ready.parties=} {all_ready.n_waiting=} {all_ready.broken=}")

ben ready
chen ready
asha ready
asha leaves (arrived #2)
ben leaves (arrived #0)
chen leaves (arrived #1)
all_ready.parties=3 all_ready.n_waiting=0 all_ready.broken=False


### 5.4.4 Queues and worker pools

`asyncio.Queue` hands items from producer tasks to consumer tasks. `await
queue.put(item)` waits while the queue is full (if it has a `maxsize`);
`await queue.get()` waits while it is empty. Three orderings, plus the
non-waiting forms and their errors:

In [280]:
for queue_class in (asyncio.Queue, asyncio.LifoQueue, asyncio.PriorityQueue):
    demo_queue = queue_class()
    for item in (3, 1, 2):
        demo_queue.put_nowait(item)
    drained_order = [demo_queue.get_nowait() for _ in range(demo_queue.qsize())]
    print(f"{queue_class.__name__:<14} -> {drained_order}")
print(f"get_nowait() on an empty queue -> {raised_error_name(asyncio.Queue().get_nowait)}")
full_queue = asyncio.Queue(maxsize=1)
full_queue.put_nowait("only")
print(
    f"put_nowait() on a full queue  -> {raised_error_name(lambda: full_queue.put_nowait('more'))}"
)

Queue          -> [3, 1, 2]
LifoQueue      -> [2, 1, 3]
PriorityQueue  -> [1, 2, 3]
get_nowait() on an empty queue -> QueueEmpty
put_nowait() on a full queue  -> QueueFull


**A worker pool**, the classic pattern: N worker tasks loop on `get()`; each
calls `task_done()` when an item is finished; `await queue.join()` waits until
every item put so far is done. Before 3.13 the workers were then stopped by
cancelling them:

In [281]:
async def packer(name: str, orders: asyncio.Queue[str], packed: list[str]) -> None:
    """Take orders forever; the pool cancels the packers when the queue is drained."""
    while True:
        order = await orders.get()
        try:
            await asyncio.sleep(0.01)  # pack it
            packed.append(f"{order} by {name}")
        finally:
            orders.task_done()


orders: asyncio.Queue[str] = asyncio.Queue()
for order_number in range(9):
    orders.put_nowait(f"order-{order_number}")
packed_orders: list[str] = []
packers = [
    asyncio.create_task(packer(f"packer-{number}", orders, packed_orders)) for number in range(3)
]
with Timer("9 orders x 10 ms with 3 packers"):
    await orders.join()
for packer_task in packers:
    packer_task.cancel()
await asyncio.gather(*packers, return_exceptions=True)
print(f"{len(packed_orders)} packed, e.g. {packed_orders[:3]}")

9 orders x 10 ms with 3 packers took 30.87 ms
9 packed, e.g. ['order-0 by packer-0', 'order-1 by packer-1', 'order-2 by packer-2']


**Backpressure.** An unbounded queue lets a fast producer pile up items in
memory. With `maxsize=2` the producer's `put()` waits until the slow consumer
makes room, so it can never get more than two items ahead:

In [282]:
bounded_queue: asyncio.Queue[int] = asyncio.Queue(maxsize=2)
flow_log: list[str] = []


async def fast_producer() -> None:
    """Put five items, blocking whenever the bounded queue is full."""
    for item in range(5):
        await bounded_queue.put(item)
        flow_log.append(f"put {item} (queue size {bounded_queue.qsize()})")


async def slow_consumer() -> None:
    """Take five items slowly."""
    for _ in range(5):
        await asyncio.sleep(0.02)
        flow_log.append(f"        got {await bounded_queue.get()}")


await asyncio.gather(fast_producer(), slow_consumer())
print("\n".join(flow_log))

put 0 (queue size 1)
put 1 (queue size 2)
        got 0
put 2 (queue size 2)
        got 1
put 3 (queue size 2)
        got 2
put 4 (queue size 2)
        got 3
        got 4


**`Queue.shutdown()`** (*new in 3.13*) replaces the cancel-the-workers dance.
After `shutdown()`, `put()` raises `QueueShutDown`; consumers still receive
the items already queued, and once the queue is empty their `get()` raises
`QueueShutDown`, which is their signal to exit. `shutdown(immediate=True)`
also drops the items still waiting.

In [283]:
async def shutdown_aware_worker(name: str, jobs: asyncio.Queue[str], report: list[str]) -> None:
    """Process jobs until the queue is shut down."""
    while True:
        try:
            job = await jobs.get()
        except asyncio.QueueShutDown:
            report.append(f"{name} exits")
            return
        await asyncio.sleep(0.01)
        report.append(f"{name} did {job}")
        jobs.task_done()


if hasattr(asyncio.Queue, "shutdown"):
    jobs: asyncio.Queue[str] = asyncio.Queue()
    for job in ("a", "b", "c"):
        jobs.put_nowait(job)
    jobs.shutdown()  # graceful: no new items, drain the rest
    print(f"put after shutdown -> {raised_error_name(lambda: jobs.put_nowait('d'))}")
    shutdown_report: list[str] = []
    await asyncio.gather(
        *(shutdown_aware_worker(name, jobs, shutdown_report) for name in ("w1", "w2"))
    )
    print(shutdown_report)
    dropped: asyncio.Queue[str] = asyncio.Queue()
    for job in ("x", "y"):
        dropped.put_nowait(job)
    dropped.shutdown(immediate=True)
    print(f"immediate: {dropped.qsize()=}, get -> {raised_error_name(dropped.get_nowait)}")
else:
    print("Queue.shutdown() needs Python 3.13+: skipped")

put after shutdown -> QueueShutDown
['w1 did a', 'w2 did b', 'w2 exits', 'w1 did c', 'w1 exits']
immediate: dropped.qsize()=0, get -> QueueShutDown


**Pipelines.** Stages connected by bounded queues: each stage reads from its
inbox and writes to its outbox, and shutdown flows downstream. When `read`
finishes it shuts down `raw`; `parse` drains `raw`, gets `QueueShutDown`, and
shuts down `parsed`; `total` drains `parsed` and returns. (With several
workers per stage, shut the outbox down only after *all* of them finished.)

In [284]:
SALES_LINES = ["chai,2", "vada,1", "chai,3", "idli,4", "vada,2", "chai,1"]


async def read_stage(lines: list[str], outbox: asyncio.Queue[str]) -> None:
    """Feed every line downstream, then shut the queue down."""
    for line in lines:
        await outbox.put(line)
    outbox.shutdown()


async def parse_stage(inbox: asyncio.Queue[str], outbox: asyncio.Queue[tuple[str, int]]) -> None:
    """Parse sku,quantity lines until upstream shuts down."""
    try:
        while True:
            sku, quantity = (await inbox.get()).split(",")
            await outbox.put((sku, int(quantity)))
    except asyncio.QueueShutDown:
        outbox.shutdown()


async def total_stage(inbox: asyncio.Queue[tuple[str, int]]) -> Counter[str]:
    """Sum quantities per SKU until upstream shuts down."""
    totals: Counter[str] = Counter()
    try:
        while True:
            sku, quantity = await inbox.get()
            totals[sku] += quantity
    except asyncio.QueueShutDown:
        return totals


raw_queue: asyncio.Queue[str] = asyncio.Queue(maxsize=2)
parsed_queue: asyncio.Queue[tuple[str, int]] = asyncio.Queue(maxsize=2)
_, _, sales_totals = await asyncio.gather(
    read_stage(SALES_LINES, raw_queue),
    parse_stage(raw_queue, parsed_queue),
    total_stage(parsed_queue),
)
print(dict(sales_totals))

{'chai': 6, 'vada': 3, 'idli': 4}


### 5.4.5 Async protocols

Every synchronous protocol from Units 1.13–1.14 has an async twin whose
methods are coroutines:

| Sync | Async | Methods |
| --- | --- | --- |
| `with` | `async with` | `__aenter__`, `__aexit__` |
| `@contextmanager` | `@asynccontextmanager` | an async generator with one `yield` |
| `for` | `async for` | `__aiter__` (plain method), `__anext__` (raises `StopAsyncIteration`) |
| generator | async generator | `async def` + `yield` |
| `closing()` | `aclosing()` | calls `aclose()` |
| `ExitStack` | `AsyncExitStack` | |

**`async with`:** connecting and disconnecting can both await. `__aexit__`
sees the exception (if any) and returns `False` to let it propagate.

In [285]:
class FakeDatabase:
    """An async context manager: connect on entry, disconnect on exit."""

    def __init__(self, log: list[str]) -> None:
        """Remember the log to write to."""
        self.log = log

    async def __aenter__(self) -> FakeDatabase:
        """Connect after a simulated handshake."""
        await asyncio.sleep(0)  # the network handshake
        self.log.append("connect")
        return self

    async def __aexit__(
        self, error_type: type[BaseException] | None, error: object, traceback: object
    ) -> bool:
        """Disconnect after a simulated round trip; never swallow errors."""
        await asyncio.sleep(0)
        self.log.append(f"disconnect (error: {error_type.__name__ if error_type else None})")
        return False  # do not swallow exceptions

    async def count_orders(self) -> int:
        """Run a fake query."""
        self.log.append("query")
        return 3


database_log: list[str] = []
async with FakeDatabase(database_log) as database:
    print(f"{await database.count_orders()=}")
print(database_log)

await database.count_orders()=3
['connect', 'query', 'disconnect (error: None)']


**`@asynccontextmanager`** writes the same thing as an async generator: code
before `yield` is the entry, code after it the exit. A transaction that
commits on success and rolls back on error:

In [286]:
@contextlib.asynccontextmanager
async def transaction(log: list[str]) -> AsyncIterator[str]:
    """Yield a transaction id, committing or rolling back afterwards."""
    await asyncio.sleep(0)
    log.append("BEGIN")
    try:
        yield "txn-1"
    except Exception:
        log.append("ROLLBACK")
        raise
    else:
        log.append("COMMIT")


transaction_log: list[str] = []
async with transaction(transaction_log) as transaction_id:
    transaction_log.append(f"insert in {transaction_id}")
try:
    async with transaction(transaction_log):
        raise ValueError("duplicate order id")
except ValueError as error:
    transaction_log.append(f"caught {error!r}")
print(transaction_log)

['BEGIN', 'insert in txn-1', 'COMMIT', 'BEGIN', 'ROLLBACK', "caught ValueError('duplicate order id')"]


**`async for`** with a hand-written async iterator: `__aiter__` returns the
iterator (it is *not* a coroutine), `__anext__` is a coroutine that raises
`StopAsyncIteration` at the end:

In [287]:
class OrderFeed:
    """Async iterator over order ids, fetching each one with an await."""

    def __init__(self, order_ids: Iterable[str]) -> None:
        """Remember the order ids to stream."""
        self._order_ids = iter(order_ids)

    def __aiter__(self) -> OrderFeed:
        """Return self: this object is its own async iterator."""
        return self

    async def __anext__(self) -> str:
        """Return the next order id after a simulated fetch."""
        try:
            order_id = next(self._order_ids)
        except StopIteration:
            raise StopAsyncIteration from None
        await asyncio.sleep(0.001)  # fetch the order
        return order_id


async for order_id in OrderFeed(["A-1", "A-2", "A-3"]):
    print(f"processing {order_id}")

processing A-1
processing A-2
processing A-3


**Async generators** are far shorter: `async def` with `yield`. They work with
`async for` and in async comprehensions (`[x async for x in gen]`):

In [288]:
async def price_ticks(sku: str, count: int) -> AsyncIterator[tuple[str, float]]:
    """Stream simulated price updates."""
    price = 20.0
    for _ in range(count):
        await asyncio.sleep(0.001)
        price = round(price * 1.05, 2)
        yield sku, price


ticks = [price async for _, price in price_ticks("chai", 4)]
print(f"{ticks=}")
print(f"{type(price_ticks('chai', 1)).__name__}, {inspect.isasyncgenfunction(price_ticks)=}")

ticks=[21.0, 22.05, 23.15, 24.31]
async_generator, inspect.isasyncgenfunction(price_ticks)=True


**`aclosing`.** If you `break` out of an `async for`, the generator is left
suspended. Its `finally` cannot run synchronously (it might need to await), so
asyncio's finalizer hook schedules `aclose()` for later, *after* your code has
moved on (at the latest, `asyncio.run` closes leftover generators at
shutdown). `contextlib.aclosing(gen)` closes it right at the end of the block:

In [289]:
closing_log: list[str] = []


async def stream_rows() -> AsyncIterator[int]:
    """Yield rows, reporting in finally when the stream is closed."""
    try:
        for row in range(1000):
            await asyncio.sleep(0)
            yield row
    finally:
        closing_log.append("  stream_rows: cursor closed")


async for row in stream_rows():
    if row == 2:
        break
closing_log.append("plain async for: loop exited")
await asyncio.sleep(0.01)
async with contextlib.aclosing(stream_rows()) as rows:
    async for row in rows:
        if row == 2:
            break
closing_log.append("aclosing: block exited")
print("\n".join(closing_log))

plain async for: loop exited
  stream_rows: cursor closed
  stream_rows: cursor closed
aclosing: block exited


### 5.4.6 Blocking the loop

Any call that does not `await` and takes a while (`time.sleep`, `requests.get`,
a big JSON parse, a CPU-heavy loop, a synchronous database driver) freezes
the **whole** loop: no other request makes progress, timers fire late, and
timeouts stop working. Three ways to spot it:

1. **A lag monitor:** a task that sleeps 10 ms in a loop and measures how late
   it wakes up.
2. **Debug mode** (`asyncio.run(main(), debug=True)`, `PYTHONASYNCIODEBUG=1`,
   `python -X dev`, or `loop.set_debug(True)`): the loop logs every callback
   or task step that runs longer than `loop.slow_callback_duration` (0.1 s by
   default), and reports where never-awaited coroutines were created.
3. **A profiler** such as py-spy (Unit 5.5).

The lag monitor first. A handler that calls `time.sleep(0.2)` directly delays
the heartbeat by about 200 ms; the same call through `asyncio.to_thread`
barely delays it:

In [290]:
async def heartbeat(stop: asyncio.Event, interval: float = 0.01) -> float:
    """Sleep in a loop; return the worst lateness (seconds) seen while running."""
    worst = 0.0
    while not stop.is_set():
        before = KERNEL_LOOP.time()
        await asyncio.sleep(interval)
        worst = max(worst, KERNEL_LOOP.time() - before - interval)
    return worst


async def worst_lag_during(work: Callable[[], Awaitable[object]]) -> float:
    """Run work() while a heartbeat runs; return the heartbeat's worst lag."""
    stop = asyncio.Event()
    monitor = asyncio.create_task(heartbeat(stop))
    await asyncio.sleep(0.02)  # let the heartbeat settle
    await work()
    stop.set()
    return await monitor


def render_invoice_pdf() -> str:
    """A blocking library call that takes 200 ms."""
    time.sleep(0.2)  # a blocking library call
    return "invoice.pdf"


async def invoice_handler_blocking() -> str:
    """Call the blocking renderer on the loop thread."""
    return render_invoice_pdf()


async def invoice_handler_threaded() -> str:
    """Run the blocking renderer in a worker thread."""
    return await asyncio.to_thread(render_invoice_pdf)


for handler in (invoice_handler_blocking, invoice_handler_threaded):
    print(f"{handler.__name__:<25} worst loop lag {await worst_lag_during(handler) * 1000:5.0f} ms")

invoice_handler_blocking  worst loop lag   201 ms


invoice_handler_threaded  worst loop lag     2 ms


**Debug mode in a script.** `asyncio.run(main(), debug=True)` with a
threshold of 50 ms. The loop names the slow task step and its duration; it
also shows where a never-awaited coroutine was created:

In [291]:
print(
    run_child(
        """
        import asyncio
        import time


        async def render_invoice():
            time.sleep(0.15)  # blocks the loop
            await asyncio.sleep(0)


        async def send_email():
            await asyncio.sleep(0)


        async def main():
            asyncio.get_running_loop().slow_callback_duration = 0.05
            send_email()  # BUG: never awaited
            await asyncio.gather(render_invoice(), asyncio.sleep(0.01))


        asyncio.run(main(), debug=True)
        """
    )
)

<scratch>/child_script.py:16: RuntimeWarning: coroutine 'send_email' was never awaited
Coroutine created at (most recent call last)
  File "<scratch>/child_script.py", line 20, in <module>
    asyncio.run(main(), debug=True)
  File "<stdlib>/asyncio/runners.py", line 205, in run
    return runner.run(main)
  File "<stdlib>/asyncio/runners.py", line 128, in run
    return self._loop.run_until_complete(task)
  File "<stdlib>/asyncio/base_events.py", line 711, in run_until_complete
    self.run_forever()
  File "<stdlib>/asyncio/base_events.py", line 682, in run_forever
    self._run_once()
  File "<stdlib>/asyncio/base_events.py", line 2065, in _run_once
    handle._run()
  File "<stdlib>/asyncio/events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
  File "<scratch>/child_script.py", line 16, in main
    send_email()  # BUG: never awaited
  send_email()  # BUG: never awaited
Executing <Task pending name='Task-2' coro=<render_invoice() running at <scratch>/child

**Debug mode on a running loop.** The kernel's loop can be switched to debug
mode for a moment. The helper below does that, lowers the threshold, and
collects the `asyncio` logger's warnings in a list instead of printing them.
The kernel's own tasks may show up too, so the cell keeps only messages
about our task, parsed into name, coroutine and duration:

In [292]:
class ListHandler(logging.Handler):
    """A logging handler that keeps formatted messages in a list."""

    def __init__(self, messages: list[str]) -> None:
        """Collect log messages in a list."""
        super().__init__()
        self.messages = messages

    def emit(self, record: logging.LogRecord) -> None:
        """Append the formatted message."""
        self.messages.append(record.getMessage())


@contextlib.contextmanager
def slow_callback_report(threshold: float = 0.05) -> Generator[list[str]]:
    """Temporarily enable debug mode on the kernel loop; yield the slow-step messages."""
    messages: list[str] = []
    asyncio_logger = logging.getLogger("asyncio")
    handler = ListHandler(messages)
    saved = (KERNEL_LOOP.get_debug(), KERNEL_LOOP.slow_callback_duration, asyncio_logger.propagate)
    KERNEL_LOOP.set_debug(True)
    KERNEL_LOOP.slow_callback_duration = threshold
    asyncio_logger.addHandler(handler)
    asyncio_logger.propagate = False
    try:
        yield messages
    finally:
        asyncio_logger.removeHandler(handler)
        KERNEL_LOOP.set_debug(saved[0])
        KERNEL_LOOP.slow_callback_duration, asyncio_logger.propagate = saved[1], saved[2]

Run the blocking handler and the threaded one under it, as named tasks:

In [293]:
SLOW_STEP = re.compile(
    r"name='(?P<task>[^']+)' coro=<(?P<coro>\w+)\(\).*took (?P<seconds>[\d.]+) seconds"
)
with slow_callback_report(threshold=0.05) as slow_steps:
    await asyncio.create_task(invoice_handler_blocking(), name="render-invoice")
    await asyncio.create_task(invoice_handler_threaded(), name="render-invoice-threaded")
for message in slow_steps:
    if (match := SLOW_STEP.search(message)) and match["task"].startswith("render-invoice"):
        print(f"slow step: task {match['task']} ({match['coro']}) took {match['seconds']} s")
threshold = KERNEL_LOOP.slow_callback_duration
print(f"debug mode restored to {KERNEL_LOOP.get_debug()}, threshold {threshold} s")

slow step: task render-invoice (invoice_handler_blocking) took 0.201 s
debug mode restored to False, threshold 0.1 s


**Fixes.** Which tool depends on *why* the call blocks:

| Blocking because | Fix | Why |
| --- | --- | --- |
| it waits on I/O in a sync library | `await asyncio.to_thread(func, *args)` | the thread waits; the loop stays free (the GIL is released while waiting) |
| it waits on I/O and an async library exists | use the async library (`httpx.AsyncClient`, `asyncpg`, `aiofiles`, `asyncio.create_subprocess_exec`) | no thread per call |
| it computes in pure Python | `await loop.run_in_executor(process_pool, func, *args)` | a thread would still hold the GIL and slow the loop (Unit 5.1); a process runs truly in parallel |
| it computes in C that releases the GIL (`hashlib`, NumPy, zlib) | `to_thread` | threads run in parallel |

`to_thread(func, *args)` is `loop.run_in_executor(None, functools.partial(...))`
on the default thread pool, plus copying `contextvars` (5.4.7). For CPU-bound
Python, the worker function must be importable by child processes (Unit 5.2),
so it lives in a `%%writefile` module:

In [294]:
%%writefile _scratch/phase5/p5_async_workers.py
"""CPU-bound worker for Unit 5.4.6 (process pools need importable functions)."""


def count_primes(limit: int) -> int:
    """Count primes below limit with trial division: pure Python, holds the GIL."""
    count = 0
    for candidate in range(2, limit):
        if all(candidate % divisor for divisor in range(2, int(candidate**0.5) + 1)):
            count += 1
    return count

Overwriting _scratch/phase5/p5_async_workers.py


Counting primes below 120 000 three ways: directly in the coroutine (the
loop is frozen for the whole computation), in the default thread pool (the
thread competes with the loop for the GIL), and in a process pool (the loop is
untouched). Timings and lags vary by machine; compare the columns:

In [295]:
async_workers = fresh_import("p5_async_workers")
prime_limit = 120_000 * DEMO_SCALE


async def primes_directly() -> int:
    """Count primes on the loop thread."""
    return async_workers.count_primes(prime_limit)


async def primes_in_thread() -> int:
    """Count primes in a worker thread (still holds the GIL)."""
    return await asyncio.to_thread(async_workers.count_primes, prime_limit)


with concurrent.futures.ProcessPoolExecutor(max_workers=1) as process_pool:
    await KERNEL_LOOP.run_in_executor(
        process_pool, async_workers.count_primes, 10
    )  # warm up the worker

    async def primes_in_process() -> int:
        """Count primes in a worker process."""
        return await KERNEL_LOOP.run_in_executor(
            process_pool, async_workers.count_primes, prime_limit
        )

    for strategy in (primes_directly, primes_in_thread, primes_in_process):
        with Timer(strategy.__name__, report=None) as strategy_timer:
            lag = await worst_lag_during(strategy)
        seconds_taken = strategy_timer.elapsed_seconds
        print(f"{strategy.__name__:<18} took {seconds_taken:.2f} s, worst lag {lag * 1000:6.0f} ms")

primes_directly    took 0.11 s, worst lag     94 ms


primes_in_thread   took 0.13 s, worst lag      8 ms


primes_in_process  took 0.13 s, worst lag      1 ms


**Async libraries.** Asyncio itself has no asynchronous file API (neither does
the OS, portably). `aiofiles` gives `async with aiofiles.open(...)` and
`await file.read()`, implemented by running each call in the loop's default
thread pool, which is what libuv does for `fs` in Node:

In [296]:
if HAS_AIOFILES:
    import aiofiles

    report_path = PHASE5_DIR / "async_report.txt"
    async with aiofiles.open(report_path, "w", encoding="utf-8") as report_file:
        await report_file.write("chai,5\nvada,3\n")
    async with aiofiles.open(report_path, encoding="utf-8") as report_file:
        print(f"read back with aiofiles: {(await report_file.read()).splitlines()}")
else:
    print("aiofiles is not installed: skipping (uv add aiofiles)")

read back with aiofiles: ['chai,5', 'vada,3']


Subprocesses have a native async API, `asyncio.create_subprocess_exec`, built
on the selector (pipes are file descriptors too); several can run at once
without threads:

In [297]:
async def run_python(snippet: str) -> str:
    """Run a Python snippet as an async subprocess; return its output."""
    process = await asyncio.create_subprocess_exec(
        sys.executable, "-c", snippet, stdout=asyncio.subprocess.PIPE, env=CHILD_ENV
    )
    stdout, _ = await process.communicate()
    return f"exit {process.returncode}: {stdout.decode().strip()}"


with Timer("three child interpreters started concurrently"):
    outputs = await asyncio.gather(
        *(
            run_python(f"import time; time.sleep(0.2); print({number} * {number})")
            for number in (2, 3, 4)
        )
    )
print(outputs)

three child interpreters started concurrently took 216.14 ms
['exit 0: 4', 'exit 0: 9', 'exit 0: 16']


### 5.4.7 `contextvars`

A global variable is shared by every concurrent request, and a thread-local
does not help when one thread runs a thousand tasks. A
**`contextvars.ContextVar`** holds a value per *context*, and asyncio gives
every Task its own copy of the context, taken when the task is created.
Node's equivalent is `AsyncLocalStorage`.

| API | Meaning |
| --- | --- |
| `var = ContextVar("name", default=...)` | create once, at module level |
| `var.get()` | the value in the current context (or the default) |
| `token = var.set(value)` / `var.reset(token)` | set, then restore the previous value |
| `with var.set(value):` | the same as set + reset (*new in 3.14*: `Token` is a context manager) |
| `contextvars.copy_context()`, `ctx.run(func)` | snapshot the context; run a function inside a snapshot |

In [298]:
tenant = contextvars.ContextVar("tenant", default="public")
print(f"default: {tenant.get()}")
token = tenant.set("acme")
print(f"after set: {tenant.get()}")
tenant.reset(token)
print(f"after reset: {tenant.get()}")
if sys.version_info >= (3, 14):
    with tenant.set("globex"):
        print(f"inside with: {tenant.get()}")
    print(f"after with: {tenant.get()}")
token = tenant.set("initech")
snapshot = contextvars.copy_context()  # remembers tenant = "initech"
tenant.reset(token)
print(f"now: {tenant.get()}; inside the snapshot: {snapshot.run(tenant.get)}")

default: public
after set: acme
after reset: public
inside with: globex
after with: public
now: public; inside the snapshot: initech


**Each task works on a copy.** A child task sees what its parent had set when
the child was created; a value the child sets never leaks back to the parent,
and two concurrent requests never see each other's values. (`gather` wraps
each coroutine in its own task, so each `handle` call gets its own copy.)

In [299]:
request_id = contextvars.ContextVar("request_id", default="-")
context_log: list[str] = []


async def child_step() -> None:
    """Read and change the request id inside a child task."""
    context_log.append(f"  child task sees {request_id.get()}")
    request_id.set("changed-by-child")


async def handle(new_request_id: str) -> None:
    """Set a request id, run a child task, then read it again."""
    request_id.set(new_request_id)
    await asyncio.create_task(child_step())
    await asyncio.sleep(0.01)  # let the other request run in between
    context_log.append(f"{new_request_id}: parent still sees {request_id.get()}")


await asyncio.gather(handle("req-1"), handle("req-2"))
print("\n".join(context_log))
print(f"the cell's own context: {request_id.get()}")

  child task sees req-1
  child task sees req-2
req-1: parent still sees req-1
req-2: parent still sees req-2
the cell's own context: -


**Log correlation.** A `logging.Filter` copies the current request id onto
every record, and the formatter prints it. Two requests interleave, yet every
line is tagged with the right id, without passing ids through function
arguments. (The handler writes to a `StringIO` so the output is shown here.)

In [300]:
class RequestIdFilter(logging.Filter):
    """Attach the current request id to every log record."""

    def filter(self, record: logging.LogRecord) -> bool:
        """Copy the current request id onto the log record."""
        record.request_id = request_id.get()
        return True


log_stream = io.StringIO()
stream_handler = logging.StreamHandler(log_stream)
stream_handler.setFormatter(logging.Formatter("%(request_id)-7s %(levelname)s %(message)s"))
stream_handler.addFilter(RequestIdFilter())
shop_logger = logging.getLogger("shop.orders.demo")
shop_logger.handlers[:] = [stream_handler]
shop_logger.setLevel(logging.INFO)
shop_logger.propagate = False


async def handle_order(new_request_id: str, sku: str) -> None:
    """Log two lines with a request id set for this request only."""
    token = request_id.set(new_request_id)
    try:
        shop_logger.info("checking stock for %s", sku)
        await asyncio.sleep(0.01)
        shop_logger.info("reserved %s", sku)
    finally:
        request_id.reset(token)


await asyncio.gather(handle_order("req-a1", "chai"), handle_order("req-b2", "vada"))
print(log_stream.getvalue().rstrip())

req-a1  INFO checking stock for chai
req-b2  INFO checking stock for vada
req-a1  INFO reserved chai
req-b2  INFO reserved vada


**Contexts and threads.** `asyncio.to_thread` runs the function inside a copy
of the caller's context; `loop.run_in_executor` does not, so the worker sees
the default value. Fix it by passing `contextvars.copy_context().run` as the
function:

In [301]:
def audit_entry() -> str:
    """Runs in a worker thread and reads the request id."""
    return f"audit for {request_id.get()} on {threading.current_thread().name.split('_')[0]}"


token = request_id.set("req-77")
try:
    print(f"to_thread:                 {await asyncio.to_thread(audit_entry)}")
    print(f"run_in_executor:           {await KERNEL_LOOP.run_in_executor(None, audit_entry)}")
    snapshot = contextvars.copy_context()
    in_snapshot = await KERNEL_LOOP.run_in_executor(None, snapshot.run, audit_entry)
    print(f"run_in_executor + ctx.run: {in_snapshot}")
finally:
    request_id.reset(token)

to_thread:                 audit for req-77 on asyncio
run_in_executor:           audit for - on asyncio
run_in_executor + ctx.run: audit for req-77 on asyncio


### 5.4.8 HTTP clients

`httpx.AsyncClient` is the async twin of the `httpx.Client` from Unit 4.6.
Create **one** client per application (or per upstream service) and share
it: it keeps a connection pool, so repeated requests reuse TCP connections
instead of paying a new handshake each time. Always close it, with `async
with` or `await client.aclose()`.

Twenty concurrent requests to `/slow?seconds=0.2` through one shared client
take about 0.2 s. The server's `/connections` counter shows the pool at work:
the first batch opens connections, the second batch reuses them.

In [302]:
async def connections_opened(client: httpx.AsyncClient) -> int:
    """Total TCP connections the shop API has accepted so far."""
    return (await client.get("/connections")).json()["connections"]


async with httpx.AsyncClient(base_url=SHOP_API, timeout=5) as shared_client:
    for batch in ("first batch", "second batch"):
        before = await connections_opened(shared_client)
        with Timer(f"{batch}: 20 concurrent requests", report=None) as batch_timer:
            responses = await asyncio.gather(
                *(shared_client.get("/slow", params={"seconds": 0.2}) for _ in range(20))
            )
        opened = await connections_opened(shared_client) - before
        statuses = Counter(response.status_code for response in responses)
        seconds_taken = batch_timer.elapsed_seconds
        print(
            f"{batch}: {seconds_taken:.2f} s, statuses {dict(statuses)}, new connections {opened}"
        )

first batch: 0.25 s, statuses {200: 20}, new connections 19


second batch: 0.25 s, statuses {200: 20}, new connections 0


**Limits.** `httpx.Limits(max_connections=5)` caps the pool. The other
requests wait for a free connection, so twenty 0.2 s requests take about
4 × 0.2 s. That protects the upstream service, the way a semaphore would:

In [303]:
limits = httpx.Limits(max_connections=5, max_keepalive_connections=5)
async with httpx.AsyncClient(base_url=SHOP_API, timeout=5, limits=limits) as limited_client:
    before = await connections_opened(limited_client)
    with Timer("20 requests through a pool of 5 connections"):
        await asyncio.gather(
            *(limited_client.get("/slow", params={"seconds": 0.2}) for _ in range(20))
        )
    print(f"new connections: {await connections_opened(limited_client) - before}")

20 requests through a pool of 5 connections took 990.56 ms
new connections: 4


**One client per request** is the most common mistake: every request opens
(and closes) its own connection. Thirty quick `/rates` calls each way:

In [304]:
async def rates_with_new_client() -> int:
    """Fetch /rates with a client created just for this call."""
    async with httpx.AsyncClient(base_url=SHOP_API) as throwaway_client:
        return (await throwaway_client.get("/rates")).status_code


async with httpx.AsyncClient(base_url=SHOP_API) as counter_client:
    before = await connections_opened(counter_client)
    with Timer("30 requests, a new client each", report=None) as per_request_timer:
        await asyncio.gather(*(rates_with_new_client() for _ in range(30)))
    per_request_connections = await connections_opened(counter_client) - before
    before = await connections_opened(counter_client)
    with Timer("30 requests, one shared client", report=None) as shared_timer:
        await asyncio.gather(*(counter_client.get("/rates") for _ in range(30)))
    shared_connections = await connections_opened(counter_client) - before
per_request_ms, shared_ms = (
    per_request_timer.elapsed_seconds * 1000,
    shared_timer.elapsed_seconds * 1000,
)
print(f"new client per request: {per_request_ms:6.1f} ms, {per_request_connections} connections")
print(f"one shared client:      {shared_ms:6.1f} ms, {shared_connections} connections")

new client per request:  119.3 ms, 30 connections
one shared client:        91.4 ms, 15 connections


**Timeouts.** httpx has four timeouts: `connect`, `read`, `write` and `pool`
(waiting for a free connection). There is no default *total* timeout, so wrap
a whole operation in `asyncio.timeout` when you need a deadline:

In [305]:
async with httpx.AsyncClient(
    base_url=SHOP_API, timeout=httpx.Timeout(5.0, read=0.1)
) as impatient_client:
    try:
        await impatient_client.get("/slow", params={"seconds": 0.5})
    except httpx.ReadTimeout as error:
        print(f"read timeout: {type(error).__name__}")

one_connection = httpx.Limits(max_connections=1)
async with httpx.AsyncClient(
    base_url=SHOP_API, timeout=httpx.Timeout(5.0, pool=0.05), limits=one_connection
) as tiny_pool:
    hog = asyncio.create_task(tiny_pool.get("/slow", params={"seconds": 0.3}))
    await asyncio.sleep(0.05)  # the hog now holds the only connection
    try:
        await tiny_pool.get("/rates")
    except httpx.PoolTimeout as error:
        print(f"pool timeout: {type(error).__name__}")
    await hog

async with httpx.AsyncClient(base_url=SHOP_API, timeout=5) as client_with_deadline:
    try:
        async with asyncio.timeout(0.15):  # total deadline for two calls
            await client_with_deadline.get("/slow", params={"seconds": 0.1})
            await client_with_deadline.get("/slow", params={"seconds": 0.1})
    except TimeoutError:
        print("total deadline: TimeoutError from asyncio.timeout")

read timeout: ReadTimeout


pool timeout: PoolTimeout


total deadline: TimeoutError from asyncio.timeout


Status errors are not raised automatically; `raise_for_status()` turns 4xx/5xx
into `httpx.HTTPStatusError`:

In [306]:
async with httpx.AsyncClient(base_url=SHOP_API) as status_client:
    busy = await status_client.get("/status/503")
    print(f"{busy.status_code=} {busy.is_success=}")
    print(raised_message(busy.raise_for_status).splitlines()[0])

busy.status_code=503 busy.is_success=False
HTTPStatusError: Server error '503 Service Unavailable' for url 'http://127.0.0.1:18046/status/503'


**aiohttp** is the other major async HTTP library (client *and* server; it
predates httpx). It is not part of this course's dependencies, so the next cell
runs only if it is installed. The client looks like this:

```python
import aiohttp

async with aiohttp.ClientSession(base_url=SHOP_API, timeout=aiohttp.ClientTimeout(total=5)) as session:
    async with session.get("/rates") as response:
        rates = await response.json()
```

| | httpx | aiohttp |
| --- | --- | --- |
| Sync API too | yes (`httpx.Client`) | no |
| HTTP/2 | yes (`http2=True`) | no |
| Server | no | yes |
| Test transport | `MockTransport` | `aioresponses` (third party) |

In [307]:
if HAS_AIOHTTP:
    import aiohttp

    async with aiohttp.ClientSession(
        base_url=SHOP_API, timeout=aiohttp.ClientTimeout(total=5)
    ) as session:
        async with session.get("/rates") as aiohttp_response:
            print(f"aiohttp: {aiohttp_response.status} {await aiohttp_response.json()}")
else:
    print("aiohttp is not installed (not a course dependency): skipping the live demo")

aiohttp is not installed (not a course dependency): skipping the live demo


### 5.4.9 Rate limiting and retries

**Token bucket.** A bucket holds up to `capacity` tokens and refills at
`rate` tokens per second; each request takes one token and waits when the
bucket is empty. Bursts up to `capacity` go through at once, then requests are
spaced `1 / rate` apart. (Unit 1.6.14's `allow_request` is a sliding-window
*check* that rejects; a token bucket *waits*.) The lock makes waiters queue
up fairly instead of all waking at once:

In [308]:
class AsyncTokenBucket:
    """Allow `rate` acquisitions per second on average, bursts up to `capacity`."""

    def __init__(self, rate: float, capacity: int) -> None:
        """Start full: capacity tokens, refilled at rate per second."""
        self.rate, self.capacity = rate, capacity
        self.tokens = float(capacity)
        self.updated = KERNEL_LOOP.time()
        self.lock = asyncio.Lock()

    async def acquire(self) -> None:
        """Wait until a token is available, then take it."""
        async with self.lock:
            while True:
                now = KERNEL_LOOP.time()
                self.tokens = min(self.capacity, self.tokens + (now - self.updated) * self.rate)
                self.updated = now
                if self.tokens >= 1:
                    self.tokens -= 1
                    return
                await asyncio.sleep((1 - self.tokens) / self.rate)

Fifteen requests to the shop API at 50 per second with a burst of 5: the
first five start immediately, the rest about 20 ms apart:

In [309]:
bucket = AsyncTokenBucket(rate=50, capacity=5)
bucket_start = KERNEL_LOOP.time()
start_times_ms: list[int] = []


async def rate_limited_get(client: httpx.AsyncClient) -> int:
    """Take a token, record the start time, fetch /rates."""
    await bucket.acquire()
    start_times_ms.append(round((KERNEL_LOOP.time() - bucket_start) * 1000))
    return (await client.get("/rates")).status_code


async with httpx.AsyncClient(base_url=SHOP_API) as limited:
    statuses = await asyncio.gather(*(rate_limited_get(limited) for _ in range(15)))
gaps_after_burst = [later - earlier for earlier, later in itertools.pairwise(start_times_ms[4:])]
print(f"start times (ms): {start_times_ms}")
average_gap = sum(gaps_after_burst) / len(gaps_after_burst)
print(f"{len(statuses)} requests; after the burst, one every {average_gap:.1f} ms (limit 20 ms)")

start times (ms): [6, 7, 7, 7, 7, 28, 48, 67, 87, 107, 127, 147, 167, 187, 207]
15 requests; after the burst, one every 20.0 ms (limit 20 ms)


**Backoff with jitter.** Retrying immediately hammers a struggling service.
*Exponential backoff* doubles the wait each attempt (capped); *jitter*
randomizes it so that a thousand clients that failed together do not all retry
at the same instant (AWS Architecture Blog, "Exponential Backoff and Jitter"):

| Strategy | Delay before retry *n* (base *b*, cap *c*) |
| --- | --- |
| exponential | `min(c, b * 2**n)` |
| full jitter | `uniform(0, min(c, b * 2**n))` |
| equal jitter | half of the exponential delay + `uniform(0, half)` |

In [310]:
def backoff_delay(
    attempt: int, strategy: str, rng: random.Random, base: float = 0.1, cap: float = 2.0
) -> float:
    """Delay in seconds before retry number `attempt` (0-based)."""
    exponential = min(cap, base * 2**attempt)
    match strategy:
        case "exponential":
            return exponential
        case "full jitter":
            return rng.uniform(0, exponential)
        case "equal jitter":
            return exponential / 2 + rng.uniform(0, exponential / 2)
    raise ValueError(f"unknown strategy {strategy!r}")


backoff_rng = random.Random(42)  # seeded: the table is the same on every run
print(
    f"{'attempt':<8}"
    + "".join(f"{strategy:>14}" for strategy in ("exponential", "full jitter", "equal jitter"))
)
for attempt in range(6):
    delays = [
        backoff_delay(attempt, strategy, backoff_rng)
        for strategy in ("exponential", "full jitter", "equal jitter")
    ]
    print(f"{attempt:<8}" + "".join(f"{delay:>13.3f}s" for delay in delays))

attempt    exponential   full jitter  equal jitter
0               0.100s        0.064s        0.051s
1               0.200s        0.055s        0.122s
2               0.400s        0.295s        0.335s
3               0.800s        0.714s        0.435s
4               1.600s        0.675s        0.824s
5               2.000s        0.437s        1.505s


A retry loop with full jitter against the flaky endpoint, which answers 503
twice and then 200. Only 503 (and connection errors) are retried; a 404
would be returned at once. Lab 5.4 exercise 3 turns this into a reusable
decorator.

In [311]:
async def fetch_flaky_with_retries(
    client: httpx.AsyncClient, key: str, attempts: int = 4
) -> dict[str, object]:
    """GET /flaky/<key>?fail=2, retrying 503s with full-jitter backoff."""
    retry_rng = random.Random(7)
    for attempt in range(attempts):
        try:
            response = await client.get(f"/flaky/{key}", params={"fail": 2})
            if response.status_code != 503:
                response.raise_for_status()
                return response.json()
            reason = "503"
        except httpx.TransportError as error:
            reason = type(error).__name__
        delay = backoff_delay(attempt, "full jitter", retry_rng, base=0.02)
        print(f"  attempt {attempt + 1}: {reason}, retrying in {delay * 1000:.0f} ms")
        await asyncio.sleep(delay)
    raise RuntimeError(f"gave up after {attempts} attempts")


async with httpx.AsyncClient(base_url=SHOP_API) as retry_client:
    await retry_client.get("/reset")  # clear the server's attempt counters
    print(await fetch_flaky_with_retries(retry_client, "unit54-retry"))

  attempt 1: 503, retrying in 6 ms
  attempt 2: 503, retrying in 6 ms
{'ok': True, 'attempt': 3}


`tenacity` (Unit 4.6) supports async code too: `AsyncRetrying` in an
`async for` loop, or the `@retry` decorator on an `async def`:

In [312]:
if HAS_TENACITY:
    from tenacity import (
        AsyncRetrying,
        retry_if_exception_type,
        stop_after_attempt,
        wait_random_exponential,
    )

    async with httpx.AsyncClient(base_url=SHOP_API) as tenacity_client:
        await tenacity_client.get("/reset")
        retrying = AsyncRetrying(
            stop=stop_after_attempt(5),
            wait=wait_random_exponential(multiplier=0.01, max=0.1),
            retry=retry_if_exception_type(httpx.HTTPStatusError),
            reraise=True,
        )
        async for attempt in retrying:
            with attempt:
                flaky_response = await tenacity_client.get(
                    "/flaky/unit54-tenacity", params={"fail": 2}
                )
                flaky_response.raise_for_status()
    attempt_number = attempt.retry_state.attempt_number
    print(f"tenacity: succeeded on attempt {attempt_number}: {flaky_response.json()}")
else:
    print("tenacity is not installed: skipping")

tenacity: succeeded on attempt 3: {'ok': True, 'attempt': 3}


### 5.4.10 Performance and testing

**uvloop** replaces asyncio's pure-Python loop with one built on libuv, the
same C library Node uses, written in Cython. It is a drop-in loop
implementation: `uvloop.run(main())`, or `asyncio.run(main(),
loop_factory=uvloop.new_event_loop)` (`loop_factory` is *new in 3.12*; the
older `uvloop.install()` / event-loop policies are deprecated). It speeds up
the loop's own overhead (I/O callbacks, transports), not your Python code. It
does not support Windows.

The benchmark runs in a child process: TCP echo round trips over loopback and
many short tasks, on each loop. Timings vary; read the ratio. First the
script's workloads:

In [313]:
UVLOOP_WORKLOADS_SOURCE = r"""
import asyncio
import os
import time

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))
ROUND_TRIPS = 5_000 * DEMO_SCALE
TASKS = 50_000 * DEMO_SCALE


async def echo(reader, writer):
    while line := await reader.readline():
        writer.write(line)
        await writer.drain()
    writer.close()


async def round_trips():
    server = await asyncio.start_server(echo, "127.0.0.1", 0)
    reader, writer = await asyncio.open_connection("127.0.0.1", server.sockets[0].getsockname()[1])
    started = time.perf_counter()
    for _ in range(ROUND_TRIPS):
        writer.write(b"ping\n")
        await reader.readline()
    elapsed = time.perf_counter() - started
    writer.close()
    await writer.wait_closed()
    server.close()
    await server.wait_closed()
    return elapsed


async def many_tasks():
    started = time.perf_counter()
    await asyncio.gather(*(asyncio.sleep(0) for _ in range(TASKS)))
    return time.perf_counter() - started
"""

The second half of the script runs both workloads on each loop with
`asyncio.Runner(loop_factory=...)` and prints the times and the ratio:

In [314]:
UVLOOP_RUNNER_SOURCE = r"""
def bench(loop_factory):
    with asyncio.Runner(loop_factory=loop_factory) as runner:
        return runner.run(round_trips()), runner.run(many_tasks())


try:
    import uvloop
except ImportError:
    uvloop = None
results = {"asyncio (default loop)": bench(None)}
if uvloop is not None:
    results[f"uvloop {uvloop.__version__}"] = bench(uvloop.new_event_loop)
print(f"{ROUND_TRIPS} TCP round trips and {TASKS} tasks on each loop:")
for label, (echo_seconds, task_seconds) in results.items():
    print(f"  {label:<23} round trips {echo_seconds:.3f} s | tasks {task_seconds:.3f} s")
if len(results) == 2:
    (echo_a, task_a), (echo_b, task_b) = results.values()
    print(f"uvloop speedup: round trips {echo_a / echo_b:.1f}x, tasks {task_a / task_b:.1f}x")
"""

Run the two halves as one script (or skip it where uvloop is unavailable):

In [315]:
if HAS_UVLOOP:
    print(run_child(UVLOOP_WORKLOADS_SOURCE + "\n\n" + UVLOOP_RUNNER_SOURCE))
else:
    reason = (
        "not supported on Windows" if sys.platform == "win32" else "not installed (uv add uvloop)"
    )
    print(f"uvloop {reason}: skipping the benchmark")

5000 TCP round trips and 50000 tasks on each loop:
  asyncio (default loop)  round trips 0.102 s | tasks 0.173 s
  uvloop 0.23.0           round trips 0.043 s | tasks 0.117 s
uvloop speedup: round trips 2.4x, tasks 1.5x


**`AsyncMock`** (*new in 3.8*) is a mock whose calls return awaitables. It
records awaits as well as calls (`assert_awaited_once_with`, `await_count`),
and `side_effect` can raise:

In [316]:
async def checkout_with(gateway: Any, amount: int) -> str:
    """Charge the gateway and return the status it reports."""
    result = await gateway.charge(amount=amount, currency="INR")
    return result["status"]


payment_gateway = AsyncMock()
payment_gateway.charge.return_value = {"status": "ok", "txn": "T-1"}
print(f"{await checkout_with(payment_gateway, 250)=}")
payment_gateway.charge.assert_awaited_once_with(amount=250, currency="INR")
print(
    f"{payment_gateway.charge.await_count=} {inspect.iscoroutinefunction(payment_gateway.charge)=}"
)
payment_gateway.charge.side_effect = ConnectionError("gateway down")
try:
    await checkout_with(payment_gateway, 99)
except ConnectionError as error:
    print(f"side_effect raised {type(error).__name__}: {error}")

await checkout_with(payment_gateway, 250)='ok'
payment_gateway.charge.await_count=1 inspect.iscoroutinefunction(payment_gateway.charge)=True
side_effect raised ConnectionError: gateway down


**`httpx.MockTransport`** swaps the network for a function (sync or async)
that receives the `Request` and returns a `Response`. The client code under
test is unchanged; nothing leaves the process (the `shop.test` host is never
resolved):

In [317]:
seen_paths: list[str] = []


async def fake_shop(request: httpx.Request) -> httpx.Response:
    """A mock shop: /rates answers, every other path is a 404."""
    seen_paths.append(request.url.path)
    if request.url.path == "/rates":
        return httpx.Response(200, json={"rates": {"USD": 0.012}})
    return httpx.Response(404, json={"error": "no route"})


async with httpx.AsyncClient(
    transport=httpx.MockTransport(fake_shop), base_url="https://shop.test"
) as mocked_client:
    rates_reply, missing_reply = await asyncio.gather(
        mocked_client.get("/rates"), mocked_client.get("/nope")
    )
print(f"{rates_reply.json()=} {missing_reply.status_code=} {seen_paths=}")

rates_reply.json()={'rates': {'USD': 0.012}} missing_reply.status_code=404 seen_paths=['/rates', '/nope']


**pytest and async tests.** A test function that is `async def` needs a
plugin to run it on an event loop:

| Plugin | Marker | Notes |
| --- | --- | --- |
| anyio (bundled with the `anyio` package, which httpx depends on) | `@pytest.mark.anyio` + an `anyio_backend` fixture | runs on asyncio or trio |
| pytest-asyncio | `@pytest.mark.asyncio` (or `asyncio_mode = "auto"`) | asyncio only; not a course dependency |

The test module below uses anyio's plugin, `AsyncMock`, `MockTransport` and a
timeout. First its header: imports, the anyio marker and backend fixture, and
the code under test:

In [318]:
ASYNC_TEST_HEADER = r'''
"""Async tests run by pytest with anyio's plugin (bundled with anyio)."""

import asyncio
from unittest.mock import AsyncMock

import httpx
import pytest

pytestmark = pytest.mark.anyio  # run every test in this module on an event loop


@pytest.fixture
def anyio_backend() -> str:
    return "asyncio"  # "trio" also works if trio is installed


async def fetch_rate(client: httpx.AsyncClient, currency: str) -> float:
    response = await client.get("/rates")
    response.raise_for_status()
    return response.json()["rates"][currency]


def rates_ok(request: httpx.Request) -> httpx.Response:
    return httpx.Response(200, json={"rates": {"USD": 0.01}})
'''

The four tests: a `MockTransport` success, a 503 that must raise, an
`AsyncMock` awaited exactly once, and a timeout:

In [319]:
ASYNC_TEST_CASES = r"""
async def test_fetch_rate_with_mock_transport() -> None:
    transport = httpx.MockTransport(rates_ok)
    async with httpx.AsyncClient(transport=transport, base_url="https://shop.test") as client:
        assert await fetch_rate(client, "USD") == 0.01


async def test_server_error_raises() -> None:
    transport = httpx.MockTransport(lambda request: httpx.Response(503))
    async with httpx.AsyncClient(transport=transport, base_url="https://shop.test") as client:
        with pytest.raises(httpx.HTTPStatusError):
            await fetch_rate(client, "USD")


async def test_gateway_is_awaited_once() -> None:
    gateway = AsyncMock()
    gateway.charge.return_value = "ok"
    assert await gateway.charge(250, currency="INR") == "ok"
    gateway.charge.assert_awaited_once_with(250, currency="INR")


async def test_slow_call_times_out() -> None:
    with pytest.raises(TimeoutError):
        async with asyncio.timeout(0.01):
            await asyncio.sleep(1)
"""

The next cell writes it to a scratch folder with its own `pytest.ini` (so the
repository's settings do not apply) and runs pytest in a child process:

In [320]:
async_test_dir = PHASE5_DIR / "async_tests"
async_test_dir.mkdir(exist_ok=True)
pytest_ini = "[pytest]\naddopts = -p no:cacheprovider\n"
(async_test_dir / "pytest.ini").write_text(pytest_ini, encoding="utf-8")
test_module = textwrap.dedent(ASYNC_TEST_HEADER + "\n\n" + ASYNC_TEST_CASES).lstrip()
(async_test_dir / "test_async_shop.py").write_text(test_module, encoding="utf-8")
pytest_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-v", "--no-header", "test_async_shop.py"],
    capture_output=True, text=True, cwd=async_test_dir, env=CHILD_ENV, timeout=120,
)  # fmt: skip
print(re.sub(r"in [\d.]+s", "in <time>", pytest_run.stdout.strip()))

============================= test session starts ==============================
collecting ... collected 4 items

test_async_shop.py::test_fetch_rate_with_mock_transport PASSED           [ 25%]
test_async_shop.py::test_server_error_raises PASSED                      [ 50%]
test_async_shop.py::test_gateway_is_awaited_once PASSED                  [ 75%]
test_async_shop.py::test_slow_call_times_out PASSED                      [100%]

============================== 4 passed in <time> ===============================


With pytest-asyncio installed, the `@pytest.mark.asyncio` form runs the same
way (otherwise the cell prints a skip message):

In [321]:
PYTEST_ASYNCIO_SOURCE = r'''
"""The same idea with pytest-asyncio's marker."""

import asyncio

import pytest


@pytest.mark.asyncio
async def test_sleep_returns_result() -> None:
    assert await asyncio.sleep(0, result="done") == "done"
'''

Run it if the plugin is installed:

In [322]:
if HAS_PYTEST_ASYNCIO:
    marker_module = textwrap.dedent(PYTEST_ASYNCIO_SOURCE).lstrip()
    (async_test_dir / "test_pytest_asyncio.py").write_text(marker_module, encoding="utf-8")
    pytest_asyncio_run = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", "test_pytest_asyncio.py"],
        capture_output=True, text=True, cwd=async_test_dir, env=CHILD_ENV, timeout=120,
    )  # fmt: skip
    print(re.sub(r"in [\d.]+s", "in <time>", pytest_asyncio_run.stdout.strip()))
else:
    print("pytest-asyncio is not installed (not a course dependency): skipping")

pytest-asyncio is not installed (not a course dependency): skipping


### 5.4.11 Interop

**Threads → loop.** asyncio objects are not thread-safe; from another
thread, talk to a loop only through:

| Call (from another thread) | Effect |
| --- | --- |
| `asyncio.run_coroutine_threadsafe(coro, loop)` | runs `coro` on the loop; returns a `concurrent.futures.Future` to block on |
| `loop.call_soon_threadsafe(callback, *args)` | queues a callback and wakes the loop (through its self-pipe) |
| `asyncio.wrap_future(concurrent_future)` | the other direction: await a thread-pool Future from async code |

Three "legacy" worker threads each ask the loop to reserve stock and block on
the answer, while the loop awaits the threads' futures:

In [323]:
async def reserve_stock(sku: str) -> str:
    """Reserve stock on the loop thread after a short wait."""
    await asyncio.sleep(0.01)
    return f"{sku} reserved on {threading.current_thread().name}"


def legacy_worker(sku: str, loop: asyncio.AbstractEventLoop) -> str:
    """Sync code in a worker thread: submit to the loop, block for the result."""
    return asyncio.run_coroutine_threadsafe(reserve_stock(sku), loop).result(timeout=2)


with concurrent.futures.ThreadPoolExecutor(
    max_workers=3, thread_name_prefix="legacy"
) as legacy_pool:
    thread_futures = [
        legacy_pool.submit(legacy_worker, sku, KERNEL_LOOP) for sku in ("chai", "vada", "idli")
    ]
    print(await asyncio.gather(*(asyncio.wrap_future(future) for future in thread_futures)))

['chai reserved on MainThread', 'vada reserved on MainThread', 'idli reserved on MainThread']


`call_soon_threadsafe` feeding an `asyncio.Queue` from a sensor thread. Calling
`put_nowait` directly from the thread would be a race (and would not wake a
loop sleeping in `select`):

In [324]:
readings: asyncio.Queue[float] = asyncio.Queue()


def sensor_thread(loop: asyncio.AbstractEventLoop) -> None:
    """Post readings into the loop from a plain thread, then shut down."""
    for temperature in (21.5, 22.0, 22.4):
        time.sleep(0.01)
        loop.call_soon_threadsafe(readings.put_nowait, temperature)
    loop.call_soon_threadsafe(readings.shutdown)


threading.Thread(target=sensor_thread, args=(KERNEL_LOOP,), daemon=True).start()
received_readings: list[float] = []
try:
    while True:
        received_readings.append(await readings.get())
except asyncio.QueueShutDown:
    print(f"readings from the thread: {received_readings}")

readings from the thread: [21.5, 22.0, 22.4]


**anyio and trio.** *Trio* is an alternative async library built around
structured concurrency from the start (nurseries, cancel scopes); its ideas
gave asyncio `TaskGroup` and `timeout`. Trio and asyncio do not interoperate
directly. *anyio* is a compatibility layer with trio-style APIs that runs on
either: libraries written against anyio (httpx, Starlette, FastAPI in
Phase 6) work on both. Inside the kernel, anyio code runs on the asyncio loop:

In [325]:
if HAS_ANYIO:
    import anyio

    anyio_log: list[str] = []

    async def anyio_job(name: str, seconds: float) -> None:
        """Sleep, then log the name."""
        await anyio.sleep(seconds)
        anyio_log.append(name)

    async with anyio.create_task_group() as task_group:  # like asyncio.TaskGroup
        task_group.start_soon(anyio_job, "slow", 0.02)
        task_group.start_soon(anyio_job, "fast", 0.01)
    with anyio.move_on_after(0.01) as cancel_scope:  # a timeout that does not raise
        await anyio.sleep(1)
    print(f"{anyio_log=} {cancel_scope.cancelled_caught=}")
    print(f"anyio.to_thread.run_sync: {await anyio.to_thread.run_sync(sum, [1, 2, 3])}")
else:
    print("anyio is not installed: skipping")

anyio_log=['fast', 'slow'] cancel_scope.cancelled_caught=True
anyio.to_thread.run_sync: 6


`anyio.run(main, backend=...)` is the entry point, like `asyncio.run`. The same
`main` runs on asyncio, and on trio when trio is installed (it is not a course
dependency, so the trio line prints a skip message here):

In [326]:
print(
    run_child(
        """
        import importlib.util

        import anyio


        async def main():
            async with anyio.create_task_group() as group:
                for delay in (0.02, 0.01):
                    group.start_soon(anyio.sleep, delay)
            return anyio.get_current_task().name


        for backend in ("asyncio", "trio"):
            if backend == "trio" and importlib.util.find_spec("trio") is None:
                print("trio:    not installed, skipping")
                continue
            main_task_name = anyio.run(main, backend=backend)
            print(f"{backend + ':':<8} anyio.run finished, main task {main_task_name!r}")
        """
    )
    if HAS_ANYIO
    else "anyio is not installed: skipping"
)

asyncio: anyio.run finished, main task '__main__.main'
trio:    not installed, skipping


#### Predict the output

**Predict the output.**

```python
async def worker():
    try:
        await asyncio.sleep(1)
    except Exception:
        print("caught")
    finally:
        print("finally")

task = asyncio.create_task(worker())
await asyncio.sleep(0)
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("cancelled")
```

<details><summary>Answer</summary>

`finally`, then `cancelled`. `CancelledError` derives from `BaseException`
(since Python 3.8), so `except Exception` does not catch it; `finally` runs,
the error propagates out of the task, and awaiting the task re-raises it.

</details>

Run the next cell to check your prediction.

In [327]:
async def worker() -> None:
    """Sleep a second; the except Exception clause will not see a cancel."""
    try:
        await asyncio.sleep(1)
    except Exception:
        print("caught")
    finally:
        print("finally")


predict_task = asyncio.create_task(worker())
await asyncio.sleep(0)
predict_task.cancel()
try:
    await predict_task
except asyncio.CancelledError:
    print("cancelled")

finally
cancelled


**Predict the output.**

```python
async def steady():
    try:
        await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("steady cancelled")
        raise

async def broken():
    await asyncio.sleep(0.01)
    raise KeyError("sku-404")

try:
    async with asyncio.TaskGroup() as group:
        group.create_task(steady())
        group.create_task(broken())
except* KeyError as errors:
    print(type(errors).__name__, len(errors.exceptions))
```

<details><summary>Answer</summary>

`steady cancelled`, then `ExceptionGroup 1`. When `broken` fails, the group
cancels `steady`, waits for it, and raises an `ExceptionGroup` containing the
one `KeyError`. `except*` binds the matching *group*, not the bare `KeyError`.

</details>

Run the next cell to check your prediction.

In [328]:
async def steady() -> None:
    """Sleep until cancelled, report it, re-raise."""
    try:
        await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("steady cancelled")
        raise


async def broken() -> None:
    """Fail with a KeyError after 10 ms."""
    await asyncio.sleep(0.01)
    raise KeyError("sku-404")


try:
    async with asyncio.TaskGroup() as group:
        group.create_task(steady())
        group.create_task(broken())
except* KeyError as errors:
    print(type(errors).__name__, len(errors.exceptions))

steady cancelled
ExceptionGroup 1


**Predict the output.**

```python
colour = contextvars.ContextVar("colour", default="outer")

async def paint():
    colour.set("painted")

async def main():
    await asyncio.create_task(paint())
    print(colour.get())
    await paint()
    print(colour.get())

await asyncio.create_task(main())
```

<details><summary>Answer</summary>

`outer`, then `painted`. A Task runs in a *copy* of the context, so the first
`paint()` (a separate task) changes only its copy. Awaiting `paint()`
directly runs it inside `main`'s own task and context, so the second
`set()` is visible to `main`. (`main` itself runs as a task so that the
kernel's context stays untouched.)

</details>

Run the next cell to check your prediction.

In [329]:
colour = contextvars.ContextVar("colour", default="outer")


async def paint() -> None:
    """Set the context variable in the current context."""
    colour.set("painted")


async def main() -> None:
    """Paint inside a task (a copied context), then in this coroutine."""
    await asyncio.create_task(paint())
    print(colour.get())
    await paint()
    print(colour.get())


await asyncio.create_task(main())

outer
painted


### Lab 5.4

Seven exercises. All HTTP goes through `httpx.MockTransport` handlers (no
network). Replace each `raise NotImplementedError`, then run the check cell;
until then the checks print FAIL, which is expected. The first setup cell
defines the general helpers: `outcome` (await, turning an exception into
`"raised Name"`), `quiet_unawaited` and `scenario_field`:

In [330]:
from collections.abc import Awaitable


async def outcome(awaitable: Awaitable[object]) -> object:
    """Await and return the result, or "raised ErrorName". Never raises an Exception."""
    try:
        return await awaitable
    except Exception as error:
        return f"raised {type(error).__name__}"


@contextlib.contextmanager
def quiet_unawaited() -> Generator[None]:
    """Hide "coroutine ... was never awaited" warnings caused by unfinished stubs."""
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", "coroutine .* was never awaited", RuntimeWarning)
        yield
        gc.collect()


def scenario_field(result: object, key: str) -> object:
    """result[key] if result is a dict (a finished scenario), else result itself (an error)."""
    return result[key] if isinstance(result, dict) else result


print("Lab 5.4 helpers ready: outcome, quiet_unawaited, scenario_field")

Lab 5.4 helpers ready: outcome, quiet_unawaited, scenario_field


The second defines `InFlightTracker`, which counts concurrent requests inside a
mock handler (the current number, the peak, and calls per key):

In [331]:
class InFlightTracker:
    """Counts concurrent requests inside a mock transport handler."""

    def __init__(self) -> None:
        """Start with all counters at zero."""
        self.reset()

    def reset(self) -> None:
        """Zero the current and peak counts and the per-key calls."""
        self.now, self.peak, self.calls = 0, 0, Counter()

    @contextlib.contextmanager
    def track(self, key: object) -> Generator[None]:
        """Count one request in flight for the duration of the with block."""
        self.now += 1
        self.peak = max(self.peak, self.now)
        self.calls[key] += 1
        try:
            yield
        finally:
            self.now -= 1


print("Lab 5.4 helper ready: InFlightTracker")

Lab 5.4 helper ready: InFlightTracker


The third defines `mock_client` (an `AsyncClient` on a `MockTransport`) and
`loop_lag_during` (a 10 ms heartbeat that measures how long the loop was
blocked):

In [332]:
def mock_client(handler: Callable[[httpx.Request], Awaitable[httpx.Response]]) -> httpx.AsyncClient:
    """An AsyncClient whose requests go to handler instead of the network."""
    return httpx.AsyncClient(
        transport=httpx.MockTransport(handler), base_url="https://api.shop.test"
    )


async def loop_lag_during(work: Callable[[], Awaitable[object]], interval: float = 0.01) -> float:
    """Run work() while a 10 ms heartbeat runs; return the heartbeat's worst lateness in seconds."""
    stop, worst = asyncio.Event(), [0.0]

    async def beat() -> None:
        """Wake every interval and record the worst lateness."""
        while not stop.is_set():
            before = KERNEL_LOOP.time()
            await asyncio.sleep(interval)
            worst[0] = max(worst[0], KERNEL_LOOP.time() - before - interval)

    monitor = asyncio.create_task(beat())
    await asyncio.sleep(0.02)
    try:
        await work()
    finally:
        stop.set()
        await monitor
    return worst[0]


print("Lab 5.4 helpers ready: mock_client, loop_lag_during")

Lab 5.4 helpers ready: mock_client, loop_lag_during


#### Exercise 1 — `fetch_all(urls, max_concurrency)`

Write `fetch_all(client, urls, max_concurrency)`. It GETs every URL with the
shared `client`, with **at most `max_concurrency` requests in flight**, and
returns a list **in input order**. Each element is the parsed JSON body of a
successful response, or the class name of the error otherwise: call
`response.raise_for_status()`, so a 500 becomes `"HTTPStatusError"` and a
refused connection `"ConnectError"`. It never raises for a single failed URL.

In [333]:
catalog_tracker = InFlightTracker()


async def catalog_api(request: httpx.Request) -> httpx.Response:
    """GET /items/<n>: 10 ms each; item 13 answers 500, item 21 refuses to connect."""
    item_id = int(request.url.path.rsplit("/", 1)[1])
    with catalog_tracker.track(item_id):
        await asyncio.sleep(0.01)
        if item_id == 13:
            return httpx.Response(500)
        if item_id == 21:
            raise httpx.ConnectError("connection refused", request=request)
        return httpx.Response(200, json={"id": item_id, "price": item_id * 10})

Your solution:

In [334]:
async def fetch_all(
    client: httpx.AsyncClient, urls: list[str], max_concurrency: int
) -> list[object]:
    """GET every URL, at most max_concurrency at a time; JSON or error name, in input order."""
    raise NotImplementedError

Check exercise 1:

In [335]:
catalog_urls = [f"/items/{item_id}" for item_id in range(30)]
expected_catalog = [
    "HTTPStatusError"
    if item_id == 13
    else "ConnectError"
    if item_id == 21
    else {"id": item_id, "price": item_id * 10}
    for item_id in range(30)
]
catalog_tracker.reset()
async with mock_client(catalog_api) as catalog_client:
    with Timer("fetch_all", report=None) as fetch_timer:
        fetched = await outcome(fetch_all(catalog_client, catalog_urls, max_concurrency=5))
    fetched_nothing = await outcome(fetch_all(catalog_client, [], max_concurrency=5))
run_checks(
    [
        ("results in input order, errors as names", lambda: fetched, expected_catalog),
        ("never more than 5 requests in flight", lambda: 0 < catalog_tracker.peak <= 5, True),
        ("actually concurrent (peak == 5)", lambda: catalog_tracker.peak, 5),
        (
            "faster than sequential 30 x 10 ms",
            lambda: isinstance(fetched, list) and fetch_timer.elapsed_seconds < 0.2,
            True,
        ),
        ("empty URL list", lambda: fetched_nothing, []),
    ]
)

FAIL  results in input order, errors as names: expected [{'id': 0, 'price': 0}, {'id': 1, 'price': 10}, {'id': 2, 'price': 20}, {'id': 3, 'price': 30}, {'id': 4, 'price': 40}, {'id': 5, 'price': 50}, {'id': 6, 'price': 60}, {'id': 7, 'price': 70}, {'id': 8, 'price': 80}, {'id': 9, 'price': 90}, {'id': 10, 'price': 100}, {'id': 11, 'price': 110}, {'id': 12, 'price': 120}, 'HTTPStatusError', {'id': 14, 'price': 140}, {'id': 15, 'price': 150}, {'id': 16, 'price': 160}, {'id': 17, 'price': 170}, {'id': 18, 'price': 180}, {'id': 19, 'price': 190}, {'id': 20, 'price': 200}, 'ConnectError', {'id': 22, 'price': 220}, {'id': 23, 'price': 230}, {'id': 24, 'price': 240}, {'id': 25, 'price': 250}, {'id': 26, 'price': 260}, {'id': 27, 'price': 270}, {'id': 28, 'price': 280}, {'id': 29, 'price': 290}], got 'raised NotImplementedError'
FAIL  never more than 5 requests in flight: expected True, got False
FAIL  actually concurrent (peak == 5): expected 5, got 0
FAIL  faster than sequential 30 x 10 ms: 

#### Exercise 2 — an async worker pool with graceful shutdown

Complete `WorkerPool`, which runs the async function `job` over submitted
items with `worker_count` worker tasks fed by a bounded `asyncio.Queue`:

- `await pool.start()` starts the workers and stores them in `pool.workers`;
- `await pool.submit(item)` queues an item (it waits while the queue is
  full: backpressure) and raises `RuntimeError` after shutdown has begun;
- `await pool.shutdown()` stops accepting items, lets the workers finish
  **everything already queued**, waits for every worker to exit, and returns
  `{item: result}`, where a job that raised maps to the exception's class
  name;
- `await pool.shutdown(immediate=True)` drops the queued items, cancels the
  jobs in progress, waits for the workers, and returns what had finished.

`Queue.shutdown()` (5.4.4) does most of the work.

Your solution:

In [336]:
class WorkerPool:
    """Fixed-size pool of worker tasks fed by a bounded queue."""

    def __init__(
        self, job: Callable[[Any], Awaitable[object]], worker_count: int = 3, queue_size: int = 10
    ) -> None:
        """Store the job, the worker count and a bounded queue."""
        self.job = job
        self.worker_count = worker_count
        self.queue: asyncio.Queue[Any] = asyncio.Queue(maxsize=queue_size)
        self.workers: list[asyncio.Task[None]] = []
        self.results: dict[Any, object] = {}

    async def start(self) -> None:
        """Start worker_count worker tasks."""
        raise NotImplementedError

    async def submit(self, item: Any) -> None:
        """Queue one item, waiting while the queue is full."""
        raise NotImplementedError

    async def shutdown(self, immediate: bool = False) -> dict[Any, object]:
        """Stop the workers (gracefully or at once) and return the results."""
        raise NotImplementedError

Check exercise 2:

In [337]:
async def double_later(item: int) -> int:
    """Double an item after 10 ms; item 7 raises ValueError."""
    await asyncio.sleep(0.01)
    if item == 7:
        raise ValueError("unlucky item")
    return item * 2


async def slow_job(item: int) -> int:
    """Return the item after 50 ms."""
    await asyncio.sleep(0.05)
    return item


async def graceful_scenario() -> dict[str, object]:
    """Run 12 jobs through a 3-worker pool and shut it down gracefully."""
    pool = WorkerPool(double_later, worker_count=3, queue_size=4)
    await pool.start()
    with Timer("12 jobs", report=None) as pool_timer:
        for item in range(12):
            await pool.submit(item)
        results = await pool.shutdown()
    return {
        "results": results,
        "seconds": pool_timer.elapsed_seconds,
        "workers": len(pool.workers),
        "all_exited": all(worker.done() for worker in pool.workers),
        "late_submit": await outcome(pool.submit(99)),
    }

The immediate-shutdown scenario queues 20 slow jobs on 2 workers and shuts down
after 10 ms with `immediate=True`. Both scenarios run here; the checks follow:

In [338]:
async def immediate_scenario() -> dict[str, object]:
    """Queue 20 slow jobs, then shut the pool down immediately."""
    pool = WorkerPool(slow_job, worker_count=2, queue_size=50)
    await pool.start()
    for item in range(20):
        await pool.submit(item)
    await asyncio.sleep(0.01)
    results = await pool.shutdown(immediate=True)
    return {"finished": len(results), "all_exited": all(worker.done() for worker in pool.workers)}


graceful, immediate = await outcome(graceful_scenario()), await outcome(immediate_scenario())

The checks: every queued item processed (errors as class names), three workers that
all exit, real concurrency, `submit` refused after shutdown, and an immediate
shutdown that drops queued items:

In [339]:
run_checks(
    [
        (
            "every queued item processed, errors as names",
            lambda: scenario_field(graceful, "results"),
            {item: "ValueError" if item == 7 else item * 2 for item in range(12)},
        ),
        (
            "3 workers, all exited after shutdown",
            lambda: (scenario_field(graceful, "workers"), scenario_field(graceful, "all_exited")),
            (3, True),
        ),
        (
            "workers ran concurrently (< 0.1 s, not 0.12 s)",
            lambda: scenario_field(graceful, "seconds") < 0.1,
            True,
        ),
        (
            "submit after shutdown -> RuntimeError",
            lambda: scenario_field(graceful, "late_submit"),
            "raised RuntimeError",
        ),
        (
            "immediate shutdown drops queued items",
            lambda: scenario_field(immediate, "finished") < 20,
            True,
        ),
        (
            "immediate shutdown: workers exited",
            lambda: scenario_field(immediate, "all_exited"),
            True,
        ),
    ]
)

FAIL  every queued item processed, errors as names: expected {0: 0, 1: 2, 2: 4, 3: 6, 4: 8, 5: 10, 6: 12, 7: 'ValueError', 8: 16, 9: 18, 10: 20, 11: 22}, got 'raised NotImplementedError'
FAIL  3 workers, all exited after shutdown: expected (3, True), got ('raised NotImplementedError', 'raised NotImplementedError')
FAIL  workers ran concurrently (< 0.1 s, not 0.12 s): raised TypeError: '<' not supported between instances of 'str' and 'float'
FAIL  submit after shutdown -> RuntimeError: expected 'raised RuntimeError', got 'raised NotImplementedError'
FAIL  immediate shutdown drops queued items: raised TypeError: '<' not supported between instances of 'str' and 'int'
FAIL  immediate shutdown: workers exited: expected True, got 'raised NotImplementedError'
0/6 checks passed


#### Exercise 3 — `retry_async`

Write the decorator factory `retry_async(...)` for async functions. The
decorated function is called at most `attempts` times in total. An exception
that is an instance of `retry_on` triggers a retry; before retry *n* (n = 1,
2, …) it awaits `sleep(delay)` with the **full-jitter** delay
`rng.uniform(0, min(max_delay, base_delay * 2 ** (n - 1)))` (use
`random.Random()` when `rng` is `None`). When the attempts run out, the last
exception is re-raised. Other exceptions are raised at once, never retried.
Keep the function's name and docstring (`functools.wraps`). `sleep` is a
parameter so tests can record delays instead of waiting.

In [340]:
RETRY_CALLS: list[str] = []


async def flaky_inventory_call(
    fail_times: int, error_type: type[Exception] = ConnectionError
) -> str:
    """Fails fail_times times with error_type, then succeeds."""
    RETRY_CALLS.append(error_type.__name__)
    if len(RETRY_CALLS) <= fail_times:
        raise error_type(f"attempt {len(RETRY_CALLS)} failed")
    return f"ok after {len(RETRY_CALLS)} calls"

Your solution:

In [341]:
def retry_async[**P, R](
    *,
    attempts: int = 3,
    base_delay: float = 0.1,
    max_delay: float = 2.0,
    retry_on: tuple[type[BaseException], ...] = (Exception,),
    sleep: Callable[[float], Awaitable[object]] = asyncio.sleep,
    rng: random.Random | None = None,
) -> Callable[[Callable[P, Awaitable[R]]], Callable[P, Awaitable[R]]]:
    """Retry an async function with full-jitter exponential backoff."""
    raise NotImplementedError

Check exercise 3:

In [342]:
recorded_sleeps: list[float] = []


async def record_sleep(seconds: float) -> None:
    """A fake sleep that records the requested delay."""
    recorded_sleeps.append(round(seconds, 6))


async def retry_case(
    fail_times: int, error_type: type[Exception] = ConnectionError
) -> dict[str, object]:
    """Run the retry decorator against a call that fails fail_times times."""
    RETRY_CALLS.clear()
    recorded_sleeps.clear()
    decorator = retry_async(
        attempts=4,
        base_delay=0.1,
        max_delay=0.25,
        retry_on=(ConnectionError,),
        sleep=record_sleep,
        rng=random.Random(7),
    )
    result = await outcome(decorator(flaky_inventory_call)(fail_times, error_type))
    return {"result": result, "calls": len(RETRY_CALLS), "sleeps": list(recorded_sleeps)}

The expected delays come from the same seeded `rng`, so full jitter is checked
exactly. Three scenarios run (recover after 2 failures, give up after 10, a
non-retryable `KeyError`), then the checks:

In [343]:
expected_rng = random.Random(7)
expected_delays = [
    round(expected_rng.uniform(0, min(0.25, 0.1 * 2**retry)), 6) for retry in range(3)
]
recovers, gives_up, not_retried = (
    await outcome(retry_case(2)),
    await outcome(retry_case(10)),
    await outcome(retry_case(5, KeyError)),
)
run_checks(
    [
        (
            "recovers after 2 failures",
            lambda: (scenario_field(recovers, "result"), scenario_field(recovers, "calls")),
            ("ok after 3 calls", 3),
        ),
        (
            "full-jitter delays from rng",
            lambda: scenario_field(recovers, "sleeps"),
            expected_delays[:2],
        ),
        (
            "gives up after 4 calls, re-raises",
            lambda: (scenario_field(gives_up, "result"), scenario_field(gives_up, "calls")),
            ("raised ConnectionError", 4),
        ),
        ("capped delays (3 sleeps)", lambda: scenario_field(gives_up, "sleeps"), expected_delays),
        (
            "other errors are not retried",
            lambda: (scenario_field(not_retried, "result"), scenario_field(not_retried, "calls")),
            ("raised KeyError", 1),
        ),
        (
            "keeps the name",
            lambda: retry_async()(flaky_inventory_call).__name__,
            "flaky_inventory_call",
        ),
    ]
)

FAIL  recovers after 2 failures: expected ('ok after 3 calls', 3), got ('raised NotImplementedError', 'raised NotImplementedError')
FAIL  full-jitter delays from rng: expected [0.032383, 0.03017], got 'raised NotImplementedError'
FAIL  gives up after 4 calls, re-raises: expected ('raised ConnectionError', 4), got ('raised NotImplementedError', 'raised NotImplementedError')
FAIL  capped delays (3 sleeps): expected [0.032383, 0.03017, 0.162734], got 'raised NotImplementedError'
FAIL  other errors are not retried: expected ('raised KeyError', 1), got ('raised NotImplementedError', 'raised NotImplementedError')
FAIL  keeps the name: raised NotImplementedError: 
0/6 checks passed


#### Exercise 4 — an async paginator

Write the async generator `paginate(client, path, page_size=10)`. The endpoint
is cursor-paginated: `GET path?limit=<page_size>` returns
`{"items": [...], "next_cursor": "<cursor>" or null}`; the next page is
`GET path?limit=<page_size>&cursor=<cursor>`. Yield the items one by one,
and fetch a page only when the caller asks for an item beyond the current one
(a caller that stops after 12 items must cause only 2 requests).

In [344]:
PAGED_ITEMS = [{"id": item_id, "sku": f"sku-{item_id:03d}"} for item_id in range(45)]
page_requests: list[dict[str, str]] = []


async def paged_api(request: httpx.Request) -> httpx.Response:
    """Cursor pagination over PAGED_ITEMS (/products) or nothing (/empty)."""
    query = dict(request.url.params)
    page_requests.append(query)
    items = PAGED_ITEMS if request.url.path == "/products" else []
    start, limit = int(query.get("cursor", 0)), int(query.get("limit", 10))
    next_start = start + limit
    next_cursor = str(next_start) if next_start < len(items) else None
    return httpx.Response(200, json={"items": items[start:next_start], "next_cursor": next_cursor})

Your solution:

In [345]:
async def paginate(
    client: httpx.AsyncClient, path: str, page_size: int = 10
) -> AsyncIterator[dict[str, Any]]:
    """Yield every item of a cursor-paginated endpoint, fetching pages lazily."""
    raise NotImplementedError
    yield  # the yield makes this an async generator

Check exercise 4:

In [346]:
async def paging_case(
    path: str, page_size: int = 10, stop_after: int | None = None
) -> dict[str, object]:
    """Collect paginated ids, optionally stopping early."""
    page_requests.clear()
    collected: list[int] = []
    async with mock_client(paged_api) as paging_client:
        async with contextlib.aclosing(paginate(paging_client, path, page_size)) as items:
            async for item in items:
                collected.append(item["id"])
                if stop_after is not None and len(collected) == stop_after:
                    break
    return {
        "ids": collected,
        "requests": len(page_requests),
        "first": page_requests[0] if page_requests else None,
    }

Four runs (every page, stop after 12 items, bigger pages, an empty listing), then
the checks: order, request count, the first request's parameters, laziness, and
the empty case:

In [347]:
every_page = await outcome(paging_case("/products"))
stopped_early = await outcome(paging_case("/products", stop_after=12))
bigger_pages = await outcome(paging_case("/products", page_size=20))
empty_listing = await outcome(paging_case("/empty"))
run_checks(
    [
        ("all 45 items in order", lambda: scenario_field(every_page, "ids"), list(range(45))),
        ("5 requests for 45 items", lambda: scenario_field(every_page, "requests"), 5),
        (
            "first request: limit only, no cursor",
            lambda: scenario_field(every_page, "first"),
            {"limit": "10"},
        ),
        (
            "lazy: stopping after 12 items costs 2 requests",
            lambda: (
                scenario_field(stopped_early, "ids"),
                scenario_field(stopped_early, "requests"),
            ),
            (list(range(12)), 2),
        ),
        ("page_size=20 -> 3 requests", lambda: scenario_field(bigger_pages, "requests"), 3),
        (
            "empty endpoint",
            lambda: (
                scenario_field(empty_listing, "ids"),
                scenario_field(empty_listing, "requests"),
            ),
            ([], 1),
        ),
    ]
)

FAIL  all 45 items in order: expected [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44], got 'raised NotImplementedError'
FAIL  5 requests for 45 items: expected 5, got 'raised NotImplementedError'
FAIL  first request: limit only, no cursor: expected {'limit': '10'}, got 'raised NotImplementedError'
FAIL  lazy: stopping after 12 items costs 2 requests: expected ([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11], 2), got ('raised NotImplementedError', 'raised NotImplementedError')
FAIL  page_size=20 -> 3 requests: expected 3, got 'raised NotImplementedError'
FAIL  empty endpoint: expected ([], 1), got ('raised NotImplementedError', 'raised NotImplementedError')
0/6 checks passed


#### Exercise 5 — find a hidden blocking call

Each handler set below contains exactly one handler that blocks the event
loop somewhere deep inside (and one that is slow but harmless: slow is not
the same as blocking). Write `find_blocking_handler(handlers, threshold)`,
which runs each handler and returns the name of the one that blocks the loop
for longer than `threshold` seconds. Use the tools from 5.4.6:
`loop_lag_during` (a lag monitor, in the lab setup) or `slow_callback_report`
(debug mode). Then write `render_invoice_fixed()`, which returns the same
value as `render_invoice()` without blocking the loop. Do not read the
source to find the culprit; measure it.

In [348]:
def _legacy_pdf(order_id: int) -> str:
    """A synchronous PDF call that blocks for 100 ms."""
    time.sleep(0.1)  # a synchronous PDF library
    return f"invoice-{order_id}.pdf"


def _tax_table(region: str) -> dict[str, object]:
    """A CPU-heavy pure-Python lookup that spins for 100 ms."""
    end = time.perf_counter() + 0.1
    while time.perf_counter() < end:  # a CPU-heavy pure-Python loop
        pass
    return {"region": region, "gst": 18}


async def load_profile() -> dict[str, object]:
    """Load a profile with a real await (harmless)."""
    await asyncio.sleep(0.15)  # slow, but it awaits: harmless
    return {"user": "asha"}


async def render_invoice() -> dict[str, object]:
    """Render the invoice with the blocking PDF call."""
    await asyncio.sleep(0.01)
    return {"file": _legacy_pdf(42)}


async def send_sms() -> dict[str, object]:
    """Send an SMS with a real await."""
    await asyncio.sleep(0.02)
    return {"sms": "sent"}


async def lookup_tax() -> dict[str, object]:
    """Look up tax with the CPU-heavy helper."""
    await asyncio.sleep(0.01)
    return _tax_table("KA")


HANDLERS_A = {"load_profile": load_profile, "render_invoice": render_invoice, "send_sms": send_sms}
HANDLERS_B = {"load_profile": load_profile, "send_sms": send_sms, "lookup_tax": lookup_tax}

Your solution:

In [349]:
async def find_blocking_handler(
    handlers: dict[str, Callable[[], Awaitable[object]]], threshold: float = 0.05
) -> str:
    """Name of the handler that blocks the event loop for longer than threshold seconds."""
    raise NotImplementedError


async def render_invoice_fixed() -> dict[str, object]:
    """Same result as render_invoice(), without blocking the loop."""
    raise NotImplementedError

Check exercise 5:

In [350]:
culprit_a = await outcome(find_blocking_handler(HANDLERS_A))
culprit_b = await outcome(find_blocking_handler(HANDLERS_B))
fixed_result = await outcome(render_invoice_fixed())
fixed_lag = await outcome(loop_lag_during(render_invoice_fixed))
run_checks(
    [
        ("set A: the blocking handler", lambda: culprit_a, "render_invoice"),
        ("set B: the blocking handler", lambda: culprit_b, "lookup_tax"),
        ("fixed version: same result", lambda: fixed_result, {"file": "invoice-42.pdf"}),
        (
            "fixed version: loop lag under 50 ms",
            lambda: isinstance(fixed_lag, float) and fixed_lag < 0.05,
            True,
        ),
    ]
)

FAIL  set A: the blocking handler: expected 'render_invoice', got 'raised NotImplementedError'
FAIL  set B: the blocking handler: expected 'lookup_tax', got 'raised NotImplementedError'
FAIL  fixed version: same result: expected {'file': 'invoice-42.pdf'}, got 'raised NotImplementedError'
FAIL  fixed version: loop lag under 50 ms: expected True, got False
0/4 checks passed


#### Exercise 6 — request-ID propagation with `contextvars`

Two pieces:

1. `make_traced_client(transport)` returns an `httpx.AsyncClient` (base URL
   `https://api.shop.test`) with a **request event hook** (an `async`
   function in `event_hooks={"request": [...]}`) that sets the header
   `X-Request-ID` to `REQUEST_ID.get()` on every outgoing request.
2. `handle_request(client, path, request_id=None)` sets `REQUEST_ID` to
   `request_id` (or to a new `uuid.uuid4().hex[:8]` when it is `None`) **for
   this request only** (reset it before returning); logs `"start {path}"` and
   `"done {path}"` with `request_log.info`; GETs `path`; runs
   `audit_trail(path)` in a worker thread with `asyncio.to_thread`; and returns
   `{"request_id": ..., "echoed": <the "x-request-id" field of the JSON
   reply>, "audit": <what audit_trail returned>}`.

The log handler records the request id current at each log call, so correct
propagation is visible in `request_records`.

In [351]:
import uuid  # noqa: F401 (for your solution)

REQUEST_ID: contextvars.ContextVar[str] = contextvars.ContextVar("lab_request_id", default="-")
request_records: list[tuple[str, str]] = []


class CaptureRequestIds(logging.Handler):
    """Record (current request id, message) for every log call."""

    def emit(self, record: logging.LogRecord) -> None:
        """Record the current request id with each log message."""
        request_records.append((REQUEST_ID.get(), record.getMessage()))


request_log = logging.getLogger("lab54.requests")
request_log.handlers[:] = [CaptureRequestIds()]
request_log.setLevel(logging.INFO)
request_log.propagate = False


def audit_trail(path: str) -> str:
    """Runs in a worker thread; returns the request id it can see."""
    return REQUEST_ID.get()


async def echo_request_id_api(request: httpx.Request) -> httpx.Response:
    """Echo the request id header back after a short wait."""
    await asyncio.sleep(0.005)  # let concurrent requests interleave
    return httpx.Response(
        200, json={"x-request-id": request.headers.get("x-request-id", "missing")}
    )

Your solution:

In [352]:
def make_traced_client(transport: httpx.AsyncBaseTransport) -> httpx.AsyncClient:
    """AsyncClient whose request hook adds X-Request-ID from REQUEST_ID."""
    raise NotImplementedError


async def handle_request(
    client: httpx.AsyncClient, path: str, request_id: str | None = None
) -> dict[str, str]:
    """Serve one request with REQUEST_ID set; return request_id, echoed header and audit value."""
    raise NotImplementedError

Check exercise 6:

In [353]:
async def request_id_scenario() -> dict[str, object]:
    """Send 20 tagged requests and one untagged; collect what each saw."""
    request_records.clear()
    async with make_traced_client(httpx.MockTransport(echo_request_id_api)) as traced_client:
        many = await asyncio.gather(
            *(
                handle_request(traced_client, f"/orders/{order_number}", f"req-{order_number:02d}")
                for order_number in range(20)
            )
        )
        generated = await handle_request(traced_client, "/orders/new")
    return {
        "many": many,
        "generated": generated,
        "caller_sees": REQUEST_ID.get(),
        "records": list(request_records),
    }


def records_match(scenario: object) -> bool:
    """True if all 42 log lines carry the id of their own request."""
    records = scenario_field(scenario, "records")
    tagged = [
        (logged_id, message) for logged_id, message in records if "/orders/new" not in message
    ]
    return len(records) == 42 and all(
        logged_id == f"req-{message.rsplit('/', 1)[1]:0>2}" for logged_id, message in tagged
    )

Run the scenario, then check: each request's own id in its header and in the worker
thread, 42 log lines each tagged with the right id, a generated 8-character id for
the request without one, and the caller's context restored afterwards:

In [354]:
traced = await outcome(request_id_scenario())
run_checks(
    [
        (
            "header carries each request's id",
            lambda: [reply["echoed"] for reply in scenario_field(traced, "many")],
            [f"req-{order_number:02d}" for order_number in range(20)],
        ),
        (
            "worker thread sees the id",
            lambda: all(
                reply["audit"] == reply["request_id"] for reply in scenario_field(traced, "many")
            ),
            True,
        ),
        ("40 + 2 log lines, each tagged with its own id", lambda: records_match(traced), True),
        (
            "generated id: 8 hex chars, echoed",
            lambda: (
                len(scenario_field(traced, "generated")["request_id"]),
                scenario_field(traced, "generated")["echoed"]
                == scenario_field(traced, "generated")["request_id"],
            ),
            (8, True),
        ),
        ("caller's context restored", lambda: scenario_field(traced, "caller_sees"), "-"),
    ]
)

FAIL  header carries each request's id: raised TypeError: string indices must be integers, not 'str'
FAIL  worker thread sees the id: raised TypeError: string indices must be integers, not 'str'
FAIL  40 + 2 log lines, each tagged with its own id: raised ValueError: not enough values to unpack (expected 2, got 1)
FAIL  generated id: 8 hex chars, echoed: raised TypeError: string indices must be integers, not 'str'
FAIL  caller's context restored: expected '-', got 'raised NotImplementedError'
0/5 checks passed


#### Exercise 7 — an API aggregator over 500 mocked endpoints

Write `aggregate_sales(client, store_ids, max_concurrency=50, timeout=0.2)`.
For every store it GETs `/stores/<id>/sales` (JSON `{"store": id, "amount":
n}`), with at most `max_concurrency` requests in flight and each attempt
limited to `timeout` seconds. Use `asyncio.timeout`: `MockTransport`
ignores httpx's own timeouts. A `503` is retried **once**; any other error
status, a timeout or a transport error marks the store as failed. Return
`{"total": <sum of amounts of the successful stores>, "ok": <their count>,
"failed": <sorted list of failed store ids>}`.

In [355]:
STORE_COUNT = 500
store_tracker = InFlightTracker()


async def sales_api(request: httpx.Request) -> httpx.Response:
    """Store sales; some stores hang, some always fail, some fail once with 503."""
    store_id = int(request.url.path.split("/")[2])
    with store_tracker.track(store_id):
        if store_id % 97 == 3:
            await asyncio.sleep(1.0)  # hangs: must be cut off by the timeout
        await asyncio.sleep(0.01)
        if store_id % 50 == 7:
            return httpx.Response(500)
        if store_id % 40 == 11 and store_tracker.calls[store_id] == 1:
            return httpx.Response(503)  # busy once, fine on retry
        return httpx.Response(200, json={"store": store_id, "amount": store_id * 37 % 1000})

Your solution:

In [356]:
async def aggregate_sales(
    client: httpx.AsyncClient, store_ids: list[int], max_concurrency: int = 50, timeout: float = 0.2
) -> dict[str, object]:
    """Fetch every store's sales concurrently; return total, ok count and failed ids."""
    raise NotImplementedError

Check exercise 7:

In [357]:
store_ids = list(range(STORE_COUNT))
failing_stores = sorted(store for store in store_ids if store % 50 == 7 or store % 97 == 3)
expected_sales = {
    "total": sum(store * 37 % 1000 for store in store_ids if store not in failing_stores),
    "ok": STORE_COUNT - len(failing_stores),
    "failed": failing_stores,
}
store_tracker.reset()
async with mock_client(sales_api) as sales_client:
    with Timer("aggregate_sales", report=None) as aggregate_timer:
        aggregated = await outcome(aggregate_sales(sales_client, store_ids))
run_checks(
    [
        ("totals, ok count and failed ids", lambda: aggregated, expected_sales),
        ("at most 50 in flight, and concurrent", lambda: 1 < store_tracker.peak <= 50, True),
        (
            "each 503 store retried exactly once",
            lambda: {store_tracker.calls[store] for store in store_ids if store % 40 == 11},
            {2},
        ),
        (
            "hung stores cut off (< 1 s overall)",
            lambda: isinstance(aggregated, dict) and aggregate_timer.elapsed_seconds < 1.0,
            True,
        ),
    ]
)
if isinstance(aggregated, dict):
    seconds_taken = aggregate_timer.elapsed_seconds
    print(f"aggregate_sales took {seconds_taken:.2f} s for {STORE_COUNT} endpoints")

FAIL  totals, ok count and failed ids: expected {'total': 243409, 'ok': 484, 'failed': [3, 7, 57, 100, 107, 157, 197, 207, 257, 294, 307, 357, 391, 407, 457, 488]}, got 'raised NotImplementedError'
FAIL  at most 50 in flight, and concurrent: expected True, got False
FAIL  each 503 store retried exactly once: expected {2}, got {0}
FAIL  hung stores cut off (< 1 s overall): expected True, got False
0/4 checks passed


### Recap — Unit 5.4

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| Structured concurrency | `TaskGroup`: children never outlive the block; one failure cancels the rest; `ExceptionGroup` + `except*` | `Promise.all` + manual `AbortController` |
| Timeouts and cancellation | `asyncio.timeout` / `timeout_at`; `CancelledError` is a `BaseException`: clean up, re-raise; `shield` for must-finish steps | `AbortSignal.timeout` |
| Coordination | `Event`, `Lock` (only around read-await-write), `Condition`, `Semaphore`, `Barrier`; not thread-safe | `p-limit`, hand-rolled |
| Queues and pools | bounded `Queue` = backpressure; `Queue.shutdown()` (3.13) for clean exits; pipelines pass shutdown downstream | streams with `highWaterMark` |
| Async protocols | `async with`, `@asynccontextmanager`, `async for`, async generators; `aclosing` for early exits | `for await`, `await using` |
| Blocking the loop | lag monitor, debug mode (`slow_callback_duration`); `to_thread` for I/O, process pool for Python CPU work | `blocked-at`, worker threads |
| `contextvars` | per-task copies of request state; `to_thread` copies the context, `run_in_executor` does not | `AsyncLocalStorage` |
| HTTP clients | one shared `httpx.AsyncClient`; `Limits`; connect/read/write/pool timeouts plus `asyncio.timeout` for totals | `undici.Agent` |
| Rate limiting and retries | token bucket waits; exponential backoff with full jitter; retry only what is retryable | `bottleneck`, `p-retry` |
| Performance and testing | `uvloop` for loop overhead; pytest + anyio plugin; `AsyncMock`; `httpx.MockTransport` | Jest/Vitest, `nock` |
| Interop | `run_coroutine_threadsafe`, `call_soon_threadsafe`, `wrap_future`; anyio runs on asyncio and trio | — |

**Further reading**

- [`asyncio` task groups, timeouts, shielding](https://docs.python.org/3/library/asyncio-task.html),
  [synchronization primitives](https://docs.python.org/3/library/asyncio-sync.html),
  [queues](https://docs.python.org/3/library/asyncio-queue.html),
  [developing with asyncio (debug mode)](https://docs.python.org/3/library/asyncio-dev.html)
- [`contextvars`](https://docs.python.org/3/library/contextvars.html),
  [`contextlib` (`asynccontextmanager`, `aclosing`, `AsyncExitStack`)](https://docs.python.org/3/library/contextlib.html),
  [`unittest.mock.AsyncMock`](https://docs.python.org/3/library/unittest.mock.html#unittest.mock.AsyncMock)
- [PEP 492](https://peps.python.org/pep-0492/) (`async with`, `async for`),
  [PEP 525](https://peps.python.org/pep-0525/) (async generators),
  [PEP 567](https://peps.python.org/pep-0567/) (context variables),
  [PEP 654](https://peps.python.org/pep-0654/) (exception groups and `except*`)
- [HTTPX async client](https://www.python-httpx.org/async/), [uvloop](https://github.com/MagicStack/uvloop),
  [AnyIO](https://anyio.readthedocs.io/), [Trio](https://trio.readthedocs.io/)

## Unit 5.5 — Performance and Memory Optimization

**Objectives**

- Follow a measure-first method: find the bottleneck with a profiler, predict the
  gain with Amdahl's law, change one thing, re-measure with clean benchmarks.
- Time code with `perf_counter`, `timeit` and `%timeit`, and read the spread, not
  just one number.
- Profile CPU with deterministic (`cProfile`) and sampling profilers (the 3.15
  **Tachyon** profiler, py-spy, scalene), per line with `line_profiler`, and read
  a flame graph.
- Profile memory with `tracemalloc` and memray; tell *peak* from *current*.
- Optimize in the right order: algorithms and data structures first, then
  interpreter-level habits, then memory layout, caching, vectorization and native
  code.
- Measure what the free-threaded build and the JIT change, and keep performance
  under watch in production and CI.

**JS bridge**

| Node.js | Python | The difference that bites |
| --- | --- | --- |
| `performance.now()`, `console.time` | `time.perf_counter()` / `perf_counter_ns()` | `time.time()` is wall-clock and can jump; never benchmark with it |
| `tinybench`, `mitata`, benchmark.js | `timeit`, `%timeit`, `pyperf` | `timeit` switches the garbage collector **off** while it measures |
| `node --cpu-prof` + DevTools | `cProfile` + `pstats` + snakeviz | `cProfile` hooks **every call**: exact counts, but it slows small functions down a lot |
| `0x`, `clinic flame`, `perf` | Tachyon (`python -m profiling.sampling`), py-spy | sampling profilers read the stack from outside: low overhead, can attach to a live process |
| heap snapshots, `--heap-prof` | `tracemalloc`, memray | `tracemalloc` sees Python allocations only; memray also sees C allocations |
| V8's JIT (always on, hidden classes) | specializing interpreter (3.11+) and an optional JIT (3.13+, off by default) | type-stable code helps both; Python's JIT gains are still modest |
| typed arrays, `Buffer` | `array`, `memoryview`, NumPy | one block of raw numbers instead of one object per number |
| N-API, WASM | C extensions, Cython, Rust/PyO3, `ctypes`/`cffi` | crossing the boundary has a per-call cost: move whole loops, not single operations |
| `worker_threads` for CPU work | free-threaded build (`python3.15t`) | on the default build, threads do not speed up pure-Python CPU work |

**How this unit runs.** Every profiler runs on a small script in `notebooks/_scratch/phase5/`
through a subprocess, and its real output is printed. Tools that are not installed,
the free-threaded build and the JIT are detected first; when one is missing its cell
prints a skip message. Workload sizes are multiplied by `DEMO_SCALE`. **Timings vary
by machine:** cells print what they measured and the ratio, never a promise.

Unit setup, part 1: imports, `DEMO_SCALE`, the scratch folder, and feature flags for
the optional tools (checked with `importlib.util.find_spec`, never imported here).

In [358]:
import array
import bisect
import functools
import gc
import heapq
import importlib
import importlib.util
import io
import json
import linecache
import math
import os
import platform
import pstats
import random
import re
import shutil
import statistics
import subprocess
import sys
import sysconfig
import textwrap
import time
import timeit
import tracemalloc
from collections import Counter, deque
from collections.abc import Callable, Iterable, Iterator
from dataclasses import dataclass
from pathlib import Path

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))  # the one knob for workload sizes
REPO_ROOT = find_repo_root()
scratch_dir = REPO_ROOT / "notebooks" / "_scratch"  # git-ignored
PHASE5_DIR = scratch_dir / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT / "notebooks")  # %%writefile cells use paths relative to notebooks/
if str(PHASE5_DIR) not in sys.path:
    sys.path.insert(0, str(PHASE5_DIR))
CHILD_ENV = {**os.environ, "PYTHON_COLORS": "0", "PYTHONUNBUFFERED": "1", "NO_COLOR": "1"}
CHILD_ENV.pop("FORCE_COLOR", None)  # ipykernel sets it; profilers would print colour codes
VENV_BIN = Path(sys.executable).parent


def has_module(name: str) -> bool:
    """True if the module can be imported (checked without importing it)."""
    try:
        return importlib.util.find_spec(name) is not None
    except ModuleNotFoundError:  # a parent package is missing
        return False


def tool_path(name: str) -> str | None:
    """A command-line tool from this virtual environment first, then from PATH."""
    candidate = VENV_BIN / name
    return str(candidate) if candidate.exists() else shutil.which(name)


HAS_NUMPY, HAS_PANDAS, HAS_POLARS = has_module("numpy"), has_module("pandas"), has_module("polars")
HAS_TACHYON = sys.version_info >= (3, 15) and has_module("profiling.sampling")
HAS_LINE_PROFILER, HAS_MEMRAY = has_module("line_profiler"), has_module("memray")
HAS_SCALENE, HAS_SNAKEVIZ = has_module("scalene"), has_module("snakeviz")
HAS_PYPERF, HAS_PYTEST_BENCHMARK = has_module("pyperf"), has_module("pytest_benchmark")
PY_SPY = tool_path("py-spy")
(PHASE5_DIR / "perf_tests").mkdir(exist_ok=True)  # for the %%writefile test module in 5.5.12

Unit setup, part 2: `Timer` (Unit 1.14), `run_tool()` (runs a command in the
scratch folder and replaces machine-specific paths with labels such as `<scratch>`)
and `run_child()` (Units 5.1–5.4: runs source code as a script in a child
interpreter).

In [359]:
class Timer:
    """Timing context manager (shared since Unit 1.14): reports on exit, keeps elapsed_seconds."""

    def __init__(self, label: str, report: Callable[[str], None] | None = print) -> None:
        """Remember the label and where to report the elapsed time."""
        self.label = label
        self.report = report
        self.elapsed_seconds = 0.0

    def __enter__(self) -> Timer:
        """Start the clock."""
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc_info: object) -> None:
        """Stop the clock and report the elapsed milliseconds."""
        self.elapsed_seconds = time.perf_counter() - self._start
        if self.report is not None:
            self.report(f"{self.label} took {self.elapsed_seconds * 1000:.2f} ms")


def tidy_output(text: str, python: str = sys.executable) -> str:
    """Replace machine-specific paths and addresses in child output with short labels."""
    replacements = (
        (str(PHASE5_DIR), "<scratch>"),
        (str(REPO_ROOT), "<repo>"),
        (str(Path(python).parent), "<python>"),
    )
    for path, label in replacements:
        text = text.replace(path, label)
    text = re.sub(r"\x1b\[[0-9;]*m", "", text)  # colour codes some tools print anyway
    text = re.sub(r"0x[0-9a-f]{6,}", "0x<address>", text)
    return re.sub(r"/[^\s\"'():]*/lib/python3\.\d+t?/", "<stdlib>/", text)


def run_tool(*command: str, extra_env: dict[str, str] | None = None, timeout: int = 120) -> str:
    """Run a command in the scratch folder; return its tidied stdout + stderr."""
    completed = subprocess.run(
        list(command),
        capture_output=True,
        text=True,
        timeout=timeout,
        env={**CHILD_ENV, **(extra_env or {})},
        cwd=PHASE5_DIR,
    )
    return tidy_output((completed.stdout + completed.stderr).rstrip())


def run_child(
    source: str, python: str = sys.executable, extra_env: dict[str, str] | None = None
) -> str:
    """Run dedented source as a script in a child interpreter; return its stdout + stderr."""
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    output = run_tool(python, str(script), extra_env=extra_env)
    return tidy_output(output, python)

Unit setup, part 3: `find_free_threaded_python()` (Unit 3.1), the interpreter label
printed next to benchmarks, and a summary of which tools this machine has.

In [360]:
def find_free_threaded_python() -> str | None:
    """Path of an installed free-threaded 3.15 interpreter, or None (shared since Unit 3.1)."""
    candidates: list[str] = []
    if uv_path := shutil.which("uv"):
        found = subprocess.run(
            [uv_path, "python", "find", "3.15t"], capture_output=True, text=True, timeout=60
        )
        if found.returncode == 0:
            candidates.append(found.stdout.strip())
    if on_path := shutil.which("python3.15t"):
        candidates.append(on_path)
    probe = "import sysconfig; print(sysconfig.get_config_var('Py_GIL_DISABLED'))"
    for candidate in candidates:
        answer = subprocess.run([candidate, "-c", probe], capture_output=True, text=True)
        if answer.stdout.strip() == "1":
            return candidate
    return None


def interpreter_label() -> str:
    """Implementation, version and build flavour: print it next to every benchmark."""
    free_threaded = bool(sysconfig.get_config_var("Py_GIL_DISABLED"))
    build = "free-threaded build" if free_threaded else "default (GIL) build"
    implementation = platform.python_implementation()
    return f"{implementation} {platform.python_version()}, {build}, {platform.machine()}"


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


FREE_THREADED_PYTHON = find_free_threaded_python()
print(interpreter_label())
print(f"{DEMO_SCALE=}  free-threaded interpreter: {'found' if FREE_THREADED_PYTHON else 'none'}")
tools = {
    "Tachyon (profiling.sampling)": HAS_TACHYON,
    "py-spy": PY_SPY is not None,
    "line_profiler": HAS_LINE_PROFILER,
    "memray": HAS_MEMRAY,
    "scalene": HAS_SCALENE,
    "snakeviz": HAS_SNAKEVIZ,
    "pyperf": HAS_PYPERF,
    "pytest-benchmark": HAS_PYTEST_BENCHMARK,
}
for tool_name, found in tools.items():
    print(
        f"  {tool_name:<30} "
        f"{'available' if found else 'not installed: its cells print a skip message'}"
    )

CPython 3.15.1, default (GIL) build, x86_64
DEMO_SCALE=1  free-threaded interpreter: found
  Tachyon (profiling.sampling)   available
  py-spy                         available
  line_profiler                  available
  memray                         available
  scalene                        available
  snakeviz                       available
  pyperf                         not installed: its cells print a skip message
  pytest-benchmark               not installed: its cells print a skip message


### 5.5.1 Method

Performance work fails in one of two ways: optimizing code that does not matter,
or "optimizing" without measuring and making things slower. The cure is a loop:

```mermaid
flowchart LR
    A[Measure end to end] --> B[Profile: where does the time go?]
    B --> C[Hypothesis + Amdahl estimate]
    C --> D[Change ONE thing]
    D --> E[Re-measure, same input]
    E -->|faster and still correct| F[Keep, commit]
    E -->|not faster| G[Revert]
    F --> A
    G --> B
```

<details><summary>ASCII fallback</summary>

```text
measure -> profile -> hypothesis (+ Amdahl) -> change one thing -> re-measure
   ^                                                               |
   +------------------- keep (faster, correct) or revert ----------+
```

</details>

**Measure first.** Your intuition about *where* the time goes is usually wrong: a
"slow" regex may cost 1 % while a forgotten linear scan inside a loop costs 90 %.

#### Amdahl's law for optimization

If a part that takes a fraction `p` of the runtime becomes `s` times faster, the
whole program becomes `1 / ((1 - p) + p / s)` times faster. Unit 5.2 used the same
formula for parallel workers; here it decides what is worth optimizing at all.

In [361]:
def overall_speedup(fraction: float, part_speedup: float) -> float:
    """Amdahl: whole-program speedup when `fraction` of it gets `part_speedup` faster."""
    return 1 / ((1 - fraction) + fraction / part_speedup)


print(
    f"{'share of runtime':>17} | {'part 2x faster':>14} | "
    f"{'part 10x':>8} | {'part infinitely fast':>20}"
)
for fraction in (0.05, 0.25, 0.50, 0.90, 0.99):
    speedups = [overall_speedup(fraction, part_speedup) for part_speedup in (2, 10, math.inf)]
    print(
        f"{fraction:>17.0%} | {speedups[0]:>13.2f}x | {speedups[1]:>7.2f}x | {speedups[2]:>19.2f}x"
    )

 share of runtime | part 2x faster | part 10x | part infinitely fast
               5% |          1.03x |    1.05x |                1.05x
              25% |          1.14x |    1.29x |                1.33x
              50% |          1.33x |    1.82x |                2.00x
              90% |          1.82x |    5.26x |               10.00x
              99% |          1.98x |    9.17x |              100.00x


Read the first row: making a function that takes 5 % of the runtime *infinitely*
fast gains 1.05×. Read the last column: the untouched part caps the total. This is
why the profiler comes first: it tells you `p`.

#### Benchmark hygiene

| Rule | Why |
| --- | --- |
| Same input, same machine, same Python build | otherwise you compare different things |
| Warm up, then repeat many times | the first calls pay for imports, caches and the specializing interpreter's warm-up |
| Report the **minimum** (or median) and the spread | noise only ever adds time; one number hides variance |
| Run benchmarks in a **fresh process** | earlier cells leave garbage, warm caches and fragmented memory |
| Keep the machine quiet | browsers, indexing and CPU frequency scaling add noise |
| Check correctness in the same run | a fast wrong answer is not an optimization |
| Measure at realistic sizes | O(n²) code looks fine at n = 100 |

Warm-up is real. First calls pay one-time costs: lazy imports, filling caches
(here: `re` compiling a pattern and caching it), and the specializing
interpreter's warm-up (Unit 3.2). `sum_of_squares` is used by later cells; the
regex function shows the warm-up (run the cell a second time and the slow first
call is gone, because the pattern is already cached):

In [362]:
def sum_of_squares(count: int) -> int:
    """A small pure-Python loop to time."""
    total = 0
    for number in range(count):
        total += number * number
    return total


def parse_order_code(code_text: str) -> tuple[str, ...] | None:
    """Split an order code such as MH-2026-004217-7; re compiles the pattern on first use."""
    match = re.fullmatch(
        r"(?P<state>[A-Z]{2})-(?P<year>\d{4})-(?P<serial>\d{6})-(?P<check>[0-9X])", code_text
    )
    return match.groups() if match else None


call_times = []
for _ in range(5):
    start = time.perf_counter_ns()
    parse_order_code("MH-2026-004217-7")
    call_times.append(time.perf_counter_ns() - start)
print("per-call ns:", call_times)
print(f"first call / fastest call = {call_times[0] / min(call_times):.0f}x")

per-call ns: [91858, 2187, 856, 517, 495]
first call / fastest call = 186x


Repeat and look at the spread. `timeit.repeat` returns one total per repetition;
the minimum is the best estimate of the code's cost, the rest is noise from the
machine:

In [363]:
totals = timeit.repeat(lambda: sum_of_squares(2_000), number=200, repeat=7)
per_call_us = sorted(total / 200 * 1e6 for total in totals)
print("per call, sorted (µs):", [round(value, 2) for value in per_call_us])
print(f"min {per_call_us[0]:.2f} µs, median {statistics.median(per_call_us):.2f} µs")
print(f"spread (max/min): {per_call_us[-1] / per_call_us[0]:.2f}x")

per call, sorted (µs): [35.24, 35.35, 35.43, 35.5, 35.73, 35.75, 36.8]
min 35.24 µs, median 35.50 µs
spread (max/min): 1.04x


`timeit` switches the garbage collector **off** while it measures, so code that
allocates many container objects looks faster under `timeit` than in a real
program. Pass `gc.enable()` as the setup to measure with the collector on:

In [364]:
build_nested = "[[index] for index in range(20_000)]"  # 20 000 GC-tracked lists
gc_off = min(timeit.repeat(build_nested, number=20, repeat=5)) / 20
gc_on = (
    min(timeit.repeat(build_nested, setup="gc.enable()", number=20, repeat=5, globals=globals()))
    / 20
)
print(f"GC off (timeit default): {gc_off * 1000:.2f} ms per call")
print(f"GC on:                   {gc_on * 1000:.2f} ms per call  ({gc_on / gc_off:.2f}x)")

GC off (timeit default): 0.60 ms per call
GC on:                   2.37 ms per call  (3.94x)


#### pyperf

[`pyperf`](https://pyperf.readthedocs.io/) is the benchmark runner CPython's own
developers use (it powers the `pyperformance` suite). It applies the hygiene rules
for you: it spawns **several worker processes**, discards warm-up runs, calibrates
the loop count, reports mean ± standard deviation, warns when the results are
unstable, and can compare two saved result files (`python -m pyperf compare_to`).
It is not part of this course's dependencies, so the cell checks for it first:

In [365]:
pyperf_command = [
    sys.executable,
    "-m",
    "pyperf",
    "timeit",
    "--fast",
    "-s",
    "values = list(range(1000))",
    "sum(values)",
]
print("$ python -m pyperf timeit --fast -s 'values = list(range(1000))' 'sum(values)'")
if HAS_PYPERF:
    print(run_tool(*pyperf_command))
else:
    print("pyperf is not installed: skipping (install with `uv add --dev pyperf`).")

$ python -m pyperf timeit --fast -s 'values = list(range(1000))' 'sum(values)'
pyperf is not installed: skipping (install with `uv add --dev pyperf`).


The core idea needs nothing but the standard library: run the measurement in
several **fresh processes** and look at how much the processes disagree. Here
`python -m timeit` (the command-line face of `timeit`) runs in four separate
interpreters:

In [366]:
per_process = []
for _ in range(4):
    line = run_tool(
        sys.executable, "-m", "timeit", "-r", "5", "-s", "values = list(range(1000))", "sum(values)"
    )
    print(line)
    per_process.append(
        float(line.split(":")[1].split()[0])
    )  # "N loops, best of 5: X usec per loop"
mean, spread = statistics.mean(per_process), statistics.stdev(per_process)
print(f"across processes: {mean:.3f} ± {spread:.3f} (same unit as above)")

100000 loops, best of 5: 2.48 usec per loop


100000 loops, best of 5: 2.01 usec per loop


100000 loops, best of 5: 2.48 usec per loop


100000 loops, best of 5: 2.53 usec per loop
across processes: 2.375 ± 0.244 (same unit as above)


### 5.5.2 Timing

#### `perf_counter` and the other clocks

| Clock | Measures | Use it for |
| --- | --- | --- |
| `time.perf_counter()` / `perf_counter_ns()` | wall time, monotonic, highest resolution | benchmarks, `Timer` |
| `time.monotonic()` | wall time, monotonic | timeouts, deadlines, rate limiters |
| `time.process_time()` | CPU time of this process (all threads), excludes sleep | "is it CPU or waiting?" |
| `time.time()` | calendar time since 1970; **can jump** (NTP, manual changes) | timestamps, never durations |

`time.get_clock_info` shows what each clock is built on and its resolution:

In [367]:
for clock_name in ("perf_counter", "monotonic", "process_time", "time"):
    info = time.get_clock_info(clock_name)
    print(
        f"{clock_name:<13} monotonic={info.monotonic!s:<5} adjustable={info.adjustable!s:<5}"
        f" resolution={info.resolution:.0e} s  ({info.implementation})"
    )

perf_counter  monotonic=True  adjustable=False resolution=1e-09 s  (clock_gettime(CLOCK_MONOTONIC))
monotonic     monotonic=True  adjustable=False resolution=1e-09 s  (clock_gettime(CLOCK_MONOTONIC))
process_time  monotonic=True  adjustable=False resolution=1e-09 s  (clock_gettime(CLOCK_PROCESS_CPUTIME_ID))
time          monotonic=False adjustable=True  resolution=1e-09 s  (clock_gettime(CLOCK_REALTIME))


Wall time and CPU time answer different questions. A sleep (or a network wait)
takes wall time but almost no CPU time; a busy loop takes both. Comparing the two
tells you whether code is waiting or computing:

In [368]:
def wall_and_cpu(work: Callable[[], object]) -> tuple[float, float]:
    """Return (wall seconds, CPU seconds) for one call of work."""
    wall_start, cpu_start = time.perf_counter(), time.process_time()
    work()
    return time.perf_counter() - wall_start, time.process_time() - cpu_start


for label, work in (
    ("sleep 0.2 s", lambda: time.sleep(0.2)),
    ("busy loop", lambda: sum_of_squares(1_000_000)),
):
    wall_seconds, cpu_seconds = wall_and_cpu(work)
    print(f"{label:<12} wall {wall_seconds:.3f} s  CPU {cpu_seconds:.3f} s")

sleep 0.2 s  wall 0.200 s  CPU 0.001 s
busy loop    wall 0.042 s  CPU 0.042 s


Use `perf_counter_ns()` (integer nanoseconds) when you subtract many small
intervals: floats lose precision on a large counter value. `Timer` from
Unit 1.14 wraps `perf_counter` in a context manager:

In [369]:
float_rng = random.Random(42)
unsorted_floats = [float_rng.random() for _ in range(200_000 * DEMO_SCALE)]
with Timer("sorting 200 000 floats"):
    sorted(unsorted_floats)
gaps = []
for _ in range(1_000):
    first_ns = time.perf_counter_ns()
    gaps.append(time.perf_counter_ns() - first_ns)
print(f"smallest gap between two perf_counter_ns() calls: {min(gaps)} ns")

sorting 200 000 floats took 23.70 ms
smallest gap between two perf_counter_ns() calls: 48 ns


#### `timeit`

`timeit.timeit(stmt, setup, number)` runs `stmt` `number` times and returns the
**total** seconds. Statements can be strings (compiled once, run in a fresh
namespace, so pass `globals=` to see your names) or zero-argument callables (the
call itself adds a little overhead):

In [370]:
string_total = timeit.timeit(
    "sorted(values)", setup="values = list(range(1000, 0, -1))", number=2_000
)
callable_total = timeit.timeit(lambda: sorted(range(1000, 0, -1)), number=2_000)
with_globals = timeit.timeit("sum_of_squares(100)", number=2_000, globals=globals())
print(f"string stmt:   {string_total / 2_000 * 1e6:.2f} µs per call")
print(f"callable stmt: {callable_total / 2_000 * 1e6:.2f} µs per call (also builds the range)")
print(f"globals=:      {with_globals / 2_000 * 1e6:.2f} µs per sum_of_squares(100)")

string stmt:   3.19 µs per call
callable stmt: 4.84 µs per call (also builds the range)
globals=:      1.37 µs per sum_of_squares(100)


`Timer.autorange()` picks the loop count for you (powers of 1, 2, 5 × 10ⁿ until the
total reaches 0.2 s), which is what the `python -m timeit` command does before it
repeats:

In [371]:
loop_count, total_seconds = timeit.Timer("'-'.join(map(str, range(100)))").autorange()
print(f"autorange chose {loop_count:,} loops, {total_seconds:.3f} s in total")
print(f"-> {total_seconds / loop_count * 1e6:.2f} µs per call")

autorange chose 100,000 loops, 0.362 s in total
-> 3.62 µs per call


#### `%timeit` and `%%timeit`

In IPython and Jupyter, `%timeit` (one line) and `%%timeit` (whole cell) do the
autorange and repeat for you and print mean ± standard deviation. `-n` fixes the
loops per run and `-r` the number of runs (fixed here to keep Run All fast); `-o`
returns a `TimeitResult` object whose `.best`, `.average` and `.stdev` you can use:

In [372]:
%timeit -n 2000 -r 5 sum(range(1_000))

2.75 μs ± 993 ns per loop (mean ± std. dev. of 5 runs, 2,000 loops each)


The cell magic times the whole cell body; the first line after `%%timeit` is setup that is not timed:

In [373]:
%%timeit -n 200 -r 5
joined = ""
for word in ["chai"] * 1_000:
    joined += word

24.4 μs ± 4.35 μs per loop (mean ± std. dev. of 5 runs, 200 loops each)


With `-o -q` the result is returned quietly, so you can compute with it:

In [374]:
timeit_result = %timeit -o -q -n 2000 -r 5 sorted(range(1_000, 0, -1))
print(f"best {timeit_result.best * 1e6:.2f} µs, average {timeit_result.average * 1e6:.2f} µs")
print(
    f"stdev {timeit_result.stdev * 1e6:.2f} µs over {timeit_result.repeat} runs"
    f" of {timeit_result.loops} loops"
)

best 5.32 µs, average 5.47 µs
stdev 0.11 µs over 5 runs of 2000 loops


### 5.5.3 CPU profiling

A **profiler** tells you where the time goes. There are two families:

| | Deterministic (tracing) | Sampling (statistical) |
| --- | --- | --- |
| How | a hook runs on **every** call and return | the stack is read every *N* µs, often from another process |
| Tools | `cProfile` (= `profiling.tracing` in 3.15), `line_profiler` | Tachyon (3.15), py-spy, scalene |
| Gives | exact call counts, per-function times | the share of samples per function or line |
| Overhead | large, and uneven: tiny functions look slower than they are | small and even; safe in production |
| Attach to a running process | no | yes (Tachyon, py-spy) |

Every profiler below runs on the same small script, in a subprocess, and its real
output is printed. The script has three deliberate hot spots: a linear-scan lookup
inside a loop, a string-building loop, and a simulated network call that waits
without using the CPU.

In [375]:
%%writefile _scratch/phase5/profile_target.py
"""Profiling target for Unit 5.5: an order report with three deliberate hot spots."""

import os
import time

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))
CITIES = ("Pune", "Delhi", "Kochi", "Patna", "Surat")


def make_customers(count: int) -> list[tuple[int, str]]:
    """(customer_id, city) pairs; ids are spread out, so no list index works."""
    return [(number * 7 + 3, CITIES[number % 5]) for number in range(count)]


def make_orders(count: int, customers: list[tuple[int, str]]) -> list[tuple[int, int]]:
    """(customer_id, amount_paise) pairs from a deterministic formula (no randomness)."""
    return [
        (customers[(number * 7919) % len(customers)][0], (number * 104729) % 50_000 + 100)
        for number in range(count)
    ]

Overwriting _scratch/phase5/profile_target.py


Hot spot 1 and the function that calls it. `%%writefile -a` appends to the same
file, so the script is still one module with the same line numbers:

In [376]:
%%writefile -a _scratch/phase5/profile_target.py


def find_city(customers: list[tuple[int, str]], customer_id: int) -> str:
    """Hot spot 1: a linear scan for every lookup."""
    for candidate_id, city in customers:
        if candidate_id == customer_id:
            return city
    raise KeyError(customer_id)


def city_totals(orders: list[tuple[int, int]], customers: list[tuple[int, str]]) -> dict[str, int]:
    """Revenue per city, calling find_city once per order."""
    totals: dict[str, int] = {}
    for customer_id, amount in orders:
        city = find_city(customers, customer_id)
        totals[city] = totals.get(city, 0) + amount
    return totals

Appending to _scratch/phase5/profile_target.py


Hot spots 2 and 3 and the entry point, appended in the same way:

In [377]:
%%writefile -a _scratch/phase5/profile_target.py


def render_lines(orders: list[tuple[int, int]]) -> str:
    """Hot spot 2: one str.format call and one string concatenation per order."""
    text = ""
    for number, (customer_id, amount) in enumerate(orders * 20):
        text += "{:>7} {:>8} {:>10.2f}\n".format(number, customer_id, amount / 100)
    return text


def fetch_exchange_rate() -> float:
    """Hot spot 3, wall clock only: a simulated 0.3 s network call (no CPU used)."""
    time.sleep(0.3)
    return 83.25


def main() -> None:
    """Build the data, fetch the rate, compute and render."""
    customers = make_customers(2_500)
    orders = make_orders(8_000 * DEMO_SCALE, customers)
    rate = fetch_exchange_rate()
    totals = city_totals(orders, customers)
    text = render_lines(orders)
    print(f"{len(orders)} orders, {len(totals)} cities, {len(text):,} chars, rate {rate}")


if __name__ == "__main__":
    main()

Appending to _scratch/phase5/profile_target.py


First, the plain run: how long does the whole script take without any profiler?

In [378]:
with Timer("profile_target.py, no profiler"):
    print(run_tool(sys.executable, "profile_target.py"))

8000 orders, 5 cities, 4,480,000 chars, rate 83.25
profile_target.py, no profiler took 485.16 ms


#### `cProfile` from the command line

`python -m cProfile -s tottime script.py` runs the script under the profiler and
prints one row per function, sorted by **own time**. Only the top of the table
matters:

In [379]:
def first_lines(text: str, line_count: int) -> str:
    """The first line_count lines of a block of text."""
    return "\n".join(text.splitlines()[:line_count])


cprofile_output = run_tool(sys.executable, "-m", "cProfile", "-s", "tottime", "profile_target.py")
print(first_lines(cprofile_output, 14))

8000 orders, 5 cities, 4,480,000 chars, rate 83.25
         184019 function calls in 0.526 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.300    0.300    0.300    0.300 {built-in method time.sleep}
     8000    0.110    0.000    0.110    0.000 profile_target.py:23(find_city)
   160000    0.080    0.000    0.080    0.000 {method 'format' of 'str' objects}
        1    0.030    0.030    0.110    0.110 profile_target.py:40(render_lines)
        1    0.002    0.002    0.113    0.113 profile_target.py:31(city_totals)
        1    0.001    0.001    0.002    0.002 profile_target.py:15(make_orders)
     8000    0.001    0.000    0.001    0.000 {method 'get' of 'dict' objects}
     8003    0.000    0.000    0.000    0.000 {built-in method builtins.len}


| Column | Meaning |
| --- | --- |
| `ncalls` | number of calls (`a/b`: total/primitive, for recursion) |
| `tottime` | time spent **inside** the function itself, excluding the functions it called |
| `percall` (first) | `tottime / ncalls` |
| `cumtime` | time from entry to exit, **including** everything it called |
| `percall` (second) | `cumtime / primitive calls` |

Read it: `find_city` has the biggest `tottime`, and it is called once per order. The
fix is not to make `find_city` faster but to stop calling it 8 000 times (a dict,
in 5.5.5). Notice too that the run took longer than without the profiler: that is
the tracing overhead.

#### Saving a profile and reading it with `pstats`

`-o file.prof` saves the raw data instead of printing it. `pstats.Stats` loads it
and can sort, filter and restrict the output; `strip_dirs()` shortens file paths:

In [380]:
print(run_tool(sys.executable, "-m", "cProfile", "-o", "target.prof", "profile_target.py"))
stats_buffer = io.StringIO()
target_stats = pstats.Stats(str(PHASE5_DIR / "target.prof"), stream=stats_buffer)
target_stats.strip_dirs().sort_stats(pstats.SortKey.CUMULATIVE).print_stats(r"profile_target", 6)
print(tidy_output(stats_buffer.getvalue().strip()))

8000 orders, 5 cities, 4,480,000 chars, rate 83.25
Tue Nov 17 14:40:50 2026    <scratch>/target.prof

         184019 function calls in 0.523 seconds

   Ordered by: cumulative time
   List reduced from 20 to 8 due to restriction <'profile_target'>
   List reduced from 8 to 6 due to restriction <6>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    0.523    0.523 profile_target.py:1(<module>)
        1    0.000    0.000    0.523    0.523 profile_target.py:54(main)
        1    0.000    0.000    0.300    0.300 profile_target.py:48(fetch_exchange_rate)
        1    0.002    0.002    0.111    0.111 profile_target.py:31(city_totals)
        1    0.029    0.029    0.110    0.110 profile_target.py:40(render_lines)
     8000    0.108    0.000    0.108    0.000 profile_target.py:23(find_city)


The argument `r"profile_target"` is a regex that keeps only rows from our file,
and `6` keeps the first six. `print_callers` answers "who calls this?" and
`print_callees` answers "what does this call?":

In [381]:
stats_buffer.seek(0)
stats_buffer.truncate()
target_stats.print_callers("find_city")
target_stats.print_callees("city_totals")
print(tidy_output(stats_buffer.getvalue().strip()))

Ordered by: cumulative time
   List reduced from 20 to 1 due to restriction <'find_city'>

Function                         was called by...
                                     ncalls  tottime  cumtime
profile_target.py:23(find_city)  <-    8000    0.108    0.108  profile_target.py:31(city_totals)


   Ordered by: cumulative time
   List reduced from 20 to 1 due to restriction <'city_totals'>

Function                           called...
                                       ncalls  tottime  cumtime
profile_target.py:31(city_totals)  ->    8000    0.108    0.108  profile_target.py:23(find_city)
                                         8000    0.001    0.001  {method 'get' of 'dict' objects}


#### The `cProfile` API, and `profiling.tracing`

In code, `cProfile.Profile()` is a context manager (3.8+), and `pstats.Stats`
accepts the profiler directly. *New in 3.15* (PEP 799): the profilers moved into a
`profiling` package; `profiling.tracing` is the new name of `cProfile`, which stays
as an alias. The demo runs in a child interpreter:

In [382]:
print(
    run_child(
        """
        import cProfile
        import pstats
        import sys

        import profile_target

        customers = profile_target.make_customers(2_500)
        orders = profile_target.make_orders(2_000, customers)
        with cProfile.Profile() as profiler:
            profile_target.city_totals(orders, customers)
        pstats.Stats(profiler).strip_dirs().sort_stats(pstats.SortKey.TIME).print_stats(3)
        if sys.version_info >= (3, 15):
            import profiling.tracing

            same_class = profiling.tracing.Profile is cProfile.Profile
            print("profiling.tracing.Profile is cProfile.Profile:", same_class)
        """
    ).strip()
)

4003 function calls in 0.025 seconds

   Ordered by: internal time
   List reduced from 5 to 3 due to restriction <3>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     2000    0.025    0.000    0.025    0.000 profile_target.py:23(find_city)
        1    0.000    0.000    0.025    0.025 profile_target.py:31(city_totals)
     2000    0.000    0.000    0.000    0.000 {method 'get' of 'dict' objects}


profiling.tracing.Profile is cProfile.Profile: True


#### snakeviz

snakeviz is a **browser** viewer for `.prof` files: an icicle or sunburst chart in
which each function's width is its share of the time and clicking zooms in. It
starts a local web server, so this notebook does not launch it. Here is the command
and the file it reads:

In [383]:
profile_file = PHASE5_DIR / "target.prof"
print(f"profile file: <scratch>/{profile_file.name}, {profile_file.stat().st_size:,} bytes")
print(f"functions recorded: {len(target_stats.stats)}, total calls: {target_stats.total_calls:,}")
print("open it in a terminal with:")
print("    uv run snakeviz notebooks/_scratch/phase5/target.prof")
if not HAS_SNAKEVIZ:
    print("snakeviz is not installed: `uv add --dev snakeviz` first.")

profile file: <scratch>/target.prof, 2,163 bytes
functions recorded: 20, total calls: 184,019
open it in a terminal with:
    uv run snakeviz notebooks/_scratch/phase5/target.prof


#### The Tachyon sampling profiler (*New in 3.15*)

Python 3.15 ships a sampling profiler in the standard library, `profiling.sampling`,
nicknamed **Tachyon** (PEP 799). It reads the target's stack from **outside** the
process many thousands of times per second, so the target runs at nearly full
speed. `run` starts a script under it; `attach PID` samples a process that is
already running (5.5.12). Outputs include a pstats-like table (the default),
`--collapsed` stacks, an HTML `--flamegraph`, `--gecko` for the Firefox Profiler and
a source `--heatmap`.

In [384]:
if HAS_TACHYON:
    tachyon_output = run_tool(
        sys.executable,
        "-m",
        "profiling.sampling",
        "run",
        "-r",
        "5khz",
        "--sort",
        "tottime",
        "-l",
        "6",
        "--no-summary",
        "profile_target.py",
    )
    print(tachyon_output)
else:
    print("Tachyon needs Python 3.15+ (profiling.sampling): skipping.")

8000 orders, 5 cities, 4,480,000 chars, rate 83.25
Captured 2,520 samples in 0.50 seconds
Sample rate: 4,999.64 samples/sec
Error rate: 1.43
Profile Stats:
       nsamples   sample%  tottime (ms)    cumul%  cumtime (ms)  filename:lineno(function)
      1501/1501      60.4       300.200      60.4       300.200  profile_target.py:50(fetch_exchange_rate)
        404/404      16.3        80.800      16.3        80.800  profile_target.py:44(render_lines)
        276/276      11.1        55.200      11.1        55.200  profile_target.py:25(find_city)
        264/264      10.6        52.800      10.6        52.800  profile_target.py:26(find_city)
          21/21       0.8         4.200       0.8         4.200  profile_target.py:43(render_lines)
            7/7       0.3         1.400       0.3         1.400  ~:0(<GC>)

Legend:
  nsamples: Direct/Cumulative samples (direct executing / on call stack)
  sample%: Percentage of total samples this function was directly executing
  tottime: Estimate

The rows are **lines**, not functions (`file:line(function)`), so a hot loop body
stands out directly. The columns are estimates: `nsamples` direct/cumulative,
`sample%` the share of samples executing that line.

**Wall time vs CPU time.** By default Tachyon samples in `wall` mode: a thread
blocked in `time.sleep` (or on a socket) still collects samples. `--mode cpu` keeps
only samples taken while the thread was running on a CPU. The simulated network
call shows the difference:

In [385]:
if HAS_TACHYON:
    for mode in ("wall", "cpu"):
        mode_output = run_tool(
            sys.executable,
            "-m",
            "profiling.sampling",
            "run",
            "--mode",
            mode,
            "--sort",
            "cumtime",
            "-l",
            "40",
            "--no-summary",
            "profile_target.py",
        )
        interesting = [
            line
            for line in mode_output.splitlines()
            if "fetch_exchange_rate" in line or "(city_totals)" in line
        ]
        print(f"--mode {mode}:")
        print(
            "\n".join(f"    {line.strip()}" for line in interesting)
            or "    (no samples in these functions)"
        )
else:
    print("Tachyon needs Python 3.15+: skipping.")

--mode wall:
    300/300      62.2       300.000      62.2       300.000  profile_target.py:50(fetch_exchange_rate)
    1/105       0.2         1.000      21.8       105.000  profile_target.py:35(city_totals)


--mode cpu:
    0/107       0.0         0.000      55.2       107.000  profile_target.py:35(city_totals)


In `wall` mode `fetch_exchange_rate` holds a large share (it waited 0.3 s); in `cpu`
mode it all but disappears. Use wall mode to find out why a request is slow, CPU
mode to find out what burns the processor. (`--mode gil` keeps samples where the
thread held the GIL, which is useful for threaded programs.)

#### Reading flame graphs

A **flame graph** draws every sampled stack as a column of boxes: the bottom box is
the entry point, each box above it is a function that box called.

- **Width** = share of samples containing that function (not time order: the x axis
  is sorted alphabetically or by size, not by when things happened).
- **Height** = stack depth. Tall is not slow; **wide** is.
- A wide box with nothing on top is spending its *own* time: that is the hot spot.
  A wide box covered by children is just passing time down.
- Colours are usually random (or by module); they carry no meaning.
- An "icicle" chart (snakeviz, some viewers) is the same picture upside down.

Tachyon writes **collapsed stacks** (one line per distinct stack: frames joined by
`;`, then a sample count), the input format of every flame-graph tool, and an
interactive HTML flame graph:

In [386]:
if HAS_TACHYON:
    for output_flag, file_name in (
        ("--collapsed", "tachyon_stacks.txt"),
        ("--flamegraph", "target_flame.html"),
    ):
        print(
            run_tool(
                sys.executable,
                "-m",
                "profiling.sampling",
                "run",
                output_flag,
                "-o",
                file_name,
                "profile_target.py",
            )
        )
    tachyon_stacks = (PHASE5_DIR / "tachyon_stacks.txt").read_text()
    print(
        "\nHTML flame graph: <scratch>/target_flame.html "
        f"({(PHASE5_DIR / 'target_flame.html').stat().st_size:,} bytes)"
    )
    print("first collapsed stack:", tidy_output(tachyon_stacks.splitlines()[0])[:150], "...")
else:
    tachyon_stacks = ""
    print("Tachyon needs Python 3.15+: skipping.")

8000 orders, 5 cities, 4,480,000 chars, rate 83.25
Captured 513 samples in 0.51 seconds
Sample rate: 1,000.00 samples/sec
Error rate: 1.75
Collapsed stack output written to tachyon_stacks.txt


8000 orders, 5 cities, 4,480,000 chars, rate 83.25
Captured 482 samples in 0.48 seconds
Sample rate: 1,000.00 samples/sec
Error rate: 1.45
Flamegraph data: 1 root function, 475 total samples, 93 unique strings
Flamegraph saved to: target_flame.html

HTML flame graph: <scratch>/target_flame.html (632,345 bytes)
first collapsed stack: tid:152739;<frozen runpy>:_run_module_as_main:201;<frozen runpy>:_run_code:87;profile_target.py:<module>:65;profile_target.py:main:58;profile_target.p ...


A flame graph is a tree of stacks, so a few lines of Python can draw it as text:
parse the collapsed stacks, add the counts up along each path, and print each
function indented under its caller with a bar as wide as its share. Frames are
reduced to function names, and everything below `main` (the interpreter's start-up
frames) is cut off:

In [387]:
def parse_collapsed(text: str) -> Counter[tuple[str, ...]]:
    """Collapsed stacks (Tachyon or py-spy format) -> {(main, ..., leaf): samples}."""
    stacks: Counter[tuple[str, ...]] = Counter()
    for line in text.splitlines():
        frames_text, _, count = line.rpartition(" ")
        if not frames_text:
            continue
        names = []
        for frame in frames_text.split(";"):
            if frame.startswith("tid:"):
                continue
            if " (" in frame:  # py-spy: "function (file:line)"
                names.append(frame.split(" (")[0])
            elif frame.count(":") >= 2:  # Tachyon: "file:function:line"
                names.append(frame.split(":")[-2])
            else:  # pseudo-frames such as <GC>
                names.append(frame)
        if "main" in names:
            stacks[tuple(names[names.index("main") :])] += int(count)
    return stacks

`flame_tree` adds the samples up along every path, then draws each function under
its caller, widest first, skipping branches under 1 %:

In [388]:
def flame_tree(stacks: Counter[tuple[str, ...]], min_share: float = 0.01, width: int = 30) -> str:
    """Draw collapsed stacks as an indented text flame graph (widest children first)."""
    totals: Counter[tuple[str, ...]] = Counter()
    for stack, count in stacks.items():
        for depth in range(1, len(stack) + 1):
            totals[stack[:depth]] += count
    grand_total = sum(stacks.values()) or 1
    lines: list[str] = []

    def draw(path: tuple[str, ...]) -> None:
        """Append one bar for path, then recurse into its children."""
        share = totals[path] / grand_total
        label = "  " * (len(path) - 1) + path[-1]
        lines.append(f"{label:<34} {'█' * round(share * width):<{width}} {share:6.1%}")
        children = [
            child_path
            for child_path in totals
            if len(child_path) == len(path) + 1 and child_path[:-1] == path
        ]
        for child in sorted(children, key=totals.__getitem__, reverse=True):
            if totals[child] / grand_total >= min_share:
                draw(child)

    for root in sorted({stack[:1] for stack in stacks}, key=totals.__getitem__, reverse=True):
        draw(root)
    return "\n".join(lines)

The Tachyon samples of `profile_target.py`, drawn as a text flame graph (wall mode):

In [389]:
print(
    flame_tree(parse_collapsed(tachyon_stacks))
    if tachyon_stacks
    else "no Tachyon samples (skipped above)"
)

main                               ██████████████████████████████ 100.0%
  fetch_exchange_rate              ██████████████████              59.8%
  city_totals                      ███████                         22.5%
    find_city                      ███████                         22.5%
  render_lines                     █████                           17.3%


Read it top-down: `main` is 100 %. `city_totals` is wide, but almost all of it is
`find_city` beneath it, so `find_city` is the plateau to attack. `fetch_exchange_rate`
is wide only because wall mode counts waiting (its child is `time.sleep`, which has
no Python frame). `render_lines` is the second CPU hot spot.

#### py-spy

[py-spy](https://github.com/benfred/py-spy) is a sampling profiler written in Rust
that, like Tachyon, reads the target's memory from outside. It works on many Python
versions (useful before 3.15) and offers `record` (flame graphs: SVG by default,
`-f raw` for collapsed stacks, `-f speedscope`), `top` (a live, `top`-like view)
and `dump` (print every thread's stack once, see 5.5.12). On Linux it needs
permission to read another process's memory (`ptrace`): fine for a child it
starts itself, `sudo` or `kernel.yama.ptrace_scope=0` for attaching.

In [390]:
if PY_SPY:
    spy_output = run_tool(
        PY_SPY,
        "record",
        "-r",
        "250",
        "-f",
        "raw",
        "-o",
        "pyspy_stacks.txt",
        "--",
        sys.executable,
        "profile_target.py",
    )
    print(
        "\n".join(
            line
            for line in spy_output.splitlines()
            if (line.startswith("py-spy>") or "orders" in line) and "flamegraph.pl" not in line
        )
    )
    if (PHASE5_DIR / "pyspy_stacks.txt").exists():
        print(flame_tree(parse_collapsed((PHASE5_DIR / "pyspy_stacks.txt").read_text())))
else:
    print("py-spy is not installed (or not on PATH): skipping. Install with `uv add --dev py-spy`.")

py-spy> Sampling process 250 times a second. Press Control-C to exit.
8000 orders, 5 cities, 4,480,000 chars, rate 83.25
py-spy> Stopped sampling because process exited
py-spy> Wrote raw flamegraph data to 'pyspy_stacks.txt'. Samples: 59 Errors: 0
main                               ██████████████████████████████ 100.0%
  city_totals                      ████████████████                52.5%
    find_city                      ████████████████                52.5%
  render_lines                     █████████                       30.5%
  fetch_exchange_rate              █████                           16.9%


The CPU work has the same shape as in Tachyon's graph: two independent sampling
profilers agree. By default py-spy **drops idle samples** (a thread sleeping or
blocked on I/O), so `fetch_exchange_rate` almost vanishes, as in Tachyon's
`--mode cpu`; `py-spy record --idle` keeps them.

#### `line_profiler`

Function-level profiles say *which* function; `line_profiler` says *which line*.
It is deterministic: it times every line of the functions you select, so select
only the suspects. The usual workflow is to decorate them with `@profile` and run
`kernprof -l -v script.py`; here the `LineProfiler` API wraps two functions of the
unchanged script, in a child interpreter. Timing every line is slow, so the child
uses a quarter of the orders:

In [391]:
if HAS_LINE_PROFILER:
    line_profile_output = run_child(
        """
        from line_profiler import LineProfiler

        import profile_target

        customers = profile_target.make_customers(2_500)
        orders = profile_target.make_orders(2_000, customers)
        line_profiler = LineProfiler(profile_target.find_city, profile_target.render_lines)
        line_profiler.runcall(profile_target.city_totals, orders, customers)
        line_profiler.runcall(profile_target.render_lines, orders)
        line_profiler.print_stats(output_unit=1e-3)
        """
    )
    print(re.sub(r"\n{2,}", "\n", line_profile_output))
else:
    print("line_profiler is not installed: skipping. Install with `uv add --dev line-profiler`.")

Timer unit: 0.001 s
Total time: 0.736786 s
File: <scratch>/profile_target.py
Function: find_city at line 23
Line #      Hits         Time  Per Hit   % Time  Line Contents
    23                                           def find_city(customers: list[tuple[int, str]], customer_id: int) -> str:
    24                                               """Hot spot 1: a linear scan for every lookup."""
    25   2493000        372.9      0.0     50.6      for candidate_id, city in customers:
    26   2493000        363.6      0.0     49.3          if candidate_id == customer_id:
    27      2000          0.3      0.0      0.0              return city
    28                                               raise KeyError(customer_id)
Total time: 0.0324066 s
File: <scratch>/profile_target.py
Function: render_lines at line 40
Line #      Hits         Time  Per Hit   % Time  Line Contents
    40                                           def render_lines(orders: list[tuple[int, int]]) -> str:
    41    

`Hits` is how many times the line ran, `% Time` its share of the function. In
`find_city` the loop header and the comparison take nearly everything: millions of
hits for 2 000 lookups, the signature of a linear scan. In `render_lines`
the `format` + `+=` line dominates.

#### scalene

[scalene](https://github.com/plasma-umass/scalene) is a sampling profiler that
reports, **per line**, time split into *Python* (interpreted bytecode), *native*
(C code such as NumPy or built-ins) and *system* (the kernel: I/O, sleeping), plus
memory allocation and copy volume. The split answers "would vectorizing help?":
time that is already native will not get much faster. `scalene run` saves a JSON
profile; `scalene view` opens it in a browser or the terminal. Memory tracking
slows scalene down a lot on this script (every `+=` reallocates a large string),
so this run is `--cpu-only`; 5.5.4 runs it with memory on the memory target. The
helper prints the lines with activity from the JSON file:

In [392]:
def show_scalene(script: str, *options: str) -> None:
    """Profile script with scalene, then print the totals and the lines with activity."""
    print(
        run_tool(
            tool_path("scalene") or "scalene", "run", *options, "-o", "scalene_profile.json", script
        ).splitlines()[0]
    )
    profile = json.loads((PHASE5_DIR / "scalene_profile.json").read_text())
    print(
        f"elapsed {profile['elapsed_time_sec']:.2f} s, "
        f"max footprint {profile['max_footprint_mb']:.1f} MB"
    )
    for file_name, file_profile in profile["files"].items():
        print(
            f"{'line':>4} {'Python':>7} {'native':>7} {'system':>7} {'alloc MB':>9}  "
            f"{Path(file_name).name}"
        )
        for line in file_profile["lines"]:
            line["line"] = (
                line["line"].replace("\\u003e", ">").replace("\\u003c", "<")
            )  # scalene escapes < and >
            shares = (line["n_cpu_percent_python"], line["n_cpu_percent_c"], line["n_sys_percent"])
            if sum(shares) + line["n_malloc_mb"] >= 0.5:
                print(
                    f"{line['lineno']:>4} "
                    + " ".join(f"{share:6.1f}%" for share in shares)
                    + f" {line['n_malloc_mb']:>9.1f}  {line['line'].strip()[:58]}"
                )


if HAS_SCALENE:
    show_scalene("profile_target.py", "--cpu-only")
else:
    print("scalene is not installed: skipping. Install with `uv add --dev scalene`.")

8000 orders, 5 cities, 4,480,000 chars, rate 83.25
elapsed 0.51 s, max footprint 0.0 MB
line  Python  native  system  alloc MB  profile_target.py
  25    9.4%    6.2%    0.8%       0.0  for candidate_id, city in customers:
  26    9.4%    6.2%    0.8%       0.0  if candidate_id == customer_id:
  44    1.9%   18.3%    0.2%       0.0  text += "{:>7} {:>8} {:>10.2f}\n".format(number, customer_
  50    0.0%    6.9%   40.0%       0.0  time.sleep(0.3)


> **Note:** scalene's numbers are statistical and its line attribution is coarser
> than `line_profiler`'s (it can split a loop's time between the loop header and its
> body). Use it for the Python/native/system split and the memory columns, and
> confirm surprises with a second tool.

#### Which profiler when

| Question | Tool |
| --- | --- |
| Which functions are slow, and how often are they called? | `cProfile` + `pstats` (+ snakeviz to browse) |
| Where does time go, without slowing the program down? | Tachyon (3.15), py-spy |
| Which **line** inside this function? | `line_profiler`; Tachyon's line rows |
| Python time, or already in C? Plus memory per line | scalene |
| What is a live production process doing right now? | `py-spy dump`, Tachyon `attach` (5.5.12) |
| Waiting or computing? | Tachyon `--mode wall` vs `--mode cpu`; `perf_counter` vs `process_time` |

### 5.5.4 Memory profiling

Two numbers describe memory over a run:

- **current**: what is allocated *now*;
- **peak**: the highest value *current* ever reached.

Peak is what decides whether a job fits on a machine (or is killed by the OOM
killer). A function can end with a small current value after a huge temporary
peak. Two tool families measure memory:

| Tool | Sees | Cost |
| --- | --- | --- |
| `tracemalloc` (stdlib) | allocations made through Python's allocators, with the source line that made them | slows allocation-heavy code a few times |
| memray | every allocation, including C/C++ (`malloc`, NumPy, extensions), as a recorded timeline | runs the program under a tracker; reports afterwards |
| RSS (`/proc`, `resource`, psutil) | what the OS has given the process | free to read, but includes the interpreter, libraries and freed-but-kept memory |

The target script builds a list of row dicts, then an index over them (freed before
the end), then a small summary:

In [393]:
%%writefile _scratch/phase5/memory_target.py
"""Memory profiling target for Unit 5.5: big temporaries, small result."""

import os

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))
ROW_COUNT = 100_000 * DEMO_SCALE
CITIES = ("Pune", "Delhi", "Kochi")


def load_rows(count: int) -> list[dict[str, object]]:
    """One dict per order, with a unique order-id string each."""
    return [{"order_id": f"ORD-{number:07d}", "city": CITIES[number % 3], "amount": number % 50_000} for number in range(count)]


def build_index(rows: list[dict[str, object]]) -> dict[object, dict[str, object]]:
    """A second big structure: order id -> row."""
    return {row["order_id"]: row for row in rows}


def summarize(rows: list[dict[str, object]]) -> dict[str, int]:
    """Revenue per city: a tiny result."""
    totals: dict[str, int] = {}
    for row in rows:
        totals[row["city"]] = totals.get(row["city"], 0) + row["amount"]
    return totals


def main() -> dict[str, int]:
    """Load, index (temporarily), summarize; only the summary survives."""
    rows = load_rows(ROW_COUNT)
    index = build_index(rows)
    print(f"indexed {len(index):,} orders")
    del index
    return summarize(rows)


if __name__ == "__main__":
    print(main())

Overwriting _scratch/phase5/memory_target.py


#### `tracemalloc`

`tracemalloc.start(frames)` records every allocation with a traceback of up to
`frames` frames. `get_traced_memory()` returns `(current, peak)`, and a
`take_snapshot()` groups the live allocations by line. The child script traces the
target's `main()`:

In [394]:
print(
    run_child(
        """
        import tracemalloc

        import memory_target

        tracemalloc.start(5)
        rows = memory_target.load_rows(memory_target.ROW_COUNT)
        current, peak = tracemalloc.get_traced_memory()
        print(f"after load_rows: current {current / 2**20:.1f} MiB, peak {peak / 2**20:.1f} MiB")
        for stat in tracemalloc.take_snapshot().statistics("lineno")[:3]:
            print(f"  {stat.size / 2**20:6.1f} MiB in {stat.count:>7,} blocks  {stat.traceback[0]}")
        index = memory_target.build_index(rows)
        del index
        totals = memory_target.summarize(rows)
        del rows
        current, peak = tracemalloc.get_traced_memory()
        print(f"at the end:      current {current / 2**20:.1f} MiB, peak {peak / 2**20:.1f} MiB")
        bookkeeping_mib = tracemalloc.get_tracemalloc_memory() / 2**20
        print(f"tracemalloc's own bookkeeping: {bookkeeping_mib:.1f} MiB")
        """
    )
)

after load_rows: current 26.2 MiB, peak 26.2 MiB
    26.2 MiB in 397,765 blocks  <scratch>/memory_target.py:12
     0.0 MiB in       3 blocks  <scratch>/child_script.py:7
at the end:      current 0.0 MiB, peak 31.7 MiB
tracemalloc's own bookkeeping: 0.0 MiB


Peak vs current is the whole story here: at the end almost nothing is left, but the
peak (rows + index alive together) is what the machine had to provide. Notice the
bookkeeping line: tracing costs memory too, so never leave it on in production.

#### Peak vs current, step by step

The same functions in the kernel. `tracemalloc.reset_peak()` (3.9+) starts a new
peak measurement without clearing the traces, so you can measure one phase at a
time:

In [395]:
import memory_target  # written by the %%writefile cell above

tracemalloc.start()
phase_rows = memory_target.load_rows(50_000 * DEMO_SCALE)
print(
    "after load_rows:   current {:5.1f} MiB  peak {:5.1f} MiB".format(
        *(byte_count / 2**20 for byte_count in tracemalloc.get_traced_memory())
    )
)
tracemalloc.reset_peak()
phase_totals = memory_target.summarize(phase_rows)
print(
    "summarize only:    current {:5.1f} MiB  peak {:5.1f} MiB".format(
        *(byte_count / 2**20 for byte_count in tracemalloc.get_traced_memory())
    )
)
del phase_rows
print(
    "after del rows:    current {:5.1f} MiB  peak {:5.1f} MiB".format(
        *(byte_count / 2**20 for byte_count in tracemalloc.get_traced_memory())
    )
)
tracemalloc.stop()
print(f"{phase_totals=}")

after load_rows:   current  13.2 MiB  peak  13.2 MiB
summarize only:    current  13.2 MiB  peak  13.2 MiB
after del rows:    current   0.0 MiB  peak  13.2 MiB
phase_totals={'Pune': 416658333, 'Delhi': 416675000, 'Kochi': 416641667}


`summarize` alone adds almost nothing to the peak: it streams over the rows. Deleting
the rows brings *current* back down, while the peak of that phase stays.

**Finding growth (leaks).** Take a snapshot, run the suspect code, take another, and
`compare_to` lists the lines whose memory grew. Here a module-level cache that is
never trimmed is the "leak":

In [396]:
response_cache: dict[int, bytes] = {}


def handle_request(request_id: int) -> int:
    """Pretend request handler that caches every response forever."""
    response_cache[request_id] = bytes(2_000)  # a new 2 kB object per request
    return len(response_cache[request_id])


tracemalloc.start()
before = tracemalloc.take_snapshot()
for request_id in range(2_000):
    handle_request(request_id)
after = tracemalloc.take_snapshot()
tracemalloc.stop()
for stat in after.compare_to(before, "lineno")[:2]:
    frame = stat.traceback[0]
    source = linecache.getline(frame.filename, frame.lineno).strip()
    print(
        f"+{stat.size_diff / 1024:7.1f} KiB  +{stat.count_diff:>5} blocks  "
        f"line {frame.lineno}: {source}"
    )

+ 4042.7 KiB  + 2001 blocks  line 6: response_cache[request_id] = bytes(2_000)  # a new 2 kB object per request
+   27.5 KiB  +  880 blocks  line 12: for request_id in range(2_000):


The top entry is the line that stores a new `bytes(2_000)` per request: about
4 MB in 2 000 new blocks (the dict's own growth comes next). In a real service you
would take the two snapshots minutes apart.

**RSS is a different number.** The operating system sees pages, not objects. RSS
includes the interpreter and loaded libraries, and memory that Python freed but
kept for reuse (small objects live in *arenas* that are only returned when
completely empty). Large blocks go straight to `mmap` and are returned at once:

In [397]:
def rss_mib() -> float | None:
    """Resident set size of this process in MiB (Linux /proc, psutil fallback)."""
    status_file = Path("/proc/self/status")
    if status_file.exists():
        for line in status_file.read_text().splitlines():
            if line.startswith("VmRSS:"):
                return int(line.split()[1]) / 1024
    if has_module("psutil"):
        import psutil

        return psutil.Process().memory_info().rss / 2**20
    return None


print(f"RSS now:                     {rss_mib():7.1f} MiB")
big_block = bytearray(200 * 2**20)  # one 200 MiB allocation (mmap)
print(f"with a 200 MiB bytearray:    {rss_mib():7.1f} MiB")
del big_block
print(f"after del (block returned):  {rss_mib():7.1f} MiB")

RSS now:                       265.9 MiB


with a 200 MiB bytearray:      465.9 MiB
after del (block returned):    265.9 MiB


The bytearray's pages are touched (zero-filled) as it is created, so RSS rises by
about 200 MiB and falls back when it is freed. A list of a million small objects
behaves differently: after `del` some of the memory may stay in the process. The
numbers vary by OS and allocator.

#### memray

[memray](https://bloomberg.github.io/memray/) (Bloomberg) records **every**
allocation, including those made by C code, into a capture file while the program
runs, then turns the file into reports: `stats` (totals and histograms), `summary`
(a table by function), `flamegraph` and `table` (HTML), and `tree`/`live`
(interactive terminal views, not used here). memray supports Linux and macOS.

In [398]:
if HAS_MEMRAY:
    memray = tool_path("memray") or "memray"
    print(
        run_tool(
            memray, "run", "--force", "-o", "memory_target.bin", "memory_target.py"
        ).splitlines()[0]
    )
    memray_stats = run_tool(memray, "stats", "memory_target.bin")
    print(first_lines(memray_stats[memray_stats.find("📏") :], 9))
else:
    print(
        "memray is not installed (Linux/macOS only): skipping. Install with `uv add --dev memray`."
    )

Writing profile results into memory_target.bin
📏 Total allocations:
	287

📦 Total memory allocated:
	45.121MB

📈 Peak memory usage:
	35.948MB



"Total memory allocated" adds up every allocation over the run (memory reused many
times counts many times); **peak memory usage** is the high-water mark. The `stats`
report also lists the locations that allocated the most:

In [399]:
if HAS_MEMRAY:
    top_start = memray_stats.find("🥇 Top 5 largest allocating locations (by size)")
    print(first_lines(memray_stats[top_start:], 6))
    print(run_tool(memray, "flamegraph", "--force", "-o", "memory_flame.html", "memory_target.bin"))
    print(
        f"HTML peak-memory flame graph: {(PHASE5_DIR / 'memory_flame.html').stat().st_size:,} bytes"
    )
else:
    print("memray is not installed: skipping.")

🥇 Top 5 largest allocating locations (by size):
	- load_rows:memory_target.py:12 -> 36.547MB
	- build_index:memory_target.py:17 -> 7.558MB
	- _get_code_from_file:<frozen runpy>:264 -> 273.920kB
	- _get_code_from_file:<frozen runpy>:263 -> 131.073kB
	- _get_code_from_file:<frozen runpy>:259 -> 131.073kB




Wrote memory_flame.html
HTML peak-memory flame graph: 122,681 bytes


`load_rows` (the dict and the f-string per row) and `build_index` lead, as
expected. The memray flame graph is a flame graph of the allocations alive **at
the peak**: the same reading rules as 5.5.3, with width meaning bytes instead of
samples.

scalene (5.5.3) also measures memory per line when it runs without `--cpu-only`:

In [400]:
if HAS_SCALENE:
    show_scalene("memory_target.py")
else:
    print("scalene is not installed: skipping.")

indexed 100,000 orders
elapsed 0.14 s, max footprint 32.2 MB
line  Python  native  system  alloc MB  memory_target.py
  12   22.4%   33.4%    6.1%      26.0  return [{"order_id": f"ORD-{number:07d}", "city": CITIES[n
  17   38.0%    0.0%    0.0%       6.2  return {row["order_id"]: row for row in rows}


The `alloc MB` column puts the row dicts (`load_rows`) first and the index second,
matching tracemalloc and memray.

> **Note:** for C extensions, `memray run --native` adds C stack frames, and
> `memray run --live` shows allocations as they happen. `pytest-memray` turns
> memory limits into test failures.

### 5.5.5 Algorithmic optimization

The largest wins come from doing **less work**, not from doing the same work
faster. A better algorithm turns 10 s into 10 ms; a micro-optimization turns 10 s
into 8 s. This section measures with a small helper that returns the best time
per call:

In [401]:
def best_seconds(work: Callable[[], object], number: int = 5, repeat: int = 3) -> float:
    """Best (minimum) seconds per call of work over `repeat` rounds of `number` calls."""
    return min(timeit.repeat(work, number=number, repeat=repeat)) / number

#### Right data structures

| Operation | `list` | `dict` / `set` | `collections.deque` |
| --- | --- | --- | --- |
| `x in container` | O(n) scan | **O(1)** hash lookup | O(n) |
| lookup by key | O(n) scan (as `find_city` did) | **O(1)** | — |
| append / pop at the end | O(1) amortized | — | O(1) |
| insert / pop at the **front** | **O(n)**: shifts every element | — | **O(1)** |
| index `container[i]` | O(1) | — | O(n) in the middle |

The `find_city` fix from 5.5.3: build a dict once (O(n)), then each lookup is O(1):

In [402]:
import profile_target  # the 5.5.3 script, importable from the scratch folder

customers = profile_target.make_customers(2_500)
orders = profile_target.make_orders(8_000 * DEMO_SCALE, customers)


def city_totals_dict(
    orders: list[tuple[int, int]], customers: list[tuple[int, str]]
) -> dict[str, int]:
    """Same result as profile_target.city_totals, with one dict instead of a scan per order."""
    city_of = dict(customers)  # customer_id -> city, built once
    totals: dict[str, int] = {}
    for customer_id, amount in orders:
        city = city_of[customer_id]
        totals[city] = totals.get(city, 0) + amount
    return totals


scan_seconds = best_seconds(lambda: profile_target.city_totals(orders, customers), number=1)
dict_seconds = best_seconds(lambda: city_totals_dict(orders, customers))
print(
    "same result:",
    city_totals_dict(orders, customers) == profile_target.city_totals(orders, customers),
)
print(
    f"linear scan {scan_seconds * 1000:.1f} ms, dict {dict_seconds * 1000:.2f} ms "
    f"-> {scan_seconds / dict_seconds:.0f}x"
)

same result: True
linear scan 115.2 ms, dict 0.46 ms -> 253x


Membership tests show the same gap, and `list.pop(0)` vs `deque.popleft()` shows the cost of shifting:

In [403]:
blocked_ids = list(range(0, 200_000, 3))
blocked_set = set(blocked_ids)
queries = range(1_000)
list_seconds = best_seconds(lambda: sum(query in blocked_ids for query in queries), number=1)
set_seconds = best_seconds(lambda: sum(query in blocked_set for query in queries))
print(f"1 000 membership tests: list {list_seconds * 1000:.1f} ms, set {set_seconds * 1000:.3f} ms")


def drain_list(count: int) -> None:
    """Empty a list from the front (each pop(0) shifts the rest)."""
    items = list(range(count))
    while items:
        items.pop(0)  # shifts every remaining element


def drain_deque(count: int) -> None:
    """Empty a deque from the front (each popleft is O(1))."""
    items = deque(range(count))
    while items:
        items.popleft()


print(
    "drain 50 000 from the front: "
    f"list {best_seconds(lambda: drain_list(50_000), number=1) * 1000:.1f} ms, "
    f"deque {best_seconds(lambda: drain_deque(50_000), number=1) * 1000:.1f} ms"
)

1 000 membership tests: list 247.7 ms, set 0.020 ms


drain 50 000 from the front: list 153.9 ms, deque 1.3 ms


#### Removing quadratic loops

A loop inside a loop over the same data is O(n²): double the input, quadruple the
time. The classic example is "which order ids appear twice?":

In [404]:
def duplicates_quadratic(order_ids: list[int]) -> set[int]:
    """Compare every pair: O(n^2)."""
    found = set()
    for position, order_id in enumerate(order_ids):
        if order_id in order_ids[position + 1 :]:  # a scan inside the loop
            found.add(order_id)
    return found


def duplicates_linear(order_ids: list[int]) -> set[int]:
    """Remember what was seen: O(n)."""
    seen: set[int] = set()
    found = set()
    for order_id in order_ids:
        if order_id in seen:
            found.add(order_id)
        seen.add(order_id)
    return found

Time both at n and 2n. The growth factor tells the complexity class, whatever the machine:

In [405]:
id_rng = random.Random(42)
for label, finder in (("quadratic", duplicates_quadratic), ("linear", duplicates_linear)):
    timings = []
    for size in (2_000, 4_000):
        order_ids = [id_rng.randrange(size * 10) for _ in range(size)]
        timings.append(best_seconds(lambda find=finder, ids=order_ids: find(ids), number=1))
    print(
        f"{label:<9} n=2000: {timings[0] * 1000:7.2f} ms  n=4000: {timings[1] * 1000:7.2f} ms  "
        f"growth x{timings[1] / timings[0]:.1f}"
    )
sample_ids = [5, 3, 5, 9, 3, 5]
print(f"same answer: {duplicates_quadratic(sample_ids) == duplicates_linear(sample_ids) == {3, 5}}")

quadratic n=2000:   12.04 ms  n=4000:   46.75 ms  growth x3.9
linear    n=2000:    0.08 ms  n=4000:    0.14 ms  growth x1.8
same answer: True


Doubling n roughly quadruples the quadratic version and doubles the linear one.
Hidden quadratic loops are common: `x in some_list` inside a loop, `list.remove`
or `list.insert(0, …)` in a loop, `str +=` in some cases, or repeated `sorted()`
inside a loop.

#### `heapq`

`heapq` keeps a list arranged as a binary min-heap: `heappush`/`heappop` cost
O(log n), and the smallest item is always at index 0. Use it for "the k largest",
priority queues, and merging sorted streams.

In [406]:
sale_rng = random.Random(7)
sale_amounts = [sale_rng.randrange(100, 1_000_000) for _ in range(500_000 * DEMO_SCALE)]
top_by_sort = sorted(sale_amounts, reverse=True)[:5]
top_by_heap = heapq.nlargest(5, sale_amounts)
print(f"{top_by_heap=}  same as sorting: {top_by_heap == top_by_sort}")
sort_seconds = best_seconds(lambda: sorted(sale_amounts, reverse=True)[:5], number=1)
heap_seconds = best_seconds(lambda: heapq.nlargest(5, sale_amounts), number=1)
print(
    f"top 5 of {len(sale_amounts):,}: full sort {sort_seconds * 1000:.0f} ms, "
    f"nlargest {heap_seconds * 1000:.0f} ms"
)

top_by_heap=[999993, 999992, 999992, 999991, 999987]  same as sorting: True


top 5 of 500,000: full sort 78 ms, nlargest 5 ms


`nlargest(k, …)` keeps only a k-sized heap: O(n log k) time and O(k) memory,
against O(n log n) and O(n) for a full sort. (For k close to n, sorting wins, and
`heapq` says so in its docs.) As a priority queue, push `(priority, sequence,
item)` tuples; the sequence number breaks ties so items themselves are never
compared:

In [407]:
delivery_queue: list[tuple[int, int, str]] = []
for sequence, (priority, order_id) in enumerate(
    [(3, "ORD-7"), (1, "ORD-2"), (2, "ORD-9"), (1, "ORD-4")]
):
    heapq.heappush(delivery_queue, (priority, sequence, order_id))
print("dispatch order:", [heapq.heappop(delivery_queue)[2] for _ in range(len(delivery_queue))])
morning, evening = [3, 8, 15], [1, 9, 12, 20]  # already sorted
print("merged lazily:", list(heapq.merge(morning, evening)))

dispatch order: ['ORD-2', 'ORD-4', 'ORD-9', 'ORD-7']
merged lazily: [1, 3, 8, 9, 12, 15, 20]


#### `bisect`

`bisect` searches a **sorted** list by halving: O(log n) per lookup.
`bisect_left(a, x)` returns the first position where `x` could be inserted (before
equal items), `bisect_right` the last; `insort` inserts while keeping the order.
A typical use is bucket lookup, such as finding the tax slab for an income:

In [408]:
slab_starts = [0, 400_000, 800_000, 1_200_000, 1_600_000, 2_000_000, 2_400_000]  # rupees
slab_rates = [0, 5, 10, 15, 20, 25, 30]  # percent


def marginal_rate(income: int) -> int:
    """Tax rate of the slab containing income (bisect_right: a slab start belongs to that slab)."""
    return slab_rates[bisect.bisect_right(slab_starts, income) - 1]


print([(income, marginal_rate(income)) for income in (0, 399_999, 400_000, 1_250_000, 9_000_000)])
prices = [10, 20, 20, 30]
print(
    f"bisect_left={bisect.bisect_left(prices, 20)}  bisect_right={bisect.bisect_right(prices, 20)}"
)
bisect.insort(prices, 25)
print(f"after insort(25): {prices}")

[(0, 0), (399999, 0), (400000, 5), (1250000, 15), (9000000, 30)]
bisect_left=1  bisect_right=3
after insort(25): [10, 20, 20, 25, 30]


`key=` (3.10+) searches by a derived value without building a separate list of
keys; the list must be sorted by that same key. Against a linear scan the
difference grows with the list:

In [409]:
price_list = sorted(
    (sale_rng.randrange(1_000_000), f"SKU-{sku_number}") for sku_number in range(200_000)
)
target_price = price_list[150_000][0]
by_bisect = price_list[bisect.bisect_left(price_list, target_price, key=lambda item: item[0])]
by_scan = next(item for item in price_list if item[0] >= target_price)
print(f"same item: {by_bisect == by_scan}")
scan_seconds = best_seconds(lambda: next(item for item in price_list if item[0] >= target_price))
bisect_seconds = best_seconds(
    lambda: bisect.bisect_left(price_list, target_price, key=lambda item: item[0]), number=1000
)
print(
    f"scan {scan_seconds * 1e6:,.0f} µs, bisect {bisect_seconds * 1e6:.2f} µs "
    f"-> {scan_seconds / bisect_seconds:,.0f}x"
)

same item: True


scan 7,540 µs, bisect 0.41 µs -> 18,574x


#### Memoization

When a function is called again and again with the same arguments, remember the
answers. Recursive problems with overlapping subproblems go from exponential to
linear. Counting the calls makes the difference visible without any timing:

In [410]:
call_counter = Counter()


def ways_to_pay(amount: int, coins: tuple[int, ...]) -> int:
    """Number of ways to pay `amount` with the coins (order matters): exponential."""
    call_counter["plain"] += 1
    if amount == 0:
        return 1
    return sum(ways_to_pay(amount - coin, coins) for coin in coins if coin <= amount)


@functools.cache
def ways_to_pay_cached(amount: int, coins: tuple[int, ...]) -> int:
    """The same recursion; each (amount, coins) pair is computed once."""
    call_counter["cached"] += 1
    if amount == 0:
        return 1
    return sum(ways_to_pay_cached(amount - coin, coins) for coin in coins if coin <= amount)


coins = (1, 2, 5)
print(ways_to_pay(22, coins), ways_to_pay_cached(22, coins), dict(call_counter))
print(ways_to_pay_cached.cache_info())

77164 77164 {'plain': 186631, 'cached': 23}
CacheInfo(hits=39, misses=23, maxsize=None, currsize=23)


Same answer, hundreds of thousands of calls against 23. `functools.cache` is an
unbounded `lru_cache`; 5.5.8 covers bounded caches, hit rates and invalidation.
Memoization trades memory for time and needs hashable arguments (hence the
`tuple` of coins, not a list).

### 5.5.6 Interpreter-level optimization

After the algorithm is right, the remaining cost is the interpreter's work per
operation: each bytecode instruction, attribute lookup, function call and
temporary object costs tens of nanoseconds. These habits reduce that work. Measure
each one; since 3.11 the specializing interpreter has made several old tricks
(local aliases especially) worth much less than they used to be.

The helper prints nanoseconds per call for a few variants of the same operation:

In [411]:
def compare_variants(
    title: str, variants: dict[str, Callable[[], object]], number: int = 200
) -> None:
    """Print best time per call of each variant (in input order) and its ratio to the first."""
    print(title)
    baseline = None
    for label, work in variants.items():
        seconds = best_seconds(work, number=number, repeat=5)
        baseline = baseline or seconds
        print(f"  {label:<34} {seconds * 1e6:9.2f} µs   x{seconds / baseline:.2f}")

#### Local aliases

A global name costs a dictionary lookup (cached by `LOAD_GLOBAL` specialization
since 3.11), and `module.attribute` costs one more lookup per use. Binding the
function to a local name once makes each use a `LOAD_FAST`, an array index:

In [412]:
def roots_global(values: list[float]) -> list[float]:
    """math.sqrt looked up on every iteration."""
    result = []
    for value in values:
        result.append(math.sqrt(value))
    return result


def roots_local(values: list[float]) -> list[float]:
    """sqrt and append bound to locals once."""
    result = []
    append, sqrt = result.append, math.sqrt
    for value in values:
        append(sqrt(value))
    return result


float_values = [float(number) for number in range(10_000)]
compare_variants(
    "10 000 square roots",
    {
        "math.sqrt + result.append": lambda: roots_global(float_values),
        "local aliases": lambda: roots_local(float_values),
    },
)

10 000 square roots


  math.sqrt + result.append             296.10 µs   x1.00


  local aliases                         319.16 µs   x1.08


Under the hood, `dis` of the loop body shows the difference: the first loop does a global load and two attribute loads per iteration, the second only fast local loads. Measured above, the gain is small on 3.15, because those loads are specialized and cached:

In [413]:
import dis

for function in (roots_global, roots_local):
    source_lines, first_line = inspect.getsourcelines(function)
    body_line = first_line + next(
        line_number
        for line_number, line in enumerate(source_lines)
        if "append(" in line and "=" not in line
    )
    body_ops = [
        instruction.opname
        for instruction in dis.get_instructions(function)
        if instruction.positions and instruction.positions.lineno == body_line
    ]
    print(f"{function.__name__:<13} loop body: {' '.join(body_ops)}")

roots_global  loop body: LOAD_FAST_BORROW LOAD_ATTR LOAD_GLOBAL LOAD_ATTR LOAD_FAST_BORROW CALL CALL POP_TOP JUMP_BACKWARD
roots_local   loop body: LOAD_FAST_BORROW PUSH_NULL LOAD_FAST_BORROW PUSH_NULL LOAD_FAST_BORROW CALL CALL POP_TOP JUMP_BACKWARD


#### Built-ins written in C

`sum`, `min`, `max`, `sorted`, `any`, `map`, `str.join` and friends run their loop
in C, without dispatching one bytecode per element. Moving a loop *into* a built-in
usually beats any amount of tuning of the Python loop:

In [414]:
def manual_sum(values: list[int]) -> int:
    """Sum with a Python loop."""
    total = 0
    for value in values:
        total += value
    return total


int_values = list(range(100_000))
compare_variants(
    "sum of 100 000 ints",
    {"Python for loop": lambda: manual_sum(int_values), "built-in sum()": lambda: sum(int_values)},
    number=20,
)
compare_variants(
    "max of 100 000 ints",
    {
        "max with a key lambda": lambda: max(int_values, key=lambda value: value),
        "plain max()": lambda: max(int_values),
    },
    number=20,
)

sum of 100 000 ints
  Python for loop                      1678.91 µs   x1.00


  built-in sum()                        372.17 µs   x0.22
max of 100 000 ints


  max with a key lambda                2312.47 µs   x1.00


  plain max()                           850.59 µs   x0.37


The `key=lambda` version is slower because it calls a Python function per element:
the loop is in C, the work per element is not. Prefer `operator.itemgetter` /
`attrgetter` over a lambda when a key is needed.

#### `str.join` instead of `+=`

Strings are immutable, so `text += piece` creates a new string. CPython has an
in-place shortcut when the string has no other reference, which often hides the
cost, but the shortcut breaks as soon as another reference exists. `"".join(parts)`
computes the final size once and copies each piece once:

In [415]:
pieces = [f"line {line_number}\n" for line_number in range(20_000)]


def concat_plus(parts: list[str]) -> str:
    """Join with += on a string that has no other reference."""
    text = ""
    for part in parts:
        text += part  # in-place resize when `text` has no other reference
    return text


def concat_plus_shared(parts: list[str]) -> str:
    """Join with += while keeping old strings alive (full copies)."""
    text, history = "", []
    for part in parts[:5_000]:
        history.append(text)  # a second reference defeats the shortcut: full copy each time
        text += part
    return text


compare_variants(
    "joining text",
    {
        "+= (shortcut applies)": lambda: concat_plus(pieces),
        "+= with a second reference (1/4 of the pieces)": lambda: concat_plus_shared(pieces),
        '"".join(parts)': lambda: "".join(pieces),
    },
    number=10,
)

joining text
  += (shortcut applies)                 562.78 µs   x1.00


  += with a second reference (1/4 of the pieces)  35078.02 µs   x62.33
  "".join(parts)                         57.13 µs   x0.10


The shortcut is a CPython implementation detail (PyPy and other implementations do
not have it), and it does not apply to `bytes`. `join` is the portable, always
linear choice; for bytes use `b"".join` or `io.BytesIO`.

#### Comprehensions

A list comprehension avoids the `result.append` lookup and call per element. Since
3.12 (PEP 709) comprehensions are also **inlined**: no hidden function object is
created and called each time the comprehension runs:

In [416]:
def squares_loop(count: int) -> list[int]:
    """Build the squares with append in a loop."""
    result = []
    for number in range(count):
        result.append(number * number)
    return result


def squares_comprehension(count: int) -> list[int]:
    """Build the squares with a list comprehension."""
    return [number * number for number in range(count)]


compare_variants(
    "10 000 squares",
    {
        "for + append": lambda: squares_loop(10_000),
        "list comprehension": lambda: squares_comprehension(10_000),
    },
)
makes_function = any(
    instruction.opname == "MAKE_FUNCTION"
    for instruction in dis.get_instructions(squares_comprehension)
)
print("comprehension compiled inline (no hidden function object):", not makes_function)

10 000 squares


  for + append                          180.97 µs   x1.00


  list comprehension                    159.90 µs   x0.88
comprehension compiled inline (no hidden function object): True


#### Avoiding exceptions in hot paths

Since 3.11 a `try` block costs **nothing** when no exception is raised ("zero-cost"
exception handling). *Raising* and catching is still expensive: an exception
object, a traceback, unwinding. So `try/except` is the fastest choice when misses
are rare and the slowest when they are common:

In [417]:
stock = {f"SKU-{sku_number}": sku_number for sku_number in range(1_000)}


def count_with_try(skus: list[str]) -> int:
    """Sum stock with try/except for missing SKUs."""
    total = 0
    for sku in skus:
        try:
            total += stock[sku]
        except KeyError:
            pass
    return total


def count_with_get(skus: list[str]) -> int:
    """Sum stock with dict.get for missing SKUs."""
    total = 0
    for sku in skus:
        total += stock.get(sku, 0)
    return total

Two workloads: every key present, and half the keys missing:

In [418]:
all_hits = [f"SKU-{lookup_number % 1_000}" for lookup_number in range(10_000)]
half_misses = [f"SKU-{lookup_number % 2_000}" for lookup_number in range(10_000)]
compare_variants(
    "all keys present",
    {
        "try/except KeyError": lambda: count_with_try(all_hits),
        "dict.get": lambda: count_with_get(all_hits),
    },
)
compare_variants(
    "half the keys missing",
    {
        "try/except KeyError": lambda: count_with_try(half_misses),
        "dict.get": lambda: count_with_get(half_misses),
    },
)

all keys present


  try/except KeyError                   243.80 µs   x1.00


  dict.get                              274.61 µs   x1.13
half the keys missing


  try/except KeyError                   471.77 µs   x1.00


  dict.get                              262.76 µs   x0.56


#### Type-stable code

The specializing adaptive interpreter (PEP 659, 3.11+) watches the types each
instruction sees and rewrites it into a specialized form: `BINARY_OP` becomes
`BINARY_OP_ADD_INT` when both operands keep being ints. When the types keep
changing, the specialization fails and the instruction falls back to the generic,
slower path. V8 does the same with hidden classes and inline caches; mixing types
in a hot loop hurts both. `dis(..., adaptive=True)` shows the rewritten bytecode:

In [419]:
def running_total(values: list) -> float:
    """Add up values; the + instruction specializes on the types it sees."""
    total = 0
    for value in values:
        total = total + value
    return total


def specialized_adds(values: list) -> list[str]:
    """Warm the function up on `values`, then list the specialized forms of its `+`."""
    code_copy = running_total.__code__.replace()  # a fresh code object: no earlier specialization
    fresh = type(running_total)(code_copy, globals())
    for _ in range(50):
        fresh(values)
    return sorted(
        {
            instruction.opname
            for instruction in dis.get_instructions(fresh, adaptive=True)
            if instruction.opname.startswith("BINARY_OP")
        }
    )


print("ints only:   ", specialized_adds(list(range(1_000))))
print("floats only: ", specialized_adds([number / 2 for number in range(1_000)]))

ints only:    ['BINARY_OP_ADD_INT']
floats only:  ['BINARY_OP_ADD_FLOAT']


The same `+` became `BINARY_OP_ADD_INT` or `BINARY_OP_ADD_FLOAT` depending on the
data it ran on (the exact opnames are a CPython detail and change between
versions). Each specialized instruction carries a cheap guard; when the guard keeps
failing because the types keep changing, the instruction is de-specialized and
re-specialized, and each pass does more work.

Attribute access shows the cost most clearly. `item.price` on objects of **one**
class hits the same inline cache entry every time; objects of four different
classes (four "shapes", in V8 terms) keep missing it, exactly like a polymorphic
call site in JavaScript:

In [420]:
def make_item_classes(count: int) -> list[type]:
    """`count` distinct classes whose instances all have a `price` attribute."""

    def __init__(self: object, price: int) -> None:
        """Store the price."""
        self.price = price

    return [type(f"Item{number}", (), {"__init__": __init__}) for number in range(count)]


def total_price(items: list) -> int:
    """Sum item.price over the items."""
    total = 0
    for item in items:
        total += item.price
    return total


one_class, four_classes = make_item_classes(1), make_item_classes(4)
same_shape = [one_class[0](index) for index in range(100_000)]
mixed_shapes = [four_classes[index % 4](index) for index in range(100_000)]
compare_variants(
    "total_price over 100 000 objects",
    {
        "one class (monomorphic)": lambda: total_price(same_shape),
        "four classes (polymorphic)": lambda: total_price(mixed_shapes),
    },
    number=20,
)

total_price over 100 000 objects


  one class (monomorphic)              1688.94 µs   x1.00


  four classes (polymorphic)           2351.57 µs   x1.39


Same values, same code, more time per item for the mixed list. Consistent types
are what *every* optimizing layer relies on: the specializer, the JIT (5.5.11),
Cython, NumPy dtypes, V8.

### 5.5.7 Memory-lean data

Every Python value is an object with a header (reference count and type pointer,
16 bytes on 64-bit builds) and every container holds 8-byte pointers to objects.
A million small records therefore cost far more than their raw data. The helper
measures how much memory a structure **keeps** after it is built (traced bytes
still allocated while the result is alive):

In [421]:
def retained_bytes(build: Callable[[], object]) -> tuple[object, int]:
    """Build an object under tracemalloc; return it and the bytes it still holds."""
    gc.collect()
    tracemalloc.start()
    before = tracemalloc.get_traced_memory()[0]
    result = build()
    kept = tracemalloc.get_traced_memory()[0] - before
    tracemalloc.stop()
    return result, kept

#### `__slots__`

A regular instance stores attributes in a per-instance dict (shared keys and inline
values since 3.11–3.13 make that cheaper than it was). `__slots__` replaces it with
fixed fields in the object itself: smaller, and no new attributes can be added.
`@dataclass(slots=True)` (3.10+) generates the slots for you; a `tuple` subclass
such as `NamedTuple` is smaller still:

In [422]:
class PlainReading:
    """A reading with a per-instance __dict__."""

    def __init__(self, sensor_id: int, value: float) -> None:
        """Store the sensor id and value."""
        self.sensor_id, self.value = sensor_id, value


class SlottedReading:
    """A reading with __slots__ (no per-instance __dict__)."""

    __slots__ = ("sensor_id", "value")

    def __init__(self, sensor_id: int, value: float) -> None:
        """Store the sensor id and value."""
        self.sensor_id, self.value = sensor_id, value


@dataclass(slots=True)
class DataclassReading:
    """A reading as a slotted dataclass."""

    sensor_id: int
    value: float

100 000 instances of each, plus the bare tuple:

In [423]:
reading_count = 100_000 * DEMO_SCALE
for label, record_type in (
    ("plain class", PlainReading),
    ("__slots__ class", SlottedReading),
    ("dataclass(slots=True)", DataclassReading),
    ("tuple", lambda *fields: fields),
):
    _, kept = retained_bytes(
        lambda make=record_type: [
            make(reading_number, reading_number * 0.5) for reading_number in range(reading_count)
        ]
    )
    print(
        f"{label:<22} {kept / reading_count:6.1f} bytes per record "
        "(list pointer and float included)"
    )
print(
    "slots forbid new attributes:",
    raised_error_name(lambda: setattr(SlottedReading(1, 2.0), "unit", "C")),
)

plain class             151.7 bytes per record (list pointer and float included)


__slots__ class         111.7 bytes per record (list pointer and float included)


dataclass(slots=True)   111.9 bytes per record (list pointer and float included)


tuple                   127.7 bytes per record (list pointer and float included)
slots forbid new attributes: AttributeError


#### `array`

`array.array(typecode)` stores raw machine numbers in one contiguous buffer, like a
JavaScript typed array: 8 bytes per `'q'` (signed 64-bit) item instead of an
8-byte pointer plus a 28-byte (or larger) `int` object. Small ints (−5…1024 in
3.15) are shared singletons, so the list side is measured with larger values:

In [424]:
int_count = 200_000 * DEMO_SCALE
int_list, list_bytes = retained_bytes(lambda: [number * 1_000 for number in range(int_count)])
int_array, array_bytes = retained_bytes(
    lambda: array.array("q", (number * 1_000 for number in range(int_count)))
)
print(f"list of ints : {list_bytes / int_count:5.1f} bytes per item")
print(
    f"array('q')   : {array_bytes / int_count:5.1f} bytes per item, itemsize={int_array.itemsize}"
)
print(
    f"same values: {list(int_array[:3]) == int_list[:3]}, "
    f"raw buffer {len(int_array.tobytes()):,} bytes"
)

list of ints :  40.1 bytes per item
array('q')   :   8.5 bytes per item, itemsize=8
same values: True, raw buffer 1,600,000 bytes


#### `memoryview`

Slicing `bytes`, `bytearray` or an `array` **copies** the data. A `memoryview`
slice is a window onto the same buffer (the buffer protocol, PEP 3118): no copy,
and writes through a view of a mutable buffer change the original. `.cast()`
reinterprets the bytes with another format:

In [425]:
payload = bytes(50 * 2**20)  # a 50 MiB "file" already in memory
_, slice_bytes = retained_bytes(lambda: payload[10 : 40 * 2**20])
_, view_bytes = retained_bytes(lambda: memoryview(payload)[10 : 40 * 2**20])
print(
    f"bytes slice copies {slice_bytes / 2**20:.1f} MiB; memoryview slice keeps {view_bytes} bytes"
)

frame = bytearray(b"HDR:temperature=21")
body = memoryview(frame)[4:]  # skip the header, no copy
body[:4] = b"TEMP"
print(frame, "| as unsigned 32-bit ints:", memoryview(bytearray(range(8))).cast("I").tolist())
payload = b""  # release the 50 MiB buffer

bytes slice copies 40.0 MiB; memoryview slice keeps 488 bytes
bytearray(b'HDR:TEMPerature=21') | as unsigned 32-bit ints: [50462976, 117835012]


#### Generators

A list comprehension materializes every element; a generator expression produces
one element at a time. When the result is consumed once (summed, written, filtered
into something small), the generator's peak memory stays constant:

In [426]:
def peak_during(work: Callable[[], object]) -> int:
    """Peak traced bytes while running work (current baseline excluded)."""
    tracemalloc.start()
    work()
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak


line_count = 300_000 * DEMO_SCALE
list_peak = peak_during(
    lambda: sum(
        [
            len(f"order {order_number},Pune,{order_number % 997}")
            for order_number in range(line_count)
        ]
    )
)
generator_peak = peak_during(
    lambda: sum(
        len(f"order {order_number},Pune,{order_number % 997}") for order_number in range(line_count)
    )
)
print(f"sum over a list comprehension: peak {list_peak / 2**20:7.2f} MiB")
print(f"sum over a generator:          peak {generator_peak / 2**20:7.2f} MiB")
generator_bytes = sys.getsizeof(order_number for order_number in range(line_count))
print(f"generator object itself: {generator_bytes} bytes")

sum over a list comprehension: peak    2.48 MiB
sum over a generator:          peak    0.00 MiB
generator object itself: 216 bytes


#### Interning

Strings parsed from input are new objects even when their text repeats: a million
rows with five city names hold a million city strings. `sys.intern` returns one
canonical object per distinct text, so repeats share it (and `is` comparisons
become possible). Identifiers and many literals are interned automatically;
runtime strings are not:

In [427]:
csv_lines = [
    f"{row_number},{('Pune', 'Delhi', 'Kochi', 'Patna', 'Surat')[row_number % 5]},"
    f"{row_number % 997}"
    for row_number in range(200_000 * DEMO_SCALE)
]
plain_cities, plain_bytes = retained_bytes(lambda: [line.split(",")[1] for line in csv_lines])
interned_cities, interned_bytes = retained_bytes(
    lambda: [sys.intern(line.split(",")[1]) for line in csv_lines]
)
print(
    f"plain split  : {plain_bytes / 2**20:5.1f} MiB, "
    f"distinct objects {len({id(city) for city in plain_cities}):,}"
)
print(
    f"sys.intern   : {interned_bytes / 2**20:5.1f} MiB, "
    f"distinct objects {len({id(city) for city in interned_cities}):,}"
)
del plain_cities, interned_cities

plain split  :  10.3 MiB, distinct objects 200,000
sys.intern   :   1.5 MiB, distinct objects 5


The interned list holds five string objects and 200 000 pointers to them. pandas'
`category` dtype and Polars' `Categorical` apply the same idea to columns.

### 5.5.8 Caching

A cache is worth having when the **hit rate** is high, the computation is expensive
compared with a lookup, and stale answers are either impossible or acceptable for a
known time. Every cache needs an answer to "when is an entry wrong?" (invalidation)
and "how big may it grow?" (eviction).

#### `lru_cache` hit rates

`functools.lru_cache(maxsize=N)` keeps the N most recently used results;
`cache_info()` reports hits, misses and the current size. The hit rate depends on
how the access pattern compares with `maxsize`. Here a few products are requested
far more often than the rest (a typical skewed pattern):

In [428]:
def product_requests(count: int, product_count: int = 500, seed: int = 42) -> list[int]:
    """Skewed product ids: low ids are requested much more often (roughly Zipf-like)."""
    request_rng = random.Random(seed)
    weights = [1 / (rank + 1) for rank in range(product_count)]
    return request_rng.choices(range(product_count), weights=weights, k=count)


def hit_rate(maxsize: int | None, requests: list[int]) -> tuple[float, object]:
    """Replay requests through a fresh lru_cache of the given size."""

    @functools.lru_cache(maxsize=maxsize)
    def load_product(product_id: int) -> str:
        """Pretend to load a product from the database."""
        return f"product {product_id}"

    for product_id in requests:
        load_product(product_id)
    info = load_product.cache_info()
    return info.hits / (info.hits + info.misses), info

The same 20 000 requests through caches of growing size:

In [429]:
requests = product_requests(20_000)
for maxsize in (8, 32, 128, 512, None):
    rate, info = hit_rate(maxsize, requests)
    print(f"maxsize={maxsize!s:<5} hit rate {rate:6.1%}   {info}")

maxsize=8     hit rate  21.3%   CacheInfo(hits=4252, misses=15748, maxsize=8, currsize=8)
maxsize=32    hit rate  44.8%   CacheInfo(hits=8958, misses=11042, maxsize=32, currsize=32)
maxsize=128   hit rate  71.1%   CacheInfo(hits=14217, misses=5783, maxsize=128, currsize=128)
maxsize=512   hit rate  97.5%   CacheInfo(hits=19500, misses=500, maxsize=512, currsize=500)
maxsize=None  hit rate  97.5%   CacheInfo(hits=19500, misses=500, maxsize=None, currsize=500)


A small cache already catches the hot items; beyond the number of distinct keys
(500 here) a bigger cache cannot help. Measure the hit rate on real traffic before
choosing `maxsize`, and remember that `maxsize=None` never evicts: it is a memory
leak if the key space is unbounded.

#### TTL caches

When the underlying data changes (exchange rates, a remote config), entries must
**expire**. `cachetools.TTLCache(maxsize, ttl, timer=...)` (installed in Phase 4)
evicts entries older than `ttl` seconds. A fake clock makes the demo
deterministic:

In [430]:
if has_module("cachetools"):
    import cachetools

    fake_now = [0.0]
    rate_cache = cachetools.TTLCache(maxsize=100, ttl=60, timer=lambda: fake_now[0])
    upstream_calls = []

    @cachetools.cached(rate_cache)
    def exchange_rate(currency: str) -> float:
        """Record the upstream call and return the rate."""
        upstream_calls.append((fake_now[0], currency))
        return {"USD": 83.25, "EUR": 90.10}[currency]

    for moment in (0, 10, 59, 61, 70):
        fake_now[0] = moment
        exchange_rate("USD")
    print(
        f"5 lookups, {len(upstream_calls)} upstream calls "
        f"at t={[when for when, _ in upstream_calls]}"
    )
else:
    print("cachetools is not installed: skipping.")

5 lookups, 2 upstream calls at t=[0, 61]


The entry fetched at t=0 serves t=10 and t=59, expires at t=60, and is fetched again
at t=61. A TTL bounds staleness; it does not prevent it.

#### Invalidation

"There are only two hard things in computer science: cache invalidation and naming
things." The main strategies:

| Strategy | How | Example |
| --- | --- | --- |
| Clear everything | `cached_function.cache_clear()` | after a bulk import |
| Delete on write | the code that changes the data removes the key | `del cache[key]` in `update_price` |
| Versioned keys | put a version number in the key; bump it on change | `(product_id, catalog_version)` |
| Expire | TTL | rates, remote config |

Delete-on-write with an explicit dictionary cache:

In [431]:
price_store = {"SKU-1": 49_900, "SKU-2": 12_500}  # the "database"
price_cache: dict[str, str] = {}


def formatted_price(sku: str) -> str:
    """Read-through cache: compute on a miss, then remember."""
    if sku not in price_cache:
        price_cache[sku] = f"₹{price_store[sku] / 100:,.2f}"
    return price_cache[sku]


def update_price(sku: str, paise: int) -> None:
    """Write path: change the data AND invalidate the cached entry."""
    price_store[sku] = paise
    price_cache.pop(sku, None)


print(formatted_price("SKU-1"))
update_price("SKU-1", 44_900)
print(formatted_price("SKU-1"), "(fresh after invalidation)")

₹499.00
₹449.00 (fresh after invalidation)


Versioned keys avoid finding every key to delete: bump one counter and old entries
simply stop being asked for (eviction cleans them up later). `lru_cache` can only be
cleared as a whole, and `functools.cached_property` is invalidated per object with
`del`:

In [432]:
catalog_version = 1


@functools.lru_cache(maxsize=256)
def catalog_page(page: int, version: int) -> str:
    """The version argument is part of the cache key."""
    return f"page {page} rendered for catalog v{version}"


print(catalog_page(1, catalog_version), catalog_page.cache_info().misses, "miss")
catalog_version += 1  # the catalog changed: new keys, old entries are never hit again
print(catalog_page(1, catalog_version), catalog_page.cache_info().misses, "misses")
catalog_page.cache_clear()
print("after cache_clear:", catalog_page.cache_info())


class Basket:
    """A basket whose total is computed once and cached."""

    def __init__(self, prices: list[int]) -> None:
        """Store the prices."""
        self.prices = prices

    @functools.cached_property
    def total(self) -> int:
        """Sum the prices (computed on first access, then cached)."""
        return sum(self.prices)


basket = Basket([100, 250])
print(basket.total, end=" -> ")
basket.prices.append(50)
print(basket.total, "(stale)", end=" -> ")
del basket.total  # invalidate this object's cached value
print(basket.total, "(recomputed)")

page 1 rendered for catalog v1 1 miss
page 1 rendered for catalog v2 2 misses
after cache_clear: CacheInfo(hits=0, misses=0, maxsize=256, currsize=0)
350 -> 350 (stale) -> 400 (recomputed)


> **Warning:** `@lru_cache` on a **method** caches on `self` too, so every instance
> that ever called it stays alive as long as the cache does (a memory leak for
> short-lived objects). Cache on a module-level function keyed by plain values, or
> use `cached_property` for per-object results.

### 5.5.9 Vectorization

**Vectorization** means expressing a computation as operations on whole arrays, so
the per-element loop runs in compiled code over raw numbers instead of in the
interpreter over objects. NumPy is the foundation; pandas and Polars build
dataframes on the same idea.

#### NumPy arrays

An `ndarray` is one typed block of memory (`dtype`, `shape`, strides). Each
operation dispatches once and then loops in C, often with SIMD instructions:

In [433]:
if HAS_NUMPY:
    import numpy as np

    element_count = 1_000_000 * DEMO_SCALE
    amounts_list = [number % 50_000 for number in range(element_count)]
    amounts_array = np.array(amounts_list, dtype=np.int64)
    print(
        f"dtype={amounts_array.dtype}, shape={amounts_array.shape}, nbytes={amounts_array.nbytes:,}"
    )
    list_seconds = best_seconds(lambda: sum(amount * 2 for amount in amounts_list), number=1)
    array_seconds = best_seconds(lambda: (amounts_array * 2).sum(), number=5)
    print(
        f"sum of doubled amounts: list {list_seconds * 1000:.1f} ms, "
        f"NumPy {array_seconds * 1000:.2f} ms -> {list_seconds / array_seconds:.0f}x"
    )
else:
    print("NumPy is not installed: skipping.")

dtype=int64, shape=(1000000,), nbytes=8,000,000
sum of doubled amounts: list 16.5 ms, NumPy 0.42 ms -> 40x


> **Note:** the speedup only exists while data stays in arrays. Converting a list
> to an array (`np.array(amounts_list)`) and back (`.tolist()`) costs a Python-level
> pass each way; a NumPy scalar in a Python loop (`for x in array:`) is *slower*
> than a plain int.

#### Broadcasting

Operations between arrays of different shapes follow the **broadcasting** rules:
compare shapes from the right; two dimensions are compatible when they are equal
or one of them is 1; size-1 dimensions are stretched (virtually, without copying).
A column of tax rates against a row of prices gives the whole table in one
expression:

In [434]:
if HAS_NUMPY:
    prices_paise = np.array([9_900, 24_900, 1_29_900, 4_99_900])  # shape (4,)
    gst_percent = np.array([[5], [12], [18]])  # shape (3, 1)
    with_gst = prices_paise + prices_paise * gst_percent // 100  # shape (3, 4)
    print(f"{prices_paise.shape} with {gst_percent.shape} -> {with_gst.shape}")
    print(with_gst)
    print("incompatible shapes:", raised_error_name(lambda: np.ones(3) + np.ones(4)))
else:
    print("NumPy is not installed: skipping.")

(4,) with (3, 1) -> (3, 4)
[[ 10395  26145 136395 524895]
 [ 11088  27888 145488 559888]
 [ 11682  29382 153282 589882]]
incompatible shapes: ValueError


A common use is normalizing columns: subtract each column's mean (shape `(3,)`)
from a `(rows, 3)` matrix. No loop, and no copy of the means per row:

In [435]:
if HAS_NUMPY:
    sales_matrix = np.random.default_rng(42).normal(
        loc=[100, 50, 10], scale=[20, 5, 2], size=(100_000, 3)
    )
    centered = sales_matrix - sales_matrix.mean(axis=0)  # (100000, 3) - (3,)
    print("column means after centering:", np.round(centered.mean(axis=0), 10))
    print("column standard deviations:", np.round(centered.std(axis=0), 2))
else:
    print("NumPy is not installed: skipping.")

column means after centering: [ 0. -0. -0.]
column standard deviations: [20.07  5.    2.  ]


#### ufuncs

A **ufunc** (universal function) is an element-wise operation implemented in C:
`np.add`, `np.multiply`, `np.sqrt`, `np.maximum`, comparisons, and many more. Every
ufunc also has methods: `reduce` (fold an axis), `accumulate` (running result),
`outer`, and `at` (unbuffered in-place updates at indices, which handles repeated
indices correctly: the grouping tool of the Phase 5 Project). `out=` writes into an
existing array to avoid a temporary:

In [436]:
if HAS_NUMPY:
    daily_sales = np.array([120, 90, 150, 80, 200, 170, 60])
    print("np.add.reduce       :", np.add.reduce(daily_sales))
    print("np.maximum.accumulate:", np.maximum.accumulate(daily_sales))
    buffer = np.empty_like(daily_sales)
    np.multiply(daily_sales, 2, out=buffer)  # no temporary array
    print("multiply with out=  :", buffer)
    store_of_sale, revenue_by_store = np.array([0, 2, 0, 1, 2, 2]), np.zeros(3, dtype=np.int64)
    np.add.at(revenue_by_store, store_of_sale, [10, 20, 30, 40, 50, 60])
    print("np.add.at grouping  :", revenue_by_store, "(repeated indices are all added)")
else:
    print("NumPy is not installed: skipping.")

np.add.reduce       : 870
np.maximum.accumulate: [120 120 150 150 200 200 200]
multiply with out=  : [240 180 300 160 400 340 120]
np.add.at grouping  : [ 40  40 130] (repeated indices are all added)


Not everything that looks like a ufunc is fast: `np.vectorize` and `np.frompyfunc`
call a Python function per element. They are conveniences, not optimizations:

In [437]:
if HAS_NUMPY:
    small = np.arange(200_000, dtype=np.float64)
    ufunc_seconds = best_seconds(lambda: np.sqrt(small) + 1, number=20)
    vectorize_seconds = best_seconds(
        lambda: np.vectorize(lambda value: math.sqrt(value) + 1)(small), number=1
    )
    print(
        f"np.sqrt ufunc {ufunc_seconds * 1000:.2f} ms, "
        f"np.vectorize(Python function) {vectorize_seconds * 1000:.1f} ms"
    )
else:
    print("NumPy is not installed: skipping.")

np.sqrt ufunc 0.19 ms, np.vectorize(Python function) 15.9 ms


#### pandas vs Polars

Both are dataframe libraries. pandas (3.x) is the long-standing standard, eager
and NumPy/Arrow-backed. Polars is written in Rust on Apache Arrow memory, runs
operations on several cores, and has a **lazy** API whose query optimizer can
push filters down and drop unused columns before any work happens. The same
group-by in each (and in plain Python for reference):

In [438]:
frame_rng = random.Random(42)
frame_size = 500_000 * DEMO_SCALE
frame_cities = [
    ("Pune", "Delhi", "Kochi", "Patna", "Surat")[frame_rng.randrange(5)] for _ in range(frame_size)
]
frame_amounts = [frame_rng.randrange(100, 50_000) for _ in range(frame_size)]


def python_group_sum() -> dict[str, int]:
    """Group-by sum in plain Python, for reference."""
    totals: dict[str, int] = {}
    for city, amount in zip(frame_cities, frame_amounts, strict=True):
        totals[city] = totals.get(city, 0) + amount
    return dict(sorted(totals.items()))


print(
    f"{frame_size:,} rows; plain Python: {best_seconds(python_group_sum, number=1) * 1000:.0f} ms"
)

500,000 rows; plain Python: 23 ms


pandas and Polars, eager and lazy; each result is checked against the plain-Python one:

In [439]:
expected_totals = python_group_sum()
if HAS_PANDAS:
    import pandas as pd

    sales_pd = pd.DataFrame({"city": frame_cities, "amount": frame_amounts})
    pandas_totals = sales_pd.groupby("city")["amount"].sum().to_dict()
    pandas_seconds = best_seconds(lambda: sales_pd.groupby("city")["amount"].sum(), number=3)
    print(
        f"pandas {pd.__version__:<7} groupby: {pandas_seconds * 1000:6.1f} ms  "
        f"correct: {pandas_totals == expected_totals}"
    )
if HAS_POLARS:
    import polars as pl

    sales_pl = pl.DataFrame({"city": frame_cities, "amount": frame_amounts})

    def eager_query() -> pl.DataFrame:
        """Polars eager group-by."""
        return sales_pl.group_by("city").agg(pl.col("amount").sum())

    def lazy_query() -> pl.DataFrame:
        """Polars lazy group-by, collected."""
        return sales_pl.lazy().group_by("city").agg(pl.col("amount").sum()).collect()

    polars_totals = dict(eager_query().sort("city").iter_rows())
    polars_correct = polars_totals == expected_totals
    for label, query in (("eager", eager_query), ("lazy", lazy_query)):
        query_ms = best_seconds(query, number=3) * 1000
        print(
            f"polars {pl.__version__:<7} {label:<6}: {query_ms:6.1f} ms  correct: {polars_correct}"
        )
if not (HAS_PANDAS or HAS_POLARS):
    print("neither pandas nor Polars is installed: skipping.")

pandas 3.1.1   groupby:   21.8 ms  correct: True
polars 2.1.0   eager :    0.8 ms  correct: True
polars 2.1.0   lazy  :    0.7 ms  correct: True


Both dataframes beat the Python loop by a wide margin; which of the two wins
depends on the operation, the data size and the core count (Polars uses several
threads). The lazy plan pays off on multi-step queries, where the optimizer can
remove work:

In [440]:
if HAS_POLARS:
    plan = (
        sales_pl.lazy()
        .filter(pl.col("amount") > 40_000)
        .group_by("city")
        .agg(pl.len().alias("big_orders"))
    )
    print(plan.explain())
else:
    print("Polars is not installed: skipping.")

AGGREGATE[maintain_order: false]
  [len().alias("big_orders")] BY [col("city")]
  FROM
  simple π 1/1 ["city"]
    FILTER col("amount") > 40000
    FROM
      DF ["city", "amount"]; PROJECT */2 COLUMNS


### 5.5.10 Native code

When the hot loop is still Python after the algorithm, data structures and
vectorization are right, move that loop into compiled code. The options differ in
how much you rewrite and what you must build:

| Option | You write | Build step | Good for |
| --- | --- | --- | --- |
| `ctypes` (stdlib) | C (or use an existing `.so`/`.dll`) | a C compiler, or nothing for existing libraries | calling existing C libraries |
| CFFI | C declarations + C | optional (ABI mode needs none) | wrapping C libraries more comfortably than ctypes |
| Cython | Python with C type annotations (`.pyx` or pure-Python mode) | Cython + C compiler | speeding up existing Python loops, wrapping C |
| Rust via PyO3 + maturin | Rust | Rust toolchain + maturin | new extensions with memory safety (Polars, pydantic-core, orjson are built this way) |
| Numba | plain Python + `@njit` | none (LLVM JIT at runtime) | numeric loops over NumPy arrays |
| mypyc | ordinary type-annotated Python | mypyc + C compiler | compiling whole typed modules (mypy and black ship compiled with it) |
| PyPy | nothing: a different interpreter | none | pure-Python programs; C extensions run slower through its compatibility layer |

> **Warning:** each crossing between Python and native code has a fixed cost
> (argument conversion, a call). Move whole loops across the boundary, not one
> operation per call.

The running example is one function, written in Python first:

In [441]:
def sum_of_squares_mod(count: int) -> int:
    """Sum of (i % 1000)**2 for i in range(count): a tight integer loop."""
    total = 0
    for index in range(count):
        remainder = index % 1000
        total += remainder * remainder
    return total


loop_count = 5_000_000 * DEMO_SCALE
with Timer("pure Python", report=None) as python_timer:
    python_result = sum_of_squares_mod(loop_count)
print(f"{python_result=}  {python_timer.elapsed_seconds * 1000:.0f} ms")

python_result=1664167500000  171 ms


#### `ctypes`

`ctypes` loads a shared library and calls its functions. You declare each
function's argument and return types; nothing checks that they match the C code, so
a wrong declaration means wrong results or a crash. The cell compiles a tiny C file
with the system compiler first (and skips without one):

In [442]:
C_SOURCE = """
#include <stdint.h>
int64_t sum_of_squares_mod(int64_t count) {
    int64_t total = 0;
    for (int64_t index = 0; index < count; index++) {
        int64_t remainder = index % 1000;
        total += remainder * remainder;
    }
    return total;
}
double dot(const double *left, const double *right, int64_t length) {
    double total = 0.0;
    for (int64_t index = 0; index < length; index++) total += left[index] * right[index];
    return total;
}
"""
C_COMPILER = shutil.which("cc") or shutil.which("gcc") or shutil.which("clang")
library_suffix = {"win32": ".dll", "darwin": ".dylib"}.get(sys.platform, ".so")
LIBRARY_PATH = PHASE5_DIR / f"libsquares{library_suffix}"
if C_COMPILER and sys.platform != "win32":
    (PHASE5_DIR / "squares.c").write_text(C_SOURCE)
    print(
        run_tool(C_COMPILER, "-O2", "-shared", "-fPIC", "-o", str(LIBRARY_PATH), "squares.c")
        or f"compiled {LIBRARY_PATH.name}"
    )
else:
    print("no C compiler found (or Windows): the ctypes and CFFI cells will skip.")

compiled libsquares.so


Load the library, declare the signature, call it:

In [443]:
import ctypes

if LIBRARY_PATH.exists():
    squares_library = ctypes.CDLL(str(LIBRARY_PATH))
    squares_library.sum_of_squares_mod.argtypes = [ctypes.c_int64]
    squares_library.sum_of_squares_mod.restype = ctypes.c_int64
    with Timer("ctypes", report=None) as c_timer:
        c_result = squares_library.sum_of_squares_mod(loop_count)
    print(f"same result: {c_result == python_result}")
    print(
        f"C via ctypes {c_timer.elapsed_seconds * 1000:.1f} ms "
        f"vs Python {python_timer.elapsed_seconds * 1000:.0f} ms"
        f" -> {python_timer.elapsed_seconds / c_timer.elapsed_seconds:.0f}x"
    )
else:
    print("library not built: skipping.")

same result: True
C via ctypes 3.3 ms vs Python 171 ms -> 52x


Passing arrays: NumPy exposes its buffer as a C pointer, so C can read a million
doubles without any copying. A plain call with no work shows the fixed per-call
cost:

In [444]:
if LIBRARY_PATH.exists() and HAS_NUMPY:
    double_pointer = ctypes.POINTER(ctypes.c_double)
    squares_library.dot.argtypes = [double_pointer, double_pointer, ctypes.c_int64]
    squares_library.dot.restype = ctypes.c_double
    left, right = np.arange(1_000_000, dtype=np.float64), np.full(1_000_000, 0.5)
    c_dot = squares_library.dot(
        left.ctypes.data_as(double_pointer), right.ctypes.data_as(double_pointer), len(left)
    )
    print(f"dot via C: {c_dot:,.1f}   NumPy agrees: {math.isclose(c_dot, float(left @ right))}")
    per_call = best_seconds(lambda: squares_library.sum_of_squares_mod(0), number=10_000)
    print(f"one ctypes call doing no work: {per_call * 1e9:.0f} ns")
else:
    print("library or NumPy missing: skipping.")

dot via C: 249,999,750,000.0   NumPy agrees: True
one ctypes call doing no work: 475 ns


#### CFFI

[CFFI](https://cffi.readthedocs.io/) takes C declarations as text, so signatures
are written once in C syntax. *ABI mode* (`ffi.dlopen`) loads an existing library
like `ctypes`; *API mode* compiles a real extension module and is faster and safer.
CFFI is not a course dependency (it is often present because other packages depend
on it), so the cell checks first:

In [445]:
if has_module("cffi") and LIBRARY_PATH.exists():
    import cffi

    ffi = cffi.FFI()
    ffi.cdef("int64_t sum_of_squares_mod(int64_t count);")
    cffi_library = ffi.dlopen(str(LIBRARY_PATH))
    print(
        f"cffi {cffi.__version__}: result matches: "
        f"{cffi_library.sum_of_squares_mod(loop_count) == python_result}"
    )
else:
    print(
        "cffi is not installed (or the library was not built): skipping. `uv add cffi` to try it."
    )

cffi 2.1.1: result matches: True


#### Cython

Cython compiles a superset of Python to C. Adding C types to the hot variables
removes the object boxing in the loop; in *pure-Python mode* the annotations are
ordinary Python, so the same file still runs uncompiled:

```python
# squares_cy.py, compiled with `cythonize -i squares_cy.py`
import cython

def sum_of_squares_mod(count: cython.longlong) -> cython.longlong:
    total: cython.longlong = 0
    index: cython.longlong
    for index in range(count):
        remainder: cython.longlong = index % 1000
        total += remainder * remainder
    return total
```

In [446]:
if has_module("Cython"):
    import Cython

    print(
        f"Cython {Cython.__version__} is installed: "
        "build with `cythonize -i squares_cy.py` (needs a C compiler)."
    )
else:
    print("Cython is not installed: skipping the build (`uv add --dev cython` to try it).")

Cython is not installed: skipping the build (`uv add --dev cython` to try it).


#### Rust via PyO3 and maturin

PyO3 provides Rust bindings for the CPython API; maturin builds and packages the
result as a wheel. `maturin new --bindings pyo3 squares` creates the project, and
`maturin develop` builds it into the active environment:

```rust
use pyo3::prelude::*;

#[pyfunction]
fn sum_of_squares_mod(count: i64) -> i64 {
    (0..count).map(|index| { let r = index % 1000; r * r }).sum()
}

#[pymodule]
fn squares(module: &Bound<'_, PyModule>) -> PyResult<()> {
    module.add_function(wrap_pyfunction!(sum_of_squares_mod, module)?)
}
```

In [447]:
rust_tools = {name: shutil.which(name) for name in ("cargo", "rustc", "maturin")}
if all(rust_tools.values()):
    print("Rust toolchain and maturin found:", ", ".join(rust_tools))
    print(
        "build with: maturin new --bindings pyo3 squares && cd squares && maturin develop --release"
    )
else:
    missing = [name for name, path in rust_tools.items() if path is None]
    print(
        f"not available here ({', '.join(missing)} missing): skipping. "
        "Install Rust from rustup.rs and `uv tool install maturin`."
    )

not available here (cargo, rustc, maturin missing): skipping. Install Rust from rustup.rs and `uv tool install maturin`.


#### Numba

Numba compiles a Python function to machine code with LLVM the first time it is
called with given argument types. It understands a numeric subset of Python and
NumPy; other code falls back or fails to compile:

```python
from numba import njit

@njit(cache=True)
def sum_of_squares_mod(count):
    total = 0
    for index in range(count):
        remainder = index % 1000
        total += remainder * remainder
    return total
```

Numba releases usually trail new CPython versions by some months.

In [448]:
if has_module("numba"):
    import numba

    print(
        f"Numba {numba.__version__} is installed: decorate the function with @numba.njit "
        "and call it twice (compile, then run)."
    )
else:
    print(
        f"Numba is not installed (no build for Python {platform.python_version()} "
        "in this environment): skipping."
    )

Numba is not installed (no build for Python 3.15.1 in this environment): skipping.


#### mypyc

mypyc ships with mypy (installed in Phase 4). It compiles ordinary, fully
type-annotated modules to C extensions: the annotations you already wrote for
`mypy --strict` become the C types. The command is `mypyc module.py`, which needs a
C compiler and `setuptools` to build the extension:

In [449]:
mypyc_requirements = {
    "mypyc": has_module("mypyc"),
    "setuptools": has_module("setuptools"),
    "C compiler": C_COMPILER is not None,
}
print(
    ", ".join(f"{name}: {'yes' if found else 'no'}" for name, found in mypyc_requirements.items())
)
if all(mypyc_requirements.values()):
    print(
        "ready: `uv run mypyc notebooks/_scratch/phase5/profile_target.py` "
        "would compile the 5.5.3 script."
    )
else:
    print(
        "cannot build with mypyc here: skipping "
        "(mypyc builds through setuptools; `uv add --dev setuptools`)."
    )

mypyc: yes, setuptools: no, C compiler: yes
cannot build with mypyc here: skipping (mypyc builds through setuptools; `uv add --dev setuptools`).


#### PyPy

PyPy is an alternative Python implementation with a **tracing JIT**: it records the
hot loops of a running program and compiles them, so unchanged pure-Python code
often runs several times faster, like a JavaScript engine. The costs: it trails
CPython's language version, C extensions go through a slower compatibility layer,
and its garbage collector does not free objects immediately (always close files
with `with`).

In [450]:
pypy = shutil.which("pypy3") or shutil.which("pypy")
if pypy:
    print(
        run_tool(
            pypy,
            "-c",
            "import sys, platform; print(platform.python_implementation(), sys.version.split()[0])",
        )
    )
else:
    print("PyPy is not installed: skipping (`uv python install pypy` provides one).")

PyPy is not installed: skipping (`uv python install pypy` provides one).


### 5.5.11 Free-threading and JIT effects

Two CPython features change performance without changing your code:

- the **free-threaded build** (`python3.15t`, PEP 703/779; Units 3.1 and 5.1)
  removes the GIL, so CPU-bound threads run in parallel, at the price of somewhat
  slower single-threaded code (per-object locking, biased reference counting);
- the **experimental JIT** (PEP 744, 3.13+) is built into the default 3.15 build
  here but off unless `PYTHON_JIT=1` is set; it compiles hot traces of
  specialized bytecode into machine code with a "copy-and-patch" technique.

The only honest way to know what they do for *your* code is to run the same
benchmark on each. The benchmark script runs a pure-Python CPU loop once on one
thread, then four copies of it on four threads:

In [451]:
%%writefile _scratch/phase5/p5_build_benchmark.py
"""CPU loop on 1 thread, then 4 threads x the same work; prints one JSON line."""

import json
import os
import sys
import sysconfig
import threading
import time
from collections.abc import Callable

LOOPS = 2_000_000 * int(os.environ.get("PHASE5_DEMO_SCALE", "1"))


def busy(loops: int) -> int:
    """The CPU-bound work: a pure-Python arithmetic loop."""
    total = 0
    for index in range(loops):
        total += (index % 7) * (index % 11)
    return total


def best_of(runs: int, work: Callable[[], object]) -> float:
    """Fastest wall time of work() over runs attempts."""
    timings = []
    for _ in range(runs):
        start = time.perf_counter()
        work()
        timings.append(time.perf_counter() - start)
    return min(timings)

Overwriting _scratch/phase5/p5_build_benchmark.py


The second half (appended to the same file) starts four threads on the same
loop and prints one JSON line with the build's flags and both timings:

In [452]:
%%writefile -a _scratch/phase5/p5_build_benchmark.py


def four_threads() -> None:
    """Run busy(LOOPS) on four threads at once and wait for all of them."""
    threads = [threading.Thread(target=busy, args=(LOOPS,)) for _ in range(4)]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()


jit = getattr(sys, "_jit", None)
print(json.dumps({
    "gil_disabled": bool(sysconfig.get_config_var("Py_GIL_DISABLED")),
    "jit_enabled": bool(jit and jit.is_enabled()),
    "one_thread": best_of(3, lambda: busy(LOOPS)),
    "four_threads": best_of(2, four_threads),
}))

Appending to _scratch/phase5/p5_build_benchmark.py


Run it on the default build with the JIT off and on, and on the free-threaded build
(when installed). `PYTHON_JIT=1` only has an effect on builds that include the JIT:

In [453]:
build_runs = [
    ("default build, JIT off", sys.executable, {"PYTHON_JIT": "0"}),
    ("default build, PYTHON_JIT=1", sys.executable, {"PYTHON_JIT": "1"}),
]
if FREE_THREADED_PYTHON:
    build_runs += [
        ("free-threaded 3.15t", FREE_THREADED_PYTHON, {"PYTHON_JIT": "0"}),
        ("free-threaded, PYTHON_JIT=1", FREE_THREADED_PYTHON, {"PYTHON_JIT": "1"}),
    ]
else:
    print("no free-threaded interpreter found: only the default build is measured.")
build_results = {}
for label, python, extra_env in build_runs:
    output = run_tool(python, "p5_build_benchmark.py", extra_env=extra_env, timeout=300)
    build_results[label] = json.loads(output.splitlines()[-1])
baseline_one = build_results["default build, JIT off"]["one_thread"]
print(
    f"{'build':<28} {'JIT on?':>7} {'1 thread':>9} "
    f"{'4 threads':>10} {'4T/1T':>6} {'1T vs baseline':>15}"
)
for label, result in build_results.items():
    ratio = result["four_threads"] / result["one_thread"]
    print(
        f"{label:<28} {result['jit_enabled']!s:>7} {result['one_thread']:8.3f}s "
        f"{result['four_threads']:9.3f}s {ratio:6.2f} {result['one_thread'] / baseline_one:14.2f}x"
    )

build                        JIT on?  1 thread  4 threads  4T/1T  1T vs baseline
default build, JIT off         False    0.067s     0.237s   3.52           1.00x
default build, PYTHON_JIT=1     True    0.053s     0.212s   4.01           0.78x
free-threaded 3.15t            False    0.058s     0.072s   1.24           0.86x
free-threaded, PYTHON_JIT=1    False    0.058s     0.085s   1.46           0.87x


How to read the table (your numbers will differ):

- **4T/1T** is the wall time of four threads' worth of work divided by one
  thread's. On the GIL build it is about 4 (or worse): the threads take turns. On
  the free-threaded build it is close to 1: the four threads really run at once.
- **1T vs baseline** compares single-thread speed with the default build, JIT off.
  The free-threaded build is usually a little slower on one thread. The JIT is
  usually a few percent faster on loops like this, sometimes no faster, and it
  cannot help code that spends its time in C functions or waiting on I/O.
- "JIT on?" is what `sys._jit.is_enabled()` reported inside the child: a build
  without the JIT ignores `PYTHON_JIT=1` (here the free-threaded build).

The decision rule from Unit 5.2 still holds: measure *your* workload on each build
before switching, and check that your C extensions support the free-threaded build
(Unit 5.1.10).

### 5.5.12 Production performance

Benchmarks on a laptop find most problems; some only appear under real traffic,
real data and real concurrency. Three practices cover production:

1. **Sampling a live process** when something is slow right now;
2. **Continuous profiling**: sample all the time at a low rate, store the results,
   compare deploys;
3. **Benchmarks in CI**, so a regression fails a build instead of reaching users.

#### Sampling in production

Sampling profilers attach from outside, need no code changes and add little
overhead, so they are safe on a production process. The "server" below is a child
process that keeps doing work; the cells attach to it with py-spy and Tachyon.
(Attaching needs `ptrace` permission on Linux: `kernel.yama.ptrace_scope` 0, the
same user and capabilities, or `sudo`; in containers, `SYS_PTRACE`.)

In [454]:
%%writefile _scratch/phase5/fake_server.py
"""A stand-in for a production process: handles 'requests' until killed."""

import time

import profile_target

CUSTOMERS = profile_target.make_customers(2_500)


def handle_request(request_number: int) -> dict[str, int]:
    """Handle one fake request: build orders and total them."""
    orders = profile_target.make_orders(500, CUSTOMERS)
    return profile_target.city_totals(orders, CUSTOMERS)


def serve_forever() -> None:
    """Handle requests forever, pausing 1 ms between them."""
    request_number = 0
    while True:
        handle_request(request_number)
        request_number += 1
        time.sleep(0.001)  # waiting for the next request


if __name__ == "__main__":
    serve_forever()

Overwriting _scratch/phase5/fake_server.py


Start it in the background and take one snapshot of its stacks with `py-spy dump`:

In [455]:
server = subprocess.Popen(
    [sys.executable, "fake_server.py"],
    cwd=PHASE5_DIR,
    env=CHILD_ENV,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1.0)  # let it start
print(f"server running: {server.poll() is None}")
if PY_SPY:
    dump = run_tool(PY_SPY, "dump", "--pid", str(server.pid))
    print(re.sub(r"\d{3,}", "<n>", "\n".join(dump.splitlines()[2:])) if "Thread" in dump else dump)
else:
    print("py-spy is not installed: skipping the dump.")

server running: True

Thread <n> (active+gil)
    find_city (profile_target.py:26)
    city_totals (profile_target.py:35)
    handle_request (fake_server.py:13)
    serve_forever (fake_server.py:20)
    <module> (fake_server.py:26)


One dump is a single sample, but it is often enough: a process that is "stuck"
shows the line it is stuck on. `py-spy record --pid PID --duration 10` and
`py-spy top --pid PID` sample over time. Tachyon does the same with `attach`
(*New in 3.15*), here for one second. In its header, *Error rate* is the share of
samples it failed to take (for example, a stack read while the process was
changing it), not a failure of the program being profiled:

In [456]:
if HAS_TACHYON and server.poll() is None:
    attach_output = run_tool(
        sys.executable,
        "-m",
        "profiling.sampling",
        "attach",
        "-d",
        "1",
        "--sort",
        "tottime",
        "-l",
        "5",
        "--no-summary",
        str(server.pid),
    )
    print(re.sub(r"\b\d{4,}\b", "<pid>", attach_output))
else:
    print("Tachyon needs Python 3.15+ (or the server is not running): skipping.")
server.terminate()
server.wait(timeout=10)
print(f"server stopped, exit code {server.returncode}")

Captured 1,000 samples in 1.00 seconds
Sample rate: 999.92 samples/sec
Error rate: 0.30
Profile Stats:
       nsamples   sample%  tottime (ms)    cumul%  cumtime (ms)  filename:lineno(function)
        466/466      46.7       466.000      46.7       466.000  profile_target.py:25(find_city)
        369/369      37.0       369.000      37.0       369.000  profile_target.py:26(find_city)
        140/140      14.0       140.000      14.0       140.000  fake_server.py:22(serve_forever)
          11/11       1.1        11.000       1.1        11.000  profile_target.py:18(make_orders)
            5/5       0.5         5.000       0.5         5.000  profile_target.py:27(find_city)

Legend:
  nsamples: Direct/Cumulative samples (direct executing / on call stack)
  sample%: Percentage of total samples this function was directly executing
  tottime: Estimated total time spent directly in this function
  cumul%: Percentage of total samples when this function was on the call stack
  cumtime: Estima

Both point at `find_city`, from a process that was not started under a profiler and
was never restarted. That is the production workflow: notice latency in metrics,
attach, find the line, fix it, deploy, and confirm in the metrics.

#### Continuous profiling

A **continuous profiler** samples every instance of a service all the time at a
low rate (for example 100 Hz, well under 1 % overhead), ships aggregated stacks to
a backend, and lets you view flame graphs per service, version and time range or
diff two deploys. Examples: Grafana Pyroscope, Datadog, Elastic, Google Cloud
Profiler, Parca (eBPF-based, needs no agent in the process). The principle fits in
a few lines: a background thread reads the main thread's current frame with
`sys._current_frames()` and counts function names. The demo runs in a child:

In [457]:
print(
    run_child(
        """
        import collections
        import sys
        import threading
        import time

        import profile_target

        samples = collections.Counter()
        main_thread_id = threading.main_thread().ident

        def sampler(stop: threading.Event, interval: float = 0.005) -> None:
            while not stop.wait(interval):
                frame = sys._current_frames().get(main_thread_id)
                if frame is not None:
                    samples[frame.f_code.co_qualname] += 1

        stop = threading.Event()
        threading.Thread(target=sampler, args=(stop,), daemon=True).start()
        profile_target.main()
        stop.set()
        total = sum(samples.values())
        for name, count in samples.most_common(4):
            print(f"{name:<22} {count / total:6.1%} of {total} samples")
        """
    )
)

8000 orders, 5 cities, 4,480,000 chars, rate 83.25
fetch_exchange_rate     75.7% of 74 samples
find_city               13.5% of 74 samples
render_lines            10.8% of 74 samples
_path_stat               1.4% of 74 samples


This sampler counts wall time (it records the main thread's frame whatever it is
doing), so the sleeping `fetch_exchange_rate` leads, and it gets fewer samples than
requested because the sampling thread itself needs the GIL to run. Real continuous
profilers sample from outside the interpreter (like py-spy) or with
eBPF, record whole stacks rather than the top frame, and aggregate per minute; the
idea is the same. *New in 3.14*: `sys.remote_exec()` (PEP 768) lets a debugger or
profiler run code inside a live process safely, which is what Tachyon and pdb's
`-p PID` build on.

#### `pytest-benchmark` in CI

[`pytest-benchmark`](https://pytest-benchmark.readthedocs.io/) adds a `benchmark`
fixture that calibrates, repeats and records timings, saves them as JSON, and can
**fail the test run** when a function got slower than a saved baseline:

```python
# tests/test_performance.py
def test_city_totals_speed(benchmark):
    result = benchmark(city_totals_dict, ORDERS, CUSTOMERS)
    assert result == EXPECTED
```

```yaml
# .github/workflows/bench.yml (one job)
- run: uv run pytest tests/test_performance.py --benchmark-autosave
- run: uv run pytest tests/test_performance.py --benchmark-compare --benchmark-compare-fail=mean:15%
```

Shared CI runners are noisy, so use generous thresholds (or a dedicated runner),
compare against a baseline measured on the same runner type, and prefer comparing
the *ratio* of two implementations over absolute times. pytest-benchmark is not a
course dependency; the cell checks first:

In [458]:
if HAS_PYTEST_BENCHMARK:
    print(
        "pytest-benchmark is installed: run `uv run pytest --benchmark-only` "
        "on a test using the `benchmark` fixture."
    )
else:
    print("pytest-benchmark is not installed: skipping its run (`uv add --dev pytest-benchmark`).")

pytest-benchmark is not installed: skipping its run (`uv add --dev pytest-benchmark`).


Without the plugin, plain pytest can still guard performance in a robust way:
compare the optimized function against the slow one **in the same run** and require
a large margin. Machine speed cancels out of the ratio. The test module is written
to the scratch folder and pytest runs it in a child process:

In [459]:
%%writefile _scratch/phase5/perf_tests/test_city_totals_perf.py
"""A CI-friendly performance guard: a ratio in the same run, not an absolute time."""

import timeit

import profile_target

CUSTOMERS = profile_target.make_customers(2_500)
ORDERS = profile_target.make_orders(4_000, CUSTOMERS)


def city_totals_dict(orders: list[tuple[int, int]], customers: list[tuple[int, str]]) -> dict[str, int]:
    """The dict-based fix from 5.5.5."""
    city_of = dict(customers)
    totals: dict[str, int] = {}
    for customer_id, amount in orders:
        totals[city_of[customer_id]] = totals.get(city_of[customer_id], 0) + amount
    return totals


def test_same_result() -> None:
    """The dict version gives the same totals."""
    assert city_totals_dict(ORDERS, CUSTOMERS) == profile_target.city_totals(ORDERS, CUSTOMERS)


def test_dict_version_is_much_faster() -> None:
    """The dict version is far faster, measured in the same run."""
    slow = min(timeit.repeat(lambda: profile_target.city_totals(ORDERS, CUSTOMERS), number=1, repeat=3))
    fast = min(timeit.repeat(lambda: city_totals_dict(ORDERS, CUSTOMERS), number=1, repeat=3))
    print(f"ratio {slow / fast:.0f}x")
    assert slow / fast > 10  # the real gap is far larger; the margin absorbs CI noise

Overwriting _scratch/phase5/perf_tests/test_city_totals_perf.py


Run it with pytest (`-s` shows the printed ratio):

In [460]:
pytest_output = run_tool(
    sys.executable,
    "-m",
    "pytest",
    "-q",
    "-s",
    "-p",
    "no:cacheprovider",
    "perf_tests",
    extra_env={"PY_COLORS": "0"},
)
print("\n".join(line for line in pytest_output.splitlines() if line.strip()))

.ratio 226x
.
2 passed in 0.30s


#### Predict the output

**Predict the output.**

```python
from functools import lru_cache

@lru_cache(maxsize=2)
def square(n):
    return n * n

for n in [1, 2, 1, 3, 2, 1]:
    square(n)
print(square.cache_info())
```

<details><summary>Answer</summary>

`CacheInfo(hits=1, misses=5, maxsize=2, currsize=2)`. Trace the cache (most recent
last): 1 miss `[1]`; 2 miss `[1, 2]`; 1 **hit** `[2, 1]`; 3 miss, evicts 2 → `[1, 3]`;
2 miss, evicts 1 → `[3, 2]`; 1 miss, evicts 3 → `[2, 1]`. A cache smaller than the
working set mostly misses.

</details>

Run the next cell to check your prediction.

In [461]:
@functools.lru_cache(maxsize=2)
def square(number: int) -> int:
    """Square a number (cached)."""
    return number * number


for number in [1, 2, 1, 3, 2, 1]:
    square(number)
print(square.cache_info())

CacheInfo(hits=1, misses=5, maxsize=2, currsize=2)


**Predict the output.**

```python
import tracemalloc

tracemalloc.start()
data = [0] * 1_000_000
del data
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(current < 100_000, peak > 7_000_000)
```

<details><summary>Answer</summary>

`True True`. The list held a million 8-byte pointers (about 8 MB, all pointing at
the same cached `0`). After `del`, *current* is back near zero, but *peak* remembers
the high-water mark. Peak is what the machine had to provide.

</details>

Run the next cell to check your prediction.

In [462]:
tracemalloc.start()
data = [0] * 1_000_000
del data
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(current < 100_000, peak > 7_000_000)

True True


**Predict the output.**

```python
import numpy as np

rates = np.array([[5], [12], [18]])
prices = np.arange(4)
print((rates * prices).shape)
print(np.ones(3) + np.ones(4))
```

<details><summary>Answer</summary>

`(3, 4)`, then a `ValueError` ("operands could not be broadcast together with
shapes (3,) (4,)"). Shapes are compared from the right: `(3, 1)` with `(4,)` works
because the 1 stretches to 4 and the missing dimension counts as 1. `(3,)` with
`(4,)` has two different sizes, neither of them 1.

</details>

Run the next cell to check your prediction.

In [463]:
if HAS_NUMPY:
    rates = np.array([[5], [12], [18]])
    prices = np.arange(4)
    print((rates * prices).shape)
    try:
        print(np.ones(3) + np.ones(4))
    except ValueError as error:
        print(f"{type(error).__name__}: {error}")
else:
    print("NumPy is not installed: skipping.")

(3, 4)
ValueError: operands could not be broadcast together with shapes (3,) (4,) 


### Lab 5.5

Five exercises. Replace each `raise NotImplementedError`, then run the check cell;
until then the checks print FAIL, which is expected. Checks that involve speed use
a large margin against a much larger real gap and print the measured ratio; memory
checks are exact properties of the code, not of the machine.

The first setup cell defines the seeded customer and order generators:

In [464]:
LAB55_DIR = PHASE5_DIR / "lab55"
LAB55_DIR.mkdir(exist_ok=True)
LAB55_CITIES = ("Pune", "Delhi", "Kochi", "Patna", "Surat", "Indore")
LAB55_EVENT_TYPES = ("view", "click", "add_to_cart", "purchase")
if HAS_NUMPY:
    import numpy as np


def make_lab_customers(count: int = 1_200, seed: int = 55) -> list[tuple[int, str, str]]:
    """(customer_id, name, city) with shuffled, spread-out ids."""
    rng = random.Random(seed)
    customer_ids = rng.sample(range(10_000, 99_999), count)
    return [
        (customer_id, f"Customer {position}", LAB55_CITIES[position % 6])
        for position, customer_id in enumerate(customer_ids)
    ]


def make_lab_orders(
    customers: list[tuple[int, str, str]], count: int, seed: int = 56
) -> list[tuple[int, int, int]]:
    """(order_id, customer_id, amount_paise) rows."""
    rng = random.Random(seed)
    return [
        (order_id, rng.choice(customers)[0], rng.randrange(1_000, 500_000))
        for order_id in range(count)
    ]


print("Lab 5.5 generators ready: make_lab_customers, make_lab_orders")

Lab 5.5 generators ready: make_lab_customers, make_lab_orders


The second writes the event log for exercise 5, creates the lab data under
`notebooks/_scratch/phase5/lab55/`, and defines `best_time_or_none` (a timing that
returns `None` instead of raising while a stub is unfinished):

In [465]:
def write_events(path: Path, count: int, seed: int = 57) -> Path:
    """Write a seeded CSV event log: timestamp,user_id,event,bytes."""
    rng = random.Random(seed)
    with path.open("w", encoding="utf-8") as handle:
        handle.write("timestamp,user_id,event,bytes\n")
        for second in range(count):
            event = rng.choice(LAB55_EVENT_TYPES)
            user_id, size = rng.randrange(3_000), rng.randrange(200, 90_000)
            handle.write(f"{1_767_225_600 + second},{user_id},{event},{size}\n")
    return path


def prepare_lab_5_5() -> tuple[list[tuple[int, str, str]], list[tuple[int, int, int]], Path]:
    """Create the lab data: customers, orders and the event-log file."""
    customers = make_lab_customers()
    orders = make_lab_orders(customers, 4_000 * DEMO_SCALE)
    events = write_events(LAB55_DIR / "events.csv", 60_000 * DEMO_SCALE)
    return customers, orders, events


def best_time_or_none(work: Callable[[], object], repeat: int = 3) -> float | None:
    """Best of `repeat` single runs in seconds, or None if work raises (an unfinished stub)."""
    try:
        return min(timeit.repeat(work, number=1, repeat=repeat))
    except Exception:
        return None


LAB55_CUSTOMERS, LAB55_ORDERS, LAB55_EVENTS = prepare_lab_5_5()
print(f"{len(LAB55_CUSTOMERS):,} customers, {len(LAB55_ORDERS):,} orders")
print(f"events file: {LAB55_EVENTS.stat().st_size:,} bytes")

1,200 customers, 4,000 orders
events file: 1,771,506 bytes


#### Exercise 1 — Profile and fix a slow function

`city_totals_slow` (given below) is correct but slow. First find out *why* with a
profiler, then fix it.

1. Write `hottest_function(function, *args)`: profile one call
   `function(*args)` with `cProfile` and return the **name** of the function with
   the largest **own time** (`tottime`). For a Python function that is its plain
   name (`"lookup_customer"`); for a built-in it is the label pstats uses, such as
   `"<built-in method builtins.sorted>"`. (`pstats.Stats(profiler).stats` maps
   `(file, line, name)` to `(primitive calls, calls, tottime, cumtime, callers)`.)
2. Write `city_totals_fast(orders, customers)`, returning exactly what
   `city_totals_slow` returns, much faster.

In [466]:
def lookup_customer(
    customers: list[tuple[int, str, str]], customer_id: int
) -> tuple[int, str, str]:
    """Find a customer by id with a linear scan."""
    for customer in customers:
        if customer[0] == customer_id:
            return customer
    raise KeyError(customer_id)


def city_totals_slow(
    orders: list[tuple[int, int, int]], customers: list[tuple[int, str, str]]
) -> dict[str, int]:
    """Revenue (paise) per customer city, sorted by city name. Correct, but slow."""
    totals: dict[str, int] = {}
    for _order_id, customer_id, amount in orders:
        city = lookup_customer(customers, customer_id)[2]
        totals[city] = totals.get(city, 0) + amount
    return dict(sorted(totals.items()))

Your solution, part 1:

In [467]:
def hottest_function(function: Callable[..., object], *args: object) -> str:
    """Profile function(*args) once; return the name of the entry with the largest tottime."""
    raise NotImplementedError

Your solution, part 2:

In [468]:
def city_totals_fast(
    orders: list[tuple[int, int, int]], customers: list[tuple[int, str, str]]
) -> dict[str, int]:
    """Same result as city_totals_slow, without a scan per order."""
    raise NotImplementedError

Check exercise 1:

In [469]:
slow_totals = city_totals_slow(LAB55_ORDERS, LAB55_CUSTOMERS)
slow_seconds = best_time_or_none(lambda: city_totals_slow(LAB55_ORDERS, LAB55_CUSTOMERS))
fast_seconds = best_time_or_none(lambda: city_totals_fast(LAB55_ORDERS, LAB55_CUSTOMERS))
run_checks(
    [
        (
            "profile finds the hot spot",
            lambda: hottest_function(city_totals_slow, LAB55_ORDERS, LAB55_CUSTOMERS),
            "lookup_customer",
        ),
        (
            "built-ins are reported too",
            lambda: "sorted" in hottest_function(sorted, list(range(200_000, 0, -1))),
            True,
        ),
        (
            "fast version: same totals",
            lambda: city_totals_fast(LAB55_ORDERS, LAB55_CUSTOMERS),
            slow_totals,
        ),
        (
            "fast version: no longer dominated by lookup_customer",
            lambda: (
                hottest_function(city_totals_fast, LAB55_ORDERS, LAB55_CUSTOMERS)
                != "lookup_customer"
            ),
            True,
        ),
        (
            "fast version: at least 5x faster",
            lambda: fast_seconds is not None and slow_seconds / fast_seconds >= 5,
            True,
        ),
    ]
)
if fast_seconds:
    print(
        f"measured: slow {slow_seconds * 1000:.1f} ms, fast {fast_seconds * 1000:.2f} ms "
        f"-> {slow_seconds / fast_seconds:.0f}x"
    )

FAIL  profile finds the hot spot: raised NotImplementedError: 
FAIL  built-ins are reported too: raised NotImplementedError: 
FAIL  fast version: same totals: raised NotImplementedError: 
FAIL  fast version: no longer dominated by lookup_customer: raised NotImplementedError: 
FAIL  fast version: at least 5x faster: expected True, got False
0/5 checks passed


#### Exercise 2 — Rewrite an O(n²) algorithm as O(n)

`count_pairs_quadratic(amounts, target)` counts the index pairs `i < j` with
`amounts[i] + amounts[j] == target` by trying every pair. Write
`count_pairs_linear(amounts, target)` with the same results in **one pass**.
(Hint: when you reach an amount, how many earlier amounts complete it?)

In [470]:
def count_pairs_quadratic(amounts: list[int], target: int) -> int:
    """Pairs i < j with amounts[i] + amounts[j] == target, by brute force: O(n^2)."""
    pair_count = 0
    for first in range(len(amounts)):
        for second in range(first + 1, len(amounts)):
            if amounts[first] + amounts[second] == target:
                pair_count += 1
    return pair_count


def growth_factor(count_pairs: Callable[[list[int], int], int], size: int) -> float | None:
    """Best time at 4 x size divided by best time at size (about 4 for O(n), 16 for O(n^2))."""
    rng = random.Random(58)
    small, large = (
        [rng.randrange(1_000) for _ in range(list_size)] for list_size in (size, 4 * size)
    )
    small_time = best_time_or_none(lambda: count_pairs(small, 1_000))
    large_time = best_time_or_none(lambda: count_pairs(large, 1_000))
    return None if small_time is None or large_time is None else large_time / small_time

Your solution:

In [471]:
def count_pairs_linear(amounts: list[int], target: int) -> int:
    """Same result as count_pairs_quadratic, in O(n)."""
    raise NotImplementedError

Check exercise 2:

In [472]:
pair_cases = [
    ([], 10),
    ([5], 10),
    ([5, 5, 5], 10),
    ([1, 9, 2, 8, 5, 5], 10),
    ([-3, 13, 0, 10, 10], 10),
    ([4, 4, 4, 4], 8),
]
pair_rng = random.Random(59)
random_amounts = [pair_rng.randrange(-50, 150) for _ in range(1_500)]
linear_growth = growth_factor(count_pairs_linear, 50_000 * DEMO_SCALE)
run_checks(
    [
        *(
            (
                f"pairs in {amounts} summing to {target}",
                lambda amounts=amounts, target=target: count_pairs_linear(amounts, target),
                count_pairs_quadratic(amounts, target),
            )
            for amounts, target in pair_cases
        ),
        (
            "1 500 random amounts",
            lambda: count_pairs_linear(random_amounts, 100),
            count_pairs_quadratic(random_amounts, 100),
        ),
        (
            "4x the input takes less than 8x the time",
            lambda: linear_growth is not None and linear_growth < 8,
            True,
        ),
    ]
)
if linear_growth:
    print(f"measured growth for 4x the input: x{linear_growth:.1f} (O(n) ~ 4, O(n^2) ~ 16)")

FAIL  pairs in [] summing to 10: raised NotImplementedError: 
FAIL  pairs in [5] summing to 10: raised NotImplementedError: 
FAIL  pairs in [5, 5, 5] summing to 10: raised NotImplementedError: 
FAIL  pairs in [1, 9, 2, 8, 5, 5] summing to 10: raised NotImplementedError: 
FAIL  pairs in [-3, 13, 0, 10, 10] summing to 10: raised NotImplementedError: 
FAIL  pairs in [4, 4, 4, 4] summing to 8: raised NotImplementedError: 
FAIL  1 500 random amounts: raised NotImplementedError: 
FAIL  4x the input takes less than 8x the time: expected True, got False
0/8 checks passed


#### Exercise 3 — A micro-optimization timing table

1. Write `timing_table(candidates, number=1000, repeat=5)`: `candidates` maps a
   label to a zero-argument callable. Time each with `timeit.repeat` and return a
   list of `(label, best seconds per call)` sorted **fastest first**.
2. Write `render_timing_table(rows)`, which turns such a list into a Markdown
   table, times in microseconds with two decimals and a thousands separator,
   relative to the fastest row (the first one) with two decimals:

```text
| Variant | Time per call | Relative |
| --- | ---: | ---: |
| join | 12.34 µs | 1.00x |
| plus | 24.68 µs | 2.00x |
```

No trailing newline; an empty list gives just the two header lines.

Your solution, part 1:

In [473]:
def timing_table(
    candidates: dict[str, Callable[[], object]], number: int = 1000, repeat: int = 5
) -> list[tuple[str, float]]:
    """(label, best seconds per call) for every candidate, fastest first."""
    raise NotImplementedError

Your solution, part 2:

In [474]:
def render_timing_table(rows: list[tuple[str, float]]) -> str:
    """Markdown table: variant, time per call in µs, ratio to the fastest row."""
    raise NotImplementedError

Check exercise 3:

In [475]:
expected_table = (
    "| Variant | Time per call | Relative |\n| --- | ---: | ---: |\n"
    "| join | 12.34 µs | 1.00x |\n| plus | 24.68 µs | 2.00x |"
)
run_checks(
    [
        (
            "render two rows",
            lambda: render_timing_table([("join", 12.34e-6), ("plus", 24.68e-6)]),
            expected_table,
        ),
        (
            "render no rows",
            lambda: render_timing_table([]),
            "| Variant | Time per call | Relative |\n| --- | ---: | ---: |",
        ),
        (
            "render a slow row",
            lambda: render_timing_table([("a", 0.5e-6), ("b", 1.5e-3)]).splitlines()[-1],
            "| b | 1,500.00 µs | 3000.00x |",
        ),
    ]
)

FAIL  render two rows: raised NotImplementedError: 
FAIL  render no rows: raised NotImplementedError: 
FAIL  render a slow row: raised NotImplementedError: 
0/3 checks passed


The `timing_table` checks: rows sorted fastest first, and times reported per call
(a 2 ms sleep must come out between 2 ms and 200 ms, not ×3 for `number=3`; the
upper bound is generous so a busy machine cannot fail it):

In [476]:
run_checks(
    [
        (
            "sorted fastest first",
            lambda: [
                label
                for label, _ in timing_table(
                    {"sleep": lambda: time.sleep(0.002), "noop": lambda: None}, number=3, repeat=2
                )
            ],
            ["noop", "sleep"],
        ),
        (
            "seconds per call, not per batch",
            lambda: (
                0.002
                <= timing_table({"sleep": lambda: time.sleep(0.002)}, number=3, repeat=2)[0][1]
                < 0.2
            ),
            True,
        ),
    ]
)

FAIL  sorted fastest first: raised NotImplementedError: 
FAIL  seconds per call, not per batch: raised NotImplementedError: 
0/2 checks passed


Once it passes, your functions produce a real table for four ways to build a CSV line:

In [477]:
csv_fields = [str(column_number) for column_number in range(30)]


def csv_line_plus() -> str:
    """Build the CSV line with += (the slow variant)."""
    line = ""
    for csv_field in csv_fields:
        line += csv_field + ","
    return line[:-1]


csv_candidates = {
    "str += in a loop": csv_line_plus,
    '",".join(fields)': lambda: ",".join(csv_fields),
    "join of a generator": lambda: ",".join(field for field in csv_fields),
    "str.format with unpacking": lambda: ("{}," * len(csv_fields))[:-1].format(*csv_fields),
}
try:
    print(render_timing_table(timing_table(csv_candidates, number=2_000)))
except NotImplementedError:
    print("finish exercise 3 to see the table")

finish exercise 3 to see the table


#### Exercise 4 — NumPy vectorization

Vectorize the two loops below. `revenue_by_store_numpy(store_ids, amounts,
store_count)` takes NumPy integer arrays and returns an `int64` array of length
`store_count` (stores without sales get 0; money stays exact). `zscores_numpy(values)`
returns `(value - mean) / standard deviation` (population standard deviation) for a
float array. No Python loops.

In [478]:
def revenue_by_store_loop(store_ids: list[int], amounts: list[int], store_count: int) -> list[int]:
    """Total amount per store id (0..store_count-1), with a Python loop."""
    totals = [0] * store_count
    for store_id, amount in zip(store_ids, amounts, strict=True):
        totals[store_id] += amount
    return totals


def zscores_loop(values: list[float]) -> list[float]:
    """Standard scores with the population standard deviation, with Python loops."""
    mean = sum(values) / len(values)
    spread = (sum((value - mean) ** 2 for value in values) / len(values)) ** 0.5
    return [(value - mean) / spread for value in values]

Your solution, part 1:

In [479]:
def revenue_by_store_numpy(
    store_ids: np.ndarray, amounts: np.ndarray, store_count: int
) -> np.ndarray:
    """Vectorized revenue_by_store_loop: an int64 array of length store_count."""
    raise NotImplementedError

Your solution, part 2:

In [480]:
def zscores_numpy(values: np.ndarray) -> np.ndarray:
    """Vectorized zscores_loop."""
    raise NotImplementedError

Check exercise 4:

In [481]:
vector_rng = random.Random(60)
sale_count = 200_000 * DEMO_SCALE
store_id_list = [vector_rng.randrange(45) for _ in range(sale_count)]  # store ids 45..49 never sell
amount_list = [vector_rng.randrange(100, 900_000) for _ in range(sale_count)]
value_list = [vector_rng.gauss(500, 120) for _ in range(50_000)]
store_id_array, amount_array, value_array = (
    np.array(store_id_list),
    np.array(amount_list),
    np.array(value_list),
)

The expected result comes from the given loop; both versions are timed once here,
then the revenue checks (exact values, `int64`, zeros for stores without sales):

In [482]:
expected_revenue = revenue_by_store_loop(store_id_list, amount_list, 50)
loop_seconds = best_time_or_none(lambda: revenue_by_store_loop(store_id_list, amount_list, 50))
numpy_seconds = best_time_or_none(lambda: revenue_by_store_numpy(store_id_array, amount_array, 50))
run_checks(
    [
        (
            "revenue matches the loop (exact)",
            lambda: revenue_by_store_numpy(store_id_array, amount_array, 50).tolist(),
            expected_revenue,
        ),
        (
            "revenue dtype is int64",
            lambda: revenue_by_store_numpy(store_id_array, amount_array, 50).dtype == np.int64,
            True,
        ),
        (
            "stores without sales are 0",
            lambda: revenue_by_store_numpy(store_id_array, amount_array, 50)[45:].tolist(),
            [0] * 5,
        ),
    ]
)

FAIL  revenue matches the loop (exact): raised NotImplementedError: 
FAIL  revenue dtype is int64: raised NotImplementedError: 
FAIL  stores without sales are 0: raised NotImplementedError: 
0/3 checks passed


The z-score checks, then the one speed check (with a wide margin) and the measured
ratio:

In [483]:
run_checks(
    [
        (
            "z-scores match the loop",
            lambda: bool(np.allclose(zscores_numpy(value_array), zscores_loop(value_list))),
            True,
        ),
        (
            "z-scores have mean 0 and std 1",
            lambda: [
                round(float(statistic(zscores_numpy(value_array))), 6)
                for statistic in (np.mean, np.std)
            ],
            [0.0, 1.0],
        ),
        (
            "vectorized revenue at least 5x faster",
            lambda: numpy_seconds is not None and loop_seconds / numpy_seconds >= 5,
            True,
        ),
    ]
)
if numpy_seconds:
    print(
        f"measured: loop {loop_seconds * 1000:.1f} ms, NumPy {numpy_seconds * 1000:.2f} ms "
        f"-> {loop_seconds / numpy_seconds:.0f}x"
    )

FAIL  z-scores match the loop: raised NotImplementedError: 
FAIL  z-scores have mean 0 and std 1: raised NotImplementedError: 
FAIL  vectorized revenue at least 5x faster: expected True, got False
0/3 checks passed


#### Exercise 5 — A 5× peak-memory reduction

`event_summary_eager(path)` reads the whole event log into memory before
summarizing it.

1. Write `peak_memory(function, *args)`, which calls `function(*args)` and returns
   `(result, peak_bytes)`: the peak traced memory **during the call**, above what
   was allocated before it. It must also work when `tracemalloc` is already
   tracing (then leave it running), and stop tracing only if it started it.
2. Write `event_summary_lean(path)` with the same result and **at least 5× less**
   peak memory.

In [484]:
def event_summary_eager(path: Path) -> dict[str, object]:
    """Event counts, total bytes and distinct users, computed from all rows in memory."""
    lines = path.read_text(encoding="utf-8").splitlines()
    header = lines[0].split(",")
    rows = [dict(zip(header, line.split(","), strict=True)) for line in lines[1:]]
    return {
        "events": dict(sorted(Counter(row["event"] for row in rows).items())),
        "total_bytes": sum(int(row["bytes"]) for row in rows),
        "unique_users": len({row["user_id"] for row in rows}),
    }

Your solution, part 1:

In [485]:
def peak_memory(function: Callable[..., object], *args: object) -> tuple[object, int]:
    """Call function(*args); return (result, peak traced bytes above the starting level)."""
    raise NotImplementedError

Your solution, part 2:

In [486]:
def event_summary_lean(path: Path) -> dict[str, object]:
    """Same result as event_summary_eager, streaming the file."""
    raise NotImplementedError

Check exercise 5:

In [487]:
def peak_memory_keeps_outer_trace() -> bool:
    """peak_memory inside an outer trace must leave that trace running."""
    tracemalloc.start()
    try:
        _, inner_peak = peak_memory(lambda: len(bytearray(2_000_000)))
        return tracemalloc.is_tracing() and inner_peak >= 2_000_000
    finally:
        tracemalloc.stop()


def peak_ratio() -> float:
    """Peak of the eager version divided by peak of the lean version."""
    return (
        peak_memory(event_summary_eager, LAB55_EVENTS)[1]
        / peak_memory(event_summary_lean, LAB55_EVENTS)[1]
    )

The checks: `peak_memory` returns the result, sees a 5 MB temporary, stops only its
own trace, and the lean summary gives the same result with at least 5× lower peak:

In [488]:
run_checks(
    [
        ("peak_memory returns the result", lambda: peak_memory(len, "chai")[0], 4),
        (
            "peak_memory sees a 5 MB temporary",
            lambda: peak_memory(lambda: len(bytearray(5_000_000)))[1] >= 5_000_000,
            True,
        ),
        (
            "peak_memory stops the trace it started",
            lambda: (peak_memory(len, "x"), tracemalloc.is_tracing())[1],
            False,
        ),
        ("peak_memory works inside an outer trace", peak_memory_keeps_outer_trace, True),
        (
            "lean summary: same result",
            lambda: event_summary_lean(LAB55_EVENTS),
            event_summary_eager(LAB55_EVENTS),
        ),
        ("lean summary: at least 5x lower peak", lambda: peak_ratio() >= 5, True),
    ]
)
try:
    print(f"measured peak ratio eager/lean: {peak_ratio():.0f}x")
except NotImplementedError:
    pass

FAIL  peak_memory returns the result: raised NotImplementedError: 
FAIL  peak_memory sees a 5 MB temporary: raised NotImplementedError: 
FAIL  peak_memory stops the trace it started: raised NotImplementedError: 
FAIL  peak_memory works inside an outer trace: raised NotImplementedError: 
FAIL  lean summary: same result: raised NotImplementedError: 
FAIL  lean summary: at least 5x lower peak: raised NotImplementedError: 
0/6 checks passed


### Recap — Unit 5.5

| Concept | One-line takeaway | JS / Node equivalent |
| --- | --- | --- |
| Method | measure, profile, estimate with Amdahl, change one thing, re-measure | same discipline |
| Benchmark hygiene | warm up, repeat, report min/median and spread, fresh processes; `timeit` turns GC off; `pyperf` automates it | `tinybench`, `mitata` |
| Timing | `perf_counter` for durations, `process_time` for CPU, never `time.time()`; `%timeit -o` | `performance.now()` |
| Deterministic profiling | `cProfile`/`profiling.tracing` + `pstats` (+ snakeviz): exact calls, real overhead | `--cpu-prof` |
| Sampling profiling | Tachyon (3.15), py-spy, scalene: low overhead, attachable, wall vs CPU modes | `0x`, `perf` |
| Line profiling | `line_profiler` for the suspects only | — |
| Flame graphs | width = share of samples, height = depth; attack wide plateaus | Chrome flame chart |
| Memory | `tracemalloc` (Python, by line, peak vs current), memray (everything, timeline), RSS is a third number | heap snapshots |
| Algorithms first | dict/set lookups, no hidden O(n²), `heapq`, `bisect`, memoization | same |
| Interpreter habits | built-ins in C, `join`, comprehensions, no exceptions in hot paths, stable types; aliases matter less since 3.11 | V8 inline caches |
| Memory-lean data | `__slots__`, `array`, `memoryview`, generators, `sys.intern` | typed arrays, `Buffer` |
| Caching | measure the hit rate; bound size; TTL for changing data; invalidate on write or version keys | `lru-cache` |
| Vectorization | NumPy arrays, broadcasting, ufuncs (`at` for grouping); pandas or Polars for frames | typed-array loops, Arquero/Danfo |
| Native code | ctypes/CFFI, Cython, PyO3, Numba, mypyc, PyPy; cross the boundary per loop, not per item | N-API, WASM |
| Builds | 3.15t scales CPU threads, costs some single-thread speed; the JIT gives modest gains; measure both | V8 tiers |
| Production | attach samplers to live processes, continuous profiling, ratio-based benchmarks in CI | Clinic.js, APMs |

**Further reading**

- [`timeit`](https://docs.python.org/3/library/timeit.html),
  [`time` clocks](https://docs.python.org/3/library/time.html#time.perf_counter),
  [The Python profilers (`cProfile`, `pstats`, `profiling`)](https://docs.python.org/3.15/library/profiling.html),
  [`tracemalloc`](https://docs.python.org/3/library/tracemalloc.html)
- [`heapq`](https://docs.python.org/3/library/heapq.html), [`bisect`](https://docs.python.org/3/library/bisect.html),
  [`functools.lru_cache`](https://docs.python.org/3/library/functools.html#functools.lru_cache),
  [`array`](https://docs.python.org/3/library/array.html), [`memoryview`](https://docs.python.org/3/library/stdtypes.html#memoryview),
  [`ctypes`](https://docs.python.org/3/library/ctypes.html)
- [PEP 799](https://peps.python.org/pep-0799/) (the `profiling` package and the sampling profiler),
  [PEP 659](https://peps.python.org/pep-0659/) (specializing adaptive interpreter),
  [PEP 709](https://peps.python.org/pep-0709/) (inlined comprehensions),
  [PEP 744](https://peps.python.org/pep-0744/) (JIT compilation),
  [PEP 703](https://peps.python.org/pep-0703/) / [PEP 779](https://peps.python.org/pep-0779/) (free-threading),
  [PEP 768](https://peps.python.org/pep-0768/) (safe external debugger interface),
  [PEP 3118](https://peps.python.org/pep-3118/) (buffer protocol)
- Tools: [pyperf](https://pyperf.readthedocs.io/), [snakeviz](https://jiffyclub.github.io/snakeviz/),
  [py-spy](https://github.com/benfred/py-spy), [line_profiler](https://github.com/pyutils/line_profiler),
  [scalene](https://github.com/plasma-umass/scalene), [memray](https://bloomberg.github.io/memray/),
  [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html), [Polars](https://docs.pola.rs/),
  [PyO3](https://pyo3.rs/), [Cython](https://cython.org/), [Numba](https://numba.pydata.org/),
  [mypyc](https://mypyc.readthedocs.io/), [pytest-benchmark](https://pytest-benchmark.readthedocs.io/)

## Phase 5 Project — 10× Faster Sales Report

**Brief.** A nightly script turns one million sales rows into a text report:
totals, revenue by region, top stores, top products, monthly revenue and the best
store/product pairs. It was written quickly and it shows: nested loops, repeated
scans over all rows, string concatenation, money as floats, and 40 store-directory
API calls made one after another. Make it **at least 10× faster with at least 3×
less peak memory**, without changing a single character of its output, and
document every step (change → time → memory → reason), including one concurrency
step.

| File (`projects/sales-report/`) | Role |
| --- | --- |
| `common.py` | product catalog, seeded data generator (`generate_chunks`), simulated store API (`fetch_store_profile`) |
| `slow_report.py` | the deliberately slow baseline (step 0) |
| `fast_report.py` | steps 1–5, each a complete `build_report(rows, seed)`; `STEPS` describes them |
| `benchmark.py` | runs every step in a fresh process, measures wall time and peak RSS, writes `RESULTS.md` |
| `tests/test_reports.py` | every step must produce the slow report byte for byte |
| `RESULTS.md` | the full-size (1 000 000 rows) benchmark, run from a terminal |

**Method (Unit 5.5.1).** Measure the baseline, profile it, fix the largest cost,
re-measure, repeat. Correctness is checked at every step by comparing report
text, so no optimization can quietly change an answer.

**How this section runs.** The scripts are real files, imported from the project
folder. Correctness checks run in the kernel with the simulated network switched
off; timings run in child processes. The benchmark runs here at
`PROJECT_ROWS = 50 000 × DEMO_SCALE` rows so Run All stays fast; the 1M-row numbers
come from `RESULTS.md`, produced in a terminal with:

```bash
cd projects/sales-report
uv run python benchmark.py --rows 1000000 --repeat 3  # writes RESULTS.md
uv run pytest tests -q                                # same report at every step
```

Project setup, part 1: imports, paths, the reduced row count, and `network_delay()`,
which switches the simulated store API's round trip for in-kernel runs.

In [489]:
import contextlib
import inspect
import os
import re
import subprocess
import sys
import textwrap
import time
import tracemalloc
from collections.abc import Callable, Iterator
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

DEMO_SCALE = int(os.environ.get("PHASE5_DEMO_SCALE", "1"))
REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / "projects" / "sales-report"
PHASE5_DIR = REPO_ROOT / "notebooks" / "_scratch" / "phase5"
PHASE5_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROWS = 50_000 * DEMO_SCALE  # reduced size for Run All; RESULTS.md holds the 1M-row run
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
CHILD_ENV = {
    **os.environ,
    "PYTHON_COLORS": "0",
    "PYTHONUNBUFFERED": "1",
    "NO_COLOR": "1",
    "PY_COLORS": "0",
}
CHILD_ENV.pop("FORCE_COLOR", None)


@contextlib.contextmanager
def network_delay(seconds: float) -> Iterator[None]:
    """Temporarily set the simulated store-API round trip (SALES_NETWORK_DELAY) in this process."""
    previous = os.environ.get("SALES_NETWORK_DELAY")
    os.environ["SALES_NETWORK_DELAY"] = str(seconds)
    try:
        yield
    finally:
        if previous is None:
            os.environ.pop("SALES_NETWORK_DELAY")
        else:
            os.environ["SALES_NETWORK_DELAY"] = previous

Project setup, part 2: helpers to run the project scripts in a child process, show
source code and measure traced peak memory; then the three project modules are
imported.

In [490]:
def run_in_project(*arguments: str, delay: float = 0.05, timeout: int = 300) -> str:
    """Run `python <arguments>` in projects/sales-report/ and return stdout + stderr."""
    completed = subprocess.run(
        [sys.executable, *arguments],
        capture_output=True,
        text=True,
        timeout=timeout,
        cwd=PROJECT_DIR,
        env={**CHILD_ENV, "SALES_NETWORK_DELAY": str(delay)},
    )
    return (completed.stdout + completed.stderr).rstrip().replace(str(REPO_ROOT), "<repo>")


def run_child(source: str) -> str:
    """Run dedented source as a script in a child interpreter (Units 5.1-5.5)."""
    script = PHASE5_DIR / "child_script.py"
    script.write_text(textwrap.dedent(source).lstrip("\n"), encoding="utf-8")
    completed = subprocess.run(
        [sys.executable, str(script)],
        capture_output=True,
        text=True,
        timeout=120,
        cwd=PHASE5_DIR,
        env=CHILD_ENV,
    )
    return (completed.stdout + completed.stderr).rstrip()


def show_source(*objects: object) -> None:
    """Print the source code of functions or classes from the project files."""
    for item in objects:
        print(inspect.getsource(item).rstrip(), end="\n\n")


def traced_peak_mib(work: Callable[[], object]) -> float:
    """Peak tracemalloc MiB while running work (NumPy reports its buffers to tracemalloc too)."""
    tracemalloc.start()
    work()
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return peak / 2**20


import common  # noqa: E402  (projects/sales-report/ is on sys.path now)
import fast_report  # noqa: E402
import slow_report  # noqa: E402

print(f"project folder: <repo>/{PROJECT_DIR.relative_to(REPO_ROOT)}")
print(f"{PROJECT_ROWS=:,}  (DEMO_SCALE={DEMO_SCALE})")
print(
    "files:",
    ", ".join(
        sorted(
            path.name for path in PROJECT_DIR.iterdir() if path.suffix in {".py", ".md", ".toml"}
        )
    ),
)

project folder: <repo>/projects/sales-report
PROJECT_ROWS=50,000  (DEMO_SCALE=1)
files: README.md, RESULTS.md, benchmark.py, common.py, fast_report.py, ruff.toml, slow_report.py


### The input: seeded rows and a slow API

Both versions read the same rows from `generate_chunks(row_count, seed)`: NumPy
columns (store index, product index, quantity, day of year) in chunks of 100 000
rows from a seeded generator, so a given `(row_count, seed)` always means the same
sales. Prices come from a fixed catalog in **integer paise**. The store directory
"API" sleeps for one round trip (`SALES_NETWORK_DELAY`, default 50 ms) per call:

In [491]:
show_source(common.generate_chunks, common.fetch_store_profile)

def generate_chunks(
    row_count: int = DEFAULT_ROW_COUNT, seed: int = DEFAULT_SEED
) -> Iterator[SalesChunk]:
    """Yield row_count seeded sales rows in chunks; the same arguments give the same rows."""
    if row_count < 0:
        raise ValueError(f"row_count must be >= 0, got {row_count}")
    rng = np.random.default_rng(seed)
    first = 0
    while first < row_count:
        size = min(CHUNK_ROWS, row_count - first)
        yield SalesChunk(
            first_order_number=first + 1,
            store_index=rng.integers(0, len(STORE_CODES), size),
            product_index=rng.integers(0, len(PRODUCTS), size),
            quantity=rng.integers(1, MAX_QUANTITY + 1, size),
            day=rng.integers(0, DAYS_IN_YEAR, size),
        )
        first += size

def fetch_store_profile(store_code: str) -> dict[str, str]:
    """Simulated GET /stores/<code>: blocks for one round trip, then returns the profile."""
    time.sleep(network_delay_seconds())  # stands in for socket I/O: rele

What one chunk and one API call look like:

In [492]:
first_chunk = next(common.generate_chunks(5, seed=42))
print(f"{len(first_chunk)} rows, first order number {first_chunk.first_order_number}")
for column in ("store_index", "product_index", "quantity", "day"):
    print(f"  {column:<14} {getattr(first_chunk, column).tolist()}")
print(f"catalog: {len(common.PRODUCTS)} products, e.g. {common.PRODUCTS[0]}")
with network_delay(0.05):
    start = time.perf_counter()
    profile = common.fetch_store_profile("ST-07")
    print(f"{profile}  ({(time.perf_counter() - start) * 1000:.0f} ms)")

5 rows, first order number 1
  store_index    [3, 30, 26, 17, 17]
  product_index  [51, 5, 41, 12, 5]
  quantity       [3, 5, 4, 4, 4]
  day            [286, 187, 46, 306, 164]
catalog: 60 products, e.g. Product(sku='SKU-001', name='Basmati rice 5 kg', price_paise=34050)
{'code': 'ST-07', 'city': 'Lucknow', 'region': 'North'}  (50 ms)


### Step 0 — The baseline and its report

The slow script materializes every row as a dict, with a float price and per-row
strings for the order id and the ISO date:

In [493]:
show_source(slow_report.generate_rows)

def generate_rows(row_count: int, seed: int) -> list[Row]:
    """Materialize every generated sale as a dict with float money and string fields."""
    rows: list[Row] = []  # SLOW: one million dicts held in memory at once
    year_start = date(YEAR, 1, 1)
    for chunk in generate_chunks(row_count, seed):
        for position in range(len(chunk)):
            product = PRODUCTS[chunk.product_index[position]]
            order_number = chunk.first_order_number + position
            rows.append(
                {
                    "order_id": "ORD-" + str(order_number).zfill(7),  # SLOW: per-row string
                    "store": STORE_CODES[chunk.store_index[position]],
                    "sku": product.sku,
                    "quantity": int(chunk.quantity[position]),
                    "unit_price": product.price_paise / 100,  # SLOW: float money
                    "date": (year_start + timedelta(days=int(chunk.day[position]))).isoformat(),
                }
            )
   

Then every metric is a separate scan over all rows, and the per-group totals are
nested loops: 40 stores × all rows, 60 products × all rows, 12 months × all rows.
The store section is typical (the `# SLOW:` comments mark each mistake):

In [494]:
show_source(slow_report.fetch_profiles, slow_report.store_section)

def fetch_profiles(store_codes: tuple[str, ...]) -> list[dict[str, str]]:
    """Call the store directory API for every store, one call after another."""
    profiles = []
    for store_code in store_codes:  # SLOW: sequential network waits
        profiles.append(fetch_store_profile(store_code))
    return profiles

def store_section(rows: list[Row], profiles: list[dict[str, str]]) -> str:
    """Top 10 stores by revenue: one scan of all rows per store."""
    store_revenue = []
    for profile in profiles:
        revenue = 0.0
        for row in rows:  # SLOW: nested loop, 40 stores x all rows
            if row["store"] == profile["code"]:
                revenue += row["quantity"] * row["unit_price"]
        store_revenue.append((profile, revenue))
    store_revenue.sort(key=lambda pair: (-round(pair[1], 2), pair[0]["code"]))
    text = "\nTOP 10 STORES\n"
    text += f"{'Rank':<6}{'Store':<7}{'City':<13}{'Region':<9}{'Revenue INR':>18}\n"
    for rank, (profile, revenue) in enume

The report itself, for 20 000 rows (the same text at every step, at any size):

In [495]:
with network_delay(0):
    reference_report = slow_report.build_report(20_000, 42)
print(reference_report)

SALES REPORT 2025
Order lines:          20,000
Units sold:           60,179
Revenue INR:   25,804,344.25
Period:      2025-01-01 to 2025-12-31

REVENUE BY REGION
Region           Revenue INR  Order lines
Central         5,197,417.50        4,003
East            5,151,395.50        4,008
North           5,230,888.00        4,072
South           5,111,239.25        3,952
West            5,113,404.00        3,965

TOP 10 STORES
Rank  Store  City         Region          Revenue INR
1     ST-17  Mumbai       West             706,902.25
2     ST-10  Chandigarh   North            702,787.00
3     ST-30  Chandigarh   North            694,022.25
4     ST-02  Hyderabad    South            689,267.25
5     ST-12  Indore       Central          688,888.75
6     ST-14  Guwahati     East             681,052.00
7     ST-22  Hyderabad    South            680,233.00
8     ST-32  Indore       Central          678,841.50
9     ST-08  Bhubaneswar  East             677,780.75
10    ST-13  Chennai      South

#### Profile before optimizing

A `cProfile` run of the baseline in a child process (with the 50 ms network delay
on), sorted by own time:

In [496]:
baseline_profile = run_child(
    f"""
    import cProfile, os, pstats, sys
    os.environ["SALES_NETWORK_DELAY"] = "0.05"
    sys.path.insert(0, {str(PROJECT_DIR)!r})
    import slow_report
    with cProfile.Profile() as profiler:
        slow_report.build_report({PROJECT_ROWS}, 42)
    pstats.Stats(profiler).strip_dirs().sort_stats("tottime").print_stats(8)
    """
)
print(re.sub(r"\n\s*\n", "\n", baseline_profile.split("Ordered by")[1]))

: internal time
   List reduced from 290 to 8 due to restriction <8>
   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
       40    2.010    0.050    2.010    0.050 {built-in method time.sleep}
        1    0.091    0.091    0.136    0.136 slow_report.py:37(generate_rows)
        1    0.083    0.083    0.083    0.083 slow_report.py:135(product_section)
        1    0.065    0.065    0.102    0.102 slow_report.py:154(month_section)
        1    0.064    0.064    0.064    0.064 slow_report.py:117(store_section)
   600024    0.038    0.000    0.038    0.000 {method 'startswith' of 'str' objects}
    50000    0.024    0.000    0.024    0.000 {method 'isoformat' of 'datetime.date' objects}
        1    0.021    0.021    0.021    0.021 slow_report.py:101(region_section)


Reading it with Amdahl in mind (5.5.1):

- `time.sleep` (inside `fetch_store_profile`) holds a fixed **2 s** (40 × 50 ms), at
  any row count. At small sizes it dominates; it can only be removed by *overlapping*
  the waits (concurrency), not by faster code.
- `product_section`, `store_section`, `month_section` and `region_section` are the
  nested scans: their cost grows with groups × rows, and at 1M rows they are most
  of the runtime.
- `generate_rows` (dicts, strings, NumPy scalars) costs time *and* is the reason
  the peak memory is about half a gigabyte at 1M rows.

So the plan: remove the nested scans (step 1), shrink the rows (step 2), stop
holding them all (step 3), overlap the network waits (step 4), and finally move the
remaining per-row loop into C (step 5).

### Step 1 — One pass with dict accumulators

Every total is updated in a **single loop**: per-store, per-product, per-month and
per-pair dicts are indexed by the row's key instead of scanning all rows once per
group (5.5.5: removing quadratic loops). Money is still a float here, converted to
integer paise once at the end so the shared renderer can be used:

In [497]:
show_source(fast_report.summarize_rows_single_pass)

def summarize_rows_single_pass(rows: list[slow_report.Row]) -> SalesSummary:
    """Step 1: the slow version's list of dicts and float money, but one loop for everything."""
    store_revenue = dict.fromkeys(STORE_CODES, 0.0)
    store_lines = dict.fromkeys(STORE_CODES, 0)
    product_revenue = dict.fromkeys(SKU_INDEX, 0.0)
    product_units = dict.fromkeys(SKU_INDEX, 0)
    month_revenue: dict[str, float] = {}
    month_lines: dict[str, int] = {}
    pair_revenue: dict[tuple[str, str], float] = {}
    summary = SalesSummary(order_lines=len(rows))
    dates = []
    for row in rows:
        quantity, store, sku, month = row["quantity"], row["store"], row["sku"], row["date"][:7]
        amount = quantity * row["unit_price"]
        summary.units += quantity
        store_revenue[store] += amount
        store_lines[store] += 1
        product_revenue[sku] += amount
        product_units[sku] += quantity
        month_revenue[month] = month_revenue.get(month, 0.0) + amount
        month_

Every step's report must equal the baseline's. The check runs each step at a few
sizes with the network off (the full check is the project's pytest suite, below):

In [498]:
def same_report_as_baseline(
    step: fast_report.Step, sizes: tuple[int, ...] = (0, 997, 20_000)
) -> bool:
    """True if the step reproduces the baseline report text at every size."""
    with network_delay(0):
        return all(step.build(rows, 42) == slow_report.build_report(rows, 42) for rows in sizes)


print(f"step 1 matches the baseline: {same_report_as_baseline(fast_report.STEPS[1])}")

step 1 matches the baseline: True


### Step 2 — Lean rows, exact money, no per-row strings

A row becomes a tuple of four small ints (store index, product index, quantity, day
of year). Money is integer paise looked up in a plain-`int` price list; the month
comes from a day → month table instead of slicing an ISO date string; no order id
or pair key string is built per row; the report is assembled with one `"\n".join`
(5.5.6). `add_rows` also binds the totals to local names before the loop:

In [499]:
show_source(fast_report.generate_lean_rows, fast_report.add_rows)

def generate_lean_rows(row_count: int, seed: int) -> list[LeanRow]:
    """Step 2: rows as tuples of small ints (no dicts, no strings, no floats)."""
    rows: list[LeanRow] = []
    for chunk in generate_chunks(row_count, seed):
        rows.extend(
            zip(
                chunk.store_index.tolist(),
                chunk.product_index.tolist(),
                chunk.quantity.tolist(),
                chunk.day.tolist(),
                strict=True,
            )
        )
    return rows

def add_rows(summary: SalesSummary, rows: list[LeanRow]) -> None:
    """Add lean rows to the totals in one loop; money stays integer paise (exact)."""
    store_paise, store_lines = summary.store_paise, summary.store_lines  # local aliases
    product_paise, product_units = summary.product_paise, summary.product_units
    month_paise, month_lines, pair_paise = (
        summary.month_paise,
        summary.month_lines,
        summary.pair_paise,
    )
    prices, month_of_day = PRICE_LIST

What a row costs in each representation (`sys.getsizeof` of the container only; the dict row also owns two unique strings and a float):

In [500]:
with network_delay(0):
    dict_row = slow_report.generate_rows(1, 42)[0]
    lean_row = fast_report.generate_lean_rows(1, 42)[0]
print(f"dict row : {sys.getsizeof(dict_row)} bytes  {dict_row}")
print(f"tuple row: {sys.getsizeof(lean_row)} bytes  {lean_row}")
print(f"step 2 matches the baseline: {same_report_as_baseline(fast_report.STEPS[2])}")

dict row : 272 bytes  {'order_id': 'ORD-0000001', 'store': 'ST-04', 'sku': 'SKU-047', 'quantity': 4, 'unit_price': 438.0, 'date': '2025-06-10'}
tuple row: 80 bytes  (3, 46, 4, 160)


step 2 matches the baseline: True


#### Why integer paise, if the float report is already identical?

The baseline's floats happen to produce the right report: every line amount is
price × whole quantity, and totals are rounded once at the end. Add one realistic
rule, 18 % GST per line rounded half-up to the paisa (Unit 1.3), and floats start
to disagree with exact arithmetic on some lines:

In [501]:
def gst_float(price_paise: int, quantity: int) -> float:
    """GST in rupees with float arithmetic and round() (what the slow style would do)."""
    return round(price_paise / 100 * quantity * 0.18, 2)


def gst_paise(price_paise: int, quantity: int) -> int:
    """GST in paise, exact: integer arithmetic, half-up rounding."""
    return (price_paise * quantity * 18 + 50) // 100


lines = [(product.price_paise, quantity) for product in common.PRODUCTS for quantity in range(1, 6)]
wrong = [
    (price_paise, quantity)
    for price_paise, quantity in lines
    if round(gst_float(price_paise, quantity) * 100) != gst_paise(price_paise, quantity)
]
print(
    f"{len(wrong)} of {len(lines)} catalog lines get a different GST with floats, e.g. {wrong[:3]}"
)

55 of 300 catalog lines get a different GST with floats, e.g. [(16325, 1), (16325, 5), (41325, 1)]


Exact integers make the report correct by construction, not by luck.

### Step 3 — Stream chunks instead of materializing the rows

Step 2 still builds a list of a million tuples before adding anything up. Since
every total is a running sum, each chunk can be aggregated and dropped (5.5.7:
generators); only one 100 000-row chunk exists at a time:

In [502]:
show_source(fast_report.summarize_stream)

def summarize_stream(row_count: int, seed: int) -> SalesSummary:
    """Step 3: aggregate each generated chunk and drop it; no list of all rows."""
    summary = SalesSummary()
    for chunk in generate_chunks(row_count, seed):
        columns = (chunk.store_index, chunk.product_index, chunk.quantity, chunk.day)
        add_rows(summary, list(zip(*(column.tolist() for column in columns), strict=True)))
    return summary



Peak traced memory of the two approaches at `PROJECT_ROWS` (the gap grows with the row count; at 1M rows see `RESULTS.md`):

In [503]:
def materialize_then_add() -> None:
    """Materialize four times the rows as tuples, then aggregate them."""
    fast_report.add_rows(
        fast_report.SalesSummary(), fast_report.generate_lean_rows(PROJECT_ROWS * 4, 42)
    )


list_peak = traced_peak_mib(materialize_then_add)
stream_peak = traced_peak_mib(lambda: fast_report.summarize_stream(PROJECT_ROWS * 4, 42))
print(
    f"{PROJECT_ROWS * 4:,} rows: list of tuples peak {list_peak:.1f} MiB, "
    f"streaming peak {stream_peak:.1f} MiB"
)
print(f"step 3 matches the baseline: {same_report_as_baseline(fast_report.STEPS[3])}")

200,000 rows: list of tuples peak 22.9 MiB, streaming peak 14.7 MiB
step 3 matches the baseline: True


### Step 4 — Concurrent store lookups (the concurrency step)

The 40 API calls are pure waiting, so they are an I/O-bound job (Unit 5.1): a
thread pool submits all 40 at once, and the main thread aggregates the rows
**while** the requests are in flight. `time.sleep`, like a socket read, releases
the GIL, so the waits overlap with each other *and* with the CPU work. Results are
collected in submission order, so the report does not depend on which call
finished first:

In [504]:
show_source(fast_report.build_with_concurrent_profiles)

def build_with_concurrent_profiles(
    summarize: Callable[[int, int], SalesSummary], row_count: int, seed: int
) -> str:
    """Start every store lookup on a thread pool, aggregate meanwhile, then collect."""
    with ThreadPoolExecutor(max_workers=len(STORE_CODES), thread_name_prefix="store-api") as pool:
        pending = [pool.submit(fetch_store_profile, code) for code in STORE_CODES]
        summary = summarize(row_count, seed)  # CPU work while the 40 requests wait
        profiles = [future.result() for future in pending]  # input order, not finish order
    return render_report(summary, profiles)



The effect on the network part alone, at the real 50 ms delay:

In [505]:
from concurrent.futures import ThreadPoolExecutor

with network_delay(0.05):
    start = time.perf_counter()
    sequential = slow_report.fetch_profiles(common.STORE_CODES)
    sequential_seconds = time.perf_counter() - start
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=40) as pool:
        concurrent_profiles = list(pool.map(common.fetch_store_profile, common.STORE_CODES))
    concurrent_seconds = time.perf_counter() - start
print(
    f"sequential {sequential_seconds:.2f} s, concurrent {concurrent_seconds:.3f} s, "
    f"same profiles: {sequential == concurrent_profiles}"
)
print(f"step 4 matches the baseline: {same_report_as_baseline(fast_report.STEPS[4])}")

sequential 2.01 s, concurrent 0.067 s, same profiles: True
step 4 matches the baseline: True


**Why threads, not asyncio or processes?** The API client here is blocking (like
`requests`), so threads are the direct fit; with an async HTTP client the same step
is `asyncio.gather` under a `Semaphore` (Units 5.3–5.4) and, since the aggregation
is CPU work, it would go to `asyncio.to_thread`. Processes buy nothing for waiting
and would have to pickle the rows. And once the aggregation is vectorized (step 5)
it takes tens of milliseconds at 1M rows, so splitting it across processes would
cost more in start-up and pickling than it saves (Amdahl again, Unit 5.2).

### Step 5 — Vectorize the per-chunk grouping

The remaining cost is the Python loop over each row (about 1 s per million rows).
NumPy does the same grouping in C: one `int64` amount column per chunk, then
`np.add.at` adds amounts, units and counts into `stores × products` and `months`
tables. `np.add.at` handles repeated indices correctly (a plain fancy-index `+=`
would count each index once per chunk), and everything stays integer, so money is
exact (5.5.9):

In [506]:
show_source(fast_report.summarize_vectorized)

def summarize_vectorized(row_count: int, seed: int) -> SalesSummary:
    """Step 5: group each chunk with np.add.at on int64 columns (exact, no Python loop)."""
    pair_paise = np.zeros((STORE_COUNT, PRODUCT_COUNT), dtype=np.int64)
    pair_units = np.zeros((STORE_COUNT, PRODUCT_COUNT), dtype=np.int64)
    pair_lines = np.zeros((STORE_COUNT, PRODUCT_COUNT), dtype=np.int64)
    month_paise = np.zeros(MONTH_COUNT, dtype=np.int64)
    month_lines = np.zeros(MONTH_COUNT, dtype=np.int64)
    first_day, last_day = None, None
    for chunk in generate_chunks(row_count, seed):
        amount = PRICE_PAISE[chunk.product_index] * chunk.quantity  # int64 paise per row
        where = (chunk.store_index, chunk.product_index)
        np.add.at(pair_paise, where, amount)
        np.add.at(pair_units, where, chunk.quantity)
        np.add.at(pair_lines, where, 1)
        months = MONTH_OF_DAY[chunk.day]
        np.add.at(month_paise, months, amount)
        np.add.at(month_lines, months, 1)
        

Check, and compare the two aggregation loops alone (no network) at `PROJECT_ROWS × 4`:

In [507]:
print(f"step 5 matches the baseline: {same_report_as_baseline(fast_report.STEPS[5])}")
timings = {}
for label, summarize in (
    ("Python loop (step 3)", fast_report.summarize_stream),
    ("NumPy (step 5)", fast_report.summarize_vectorized),
):
    start = time.perf_counter()
    summary = summarize(PROJECT_ROWS * 4, 42)
    timings[label] = time.perf_counter() - start
    print(
        f"{label:<22} {timings[label] * 1000:7.1f} ms   "
        f"revenue {fast_report.format_paise(summary.revenue_paise)}"
    )
print(f"ratio: {timings['Python loop (step 3)'] / timings['NumPy (step 5)']:.0f}x")

step 5 matches the baseline: True
Python loop (step 3)      67.4 ms   revenue 256,872,244.25


NumPy (step 5)             8.6 ms   revenue 256,872,244.25
ratio: 8x


### The safety net: identical reports, tested

`tests/test_reports.py` compares every step with the slow report at several sizes
and seeds (including 0 rows, 1 row and a size that spans two chunks), checks the
generator's determinism, the exact money formatting, the grand total against an
independent sum, and that the store lookups really overlap:

In [508]:
test_file = PROJECT_DIR / "tests" / "test_reports.py"
print("\n".join(test_file.read_text(encoding="utf-8").splitlines()[10:42]))


SIZES = [(0, 1), (1, 2), (997, 3), (25_000, 4), (130_000, 42)]  # 130k spans two chunks


@pytest.mark.parametrize(("row_count", "seed"), SIZES)
def test_every_step_matches_the_slow_report(row_count: int, seed: int) -> None:
    expected = slow_report.build_report(row_count, seed)
    for step in fast_report.STEPS[1:]:
        assert step.build(row_count, seed) == expected, f"step {step.number} differs"


def test_final_version_is_the_last_step() -> None:
    assert fast_report.build_report is fast_report.STEPS[-1].build
    assert [step.number for step in fast_report.STEPS] == [0, 1, 2, 3, 4, 5]


def test_generator_is_deterministic_and_seed_dependent() -> None:
    def first_chunk(seed: int) -> list[int]:
        chunk = next(common.generate_chunks(1_000, seed))
        return chunk.product_index.tolist()

    assert first_chunk(42) == first_chunk(42)
    assert first_chunk(42) != first_chunk(43)
    assert sum(len(chunk) for chunk in common.generate_chunks(250_001, 1)) == 250_001



Run the suite with pytest in a child process:

In [509]:
print(run_in_project("-m", "pytest", "tests", "-q", "-p", "no:cacheprovider", timeout=600))

...........                                                              [100%]
11 passed in 1.61s


### The benchmark at a reduced size

`benchmark.py` runs each step in a **fresh process** (5.5.1), measures wall time
around `build_report` and the peak resident set size, and checks that every step's
report has the same SHA-256. At `PROJECT_ROWS` rows the fixed costs dominate: the
2 s of sequential network waits in steps 0–3 and the ~35 MiB interpreter + NumPy
floor in the memory column. So expect a large speedup from step 4 and a modest
memory ratio here:

In [510]:
reduced = run_in_project("benchmark.py", "--rows", str(PROJECT_ROWS), "--output", "-")
print("\n".join(line for line in reduced.splitlines() if line.startswith("step ")))
display(Markdown(reduced[reduced.index("| Step") :]))

step 0:    2.318 s      61.7 MiB
step 1:    2.115 s      61.4 MiB
step 2:    2.042 s      46.2 MiB
step 3:    2.055 s      46.1 MiB
step 4:    0.063 s      47.2 MiB
step 5:    0.067 s      43.2 MiB


| Step | Change | Time (best) | Time (median) | Peak memory | Reason |
| --- | --- | --- | --- | --- | --- |
| 0 | Baseline: `slow_report.py` | 2.318 s | 2.318 s | 61.7 MiB | nested loops, repeated scans, 1M dicts, float money, string `+=`, 40 sequential API calls |
| 1 | One pass with dict accumulators | 2.115 s (1.1x faster) | 2.115 s (1.1x faster) | 61.4 MiB (1.0x less) | O(groups x rows) nested loops and repeated scans become one O(rows) pass |
| 2 | Lean tuple rows, integer paise, `join` | 2.042 s (1.1x faster) | 2.042 s (1.1x faster) | 46.2 MiB (1.3x less) | no per-row dicts, strings or floats: fewer allocations, exact money |
| 3 | Stream chunks, no list of all rows | 2.055 s (1.1x faster) | 2.055 s (1.1x faster) | 46.1 MiB (1.3x less) | only one 100k-row chunk alive at a time: peak memory stops growing with rows |
| 4 | Concurrent store lookups (thread pool) | 0.063 s (36.8x faster) | 0.063 s (36.8x faster) | 47.2 MiB (1.3x less) | 40 x 50 ms waits overlap each other and the CPU work (`sleep`, like socket I/O, releases the GIL) |
| 5 | Vectorize grouping with NumPy `np.add.at` | 0.067 s (34.7x faster) | 0.067 s (34.7x faster) | 43.2 MiB (1.4x less) | per-row Python bytecode replaced by C loops over int64 columns |

**Result: 34.7x faster and 1.4x less peak memory** (best runs, step 0 -> step 5; 34.7x faster by median); target >= 10x faster and >= 3x less memory: NOT met.

Reports identical across all steps and runs: yes (SHA-256 `7bc8ff8b9f886c1e...`).

**Run-to-run variation.** One run per step, so no spread was measured (use `--repeat 3`).

Wall times move with CPU frequency scaling, other load and cache state. The relative
spread is largest for the shortest steps, where a few milliseconds of noise are a big
share, and separate sessions on the same machine can differ by tens of percent for
every step, the baseline included. The best run is the least disturbed one and the
median a typical one; the targets are judged on the best runs, with the median ratio
printed next to them. Read the speedup as an order of magnitude.

`tests/test_reports.py::test_store_lookups_overlap` asserts that the final version with
40 simulated 50 ms store calls finishes in under 0.5 s, against 2 s if the calls ran
one after another. The threshold is deliberately loose: the overlapped calls take
about 50 ms, so only a regression to sequential calls (or a badly overloaded machine)
can fail it.
step 0:    2.318 s      61.7 MiB
step 1:    2.115 s      61.4 MiB
step 2:    2.042 s      46.2 MiB
step 3:    2.055 s      46.1 MiB
step 4:    0.063 s      47.2 MiB
step 5:    0.067 s      43.2 MiB

### The full-size result (1 000 000 rows)

`RESULTS.md` was written by `benchmark.py --rows 1000000 --repeat 3` run from a
terminal (about a minute: three fresh processes per step, most of the time in the
baseline). It records the best and the median run of each step and how much the
runs varied. Your machine will give other numbers; the shape (which step helps
time, which helps memory) should hold:

In [511]:
results_text = (PROJECT_DIR / "RESULTS.md").read_text(encoding="utf-8")
display(Markdown(results_text))
verdict = re.search(
    r"\*\*Result: ([\d.,]+)x faster and ([\d.,]+)x less peak memory\*\*", results_text
)
if verdict:
    speedup, memory_ratio = (float(value.replace(",", "")) for value in verdict.groups())
    print(f"targets (>= 10x faster, >= 3x less memory) met: {speedup >= 10 and memory_ratio >= 3}")

# Sales report benchmark

- Rows: **1,000,000**, seed 42, 3 run(s) per step (best and median shown), each run in a fresh process; measured 2026-11-17
- Python 3.15.1 (default (GIL) build), NumPy 2.5.4, Linux x86_64, 16 CPUs
- Simulated store API: 40 calls x 0.050 s each
- Peak memory is the highest peak RSS of the runs; Python + NumPy imports alone: 35.5 MiB

| Step | Change | Time (best) | Time (median) | Peak memory | Reason |
| --- | --- | --- | --- | --- | --- |
| 0 | Baseline: `slow_report.py` | 7.235 s | 7.316 s | 469.2 MiB | nested loops, repeated scans, 1M dicts, float money, string `+=`, 40 sequential API calls |
| 1 | One pass with dict accumulators | 3.730 s (1.9x faster) | 3.879 s (1.9x faster) | 470.4 MiB (1.0x less) | O(groups x rows) nested loops and repeated scans become one O(rows) pass |
| 2 | Lean tuple rows, integer paise, `join` | 2.374 s (3.0x faster) | 2.389 s (3.1x faster) | 131.4 MiB (3.6x less) | no per-row dicts, strings or floats: fewer allocations, exact money |
| 3 | Stream chunks, no list of all rows | 2.348 s (3.1x faster) | 2.362 s (3.1x faster) | 53.4 MiB (8.8x less) | only one 100k-row chunk alive at a time: peak memory stops growing with rows |
| 4 | Concurrent store lookups (thread pool) | 0.415 s (17.4x faster) | 0.426 s (17.2x faster) | 54.4 MiB (8.6x less) | 40 x 50 ms waits overlap each other and the CPU work (`sleep`, like socket I/O, releases the GIL) |
| 5 | Vectorize grouping with NumPy `np.add.at` | 0.078 s (92.7x faster) | 0.079 s (92.2x faster) | 48.6 MiB (9.7x less) | per-row Python bytecode replaced by C loops over int64 columns |

**Result: 92.7x faster and 9.7x less peak memory** (best runs, step 0 -> step 5; 92.2x faster by median); target >= 10x faster and >= 3x less memory: met.

Reports identical across all steps and runs: yes (SHA-256 `edc9a13246dff7e5...`).

**Run-to-run variation.** Spread of the 3 runs, (slowest - fastest) / median: step 0 7%, step 1 6%, step 2 1%, step 3 1%, step 4 3%, step 5 3%.

Wall times move with CPU frequency scaling, other load and cache state. The relative
spread is largest for the shortest steps, where a few milliseconds of noise are a big
share, and separate sessions on the same machine can differ by tens of percent for
every step, the baseline included. The best run is the least disturbed one and the
median a typical one; the targets are judged on the best runs, with the median ratio
printed next to them. Read the speedup as an order of magnitude.

`tests/test_reports.py::test_store_lookups_overlap` asserts that the final version with
40 simulated 50 ms store calls finishes in under 0.5 s, against 2 s if the calls ran
one after another. The threshold is deliberately loose: the overlapped calls take
about 50 ms, so only a regression to sequential calls (or a badly overloaded machine)
can fail it.


targets (>= 10x faster, >= 3x less memory) met: True


### What the steps teach

| Step | What it fixed | Lesson | Unit |
| --- | --- | --- | --- |
| 1 | nested group × row scans | the algorithm first: an O(groups × rows) job became O(rows) | 5.5.5 |
| 2 | dict rows, per-row strings, float money | fewer, smaller objects are faster *and* leaner; exact money by construction | 5.5.6–5.5.7, 1.3 |
| 3 | the list of all rows | streaming makes peak memory independent of the input size | 5.5.7 |
| 4 | 40 sequential waits | overlap I/O with threads; a fixed cost no CPU optimization can touch | 5.1, 5.4 |
| 5 | the per-row Python loop | vectorize the hot loop; keep integers in int64 | 5.5.9 |

Two more lessons hide in the table: the order matters less than measuring after
every change (at 50 000 rows step 4 looks like the only one that counts; at 1M rows
step 1 and step 5 carry the CPU work), and every step kept the output **identical**,
which is what makes it safe to ship.

### Phase 5 review — predict the output

Twenty questions across the whole phase (4 on Unit 5.1, 4 on Unit 5.2, 4 on Unit 5.3, 4 on Unit 5.4, 4 on Unit 5.5). Decide on the answer
before opening it, then run the proof cell. Timing-sensitive answers use delays
large enough to be deterministic.

#### Question 1 (Unit 5.1)

**Predict the output.**

```python
import threading

order = []
worker = threading.Thread(target=lambda: order.append("worker"))
worker.run()  # run(), not start()
order.append("main")
print(order, worker.is_alive())
```

<details><summary>Answer</summary>

`['worker', 'main'] False`. `run()` is the method that `start()` calls **in the new
thread**; calling it directly runs the target in the *current* thread, like any
function call. No thread was started, so it is not alive.

</details>

Run the next cell to check your prediction.

In [512]:
import threading

order = []
worker = threading.Thread(target=lambda: order.append("worker"))
worker.run()  # run(), not start()
order.append("main")
print(order, worker.is_alive())

['worker', 'main'] False


#### Question 2 (Unit 5.1)

**Predict the output.**

```python
import threading

lock, rlock = threading.Lock(), threading.RLock()
print(lock.acquire(), lock.acquire(timeout=0.1))
print(rlock.acquire(), rlock.acquire(timeout=0.1))
```

<details><summary>Answer</summary>

`True False`, then `True True`. A `Lock` is not re-entrant: the second acquire by the
same thread waits for itself and times out. An `RLock` counts acquisitions by its
owning thread, so the owner may acquire it again (and must release it as many times).

</details>

Run the next cell to check your prediction.

In [513]:
import threading

lock, rlock = threading.Lock(), threading.RLock()
print(lock.acquire(), lock.acquire(timeout=0.1))
print(rlock.acquire(), rlock.acquire(timeout=0.1))
lock.release()
rlock.release()
rlock.release()

True False
True True


#### Question 3 (Unit 5.1)

**Predict the output.**

```python
import time
from concurrent.futures import ThreadPoolExecutor


def nap(seconds: float) -> float:
    """Sleep, then return the duration."""
    time.sleep(seconds)
    return seconds


with ThreadPoolExecutor(max_workers=3) as pool:
    print(list(pool.map(nap, [0.3, 0.1, 0.2])))
```

<details><summary>Answer</summary>

`[0.3, 0.1, 0.2]`. `Executor.map` returns results in **input order**, whatever order
the calls finish in (the 0.1 s call finishes first, but its result waits for the
0.3 s one). Use `as_completed` for finish order.

</details>

Run the next cell to check your prediction.

In [514]:
import time
from concurrent.futures import ThreadPoolExecutor


def nap(seconds: float) -> float:
    """Sleep, then return the duration."""
    time.sleep(seconds)
    return seconds


with ThreadPoolExecutor(max_workers=3) as pool:
    print(list(pool.map(nap, [0.3, 0.1, 0.2])))

[0.3, 0.1, 0.2]


#### Question 4 (Unit 5.1)

**Predict the output.**

```python
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as pool:
    future = pool.submit(int, "12x")
print(future.done(), type(future.exception()).__name__)
```

<details><summary>Answer</summary>

`True ValueError`. An exception raised in a worker does not crash the program or
print anything: it is stored in the future. `future.exception()` returns it, and
`future.result()` would re-raise it. Futures nobody checks hide errors.

</details>

Run the next cell to check your prediction.

In [515]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as pool:
    future = pool.submit(int, "12x")
print(future.done(), type(future.exception()).__name__)

True ValueError


#### Question 5 (Unit 5.2)

**Predict the output.**

```python
import multiprocessing as mp

counter = 0

def bump():
    global counter
    counter += 1
    print("child sees", counter)

if __name__ == "__main__":
    process = mp.Process(target=bump)
    process.start()
    process.join()
    print("parent sees", counter)
```

<details><summary>Answer</summary>

`child sees 1`, then `parent sees 0`. A process has its own memory: the child changed
**its copy** of `counter`. Sharing needs explicit IPC (queues, pipes, shared memory,
a manager). The proof runs the script in a child interpreter (a notebook function
cannot be a process target under spawn/forkserver).

</details>

Run the next cell to check your prediction.

In [516]:
print(
    run_child(
        """
        import multiprocessing as mp

        counter = 0


        def bump() -> None:
            global counter
            counter += 1
            print("child sees", counter, flush=True)


        if __name__ == "__main__":
            process = mp.Process(target=bump)
            process.start()
            process.join()
            print("parent sees", counter)
        """
    )
)

child sees 1
parent sees 0


#### Question 6 (Unit 5.2)

**Predict the output.**

```python
import pickle

try:
    pickle.dumps(lambda price: price * 1.18)
except Exception as error:
    print(type(error).__name__)
```

<details><summary>Answer</summary>

`PicklingError`. Functions are pickled **by reference** (module + qualified name), and
a lambda has no importable name. That is why process-pool workers must be module-level
functions in an importable module (Unit 5.2), and why `functools.partial` of such a
function works where a lambda fails.

</details>

Run the next cell to check your prediction.

In [517]:
import pickle

try:
    pickle.dumps(lambda price: price * 1.18)
except Exception as error:
    print(type(error).__name__)

PicklingError


#### Question 7 (Unit 5.2)

**Predict the output.**

```python
parallel_fraction, workers = 0.9, 8
print(round(1 / ((1 - parallel_fraction) + parallel_fraction / workers), 2))
```

<details><summary>Answer</summary>

`4.71`. Amdahl's law: 10 % of the work stays serial, so 8 workers give 1 / (0.1 +
0.1125) ≈ 4.71×, not 8×, and no number of workers can beat 10×.

</details>

Run the next cell to check your prediction.

In [518]:
parallel_fraction, workers = 0.9, 8
print(round(1 / ((1 - parallel_fraction) + parallel_fraction / workers), 2))

4.71


#### Question 8 (Unit 5.2)

**Predict the output.**

```python
import multiprocessing as mp
import sys

process = mp.Process(target=sys.exit, args=(3,))
process.start()
process.join()
print(process.exitcode)
```

<details><summary>Answer</summary>

`3`. `sys.exit(3)` in the child raises `SystemExit(3)`, and the process exits with
that status. `exitcode` is `0` for success, the exit status otherwise, and a negative
signal number (such as `-15` for SIGTERM) when the child was killed. `sys.exit` works
as a target because it is importable by name.

</details>

Run the next cell to check your prediction.

In [519]:
import multiprocessing as mp
import sys

process = mp.Process(target=sys.exit, args=(3,))
process.start()
process.join()
print(process.exitcode)

3


#### Question 9 (Unit 5.3)

**Predict the output.**

```python
import asyncio

async def worker(name):
    print(f"{name} start")
    await asyncio.sleep(0)
    print(f"{name} end")

async def main():
    task = asyncio.create_task(worker("task"))
    print("main before")
    await asyncio.sleep(0)
    print("main after")
    await task

asyncio.run(main())
```

<details><summary>Answer</summary>

`main before`, `task start`, `main after`, `task end`. `create_task` only schedules the
task. Each `await asyncio.sleep(0)` puts the current task at the **back** of the ready
queue: main yields, the task runs to its first `await`, main resumes, then the task
finishes. (The proof uses top-level `await main()`, because the kernel's loop is
already running.)

</details>

Run the next cell to check your prediction.

In [520]:
import asyncio


async def worker(name: str) -> None:
    """Print around one yield to the loop."""
    print(f"{name} start")
    await asyncio.sleep(0)
    print(f"{name} end")


async def main() -> None:
    """Schedule the worker and interleave with it."""
    task = asyncio.create_task(worker("task"))
    print("main before")
    await asyncio.sleep(0)
    print("main after")
    await task


await main()

main before
task start
main after
task end


#### Question 10 (Unit 5.3)

**Predict the output.**

```python
async def total() -> int:
    """Return 42."""
    return 42


result = total()
print(type(result).__name__)
result.close()
```

<details><summary>Answer</summary>

`coroutine`. Calling an `async def` function runs **none** of its body: it returns a
coroutine object, which runs only when awaited or wrapped in a task. (Unlike a JS
`async` function, which runs synchronously up to its first `await` and returns a
Promise.) `close()` avoids the "never awaited" warning.

</details>

Run the next cell to check your prediction.

In [521]:
async def total() -> int:
    """Return 42."""
    return 42


result = total()
print(type(result).__name__)
result.close()

coroutine


#### Question 11 (Unit 5.3)

**Predict the output.**

```python
import asyncio

async def after(seconds, value):
    await asyncio.sleep(seconds)
    return value

print(await asyncio.gather(after(0.2, "slow"), after(0.1, "fast")))
```

<details><summary>Answer</summary>

`['slow', 'fast']`. Like `Promise.all`, `gather` returns results in the order the
awaitables were **passed**, not the order they finished. The whole call takes about
0.2 s, because both sleeps run concurrently.

</details>

Run the next cell to check your prediction.

In [522]:
import asyncio


async def after(seconds: float, value: str) -> str:
    """Return value after a delay."""
    await asyncio.sleep(seconds)
    return value


print(await asyncio.gather(after(0.2, "slow"), after(0.1, "fast")))

['slow', 'fast']


#### Question 12 (Unit 5.3)

**Predict the output.**

```python
import asyncio

# in Jupyter, where an event loop is already running:
pending = asyncio.sleep(0)
try:
    asyncio.run(pending)
except RuntimeError as error:
    print(error)
```

<details><summary>Answer</summary>

`asyncio.run() cannot be called from a running event loop`. `asyncio.run` creates and
owns a new loop, and one thread can run only one loop at a time. In a notebook use
top-level `await`; in a script, call `asyncio.run(main())` once at the entry point.

</details>

Run the next cell to check your prediction.

In [523]:
import asyncio

pending = asyncio.sleep(0)
try:
    asyncio.run(pending)
except RuntimeError as error:
    print(error)
finally:
    pending.close()

asyncio.run() cannot be called from a running event loop


#### Question 13 (Unit 5.4)

**Predict the output.**

```python
import asyncio

try:
    async with asyncio.timeout(0.05):
        await asyncio.sleep(1)
except TimeoutError:
    print("timed out")
print("after")
```

<details><summary>Answer</summary>

`timed out`, then `after`. When the deadline passes, the sleeping code receives a
`CancelledError`, and `asyncio.timeout` converts it into the built-in `TimeoutError`
at the block's edge (3.11+). The rest of the program carries on.

</details>

Run the next cell to check your prediction.

In [524]:
import asyncio

try:
    async with asyncio.timeout(0.05):
        await asyncio.sleep(1)
except TimeoutError:
    print("timed out")
print("after")

timed out
after


#### Question 14 (Unit 5.4)

**Predict the output.**

```python
import asyncio

gate = asyncio.Semaphore(2)
state = {"running": 0, "peak": 0}

async def job():
    async with gate:
        state["running"] += 1
        state["peak"] = max(state["peak"], state["running"])
        await asyncio.sleep(0.01)
        state["running"] -= 1

await asyncio.gather(*(job() for _ in range(6)))
print(state["peak"])
```

<details><summary>Answer</summary>

`2`. Six jobs start together, but the semaphore lets only two inside the `async with`
block at a time: the asyncio version of `p-limit(2)`.

</details>

Run the next cell to check your prediction.

In [525]:
import asyncio

gate = asyncio.Semaphore(2)
state = {"running": 0, "peak": 0}


async def job() -> None:
    """Count how many jobs are inside the semaphore at once."""
    async with gate:
        state["running"] += 1
        state["peak"] = max(state["peak"], state["running"])
        await asyncio.sleep(0.01)
        state["running"] -= 1


await asyncio.gather(*(job() for _ in range(6)))
print(state["peak"])

2


#### Question 15 (Unit 5.4)

**Predict the output.**

```python
import asyncio

queue = asyncio.Queue()
queue.put_nowait("order-1")
queue.shutdown()
print(queue.get_nowait())
try:
    queue.put_nowait("order-2")
except asyncio.QueueShutDown:
    print("QueueShutDown")
```

<details><summary>Answer</summary>

`order-1`, then `QueueShutDown`. `Queue.shutdown()` (*New in 3.13*) stops new `put`s at
once, but items already queued can still be taken; `shutdown(immediate=True)` would
drop them. Workers stop when `get()` raises `QueueShutDown` on an empty, shut-down
queue.

</details>

Run the next cell to check your prediction.

In [526]:
import asyncio

queue = asyncio.Queue()
queue.put_nowait("order-1")
queue.shutdown()
print(queue.get_nowait())
try:
    queue.put_nowait("order-2")
except asyncio.QueueShutDown:
    print("QueueShutDown")

order-1
QueueShutDown


#### Question 16 (Unit 5.4)

**Predict the output.**

```python
import asyncio
import contextvars

request_id = contextvars.ContextVar("request_id", default="-")

async def handler():
    request_id.set("req-7")

await asyncio.create_task(handler())
print(request_id.get())
```

<details><summary>Answer</summary>

`-`. Each task runs in a **copy** of the context that was current when it was created.
The handler's `set` changes only its copy, so the caller still sees the default. This
is what keeps concurrent requests' IDs apart.

</details>

Run the next cell to check your prediction.

In [527]:
import asyncio
import contextvars

request_id = contextvars.ContextVar("request_id", default="-")


async def handler() -> None:
    """Set the request id inside the task's own context copy."""
    request_id.set("req-7")


await asyncio.create_task(handler())
print(request_id.get())

-


#### Question 17 (Unit 5.5)

**Predict the output.**

```python
import time

start = time.process_time()
time.sleep(0.2)
print(round(time.process_time() - start, 1))
```

<details><summary>Answer</summary>

`0.0`. `process_time` counts CPU time, and a sleeping process uses almost none.
`perf_counter` would show 0.2. Comparing the two tells you whether code is waiting or
computing (5.5.2).

</details>

Run the next cell to check your prediction.

In [528]:
import time

start = time.process_time()
time.sleep(0.2)
print(round(time.process_time() - start, 1))

0.0


#### Question 18 (Unit 5.5)

**Predict the output.**

```python
import heapq

heap = [5, 1, 4]
heapq.heapify(heap)
heapq.heappush(heap, 0)
print(heap[0], heapq.heappop(heap), heapq.heappop(heap))
```

<details><summary>Answer</summary>

`0 0 1`. A heap keeps its **smallest** item at index 0; `heappop` removes and returns
it, and the next smallest moves to the front. The rest of the list is only partially
ordered.

</details>

Run the next cell to check your prediction.

In [529]:
import heapq

heap = [5, 1, 4]
heapq.heapify(heap)
heapq.heappush(heap, 0)
print(heap[0], heapq.heappop(heap), heapq.heappop(heap))

0 0 1


#### Question 19 (Unit 5.5)

**Predict the output.**

```python
squares = (number * number for number in range(4))
print(sum(squares), sum(squares))
```

<details><summary>Answer</summary>

`14 0`. A generator expression produces its items once. The first `sum` consumes all
of them (0 + 1 + 4 + 9); the second finds the generator exhausted. Generators save
memory (5.5.7) but are one-shot, unlike a list.

</details>

Run the next cell to check your prediction.

In [530]:
squares = (number * number for number in range(4))
print(sum(squares), sum(squares))

14 0


#### Question 20 (Unit 5.5)

**Predict the output.**

```python
import numpy as np

totals = np.zeros(3, dtype=np.int64)
stores = np.array([0, 2, 0, 0])
totals[stores] += 1
print(totals)
np.add.at(totals, stores, 1)
print(totals)
```

<details><summary>Answer</summary>

`[1 0 1]`, then `[4 0 2]`. Fancy-index `+=` is buffered: it reads `totals[[0, 2, 0, 0]]`,
adds 1, and writes back, so store 0 is incremented **once** however often it appears.
`np.add.at` is unbuffered and applies every occurrence, which is why the project's
step 5 groups with it.

</details>

Run the next cell to check your prediction.

In [531]:
import numpy as np

totals = np.zeros(3, dtype=np.int64)
stores = np.array([0, 2, 0, 0])
totals[stores] += 1
print(totals)
np.add.at(totals, stores, 1)
print(totals)

[1 0 1]
[4 0 2]


## Phase 5 — Wrap-up

| You can now… | Where |
| --- | --- |
| choose threads, processes, subinterpreters or asyncio for a workload, and back the choice with a benchmark | 5.1, 5.2, 5.4 |
| explain the event loop's ready queue, tasks and futures as precisely as Node's phases and microtasks | 5.3 |
| run asyncio safely in production: task groups, timeouts, backpressure, retries, tests | 5.4 |
| profile CPU and memory, read a flame graph, and optimize in the right order | 5.5 |
| take a real script from slow to fast without changing its output, and prove both | Project |

**Next:** Phase 6 builds web services with FastAPI on top of the asyncio, HTTP
client and testing skills from this phase.